# Healthcare Challenge 3 - Baseline Submission

This notebook provides a simple baseline for **Healthcare Challenge 3: Discharge Readiness Prediction**.

**Goal**: Predict `discharge_ready_day11` (0/1) for each hospital stay
**Metric**: Macro-F1 Score - Higher is better

## Instructions:
1. **Replace API credentials** in the first cell with your team's API key and name
2. **Run all cells** to generate and submit baseline predictions
3. **Check the output** for your submission score

This baseline uses only tabular stay data with a simple Random Forest classifier.


In [1]:
# 1. Initialize Client and Load Data

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from agentds import BenchmarkClient

# 🔑 REPLACE WITH YOUR CREDENTIALS
client = BenchmarkClient(
    api_key="adsb_AJp9fgtjuTRZ4Tk0mNnYDWxG_1759039055",        # Get from your team dashboard
    team_name="clai"     # Your exact team name
)

# Iteration 1
- 0.7068
- 0.7070

## EDA 1
- 0.6815

In [2]:
import os, json, random
from datetime import datetime, timezone
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.base import clone
import joblib
import sklearn

# =====================
# Config (v1 / EDA1)
# =====================
VX = "v1"
TEAM = "clai"
TASK = "ch3"
PHASE = "EDA1"

RANDOM_STATE = 42
N_SPLITS = 5

TFIDF_MAX_FEATURES = 500
TFIDF_NGRAM_RANGE = (1, 2)
TFIDF_MIN_DF = 2

LR_PARAMS = dict(
    solver="saga",
    penalty="l2",
    C=1.0,
    class_weight="balanced",
    max_iter=20000,
    n_jobs=1,              # determinism > speed
    random_state=RANDOM_STATE
)

# =====================
# Paths / Directories
# =====================
def resolve_input_path(filename: str) -> str:
    candidates = [
        filename,
        os.path.join("healthcare", filename),
        os.path.join("/mnt/data", filename),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Could not find {filename} in {candidates}")

DATA_PATHS = {
    "stays_train": resolve_input_path("stays_train.csv"),
    "stays_test": resolve_input_path("stays_test.csv"),
    "patients": resolve_input_path("patients.csv"),
    "vitals": resolve_input_path("vitals_timeseries.json"),
}

submission_path = f"{TEAM}_{TASK}_{VX}_submission.csv"
iter_log_path = f"{TEAM}_{TASK}_{VX}_iteration_detail.jsonl"

ckpt_root = os.path.join("checkpoints", f"{TEAM}_{TASK}_{VX}")
art_root = os.path.join("artifacts", f"{TEAM}_{TASK}_{VX}")
os.makedirs(ckpt_root, exist_ok=True)
os.makedirs(art_root, exist_ok=True)

# =====================
# Iteration ID
# =====================
def next_iteration_id(log_path: str) -> int:
    if not os.path.exists(log_path):
        return 0
    # Read last non-empty line efficiently
    with open(log_path, "rb") as f:
        f.seek(0, os.SEEK_END)
        if f.tell() == 0:
            return 0
        pos = f.tell() - 1
        while pos > 0:
            f.seek(pos)
            if f.read(1) == b"\n":
                break
            pos -= 1
        f.seek(pos + 1)
        last_line = f.readline().decode("utf-8", errors="ignore").strip()
    try:
        obj = json.loads(last_line)
        return int(obj.get("iteration_id", -1)) + 1
    except Exception:
        # Fallback: count valid JSON lines
        n = 0
        with open(log_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    n += 1
        return n

iteration_id = next_iteration_id(iter_log_path)
timestamp = datetime.now(timezone.utc).isoformat()

# =====================
# Reproducibility
# =====================
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print(f"[{TEAM}/{TASK}/{VX}] Iteration {iteration_id} | Phase={PHASE} | UTC={timestamp}")
print("scikit-learn:", sklearn.__version__)
print("Input paths:", json.dumps(DATA_PATHS, indent=2))

# =====================
# Load data
# =====================
stays_train = pd.read_csv(DATA_PATHS["stays_train"])
stays_test = pd.read_csv(DATA_PATHS["stays_test"])
patients = pd.read_csv(DATA_PATHS["patients"])
with open(DATA_PATHS["vitals"], "r", encoding="utf-8") as f:
    vitals_records = json.load(f)

print("\n=== Raw shapes ===")
print("stays_train:", stays_train.shape)
print("stays_test :", stays_test.shape)
print("patients   :", patients.shape)
print("vitals recs:", len(vitals_records))

# =====================
# Vitals feature extraction
# =====================
VITAL_COLS = ["hr", "sbp", "dbp", "temp_c", "rr"]

def compute_vitals_features(records):
    rows = []
    for rec in records:
        stay_id = rec.get("stay_id")
        days = sorted(rec.get("days", []), key=lambda d: d.get("day", 0))
        feats = {"stay_id": stay_id}

        day_idx = []
        values = {c: [] for c in VITAL_COLS}
        notes = []

        for d in days:
            day = d.get("day")
            day_idx.append(day)
            notes.append("" if d.get("note") is None else str(d.get("note")))
            for c in VITAL_COLS:
                values[c].append(d.get(c, np.nan))

        x = np.array(day_idx, dtype=float) if len(day_idx) else np.array([], dtype=float)

        for c in VITAL_COLS:
            arr = np.array(values[c], dtype=float) if len(values[c]) else np.array([np.nan], dtype=float)

            feats[f"{c}_mean"] = float(np.nanmean(arr)) if np.isfinite(np.nanmean(arr)) else np.nan
            feats[f"{c}_std"]  = float(np.nanstd(arr))  if np.isfinite(np.nanstd(arr))  else np.nan
            feats[f"{c}_min"]  = float(np.nanmin(arr))  if np.isfinite(np.nanmin(arr))  else np.nan
            feats[f"{c}_max"]  = float(np.nanmax(arr))  if np.isfinite(np.nanmax(arr))  else np.nan

            feats[f"{c}_first"] = float(arr[0]) if len(arr) else np.nan
            feats[f"{c}_last"]  = float(arr[-1]) if len(arr) else np.nan
            if np.isfinite(feats[f"{c}_first"]) and np.isfinite(feats[f"{c}_last"]):
                feats[f"{c}_delta"] = feats[f"{c}_last"] - feats[f"{c}_first"]
            else:
                feats[f"{c}_delta"] = np.nan

            mask = np.isfinite(arr) & np.isfinite(x) if len(arr) == len(x) else np.zeros_like(arr, dtype=bool)
            if mask.sum() >= 2:
                feats[f"{c}_slope"] = float(np.polyfit(x[mask], arr[mask], 1)[0])
            else:
                feats[f"{c}_slope"] = np.nan

            feats[f"{c}_missing"] = int(np.isnan(arr).sum())

        notes_joined = " ".join([n.strip() for n in notes if n is not None]).strip()
        feats["notes_1_10"] = notes_joined
        feats["notes_len"] = int(len(notes_joined))
        feats["notes_word_count"] = int(len(notes_joined.split())) if notes_joined else 0

        rows.append(feats)

    return pd.DataFrame(rows)

vitals_df = compute_vitals_features(vitals_records)

print("\n=== Vitals feature table ===")
print("vitals_df:", vitals_df.shape)
print("Example columns:", list(vitals_df.columns)[:12], "...")

# =====================
# Join tables (leakage-safe core joins)
# =====================
join_notes = []
def safe_merge(left, right, on, how, validate, tag):
    try:
        out = left.merge(right, on=on, how=how, validate=validate)
        join_notes.append({"tag": tag, "on": on, "how": how, "validate": validate, "status": "ok"})
        return out
    except Exception as e:
        join_notes.append({"tag": tag, "on": on, "how": how, "validate": validate, "status": f"FAILED: {type(e).__name__}: {e}"})
        # fall back to merge without validate so we can continue EDA
        return left.merge(right, on=on, how=how)

train = safe_merge(stays_train, patients, on="patient_id", how="left", validate="many_to_one", tag="stays_train+patients")
train = safe_merge(train, vitals_df, on="stay_id", how="left", validate="one_to_one", tag="train+vitals")

test = safe_merge(stays_test, patients, on="patient_id", how="left", validate="many_to_one", tag="stays_test+patients")
test = safe_merge(test, vitals_df, on="stay_id", how="left", validate="one_to_one", tag="test+vitals")

print("\n=== Join checks ===")
print(pd.DataFrame(join_notes))

print("\n=== Basic EDA ===")
target = "discharge_ready_day11"
print("Train target prevalence:", float(train[target].mean()), f"({int(train[target].sum())}/{len(train)})")
print("\nMissingness (top 12 train columns):")
miss = train.isna().mean().sort_values(ascending=False).head(12)
print(miss.to_string())

# group rates for quick signal
grp = train.groupby(["admission_reason", "unit_type"])[target].agg(["mean","count"]).reset_index()
grp.to_csv(os.path.join(art_root, f"eda1_group_rates_iter{iteration_id}.csv"), index=False)
print("\nSaved group-rate artifact:", os.path.join(art_root, f"eda1_group_rates_iter{iteration_id}.csv"))

# =====================
# Baseline model (still part of EDA1)
# =====================
y = train[target].astype(int).values
X = train.drop(columns=[target])

cat_cols = ["sex", "insurance", "zip3", "unit_type", "admission_reason"]
text_col = "notes_1_10"

# numeric columns: all numeric except identifiers
exclude = set(["stay_id", "patient_id"] + cat_cols + [text_col])
num_cols = [c for c in X.columns if c not in exclude and pd.api.types.is_numeric_dtype(X[c])]

def squeeze_text(x):
    # x may be a DataFrame or ndarray
    if isinstance(x, pd.DataFrame):
        s = x.iloc[:, 0]
    else:
        s = pd.Series(np.asarray(x).ravel())
    return s.fillna("").astype(str).values

text_transformer = Pipeline(steps=[
    ("squeeze", FunctionTransformer(squeeze_text, validate=False)),
    ("tfidf", TfidfVectorizer(
        max_features=TFIDF_MAX_FEATURES,
        ngram_range=TFIDF_NGRAM_RANGE,
        min_df=TFIDF_MIN_DF
    ))
])

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
        ("txt", text_transformer, [text_col]),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

model = LogisticRegression(**LR_PARAMS)
pipe = Pipeline(steps=[("preprocess", preprocess), ("model", model)])

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_proba = np.zeros(len(X), dtype=float)
fold_scores_05 = []
fold_sizes = []

print("\n=== CV (OOF) ===")
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    m = clone(pipe)
    m.fit(X_tr, y_tr)
    proba = m.predict_proba(X_va)[:, 1]
    oof_proba[va_idx] = proba

    pred_05 = (proba >= 0.5).astype(int)
    f1_05 = f1_score(y_va, pred_05, average="macro")
    fold_scores_05.append(float(f1_05))
    fold_sizes.append(int(len(va_idx)))

    print(f"Fold {fold}: n_val={len(va_idx)} macroF1@0.5={f1_05:.4f}")

mean_f1_05 = float(np.mean(fold_scores_05))

# threshold tuning on OOF (no train leakage per-sample)
threshold_grid = np.linspace(0.05, 0.95, 181)
macro_f1_grid = [float(f1_score(y, (oof_proba >= t).astype(int), average="macro")) for t in threshold_grid]
best_i = int(np.argmax(macro_f1_grid))
best_threshold = float(threshold_grid[best_i])
best_macro_f1 = float(macro_f1_grid[best_i])

# per-fold at chosen global threshold
fold_scores_best = []
for fold, (_, va_idx) in enumerate(skf.split(X, y), start=1):
    y_va = y[va_idx]
    p_va = oof_proba[va_idx]
    f1_b = f1_score(y_va, (p_va >= best_threshold).astype(int), average="macro")
    fold_scores_best.append(float(f1_b))

cm = confusion_matrix(y, (oof_proba >= best_threshold).astype(int)).tolist()

print("\nOOF macroF1 mean@0.5 :", f"{mean_f1_05:.4f}", "| per-fold:", [round(s,4) for s in fold_scores_05])
print("OOF BEST threshold   :", f"{best_threshold:.3f}")
print("OOF macroF1@best_thr :", f"{best_macro_f1:.4f}", "| per-fold:", [round(s,4) for s in fold_scores_best])
print("OOF confusion matrix :", cm)

# =====================
# Fit on full train, predict test, write submission
# =====================
pipe.fit(X, y)
test_proba = pipe.predict_proba(test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

sub = pd.DataFrame({
    "stay_id": test["stay_id"].astype(int),
    "discharge_ready_day11": test_pred.astype(int)
})
sub.to_csv(submission_path, index=False)
print("\nWrote submission:", submission_path, "| positive_rate:", float(sub["discharge_ready_day11"].mean()))

# =====================
# Checkpointing (P* logic)
# =====================
iter_dir = os.path.join(ckpt_root, f"iter{iteration_id:04d}")
os.makedirs(iter_dir, exist_ok=True)

feature_schema = {
    "num_cols": num_cols,
    "cat_cols": cat_cols,
    "text_col": text_col,
    "tfidf": dict(max_features=TFIDF_MAX_FEATURES, ngram_range=TFIDF_NGRAM_RANGE, min_df=TFIDF_MIN_DF)
}

val_metrics = {
    "macro_f1_oof_best": best_macro_f1,
    "best_threshold": best_threshold,
    "macro_f1_oof_mean_at_0.5": mean_f1_05,
    "per_fold_macro_f1_at_0.5": fold_scores_05,
    "per_fold_macro_f1_at_best_threshold": fold_scores_best,
    "confusion_matrix_oof_at_best_threshold": cm,
    "n_splits": N_SPLITS,
    "random_state": RANDOM_STATE
}

config = {
    "version": VX,
    "team": TEAM,
    "task": TASK,
    "phase": PHASE,
    "iteration_id": iteration_id,
    "timestamp_utc": timestamp,
    "data_paths": DATA_PATHS,
    "join_notes": join_notes,
    "model_class": "LogisticRegression",
    "model_params": LR_PARAMS,
    "feature_schema": feature_schema,
    "validation": {"cv": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": RANDOM_STATE},
    "thresholding": {"method": "OOF grid search", "grid": [float(threshold_grid[0]), float(threshold_grid[-1]), int(len(threshold_grid))]}
}

joblib.dump(pipe, os.path.join(iter_dir, "pipeline.joblib"))
with open(os.path.join(iter_dir, "config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)
with open(os.path.join(iter_dir, "feature_schema.json"), "w", encoding="utf-8") as f:
    json.dump(feature_schema, f, indent=2)
with open(os.path.join(iter_dir, "val_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(val_metrics, f, indent=2)

oof_df = pd.DataFrame({
    "stay_id": train["stay_id"].astype(int),
    "y_true": y.astype(int),
    "oof_proba": oof_proba.astype(float),
})
oof_df.to_csv(os.path.join(iter_dir, "oof_predictions.csv"), index=False)

# Update P* pointer if improved
pstar_path = os.path.join(ckpt_root, "pstar.json")
pstar = None
if os.path.exists(pstar_path):
    try:
        with open(pstar_path, "r", encoding="utf-8") as f:
            pstar = json.load(f)
    except Exception:
        pstar = None

prev_best = float(pstar.get("best_macro_f1", -1.0)) if isinstance(pstar, dict) else -1.0
is_new_pstar = best_macro_f1 > prev_best + 1e-12

if is_new_pstar:
    new_pstar = {
        "best_iteration_id": iteration_id,
        "best_macro_f1": best_macro_f1,
        "best_threshold": best_threshold,
        "timestamp_utc": timestamp
    }
    with open(pstar_path, "w", encoding="utf-8") as f:
        json.dump(new_pstar, f, indent=2)
    print("\nP* UPDATED:", new_pstar)
else:
    print("\nP* unchanged. Current macroF1:", best_macro_f1, "| Previous best:", prev_best)

# =====================
# Iteration detail log (append-only)
# =====================
iteration_record = {
    "version": VX,
    "iteration_id": iteration_id,
    "timestamp": timestamp,
    "phase": PHASE,
    "data_paths_used": DATA_PATHS,
    "join_keys": {"patients": "patient_id", "vitals": "stay_id"},
    "join_notes": join_notes,
    "leakage_checks_performed": [
        "Used only stays_train/stays_test + patients.csv + vitals_timeseries.json (days 1-10).",
        "Target column (discharge_ready_day11) only used from stays_train; never joined into test.",
        "No CH1/CH2 artifacts used in this iteration."
    ],
    "feature_summary": {
        "numeric_feature_count": len(num_cols),
        "categorical_feature_count": len(cat_cols),
        "text_feature_type": "TF-IDF",
        "tfidf_max_features": TFIDF_MAX_FEATURES,
        "tfidf_ngram_range": TFIDF_NGRAM_RANGE,
        "tfidf_min_df": TFIDF_MIN_DF,
        "vitals_aggregates": ["mean","std","min","max","first","last","delta","slope","missing"]
    },
    "models_attempted": [
        {
            "name": "LogisticRegression(saga) + ColumnTransformer(num+OHE+TFIDF)",
            "params": LR_PARAMS,
            "cv_macro_f1_mean_at_0.5": mean_f1_05,
            "cv_macro_f1_per_fold_at_0.5": fold_scores_05,
            "oof_best_threshold": best_threshold,
            "cv_macro_f1_oof_at_best_threshold": best_macro_f1,
            "cv_macro_f1_per_fold_at_best_threshold": fold_scores_best,
            "notes": "OOF threshold tuned via grid search (0.05..0.95)."
        }
    ],
    "selected_model": "LogisticRegression(saga) + num/cat/text features",
    "validation_protocol": {
        "splitter": "StratifiedKFold",
        "n_splits": N_SPLITS,
        "shuffle": True,
        "random_state": RANDOM_STATE,
        "threshold_selection": "Maximize macro-F1 on pooled out-of-fold probabilities."
    },
    "metrics": {
        "macro_f1_oof_best": best_macro_f1,
        "macro_f1_per_fold_at_best_threshold": fold_scores_best,
        "confusion_matrix_oof_at_best_threshold": cm,
        "chosen_threshold": best_threshold
    },
    "checkpoint": {
        "iter_dir": iter_dir,
        "pstar_path": pstar_path,
        "is_new_pstar": is_new_pstar
    },
    "deltas_vs_previous": "Initial EDA1 baseline (no previous iteration).",
    "known_defects_limitations": [
        "High-dimensional TF-IDF may overfit; we have not yet tried dimensionality reduction or stronger regularization.",
        "Vitals features are simple aggregates; may miss day-specific patterns.",
        "No hyperparameter search yet; single baseline model only."
    ],
    "next_step_hypothesis": "In EDA2, test richer time-series trend/variability features and text cleaning/keyword features; then compare linear SVM or tree-based models on aggregates."
}

with open(iter_log_path, "a", encoding="utf-8") as f:
    f.write(json.dumps(iteration_record) + "\n")

print("\nAppended iteration log:", iter_log_path)
print("Saved checkpoint dir    :", iter_dir)

[clai/ch3/v1] Iteration 0 | Phase=EDA1 | UTC=2026-03-01T01:44:25.755672+00:00
scikit-learn: 1.7.2
Input paths: {
  "stays_train": "stays_train.csv",
  "stays_test": "stays_test.csv",
  "patients": "patients.csv",
  "vitals": "vitals_timeseries.json"
}

=== Raw shapes ===
stays_train: (1000, 5)
stays_test : (1000, 4)
patients   : (4000, 5)
vitals recs: 2000

=== Vitals feature table ===
vitals_df: (2000, 49)
Example columns: ['stay_id', 'hr_mean', 'hr_std', 'hr_min', 'hr_max', 'hr_first', 'hr_last', 'hr_delta', 'hr_slope', 'hr_missing', 'sbp_mean', 'sbp_std'] ...

=== Join checks ===
                    tag          on   how     validate status
0  stays_train+patients  patient_id  left  many_to_one     ok
1          train+vitals     stay_id  left   one_to_one     ok
2   stays_test+patients  patient_id  left  many_to_one     ok
3           test+vitals     stay_id  left   one_to_one     ok

=== Basic EDA ===
Train target prevalence: 0.656 (656/1000)

Missingness (top 12 train columns):
hr

## EDA 2
- 0.6829

In [4]:
# clai_ch3_v1 — Iteration 1 (EDA2) end-to-end: load → EDA2 → deterministic CV → checkpoint → submission → append log
import os, json, random, joblib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix

# ----------------------------
# Config (v1 / EDA2)
# ----------------------------
VERSION = "v1"
PHASE = "EDA2"
SEED = 42

HM_MESSAGE = (
    "HM provided real leaderboard score for iter0 submission: Macro-F1=0.6815. "
    "Instruction: proceed to EDA2 (final EDA round) and incorporate HM feedback."
)
REAL_SCORE = 0.6815

np.random.seed(SEED)
random.seed(SEED)

# ----------------------------
# Path helpers
# ----------------------------
def resolve_file(filename: str) -> Path:
    """Find filename in common project layouts (cwd, ./healthcare, /mnt/data, /mnt/data/healthcare)."""
    candidates = [
        Path.cwd() / filename,
        Path.cwd() / "healthcare" / filename,
        Path("/mnt/data") / filename,
        Path("/mnt/data") / "healthcare" / filename,
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find required file: {filename}. Tried: {candidates}")

# Required input files
stays_train_path = resolve_file("stays_train.csv")
stays_test_path  = resolve_file("stays_test.csv")
patients_path    = resolve_file("patients.csv")
vitals_path      = resolve_file("vitals_timeseries.json")

# Optional EDA1 artifact (provided by HM/user)
eda1_group_rates_path = None
for cand in [Path.cwd() / "eda1_group_rates_iter0.csv", Path("/mnt/data") / "eda1_group_rates_iter0.csv"]:
    if cand.exists():
        eda1_group_rates_path = cand
        break

# Outputs
log_path = Path(f"clai_ch3_{VERSION}_iteration_detail.jsonl")
submission_path = Path(f"clai_ch3_{VERSION}_submission.csv")

checkpoint_root = Path("checkpoints") / f"clai_ch3_{VERSION}"
artifacts_root  = Path("artifacts") / f"clai_ch3_{VERSION}"
checkpoint_root.mkdir(parents=True, exist_ok=True)
artifacts_root.mkdir(parents=True, exist_ok=True)

# Determine iteration_id (append-only)
def next_iteration_id(p: Path) -> int:
    if not p.exists():
        return 0
    mx = -1
    with p.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                mx = max(mx, int(obj.get("iteration_id", -1)))
            except Exception:
                continue
    return mx + 1

ITER_ID = next_iteration_id(log_path)
iter_dir = checkpoint_root / f"iter{ITER_ID:03d}"
iter_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now(timezone.utc).isoformat()

# ----------------------------
# Load data
# ----------------------------
stays_train = pd.read_csv(stays_train_path)
stays_test  = pd.read_csv(stays_test_path)
patients    = pd.read_csv(patients_path)

with open(vitals_path, "r", encoding="utf-8") as f:
    vitals_list = json.load(f)

print(f"[INFO] Loaded stays_train: {stays_train.shape}  stays_test: {stays_test.shape}  patients: {patients.shape}")
print(f"[INFO] vitals_timeseries records: {len(vitals_list)} (expected 2000)")

# ----------------------------
# Feature engineering from vitals_timeseries.json
# ----------------------------
VITALS = ["hr", "sbp", "dbp", "temp_c", "rr"]
AGGS = ["mean", "std", "min", "max", "first", "last", "delta", "slope", "missing"]

def build_vitals_and_notes(vitals_recs):
    rows_vitals = []
    rows_notes = []
    max_day_seen = 0
    for rec in vitals_recs:
        sid = rec["stay_id"]
        days = sorted(rec["days"], key=lambda x: x["day"])
        if len(days) != 10:
            # keep going but record via prints; should not happen
            print(f"[WARN] stay_id={sid} has {len(days)} days (expected 10).")
        max_day_seen = max(max_day_seen, max([d.get("day", 0) for d in days]) if days else 0)

        # Notes windows
        notes_by_day = {d["day"]: (d.get("note") if isinstance(d.get("note"), str) else "") for d in days}
        note_all  = " ".join([notes_by_day.get(i, "") for i in range(1, 11)]).strip()
        note_early = " ".join([notes_by_day.get(i, "") for i in range(1, 6)]).strip()
        note_late  = " ".join([notes_by_day.get(i, "") for i in range(6, 11)]).strip()
        note_day10 = notes_by_day.get(10, "").strip()

        rows_notes.append({
            "stay_id": sid,
            "note_all": note_all,
            "note_early": note_early,
            "note_late": note_late,
            "note_day10": note_day10,
            "note_all_len": int(len(note_all.split())) if note_all else 0,
            "note_day10_len": int(len(note_day10.split())) if note_day10 else 0,
        })

        # Vitals aggregates (per stay)
        feats = {"stay_id": sid}
        x = np.arange(1, len(days) + 1, dtype=float)  # day index 1..10
        for v in VITALS:
            vals = [d.get(v) for d in days]
            arr = np.array([np.nan if vv is None else float(vv) for vv in vals], dtype=float)

            feats[f"{v}_mean"] = float(np.nanmean(arr))
            feats[f"{v}_std"]  = float(np.nanstd(arr))
            feats[f"{v}_min"]  = float(np.nanmin(arr))
            feats[f"{v}_max"]  = float(np.nanmax(arr))
            feats[f"{v}_first"] = float(arr[0]) if not np.isnan(arr[0]) else np.nan
            feats[f"{v}_last"]  = float(arr[-1]) if not np.isnan(arr[-1]) else np.nan
            feats[f"{v}_delta"] = float(arr[-1] - arr[0]) if (not np.isnan(arr[-1]) and not np.isnan(arr[0])) else np.nan
            feats[f"{v}_missing"] = float(np.isnan(arr).mean())

            mask = ~np.isnan(arr)
            if mask.sum() >= 2:
                slope = float(np.polyfit(x[mask], arr[mask], 1)[0])
            else:
                slope = np.nan
            feats[f"{v}_slope"] = slope

        rows_vitals.append(feats)

    vitals_df = pd.DataFrame(rows_vitals)
    notes_df  = pd.DataFrame(rows_notes)
    return vitals_df, notes_df, max_day_seen

vitals_df, notes_df, max_day_seen = build_vitals_and_notes(vitals_list)

# ----------------------------
# Join all tables
# ----------------------------
train_df = (
    stays_train
    .merge(patients, on="patient_id", how="left")
    .merge(vitals_df, on="stay_id", how="left")
    .merge(notes_df,  on="stay_id", how="left")
)
test_df = (
    stays_test
    .merge(patients, on="patient_id", how="left")
    .merge(vitals_df, on="stay_id", how="left")
    .merge(notes_df,  on="stay_id", how="left")
)

# ----------------------------
# EDA2 prints & leakage checks
# ----------------------------
target_rate = float(train_df["discharge_ready_day11"].mean())
print(f"[EDA2] Train target rate (discharge_ready_day11=1): {target_rate:.4f} (n={len(train_df)})")

# Join completeness
missing_pat_train = int(train_df["age"].isna().sum())
missing_pat_test  = int(test_df["age"].isna().sum())
missing_vitals_train = int(train_df[[c for c in train_df.columns if c.endswith("_mean")]].isna().any(axis=1).sum())
missing_vitals_test  = int(test_df[[c for c in test_df.columns if c.endswith("_mean")]].isna().any(axis=1).sum())
print(f"[EDA2] Missing patient rows - train: {missing_pat_train}, test: {missing_pat_test}")
print(f"[EDA2] Missing vitals aggregates - train rows w/ any NA mean: {missing_vitals_train}, test: {missing_vitals_test}")

# Train/test distribution shift (categoricals)
def dist_table(df, col):
    vc = df[col].value_counts(dropna=False)
    return (vc / vc.sum()).rename("pct").reset_index().rename(columns={"index": col})

for col in ["unit_type", "admission_reason", "sex", "insurance"]:
    dt_train = dist_table(train_df, col).rename(columns={"pct": "pct_train"})
    dt_test  = dist_table(test_df, col).rename(columns={"pct": "pct_test"})
    dt = dt_train.merge(dt_test, on=col, how="outer").fillna(0.0)
    print(f"\n[EDA2] Distribution shift check: {col}")
    print(dt.sort_values("pct_test", ascending=False).to_string(index=False))

# Vitals missingness (long)
# quick approx: derive from aggregates (missing already computed per vital)
miss_cols = [f"{v}_missing" for v in VITALS]
print("\n[EDA2] Mean missingness per vital across stays (train only):")
print(train_df[miss_cols].mean().to_string())

# Suspicious text token scan (basic leakage heuristic)
suspicious_terms = ["discharge", "dc", "ready", "day11", "day 11", "home today"]
notes_all_text = (train_df["note_all"].fillna("") + " " + test_df["note_all"].fillna("")).str.lower()
susp_hits = {t: int(notes_all_text.str.contains(t).sum()) for t in suspicious_terms}
print("\n[EDA2] Suspicious term counts across ALL notes (train+test):")
print(susp_hits)

# Strong leakage checks for this dataset
train_pat = set(stays_train["patient_id"])
test_pat  = set(stays_test["patient_id"])
patient_overlap = len(train_pat & test_pat)

train_stays = set(stays_train["stay_id"])
test_stays  = set(stays_test["stay_id"])
vitals_stays = set(vitals_df["stay_id"])

leakage_checks = [
    f"Checked vitals_timeseries max day == {max_day_seen} (expected 10).",
    f"Checked patient_id overlap train/test == {patient_overlap} (expected 0).",
    f"Checked vitals coverage: missing stays train={len(train_stays - vitals_stays)}, test={len(test_stays - vitals_stays)} (expected 0,0).",
    "Heuristic scan for suspicious terms in notes (discharge/dc/ready/day11) — counts printed; no evidence of explicit target tokens expected for this synthetic data.",
]

# Save EDA2 artifact: group shift table
group_shift = (
    stays_train.groupby(["admission_reason", "unit_type"]).size().reset_index(name="count_train")
    .merge(stays_test.groupby(["admission_reason", "unit_type"]).size().reset_index(name="count_test"),
           on=["admission_reason", "unit_type"], how="outer").fillna(0)
)
group_shift["pct_train"] = group_shift["count_train"] / group_shift["count_train"].sum()
group_shift["pct_test"]  = group_shift["count_test"]  / group_shift["count_test"].sum()
group_shift.to_csv(artifacts_root / "eda2_group_shift.csv", index=False)

# If provided, load EDA1 group-rate artifact (for continuity)
if eda1_group_rates_path is not None:
    eda1_rates = pd.read_csv(eda1_group_rates_path)
    print("\n[EDA2] Loaded EDA1 group rates artifact (iter0):")
    print(eda1_rates.to_string(index=False))

# ----------------------------
# Modeling probes (still EDA2): compare a few configurations deterministically
# Primary local selection metric: mean Macro-F1 @ threshold=0.5 (aligned to iter0 real score)
# ----------------------------
VITAL_FEAT_COLS = [c for c in train_df.columns if any(c.endswith(suf) for suf in [
    "_mean","_std","_min","_max","_first","_last","_delta","_slope","_missing"
])]
BASE_NUM_COLS = ["age", "note_all_len", "note_day10_len"]
CAT_BASE = ["unit_type", "admission_reason", "sex", "insurance"]

def build_pipeline(cfg: dict):
    text_col = cfg["text_col"]
    tfidf_max_features = int(cfg["tfidf_max_features"])
    ngram_range = tuple(cfg.get("ngram_range", (1,2)))
    C = float(cfg["C"])
    class_weight = cfg.get("class_weight", None)
    zip_as_numeric = bool(cfg.get("zip_as_numeric", False))

    num_cols = BASE_NUM_COLS + VITAL_FEAT_COLS + (["zip3"] if zip_as_numeric else [])
    cat_cols = CAT_BASE + ([] if zip_as_numeric else ["zip3"])

    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imp", SimpleImputer(strategy="median")),
                ("sc", StandardScaler())
            ]), num_cols),
            ("cat", Pipeline([
                ("imp", SimpleImputer(strategy="most_frequent")),
                ("oh", OneHotEncoder(handle_unknown="ignore"))
            ]), cat_cols),
            ("txt", TfidfVectorizer(
                max_features=tfidf_max_features,
                ngram_range=ngram_range,
                min_df=2,
                sublinear_tf=True
            ), text_col),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    clf = LogisticRegression(
        solver="saga",
        penalty="l2",
        C=C,
        class_weight=class_weight,
        max_iter=20000,
        n_jobs=1,
        random_state=SEED,
    )

    pipe = Pipeline([("pre", pre), ("clf", clf)])
    return pipe, num_cols, cat_cols

def evaluate_cfg(cfg: dict):
    X = train_df.drop(columns=["discharge_ready_day11"])
    y = train_df["discharge_ready_day11"].astype(int).values

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    oof = np.zeros(len(y), dtype=float)
    fold_id = np.zeros(len(y), dtype=int)
    per_fold_05 = []

    for fold, (tr, va) in enumerate(cv.split(X, y)):
        pipe, _, _ = build_pipeline(cfg)
        pipe.fit(X.iloc[tr], y[tr])
        proba = pipe.predict_proba(X.iloc[va])[:, 1]
        oof[va] = proba
        fold_id[va] = fold
        per_fold_05.append(float(f1_score(y[va], (proba >= 0.5).astype(int), average="macro")))

    mean_05 = float(np.mean(per_fold_05))

    # Threshold sweep (fine) on OOF predictions (secondary metric; can be unstable as seen in iter0)
    ths = np.linspace(0.30, 0.70, 41)
    best_t, best_f1 = None, -1.0
    for t in ths:
        f1 = float(f1_score(y, (oof >= t).astype(int), average="macro"))
        if f1 > best_f1:
            best_f1 = f1
            best_t = float(t)

    per_fold_best = []
    for fold in range(5):
        idx = np.where(fold_id == fold)[0]
        per_fold_best.append(float(f1_score(y[idx], (oof[idx] >= best_t).astype(int), average="macro")))

    cm_best = confusion_matrix(y, (oof >= best_t).astype(int)).tolist()

    return {
        "cfg": cfg,
        "cv_macro_f1_mean_at_0.5": mean_05,
        "cv_macro_f1_per_fold_at_0.5": per_fold_05,
        "oof_best_threshold": best_t,
        "cv_macro_f1_oof_at_best_threshold": float(best_f1),
        "cv_macro_f1_per_fold_at_best_threshold": per_fold_best,
        "confusion_matrix_oof_at_best_threshold": cm_best,
        "pred_pos_rate_at_0.5": float(((oof >= 0.5).astype(int)).mean()),
        "pred_pos_rate_at_best_t": float(((oof >= best_t).astype(int)).mean()),
    }

# Model candidates for EDA2
baseline_cfg = {
    "name": "Baseline(iter0-style): note_all + class_weight=balanced + zip categorical + C=1.0",
    "text_col": "note_all",
    "tfidf_max_features": 500,
    "ngram_range": (1, 2),
    "class_weight": "balanced",
    "C": 1.0,
    "zip_as_numeric": False,
}
candidate_cfg = {
    "name": "Candidate: note_late(6-10) + no class_weight + zip numeric + C=0.3",
    "text_col": "note_late",
    "tfidf_max_features": 500,
    "ngram_range": (1, 2),
    "class_weight": None,
    "C": 0.3,
    "zip_as_numeric": True,
}
candidate2_cfg = {
    "name": "Ablation: note_all + no class_weight + zip numeric + C=0.3",
    "text_col": "note_all",
    "tfidf_max_features": 500,
    "ngram_range": (1, 2),
    "class_weight": None,
    "C": 0.3,
    "zip_as_numeric": True,
}

configs = [baseline_cfg, candidate_cfg, candidate2_cfg]

models_attempted = []
eval_results = []
print("\n[MODEL PROBES] Starting deterministic 5-fold CV evaluations...")
for cfg in configs:
    res = evaluate_cfg(cfg)
    eval_results.append(res)
    models_attempted.append({
        "name": cfg["name"],
        "params": {k: (v if k != "ngram_range" else list(v)) for k, v in cfg.items() if k != "name"},
        "cv_macro_f1_mean_at_0.5": res["cv_macro_f1_mean_at_0.5"],
        "cv_macro_f1_per_fold_at_0.5": res["cv_macro_f1_per_fold_at_0.5"],
        "oof_best_threshold": res["oof_best_threshold"],
        "cv_macro_f1_oof_at_best_threshold": res["cv_macro_f1_oof_at_best_threshold"],
        "cv_macro_f1_per_fold_at_best_threshold": res["cv_macro_f1_per_fold_at_best_threshold"],
        "confusion_matrix_oof_at_best_threshold": res["confusion_matrix_oof_at_best_threshold"],
        "notes": "Primary selection metric: Macro-F1@0.5. Secondary: OOF threshold sweep (0.30..0.70 step 0.01).",
    })
    print(f"  - {cfg['name']}")
    print(f"    mean Macro-F1@0.5 = {res['cv_macro_f1_mean_at_0.5']:.6f} | "
          f"OOF best t={res['oof_best_threshold']:.2f} → Macro-F1={res['cv_macro_f1_oof_at_best_threshold']:.6f}")

# Select best by primary metric
best_res = max(eval_results, key=lambda r: r["cv_macro_f1_mean_at_0.5"])
selected_cfg = best_res["cfg"]
selected_model_name = selected_cfg["name"]

print("\n[SELECTION] Selected model by primary metric (Macro-F1@0.5):")
print(f"  {selected_model_name}")
print(f"  Macro-F1@0.5 = {best_res['cv_macro_f1_mean_at_0.5']:.6f}")
print(f"  (Secondary) OOF best threshold t={best_res['oof_best_threshold']:.2f} → Macro-F1={best_res['cv_macro_f1_oof_at_best_threshold']:.6f}")

# Submission threshold policy for EDA2:
# Due to iter0 evidence that aggressive threshold tuning can mislead vs leaderboard,
# we use 0.50 for the official submission, and save an alternate submission at OOF-best t for HM to optionally test.
chosen_threshold = 0.50
alt_threshold = float(best_res["oof_best_threshold"])

# ----------------------------
# Fit final selected pipeline on full train, predict test, write submission(s)
# ----------------------------
X_full = train_df.drop(columns=["discharge_ready_day11"])
y_full = train_df["discharge_ready_day11"].astype(int).values

pipe, num_cols_used, cat_cols_used = build_pipeline(selected_cfg)
pipe.fit(X_full, y_full)

proba_test = pipe.predict_proba(test_df)[:, 1]
pred_test_official = (proba_test >= chosen_threshold).astype(int)
pred_test_alt = (proba_test >= alt_threshold).astype(int)

sub_official = pd.DataFrame({
    "stay_id": stays_test["stay_id"].astype(int).values,
    "discharge_ready_day11": pred_test_official.astype(int),
})
sub_official.to_csv(submission_path, index=False)

alt_submission_path = artifacts_root / f"clai_ch3_{VERSION}_submission_thr{alt_threshold:.2f}.csv"
sub_alt = pd.DataFrame({
    "stay_id": stays_test["stay_id"].astype(int).values,
    "discharge_ready_day11": pred_test_alt.astype(int),
})
sub_alt.to_csv(alt_submission_path, index=False)

print(f"\n[OUTPUT] Wrote official submission: {submission_path.resolve()}")
print(f"[OUTPUT] Wrote alternate submission: {alt_submission_path.resolve()}")

# ----------------------------
# Checkpoint saving + P* update (v1)
# ----------------------------
model_path = iter_dir / "pipeline.joblib"
joblib.dump(pipe, model_path)

config_out = {
    "version": VERSION,
    "iteration_id": ITER_ID,
    "phase": PHASE,
    "seed": SEED,
    "selected_model_name": selected_model_name,
    "selected_cfg": {k: (v if k != "ngram_range" else list(v)) for k, v in selected_cfg.items()},
    "chosen_threshold_official": chosen_threshold,
    "alt_threshold_oof_best": alt_threshold,
    "paths": {
        "stays_train": str(stays_train_path),
        "stays_test": str(stays_test_path),
        "patients": str(patients_path),
        "vitals_timeseries": str(vitals_path),
    },
}
(config_path := iter_dir / "config.json").write_text(json.dumps(config_out, indent=2), encoding="utf-8")

schema_out = {
    "numeric_cols": num_cols_used,
    "categorical_cols": cat_cols_used,
    "text_col": selected_cfg["text_col"],
    "vitals_aggregates": AGGS,
    "vitals_channels": VITALS,
}
(schema_path := iter_dir / "feature_schema.json").write_text(json.dumps(schema_out, indent=2), encoding="utf-8")

metrics_out = {
    "primary_metric": "macro_f1_at_threshold_0.5",
    "macro_f1_mean_at_0.5": best_res["cv_macro_f1_mean_at_0.5"],
    "macro_f1_per_fold_at_0.5": best_res["cv_macro_f1_per_fold_at_0.5"],
    "oof_best_threshold": best_res["oof_best_threshold"],
    "macro_f1_oof_at_best_threshold": best_res["cv_macro_f1_oof_at_best_threshold"],
    "macro_f1_per_fold_at_best_threshold": best_res["cv_macro_f1_per_fold_at_best_threshold"],
    "confusion_matrix_oof_at_best_threshold": best_res["confusion_matrix_oof_at_best_threshold"],
}
(metrics_path := iter_dir / "metrics.json").write_text(json.dumps(metrics_out, indent=2), encoding="utf-8")

# P* tracking file (based on the primary metric; updated guardrail after observing iter0 leaderboard alignment)
pstar_path = checkpoint_root / "pstar.json"
prev_pstar = {"best_primary_macro_f1_at_0.5": -1.0, "iteration_id": None}
if pstar_path.exists():
    try:
        prev_pstar = json.loads(pstar_path.read_text(encoding="utf-8"))
    except Exception:
        pass

prev_score = float(prev_pstar.get("best_primary_macro_f1_at_0.5", -1.0))
new_score = float(best_res["cv_macro_f1_mean_at_0.5"])
pstar_updated = new_score > prev_score + 1e-12

if pstar_updated:
    pstar_model_path = checkpoint_root / "pstar_pipeline.joblib"
    joblib.dump(pipe, pstar_model_path)
    new_pstar = {
        "best_primary_macro_f1_at_0.5": new_score,
        "iteration_id": ITER_ID,
        "timestamp": timestamp,
        "selected_model_name": selected_model_name,
        "chosen_threshold_official": chosen_threshold,
        "alt_threshold_oof_best": alt_threshold,
        "pstar_pipeline_path": str(pstar_model_path),
    }
    pstar_path.write_text(json.dumps(new_pstar, indent=2), encoding="utf-8")

print(f"\n[P*] Previous P* Macro-F1@0.5 = {prev_score:.6f}")
print(f"[P*] Current  Macro-F1@0.5 = {new_score:.6f}  → updated={pstar_updated}")

# ----------------------------
# Append iteration detail log (append-only)
# ----------------------------
feature_summary = {
    "numeric_feature_count": int(len(num_cols_used)),
    "categorical_feature_count": int(len(cat_cols_used)),
    "text_feature_type": "TF-IDF",
    "text_window": selected_cfg["text_col"],
    "tfidf_max_features": int(selected_cfg["tfidf_max_features"]),
    "tfidf_ngram_range": list(selected_cfg["ngram_range"]),
    "vitals_aggregates": AGGS,
}

validation_protocol = {
    "cv": "StratifiedKFold",
    "n_splits": 5,
    "shuffle": True,
    "random_state": SEED,
    "primary_selection_metric": "macro_f1_at_threshold_0.5",
    "secondary_metric": "oof_threshold_sweep_macro_f1 (0.30..0.70 step 0.01)",
}

log_obj = {
    "version": VERSION,
    "iteration_id": ITER_ID,
    "timestamp": timestamp,
    "phase": PHASE,
    "hm_message": HM_MESSAGE,
    "real_score": REAL_SCORE,

    "data_paths_used": {
        "stays_train": str(stays_train_path),
        "stays_test": str(stays_test_path),
        "patients": str(patients_path),
        "vitals_timeseries": str(vitals_path),
        "eda1_group_rates_iter0": str(eda1_group_rates_path) if eda1_group_rates_path is not None else None,
    },
    "join_keys": {
        "stays_to_patients": "patient_id",
        "stays_to_vitals_timeseries": "stay_id",
    },
    "join_notes": "Left-joined patients by patient_id; vitals aggregates + note windows by stay_id. Verified full coverage of stay_id in vitals JSON.",

    "leakage_checks_performed": leakage_checks,

    "feature_summary": feature_summary,
    "models_attempted": models_attempted,
    "selected_model": selected_model_name,

    "validation_protocol": validation_protocol,
    "metrics": {
        "macro_f1_mean_at_0.5": best_res["cv_macro_f1_mean_at_0.5"],
        "macro_f1_per_fold_at_0.5": best_res["cv_macro_f1_per_fold_at_0.5"],
        "oof_best_threshold": best_res["oof_best_threshold"],
        "macro_f1_oof_at_best_threshold": best_res["cv_macro_f1_oof_at_best_threshold"],
        "macro_f1_per_fold_at_best_threshold": best_res["cv_macro_f1_per_fold_at_best_threshold"],
        "confusion_matrix_oof_at_best_threshold": best_res["confusion_matrix_oof_at_best_threshold"],
        "chosen_threshold_for_submission": chosen_threshold,
        "alt_threshold_saved": alt_threshold,
    },

    "checkpoint": {
        "iter_dir": str(iter_dir),
        "pipeline_path": str(model_path),
        "config_path": str(config_path),
        "schema_path": str(schema_path),
        "metrics_path": str(metrics_path),
        "pstar_path": str(pstar_path),
        "pstar_updated": bool(pstar_updated),
        "prev_pstar_primary_macro_f1_at_0.5": prev_score,
        "new_pstar_primary_macro_f1_at_0.5": new_score,
    },

    "deltas_vs_previous": (
        "Incorporated HM real score (0.6815) for iter0; observed fixed-threshold CV aligned better than "
        "OOF threshold-tuned score, so we now treat Macro-F1@0.5 as primary selection metric. "
        "Explored note-windowing; late notes (days6-10) + no class_weight + stronger regularization (C=0.3) "
        "+ treating zip3 as numeric improved CV Macro-F1@0.5."
    ),
    "known_defects_limitations": [
        "Leaderboard/generalization gap remains possible; iter0 showed threshold-tuning instability vs LB.",
        "Text features may overfit to synthetic phrasing; keep model simple and validate conservatively.",
        "Only basic vitals aggregates used; no higher-order temporal features yet.",
    ],
    "next_step_hypothesis": (
        "After EDA2, proceed to Modeling iterations: try elastic-net logistic or calibrated linear SVM; "
        "add richer temporal features (e.g., day-to-day variability, last-3-day stats), and consider a more robust "
        "threshold policy (possibly fixed 0.5 unless validated improvement is stable)."
    ),

    "outputs": {
        "official_submission": str(submission_path),
        "alternate_submission": str(alt_submission_path),
        "artifacts_dir": str(artifacts_root),
    },
}

with log_path.open("a", encoding="utf-8") as f:
    f.write(json.dumps(log_obj) + "\n")

print(f"\n[LOG] Appended iteration log: {log_path.resolve()} (iteration_id={ITER_ID})")

[INFO] Loaded stays_train: (1000, 5)  stays_test: (1000, 4)  patients: (4000, 5)
[INFO] vitals_timeseries records: 2000 (expected 2000)
[EDA2] Train target rate (discharge_ready_day11=1): 0.6560 (n=1000)
[EDA2] Missing patient rows - train: 0, test: 0
[EDA2] Missing vitals aggregates - train rows w/ any NA mean: 0, test: 0

[EDA2] Distribution shift check: unit_type
unit_type  pct_train  pct_test
 med_surg      0.792     0.817
 stepdown      0.208     0.183

[EDA2] Distribution shift check: admission_reason
admission_reason  pct_train  pct_test
              HF      0.280     0.300
       Pneumonia      0.263     0.261
    DiabetesComp      0.242     0.239
          PostOp      0.131     0.122
            COPD      0.084     0.078

[EDA2] Distribution shift check: sex
sex  pct_train  pct_test
  F      0.517     0.528
  M      0.483     0.472

[EDA2] Distribution shift check: insurance
insurance  pct_train  pct_test
  private      0.483     0.524
   public      0.458     0.419
 self_pay

## Train

In [14]:
# clai_ch3_v1 — Modeling iteration (formal submission pipeline, deterministic, checkpointed)

import os, json, random, warnings, subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.base import clone

from xgboost import XGBClassifier
import joblib

warnings.filterwarnings("ignore")

# =========================
# Config
# =========================
TEAM_TAG = "clai"
VERSION = "v1"
CHALLENGE = "ch3"
SEED = 42

TARGET_COL = "discharge_ready_day11"
TEXT_COL = "note_all"

# Threshold policy:
# - PRIMARY metric + default submission uses 0.5 to reduce optimistic threshold-overfit
THRESH_MAIN = 0.50
THRESH_GRID = np.linspace(0.10, 0.90, 17)

# XGB params (sparse-aware, regularized)
XGB_PARAMS = dict(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_lambda=2.0,
    reg_alpha=0.0,
    min_child_weight=1,
    gamma=0.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=SEED,
    n_jobs=1,  # deterministic
)

TFIDF_PARAMS = dict(
    max_features=8000,
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
)

OHE_PARAMS = dict(
    handle_unknown="ignore",
    min_frequency=5,
)

# HM-provided leaderboard feedback (recorded for audit)
HM_MESSAGE = {
    "timestamp_local": "2026-03-01",
    "note": "HM reports validation tends to be optimistic; anchor primary threshold at 0.5.",
    "real_scores": {
        "official_submission_real_f1": 0.6829,
        "alternative_submission_real_f1": 0.6851,
        "earlier_baseline_real_f1": 0.6815
    }
}

# =========================
# Helpers
# =========================
def locate_file(name: str) -> Path:
    """Try common project-root layouts + /mnt/data fallback."""
    candidates = [
        Path(name),
        Path.cwd() / name,
        Path.cwd() / "data" / name,
        Path.cwd() / "ch3" / name,
        Path("/mnt/data") / name,
    ]
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError(f"Could not locate '{name}'. Tried: {candidates}")

def safe_git_hash() -> str | None:
    try:
        out = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL)
        return out.decode("utf-8").strip()
    except Exception:
        return None

def load_iteration_id(log_path: Path) -> tuple[int, dict | None]:
    if not log_path.exists() or log_path.stat().st_size == 0:
        return 0, None
    last_obj = None
    last_id = -1
    with log_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                last_obj = obj
                last_id = int(obj.get("iteration_id", last_id))
            except Exception:
                continue
    return last_id + 1 if last_id >= 0 else 0, last_obj

def extract_prev_macro_f1_at_0_5(prev_obj: dict | None) -> float | None:
    if not prev_obj:
        return None
    # Prefer same metric field if present
    m = prev_obj.get("metrics", {}) if isinstance(prev_obj.get("metrics", {}), dict) else {}
    if "macro_f1_at_0.5" in m:
        return float(m["macro_f1_at_0.5"])
    # Fallback: prior iteration logged this under models_attempted[0]
    ma = prev_obj.get("models_attempted", [])
    if isinstance(ma, list) and len(ma) > 0 and isinstance(ma[0], dict):
        if "cv_macro_f1_mean_at_0.5" in ma[0]:
            return float(ma[0]["cv_macro_f1_mean_at_0.5"])
    return None

def build_vitals_features(vitals_json: list[dict]) -> pd.DataFrame:
    """
    Build stay-level features from 10-day vitals + daily notes.
    - Numeric: aggregates, slopes, deltas, last3 means, missingness
    - Clinically-inspired: abnormal day counts, shock index
    - Text: day-marked concatenation of notes
    """
    rows = []
    for item in vitals_json:
        stay_id = item.get("stay_id")
        days = item.get("days", [])
        if stay_id is None or not isinstance(days, list) or len(days) == 0:
            continue

        row = {"stay_id": stay_id}
        notes = []
        arrays = {v: [] for v in ["hr", "sbp", "dbp", "temp_c", "rr"]}

        # Expect days 1..10
        for d in days:
            day = d.get("day")
            notes.append(f"__DAY_{day}__ {d.get('note') or ''}")
            for v in arrays:
                val = d.get(v)
                arrays[v].append(np.nan if val is None else float(val))

        row["note_all"] = " ".join(notes)

        hr = np.array(arrays["hr"], dtype=float)
        sbp = np.array(arrays["sbp"], dtype=float)
        temp = np.array(arrays["temp_c"], dtype=float)
        rr = np.array(arrays["rr"], dtype=float)

        # Abnormal counts (simple heuristic thresholds)
        row["tachy_hr_cnt"] = float(np.nansum(hr > 100))
        row["hypotension_sbp_cnt"] = float(np.nansum(sbp < 90))
        row["fever_cnt"] = float(np.nansum(temp >= 38.0))
        row["tachypnea_cnt"] = float(np.nansum(rr > 20))

        # Shock index
        with np.errstate(divide="ignore", invalid="ignore"):
            si = hr / sbp
        row["shock_index_mean"] = float(np.nanmean(si)) if np.isfinite(np.nanmean(si)) else np.nan
        if len(hr) > 0 and len(sbp) > 0 and (not np.isnan(hr[-1])) and (not np.isnan(sbp[-1])) and sbp[-1] != 0:
            row["shock_index_last"] = float(hr[-1] / sbp[-1])
        else:
            row["shock_index_last"] = np.nan

        # Trends / aggregates per vital
        x = np.arange(1, len(days) + 1, dtype=float)
        for vital, arr_list in arrays.items():
            arr = np.array(arr_list, dtype=float)
            row[f"{vital}_mean"] = float(np.nanmean(arr))
            row[f"{vital}_std"] = float(np.nanstd(arr))
            row[f"{vital}_min"] = float(np.nanmin(arr))
            row[f"{vital}_max"] = float(np.nanmax(arr))
            row[f"{vital}_first"] = float(arr[0]) if len(arr) > 0 and not np.isnan(arr[0]) else np.nan
            row[f"{vital}_last"] = float(arr[-1]) if len(arr) > 0 and not np.isnan(arr[-1]) else np.nan
            row[f"{vital}_missing"] = float(np.isnan(arr).sum())
            if len(arr) > 1 and (not np.isnan(arr[-1])) and (not np.isnan(arr[0])):
                row[f"{vital}_delta"] = float(arr[-1] - arr[0])
            else:
                row[f"{vital}_delta"] = np.nan

            m = ~np.isnan(arr)
            if m.sum() >= 2:
                xm = x[m] - x[m].mean()
                ym = arr[m] - arr[m].mean()
                denom = (xm * xm).sum()
                row[f"{vital}_slope"] = float((xm * ym).sum() / denom) if denom != 0 else np.nan
            else:
                row[f"{vital}_slope"] = np.nan

            row[f"{vital}_last3_mean"] = float(np.nanmean(arr[-3:])) if len(arr) >= 3 else float(np.nanmean(arr))

        # Note length proxies (sometimes useful)
        row["note_len"] = int(len(row["note_all"]))
        row["note_words"] = int(len(row["note_all"].split()))

        rows.append(row)

    return pd.DataFrame(rows)

def ensure_dirs():
    Path(f"checkpoints/{TEAM_TAG}_{CHALLENGE}_{VERSION}").mkdir(parents=True, exist_ok=True)
    Path(f"artifacts/{TEAM_TAG}_{CHALLENGE}_{VERSION}").mkdir(parents=True, exist_ok=True)

# =========================
# Determinism
# =========================
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# =========================
# Paths / iteration bookkeeping
# =========================
ensure_dirs()

submission_path = Path(f"{TEAM_TAG}_{CHALLENGE}_{VERSION}_submission.csv")
iter_log_path = Path(f"{TEAM_TAG}_{CHALLENGE}_{VERSION}_iteration_detail.jsonl")

iteration_id, prev_obj = load_iteration_id(iter_log_path)
prev_macro05 = extract_prev_macro_f1_at_0_5(prev_obj)

# =========================
# Load data
# =========================
stays_train_path = locate_file("stays_train.csv")
stays_test_path = locate_file("stays_test.csv")
patients_path = locate_file("patients.csv")
vitals_path = locate_file("vitals_timeseries.json")

stays_train = pd.read_csv(stays_train_path)
stays_test = pd.read_csv(stays_test_path)
patients = pd.read_csv(patients_path)
with open(vitals_path, "r", encoding="utf-8") as f:
    vitals_json = json.load(f)

vitals_df = build_vitals_features(vitals_json)

# Joins (safe: patient_id + stay_id; no CH1/CH2 joins here)
train_df = stays_train.merge(patients, on="patient_id", how="left").merge(vitals_df, on="stay_id", how="left")
test_df = stays_test.merge(patients, on="patient_id", how="left").merge(vitals_df, on="stay_id", how="left")

# Simple combined categorical
train_df["unit_reason"] = train_df["unit_type"].astype(str) + "__" + train_df["admission_reason"].astype(str)
test_df["unit_reason"] = test_df["unit_type"].astype(str) + "__" + test_df["admission_reason"].astype(str)

# Basic sanity
assert TARGET_COL in train_df.columns, f"Missing target col {TARGET_COL}"
assert "stay_id" in train_df.columns and "stay_id" in test_df.columns, "Missing stay_id"
assert TEXT_COL in train_df.columns and TEXT_COL in test_df.columns, f"Missing text col {TEXT_COL}"

# Feature columns
cat_cols = ["unit_type", "admission_reason", "sex", "insurance", "unit_reason", "zip3"]
num_cols = [c for c in train_df.columns if c not in ["stay_id", "patient_id", TARGET_COL, TEXT_COL] + cat_cols]

X = train_df.drop(columns=[TARGET_COL])
y = train_df[TARGET_COL].astype(int).values

# =========================
# Model pipeline
# =========================
preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
        ("cat", OneHotEncoder(**OHE_PARAMS), cat_cols),
        ("txt", TfidfVectorizer(**TFIDF_PARAMS), TEXT_COL),
    ],
    remainder="drop",
)

model = XGBClassifier(**XGB_PARAMS)
pipe = Pipeline([("preprocess", preprocess), ("clf", model)])

# =========================
# Deterministic validation (OOF)
# =========================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof_proba = np.zeros(len(y), dtype=float)
per_fold_macro05 = []
per_fold_cm05 = []
per_fold_best_thr = []

for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
    fold_model = clone(pipe)
    fold_model.fit(X.iloc[tr_idx], y[tr_idx])

    p = fold_model.predict_proba(X.iloc[va_idx])[:, 1]
    oof_proba[va_idx] = p

    # threshold=0.5 is primary metric for realism
    pred05 = (p >= THRESH_MAIN).astype(int)
    per_fold_macro05.append(float(f1_score(y[va_idx], pred05, average="macro")))
    per_fold_cm05.append(confusion_matrix(y[va_idx], pred05).tolist())

    # fold best threshold (for stability diagnostic)
    best_f, best_t = -1.0, float(THRESH_MAIN)
    for thr in THRESH_GRID:
        f = f1_score(y[va_idx], (p >= thr).astype(int), average="macro")
        if f > best_f:
            best_f, best_t = float(f), float(thr)
    per_fold_best_thr.append(best_t)

macro_f1_at_05 = float(f1_score(y, (oof_proba >= THRESH_MAIN).astype(int), average="macro"))
cm_at_05 = confusion_matrix(y, (oof_proba >= THRESH_MAIN).astype(int)).tolist()

# Global best threshold on OOF (logged, but NOT the primary submission threshold)
best_oof_f, best_oof_t = -1.0, float(THRESH_MAIN)
for thr in THRESH_GRID:
    f = f1_score(y, (oof_proba >= thr).astype(int), average="macro")
    if f > best_oof_f:
        best_oof_f, best_oof_t = float(f), float(thr)

thr_median = float(np.median(per_fold_best_thr))
macro_f1_at_median = float(f1_score(y, (oof_proba >= thr_median).astype(int), average="macro"))

# Delta vs previous (if available)
delta_macro05 = None if prev_macro05 is None else float(macro_f1_at_05 - prev_macro05)

# =========================
# Fit final model and predict test
# =========================
final_model = clone(pipe)
final_model.fit(X, y)
test_proba = final_model.predict_proba(test_df)[:, 1]

test_pred_main = (test_proba >= THRESH_MAIN).astype(int)

sub = pd.DataFrame({
    "stay_id": test_df["stay_id"].astype(int),
    TARGET_COL: test_pred_main.astype(int)
})
sub.to_csv(submission_path, index=False)

# Optional alt submission if median threshold differs (kept in artifacts)
art_dir = Path(f"artifacts/{TEAM_TAG}_{CHALLENGE}_{VERSION}")
alt_path = art_dir / f"{TEAM_TAG}_{CHALLENGE}_{VERSION}_submission_alt_thr{thr_median:.2f}.csv"
if abs(thr_median - THRESH_MAIN) > 1e-9:
    test_pred_alt = (test_proba >= thr_median).astype(int)
    sub_alt = pd.DataFrame({"stay_id": test_df["stay_id"].astype(int), TARGET_COL: test_pred_alt.astype(int)})
    sub_alt.to_csv(alt_path, index=False)
else:
    # still write a copy for traceability
    sub.to_csv(alt_path, index=False)

# =========================
# Checkpointing + P* tracking
# =========================
ckpt_root = Path(f"checkpoints/{TEAM_TAG}_{CHALLENGE}_{VERSION}")
ckpt_dir = ckpt_root / f"iter{iteration_id:04d}"
ckpt_dir.mkdir(parents=True, exist_ok=True)

schema = {
    "text_col": TEXT_COL,
    "categorical_cols": cat_cols,
    "numeric_cols": num_cols,
}

validation = {
    "macro_f1_at_0.5": macro_f1_at_05,
    "macro_f1_per_fold_at_0.5": per_fold_macro05,
    "confusion_matrix_at_0.5": cm_at_05,
    "per_fold_confusion_matrix_at_0.5": per_fold_cm05,
    "best_threshold_oof": best_oof_t,
    "macro_f1_oof_best_on_grid": best_oof_f,
    "per_fold_best_thresholds": per_fold_best_thr,
    "threshold_median_per_fold": thr_median,
    "macro_f1_oof_at_median_threshold": macro_f1_at_median,
    "threshold_grid": [float(x) for x in THRESH_GRID],
    "seed": SEED,
    "cv": {"type": "StratifiedKFold", "n_splits": 5, "shuffle": True, "random_state": SEED},
}

config = {
    "team_tag": TEAM_TAG,
    "version": VERSION,
    "challenge": CHALLENGE,
    "seed": SEED,
    "threshold_main": THRESH_MAIN,
    "tfidf_params": TFIDF_PARAMS,
    "onehot_params": OHE_PARAMS,
    "xgb_params": XGB_PARAMS,
    "primary_validation_metric": "macro_f1_at_0.5",
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "git_hash": safe_git_hash(),
}

# Save checkpoint bundle
with open(ckpt_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
with open(ckpt_dir / "feature_schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=2, ensure_ascii=False)
with open(ckpt_dir / "validation.json", "w", encoding="utf-8") as f:
    json.dump(validation, f, indent=2, ensure_ascii=False)

joblib.dump(final_model, ckpt_dir / "model.joblib")

# Track P* by macro_f1_at_0.5
pstar_path = ckpt_root / "pstar.json"
prev_pstar = None
if pstar_path.exists():
    try:
        prev_pstar = json.loads(pstar_path.read_text(encoding="utf-8"))
    except Exception:
        prev_pstar = None

prev_best = float(prev_pstar.get("best_score", -np.inf)) if isinstance(prev_pstar, dict) else -np.inf
pstar_updated = bool(macro_f1_at_05 > prev_best + 1e-12)

pstar_record = {
    "metric": "macro_f1_at_0.5",
    "best_score": float(macro_f1_at_05) if pstar_updated else float(prev_best),
    "best_iteration_id": int(iteration_id) if pstar_updated else int(prev_pstar.get("best_iteration_id", -1) if isinstance(prev_pstar, dict) else -1),
    "best_checkpoint_dir": str(ckpt_dir) if pstar_updated else str(prev_pstar.get("best_checkpoint_dir") if isinstance(prev_pstar, dict) else ""),
    "updated_at": datetime.now().isoformat(timespec="seconds"),
}
pstar_path.write_text(json.dumps(pstar_record, indent=2, ensure_ascii=False), encoding="utf-8")

# =========================
# Append iteration detail log (append-only)
# =========================
iteration_detail = {
    "version": VERSION,
    "iteration_id": int(iteration_id),
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "phase": "Modeling",
    "data_paths_used": {
        "stays_train": str(stays_train_path),
        "stays_test": str(stays_test_path),
        "patients": str(patients_path),
        "vitals_timeseries": str(vitals_path),
    },
    "join_keys": [
        {"left": "stays_train", "right": "patients", "key": "patient_id", "how": "left"},
        {"left": "stays_[train/test]", "right": "vitals_features", "key": "stay_id", "how": "left"},
    ],
    "leakage_checks_performed": [
        "No CH1/CH2 feature joins used in this iteration.",
        "All features derived only from patients.csv and vitals_timeseries.json (days 1-10) plus stays_[train/test].",
        "TF-IDF/OneHot/Imputer fit is performed inside each CV fold via sklearn Pipeline/ColumnTransformer.",
        "Target column excluded from all feature construction.",
    ],
    "feature_summary": {
        "numeric_feature_count": int(len(num_cols)),
        "categorical_feature_count": int(len(cat_cols)),
        "text_feature": {"column": TEXT_COL, **TFIDF_PARAMS},
        "engineered_numeric_examples": [
            "hr/sbp/dbp/temp_c/rr: mean,std,min,max,first,last,delta,slope,last3_mean,missing",
            "tachy_hr_cnt, hypotension_sbp_cnt, fever_cnt, tachypnea_cnt",
            "shock_index_mean, shock_index_last",
            "note_len, note_words",
        ],
    },
    "models_attempted": [
        {
            "name": "XGBClassifier",
            "params": XGB_PARAMS,
            "cv_macro_f1_at_0.5": macro_f1_at_05,
            "cv_macro_f1_per_fold_at_0.5": per_fold_macro05,
            "cv_macro_f1_oof_best_on_grid": best_oof_f,
            "best_threshold_oof_on_grid": best_oof_t,
            "median_threshold_per_fold": thr_median,
            "macro_f1_oof_at_median_threshold": macro_f1_at_median,
            "notes": "Primary metric anchored at 0.5 to reduce optimistic threshold tuning drift.",
        }
    ],
    "selected_model": {
        "name": "XGBClassifier",
        "threshold_main": THRESH_MAIN,
        "threshold_median_per_fold": thr_median,
    },
    "validation_protocol": {
        "cv": {"type": "StratifiedKFold", "n_splits": 5, "shuffle": True, "random_state": SEED},
        "primary_threshold": THRESH_MAIN,
        "threshold_grid_logged": [float(x) for x in THRESH_GRID],
    },
    "metrics": {
        "macro_f1_at_0.5": macro_f1_at_05,
        "macro_f1_per_fold_at_0.5": per_fold_macro05,
        "confusion_matrix_at_0.5": cm_at_05,
        "macro_f1_oof_best_on_grid": best_oof_f,
        "best_threshold_oof_on_grid": best_oof_t,
        "threshold_median_per_fold": thr_median,
        "macro_f1_oof_at_median_threshold": macro_f1_at_median,
    },
    "deltas_vs_previous_iteration": {
        "prev_iteration_id": int(prev_obj.get("iteration_id")) if isinstance(prev_obj, dict) and "iteration_id" in prev_obj else None,
        "prev_macro_f1_at_0.5": prev_macro05,
        "delta_macro_f1_at_0.5": delta_macro05,
        "what_changed": [
            "Model family: LogisticRegression -> XGBClassifier",
            "Richer vitals feature engineering: slopes/deltas/last3 + abnormal counts + shock index",
            "Text: day-marked concatenation of daily notes; TF-IDF max_features increased",
            "Threshold policy: use 0.5 as primary (reduce optimistic tuning); log grid diagnostics",
            "Full checkpoint bundle + P* tracking enabled",
        ],
    },
    "hm_message": HM_MESSAGE,
    "real_score": HM_MESSAGE.get("real_scores"),
    "pstar": {
        "metric": pstar_record["metric"],
        "updated": pstar_updated,
        "best_score": pstar_record["best_score"],
        "best_iteration_id": pstar_record["best_iteration_id"],
        "best_checkpoint_dir": pstar_record["best_checkpoint_dir"],
    },
    "artifacts_written": {
        "submission_main": str(submission_path),
        "submission_alt": str(alt_path),
        "checkpoint_dir": str(ckpt_dir),
        "model_joblib": str(ckpt_dir / "model.joblib"),
    },
    "known_defects_limitations": [
        "Still relies on IID stratified CV; if note templates shift between train/test, CV may remain optimistic.",
        "No explicit probability calibration; threshold=0.5 assumes reasonably calibrated scores.",
    ],
    "next_step_hypothesis": "If real leaderboard still lags, add char-level TF-IDF branch (robust to template shift) or try CatBoost text features + calibration; consider repeated CV stability checks.",
}

with iter_log_path.open("a", encoding="utf-8") as f:
    f.write(json.dumps(iteration_detail, ensure_ascii=False) + "\n")

# =========================
# Print run summary
# =========================
print("==== clai_ch3_v1 Modeling run complete ====")
print(f"Iteration ID: {iteration_id}")
print(f"Train shape: {train_df.shape} | Test shape: {test_df.shape}")
print(f"OOF Macro-F1 @0.5: {macro_f1_at_05:.6f}")
print(f"Per-fold Macro-F1 @0.5: {[round(x,6) for x in per_fold_macro05]}")
print(f"Confusion matrix @0.5: {cm_at_05}")
print(f"Best OOF threshold on grid: {best_oof_t:.2f} (Macro-F1 {best_oof_f:.6f})")
print(f"Median fold-best threshold: {thr_median:.2f} (OOF Macro-F1 {macro_f1_at_median:.6f})")
if prev_macro05 is not None:
    print(f"Delta vs prev Macro-F1@0.5: {delta_macro05:+.6f} (prev={prev_macro05:.6f})")
print(f"P* updated: {pstar_updated} | P* best_score={pstar_record['best_score']:.6f}")
print(f"Saved submission: {submission_path.resolve()}")
print(f"Saved alt submission: {alt_path.resolve()}")
print(f"Checkpoint dir: {ckpt_dir.resolve()}")
print(f"Iteration log appended: {iter_log_path.resolve()}")

==== clai_ch3_v1 Modeling run complete ====
Iteration ID: 2
Train shape: (1000, 69) | Test shape: (1000, 68)
OOF Macro-F1 @0.5: 0.724286
Per-fold Macro-F1 @0.5: [0.778654, 0.720738, 0.728869, 0.68, 0.705796]
Confusion matrix @0.5: [[178, 166], [62, 594]]
Best OOF threshold on grid: 0.55 (Macro-F1 0.731536)
Median fold-best threshold: 0.55 (OOF Macro-F1 0.731536)
Delta vs prev Macro-F1@0.5: +0.044845 (prev=0.679441)
P* updated: True | P* best_score=0.724286
Saved submission: D:\AgentDs\agent_ds_healthcare\clai_ch3_v1_submission.csv
Saved alt submission: D:\AgentDs\agent_ds_healthcare\artifacts\clai_ch3_v1\clai_ch3_v1_submission_alt_thr0.55.csv
Checkpoint dir: D:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v1\iter0002
Iteration log appended: D:\AgentDs\agent_ds_healthcare\clai_ch3_v1_iteration_detail.jsonl


# Iteration 2
- 0.7640

In [19]:
import os, json, random, hashlib, subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.exceptions import ConvergenceWarning
import joblib
import warnings

# ======================
# Config (v1 official)
# ======================
VX = "v1"
TEAM_TAG = "clai"
RUN_TAG = f"{TEAM_TAG}_ch3_{VX}"
SEED = 42
N_SPLITS = 5

TFIDF_ALL = dict(max_features=6000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
TFIDF_LAST3 = dict(max_features=3000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)

LR_PARAMS = dict(
    solver="saga",
    max_iter=3000,
    class_weight="balanced",
    n_jobs=1,               # deterministic + avoids parallel overhead
    C=3.0,
    random_state=SEED,
)

THR_GRID = np.linspace(0.10, 0.90, 161)

# ======================
# Reproducibility
# ======================
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# ======================
# Paths / folders
# ======================
def resolve_input_path(filename: str) -> Path:
    # Find input file in CWD; fallback to /mnt/data (for this sandbox).
    p = Path(filename)
    if p.exists():
        return p
    p2 = Path("/mnt/data") / filename
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Missing required input: {filename}")

iter_log_path = Path(f"{RUN_TAG}_iteration_detail.jsonl")
# If the log exists only in /mnt/data (sandbox), append there to preserve history
if (not iter_log_path.exists()) and (Path("/mnt/data") / iter_log_path.name).exists():
    iter_log_path = Path("/mnt/data") / iter_log_path.name

submission_path = Path(f"{RUN_TAG}_submission.csv")

ckpt_root = Path("checkpoints") / RUN_TAG
artifact_root = Path("artifacts") / RUN_TAG
ckpt_root.mkdir(parents=True, exist_ok=True)
artifact_root.mkdir(parents=True, exist_ok=True)

def get_next_iteration_id(log_path: Path) -> int:
    if not log_path.exists():
        return 0
    last_id = -1
    with log_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                last_id = max(last_id, int(obj.get("iteration_id", -1)))
            except Exception:
                continue
    return last_id + 1

iteration_id = get_next_iteration_id(iter_log_path)
timestamp = datetime.now(timezone.utc).isoformat()

# ======================
# Load core data
# ======================
stays_train = pd.read_csv(resolve_input_path("stays_train.csv"))
stays_test = pd.read_csv(resolve_input_path("stays_test.csv"))
patients = pd.read_csv(resolve_input_path("patients.csv"))
with open(resolve_input_path("vitals_timeseries.json"), "r", encoding="utf-8") as f:
    vitals = json.load(f)

# ======================
# Feature engineering from vitals_timeseries.json
# ======================
VITAL_COLS = ["hr", "sbp", "dbp", "temp_c", "rr"]

def featurize_vitals(vitals_list):
    rows = []
    for rec in vitals_list:
        sid = rec.get("stay_id")
        days = sorted(rec.get("days", []), key=lambda d: d.get("day", 0))

        notes = [(d.get("note") or "") for d in days]
        feat = {
            "stay_id": sid,
            "note_all": " ".join(notes).strip(),
            "note_last3": " ".join(notes[-3:]).strip(),  # days 8-10 emphasis
        }

        for vital in VITAL_COLS:
            arr = np.array(
                [np.nan if d.get(vital) is None else float(d.get(vital)) for d in days],
                dtype=float,
            )

            feat[f"{vital}_mean"] = float(np.nanmean(arr))
            feat[f"{vital}_std"] = float(np.nanstd(arr))
            feat[f"{vital}_min"] = float(np.nanmin(arr))
            feat[f"{vital}_max"] = float(np.nanmax(arr))
            feat[f"{vital}_last"] = float(arr[-1]) if not np.isnan(arr[-1]) else np.nan
            feat[f"{vital}_missing"] = int(np.isnan(arr).sum())

            idx = np.where(~np.isnan(arr))[0]
            if len(idx) >= 2:
                x = (idx + 1).astype(float)
                yv = arr[idx].astype(float)
                feat[f"{vital}_slope"] = float(np.polyfit(x, yv, 1)[0])
            else:
                feat[f"{vital}_slope"] = np.nan

            arr3 = arr[-3:]
            feat[f"{vital}_last3_mean"] = float(np.nanmean(arr3))
            feat[f"{vital}_last3_std"] = float(np.nanstd(arr3))

            if not np.isnan(arr[-1]) and not np.isnan(arr[0]):
                feat[f"{vital}_delta_last_first"] = float(arr[-1] - arr[0])
            else:
                feat[f"{vital}_delta_last_first"] = np.nan

        rows.append(feat)

    return pd.DataFrame(rows)

vitals_df = featurize_vitals(vitals)

# ======================
# Join tables (strict validation)
# ======================
train = (
    stays_train.merge(patients, on="patient_id", how="left", validate="many_to_one")
    .merge(vitals_df, on="stay_id", how="left", validate="one_to_one")
)
test = (
    stays_test.merge(patients, on="patient_id", how="left", validate="many_to_one")
    .merge(vitals_df, on="stay_id", how="left", validate="one_to_one")
)

# Leakage + integrity checks
assert "discharge_ready_day11" in train.columns
assert "discharge_ready_day11" not in test.columns
assert train["stay_id"].is_unique and test["stay_id"].is_unique
assert train.shape[0] == stays_train.shape[0] and test.shape[0] == stays_test.shape[0]

# Fill text NA
for c in ["note_all", "note_last3"]:
    train[c] = train[c].fillna("")
    test[c] = test[c].fillna("")

# ======================
# Build model inputs
# ======================
X = train.drop(columns=["discharge_ready_day11"])
y = train["discharge_ready_day11"].astype(int).values
groups = train["patient_id"].values

text_cols = ["note_all", "note_last3"]
cat_cols = ["unit_type", "admission_reason", "sex", "insurance", "zip3"]
num_cols = [c for c in X.columns if c not in (["stay_id", "patient_id"] + text_cols + cat_cols)]

preprocess = ColumnTransformer(
    transformers=[
        ("txt_all", TfidfVectorizer(**TFIDF_ALL), "note_all"),
        ("txt_last3", TfidfVectorizer(**TFIDF_LAST3), "note_last3"),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", Pipeline(
            steps=[
                ("imp", SimpleImputer(strategy="median")),
                ("sc", StandardScaler(with_mean=False)),
            ]
        ), num_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

clf = LogisticRegression(**LR_PARAMS)
pipe = Pipeline(steps=[("preprocess", preprocess), ("clf", clf)])

# ======================
# Deterministic validation
# ======================
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_prob = np.zeros(len(X), dtype=float)
fold_sizes = []
conv_warns = 0

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always", ConvergenceWarning)
    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y, groups)):
        pipe.fit(X.iloc[tr_idx], y[tr_idx])
        prob = pipe.predict_proba(X.iloc[va_idx])[:, 1]
        oof_prob[va_idx] = prob
        fold_sizes.append((len(tr_idx), len(va_idx)))
    conv_warns = sum(issubclass(x.category, ConvergenceWarning) for x in w)

best_thr = 0.5
best_oof_f1 = -1.0
for thr in THR_GRID:
    pred = (oof_prob >= thr).astype(int)
    f1 = f1_score(y, pred, average="macro")
    if f1 > best_oof_f1:
        best_oof_f1 = float(f1)
        best_thr = float(thr)

fold_f1 = []
for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y, groups)):
    pred = (oof_prob[va_idx] >= best_thr).astype(int)
    fold_f1.append(float(f1_score(y[va_idx], pred, average="macro")))

macro_f1_mean = float(np.mean(fold_f1))
cm = confusion_matrix(y, (oof_prob >= best_thr).astype(int))

print("==== Validation (deterministic) ====")
print(f"CV: StratifiedGroupKFold(n_splits={N_SPLITS}, group=patient_id, seed={SEED})")
print("Fold sizes (train, val):", fold_sizes)
print("Macro-F1 per fold:", [round(x, 6) for x in fold_f1])
print("Macro-F1 mean:", round(macro_f1_mean, 6))
print("OOF-best threshold:", best_thr, "OOF Macro-F1:", round(best_oof_f1, 6))
print("Confusion matrix [[TN, FP],[FN, TP]]:")
print(cm)
print("Train positive rate:", round(float(y.mean()), 4))
print("OOF predicted positive rate:", round(float(((oof_prob >= best_thr).astype(int)).mean()), 4))
print("ConvergenceWarnings captured during CV fits:", int(conv_warns))
print("\nClassification report (OOF @ best_thr):")
print(classification_report(y, (oof_prob >= best_thr).astype(int), digits=4, zero_division=0))

# ======================
# Train full + predict test + write submission
# ======================
pipe.fit(X, y)
test_prob = pipe.predict_proba(test[X.columns])[:, 1]
test_pred = (test_prob >= best_thr).astype(int)

sub = pd.DataFrame(
    {"stay_id": test["stay_id"].astype(int), "discharge_ready_day11": test_pred.astype(int)}
)
sub.to_csv(submission_path, index=False)
print(f"\nWrote submission -> {submission_path.resolve()} (rows={len(sub)})")

# ======================
# Checkpoint bundle
# ======================
iter_ckpt_dir = ckpt_root / f"iter{iteration_id:04d}"
iter_ckpt_dir.mkdir(parents=True, exist_ok=True)

git_hash = None
try:
    git_hash = subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL, text=True).strip()
except Exception:
    git_hash = None

joblib.dump(pipe, iter_ckpt_dir / "pipeline.joblib")

config = {
    "version": VX,
    "seed": SEED,
    "n_splits": N_SPLITS,
    "tfidf_all": TFIDF_ALL,
    "tfidf_last3": TFIDF_LAST3,
    "logreg_params": LR_PARAMS,
    "thr_grid": {"min": float(THR_GRID.min()), "max": float(THR_GRID.max()), "n": int(len(THR_GRID))},
    "git_hash": git_hash,
}
schema = {
    "text_cols": text_cols,
    "cat_cols": cat_cols,
    "num_cols": num_cols,
    "vital_cols": VITAL_COLS,
    "all_model_input_cols": list(X.columns),
}
val_metrics = {
    "macro_f1_mean": macro_f1_mean,
    "macro_f1_per_fold": fold_f1,
    "oof_macro_f1": best_oof_f1,
    "chosen_threshold": best_thr,
    "confusion_matrix": cm.tolist(),
    "train_positive_rate": float(y.mean()),
    "oof_pred_positive_rate": float(((oof_prob >= best_thr).astype(int)).mean()),
    "convergence_warnings": int(conv_warns),
}

(iter_ckpt_dir / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
(iter_ckpt_dir / "feature_schema.json").write_text(json.dumps(schema, indent=2), encoding="utf-8")
(iter_ckpt_dir / "val_metrics.json").write_text(json.dumps(val_metrics, indent=2), encoding="utf-8")

# Artifacts
np.save(artifact_root / f"iter{iteration_id:04d}_oof_prob.npy", oof_prob)
pd.DataFrame({"stay_id": train["stay_id"].astype(int), "y_true": y.astype(int), "oof_prob": oof_prob}).to_csv(
    artifact_root / f"iter{iteration_id:04d}_oof_predictions.csv", index=False
)

# P* tracking
best_state_path = ckpt_root / "best_state.json"
best_state = {"best_macro_f1": None, "best_iteration_id": None, "best_ckpt_dir": None}
if best_state_path.exists():
    try:
        best_state = json.loads(best_state_path.read_text(encoding="utf-8"))
    except Exception:
        pass

prev_best = best_state.get("best_macro_f1")
p_star_updated = False
if prev_best is None or best_oof_f1 > float(prev_best):
    best_state = {
        "best_macro_f1": float(best_oof_f1),
        "best_iteration_id": int(iteration_id),
        "best_ckpt_dir": str(iter_ckpt_dir),
        "timestamp": timestamp,
    }
    best_state_path.write_text(json.dumps(best_state, indent=2), encoding="utf-8")
    p_star_updated = True

print("\nCheckpoint saved ->", iter_ckpt_dir.resolve())
print("P* updated:", p_star_updated, "| best_state:", best_state)

# ======================
# Iteration detail log (append-only)
# ======================
hm_message = (
    "HM reported real Macro-F1 scores: official=0.6829, alternative=0.6851; "
    "later submission1=0.7068, alt=0.7070. "
    "Also noted local CV was optimistic vs real and requested a rethink."
)
real_score = {"official": 0.6829, "alternative": 0.6851, "submission1": 0.7068, "alt_submission": 0.7070}

entry = {
    "version": VX,
    "iteration_id": int(iteration_id),
    "timestamp": timestamp,
    "phase": "Modeling",
    "data_paths_used": {
        "stays_train": str(resolve_input_path("stays_train.csv")),
        "stays_test": str(resolve_input_path("stays_test.csv")),
        "patients": str(resolve_input_path("patients.csv")),
        "vitals": str(resolve_input_path("vitals_timeseries.json")),
    },
    "join_keys": {"patients": "patient_id", "vitals": "stay_id"},
    "leakage_checks_performed": [
        "Used ONLY stays_train/stays_test + patients.csv + vitals_timeseries.json (days 1-10).",
        "No CH1/CH2 artifacts used; no discharge_notes.json used.",
        "Target used ONLY from stays_train; never merged into test.",
        "Validation uses StratifiedGroupKFold grouped by patient_id to reduce leakage via repeated patients across folds.",
    ],
    "feature_summary": {
        "categoricals": {"cols": cat_cols, "encoder": "OneHotEncoder(handle_unknown='ignore')"},
        "text": {
            "cols": text_cols,
            "note_all_vectorizer": TFIDF_ALL,
            "note_last3_vectorizer": TFIDF_LAST3,
        },
        "numeric": {
            "count": int(len(num_cols)),
            "vitals_aggregations": [
                "mean", "std", "min", "max", "last", "missing",
                "slope", "last3_mean", "last3_std", "delta_last_first",
            ],
        },
    },
    "models_attempted": [
        {
            "name": "LogisticRegression(saga) + TFIDF(text) + vitals trend features",
            "params": LR_PARAMS,
            "cv_macro_f1_oof": float(best_oof_f1),
            "cv_macro_f1_per_fold_at_thr": fold_f1,
            "threshold": float(best_thr),
            "notes": "Threshold tuned on global OOF probabilities to maximize macro-F1.",
            "convergence_warnings": int(conv_warns),
        }
    ],
    "selected_model": {
        "name": "LogisticRegression(saga)",
        "params": LR_PARAMS,
        "preprocess": {
            "tfidf_all": TFIDF_ALL,
            "tfidf_last3": TFIDF_LAST3,
            "categoricals": cat_cols,
            "numerics_count": int(len(num_cols)),
        },
    },
    "validation_protocol": {
        "cv": "StratifiedGroupKFold",
        "n_splits": N_SPLITS,
        "shuffle": True,
        "random_state": SEED,
        "group_col": "patient_id",
        "threshold_selection": "global_oof_max_macro_f1",
    },
    "metrics": {
        "macro_f1_mean": macro_f1_mean,
        "macro_f1_per_fold": fold_f1,
        "oof_macro_f1": float(best_oof_f1),
        "chosen_threshold": float(best_thr),
        "confusion_matrix": cm.tolist(),
    },
    "deltas_vs_previous_iteration": {
        "what_changed": [
            "Added multi-window vitals numeric trend features (overall + last3).",
            "Added two text channels: all notes + last3-day notes with separate TF-IDF vectorizers.",
            "Switched validation to StratifiedGroupKFold grouped by patient_id; tuned threshold on OOF for macro-F1.",
        ]
    },
    "checkpoint": {
        "iter_ckpt_dir": str(iter_ckpt_dir),
        "p_star_updated": bool(p_star_updated),
        "best_state_path": str(best_state_path),
        "best_state": best_state,
    },
    "known_defects_limitations": [
        "Threshold optimized on OOF may still be slightly optimistic; consider nested threshold tuning if real/CV gap persists.",
        "No explicit time-based split available; patient-grouped CV is the main generalization control.",
    ],
    "next_step_hypothesis": (
        "If we need >0.75, try (1) small C sweep (e.g., 1.5/2/3/4), "
        "(2) add day-tagged text (D1..D10) to preserve chronology, "
        "(3) try SVD+LightGBM as a second model family."
    ),
    "hm_message": hm_message,
    "real_score": real_score,
}

with iter_log_path.open("a", encoding="utf-8") as f:
    f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Appended iteration log -> {iter_log_path.resolve()}")

==== Validation (deterministic) ====
CV: StratifiedGroupKFold(n_splits=5, group=patient_id, seed=42)
Fold sizes (train, val): [(800, 200), (800, 200), (800, 200), (800, 200), (800, 200)]
Macro-F1 per fold: [0.747793, 0.724106, 0.73067, 0.73325, 0.764429]
Macro-F1 mean: 0.74005
OOF-best threshold: 0.41000000000000003 OOF Macro-F1: 0.741266
Confusion matrix [[TN, FP],[FN, TP]]:
[[216 128]
 [101 555]]
Train positive rate: 0.656
OOF predicted positive rate: 0.683
ConvergenceWarnings captured during CV fits: 4

Classification report (OOF @ best_thr):
              precision    recall  f1-score   support

           0     0.6814    0.6279    0.6536       344
           1     0.8126    0.8460    0.8290       656

    accuracy                         0.7710      1000
   macro avg     0.7470    0.7370    0.7413      1000
weighted avg     0.7675    0.7710    0.7686      1000


Wrote submission -> D:\AgentDs\agent_ds_healthcare\clai_ch3_v1_submission.csv (rows=1000)

Checkpoint saved -> D:\AgentD

# Iteration 3
- 0.7417

In [21]:
import os, json, math, shutil, subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.base import clone

try:
    import joblib
except ImportError:
    from sklearn.externals import joblib

# -----------------------------
# Config (clai_ch3_v1)
# -----------------------------
VERSION = "v1"
TEAM_TAG = "clai"
TASK = "ch3"
SEED = 42
N_SPLITS = 5

TFIDF_SPECS = {
    "txt_all":   dict(max_features=800, ngram_range=(1,2), min_df=2, sublinear_tf=True),
    "txt_last3": dict(max_features=800, ngram_range=(1,2), min_df=1, sublinear_tf=True),
    "txt_day10": dict(max_features=400, ngram_range=(1,2), min_df=1, sublinear_tf=True),
}

MODEL_PARAMS = dict(
    solver="saga",
    penalty="l2",
    C=0.2,  # mild regularization to reduce overfit on expanded feature set
    class_weight="balanced",
    max_iter=20000,
    random_state=SEED,
    n_jobs=1,  # deterministic
)

THR_GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)

# -----------------------------
# Paths / directories
# -----------------------------
ROOT = Path(".").resolve()

SUBMISSION_PATH = ROOT / f"{TEAM_TAG}_{TASK}_{VERSION}_submission.csv"
ITER_LOG_PATH   = ROOT / f"{TEAM_TAG}_{TASK}_{VERSION}_iteration_detail.jsonl"

CKPT_ROOT = ROOT / "checkpoints" / f"{TEAM_TAG}_{TASK}_{VERSION}"
ART_ROOT  = ROOT / "artifacts"   / f"{TEAM_TAG}_{TASK}_{VERSION}"
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
ART_ROOT.mkdir(parents=True, exist_ok=True)

# Determine iteration_id early (append-only log)
iteration_id = 0
prev_record = None
if ITER_LOG_PATH.exists():
    try:
        last_good = None
        max_id = -1
        with open(ITER_LOG_PATH, "r", encoding="utf-8") as f:
            for ln in f:
                ln = ln.strip()
                if not ln:
                    continue
                try:
                    obj = json.loads(ln)
                    last_good = obj
                    max_id = max(max_id, int(obj.get("iteration_id", -1)))
                except Exception:
                    continue
        if max_id >= 0:
            iteration_id = max_id + 1
        prev_record = last_good
    except Exception:
        iteration_id = 0
        prev_record = None

ckpt_dir = CKPT_ROOT / f"iter{iteration_id:04d}"
ckpt_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Data dir auto-detect
# -----------------------------
DATA_DIR_CANDIDATES = [ROOT, Path("/mnt/data")]
def pick_data_dir():
    needed = ["stays_train.csv", "stays_test.csv", "patients.csv", "vitals_timeseries.json"]
    for d in DATA_DIR_CANDIDATES:
        if all((d / fn).exists() for fn in needed):
            return d
    raise FileNotFoundError(f"Could not find required files: {needed}. Tried: {DATA_DIR_CANDIDATES}")

DATA_DIR = pick_data_dir()

# -----------------------------
# Helpers
# -----------------------------
VITAL_COLS = ["hr", "sbp", "dbp", "temp_c", "rr"]

def safe_nan_stats(arr: np.ndarray):
    """Return robust stats even if all-nan."""
    arr = arr.astype(float)
    mask = ~np.isnan(arr)
    out = {"missing": int(np.isnan(arr).sum())}
    if mask.sum() == 0:
        out.update({"mean": np.nan, "std": np.nan, "min": np.nan, "max": np.nan,
                    "first": np.nan, "last": np.nan, "delta": np.nan, "slope": np.nan})
        return out

    vals = arr[mask]
    out["mean"]  = float(np.nanmean(arr))
    out["std"]   = float(np.nanstd(arr))
    out["min"]   = float(np.nanmin(arr))
    out["max"]   = float(np.nanmax(arr))
    out["first"] = float(vals[0])
    out["last"]  = float(vals[-1])
    out["delta"] = float(out["last"] - out["first"])

    # slope: linear regression vs time index using non-nan points
    if mask.sum() < 2:
        out["slope"] = np.nan
    else:
        x = np.flatnonzero(mask).astype(float)
        y = arr[mask].astype(float)
        x_mean = x.mean()
        y_mean = y.mean()
        denom = ((x - x_mean) ** 2).sum()
        out["slope"] = float(((x - x_mean) * (y - y_mean)).sum() / denom) if denom != 0 else 0.0
    return out

def extract_vitals_features(vitals_json_path: Path) -> pd.DataFrame:
    with open(vitals_json_path, "r", encoding="utf-8") as f:
        vitals_data = json.load(f)

    rows = []
    for rec in vitals_data:
        stay_id = rec["stay_id"]
        days = sorted(rec["days"], key=lambda d: d.get("day", 0))
        row = {"stay_id": stay_id}

        # Notes: all days, last3 days, day10 only
        notes_all, notes_last3 = [], []
        note_day10 = ""
        for d in days:
            note = d.get("note", "")
            note = "" if note is None else str(note)
            notes_all.append(note)
            if d.get("day") is not None and int(d.get("day")) >= 8:
                notes_last3.append(note)
            if d.get("day") is not None and int(d.get("day")) == 10:
                note_day10 = note
        row["note_all"] = " ".join(notes_all).strip()
        row["note_last3"] = " ".join(notes_last3).strip()
        row["note_day10"] = str(note_day10).strip()

        # Vitals numeric features: full 10d + last3d + day10 lag
        for v in VITAL_COLS:
            arr_full = np.array([d.get(v, np.nan) if d.get(v) is not None else np.nan for d in days], dtype=float)
            stats_full = safe_nan_stats(arr_full)
            for k, val in stats_full.items():
                row[f"{v}_{k}"] = val

            days_last3 = [d for d in days if d.get("day") is not None and int(d.get("day")) >= 8]
            arr_last3 = np.array([d.get(v, np.nan) if d.get(v) is not None else np.nan for d in days_last3], dtype=float)
            stats_last3 = safe_nan_stats(arr_last3)
            for k, val in stats_last3.items():
                row[f"{v}_{k}_last3"] = val

            d10_val = np.nan
            for d in days:
                if d.get("day") is not None and int(d.get("day")) == 10:
                    x = d.get(v, np.nan)
                    d10_val = np.nan if x is None else float(x)
                    break
            row[f"{v}_day10"] = d10_val

        rows.append(row)

    return pd.DataFrame(rows)

def threshold_sweep(y_true: np.ndarray, prob: np.ndarray, thr_grid: np.ndarray):
    best_thr, best_f1, best_cm = 0.5, -1.0, None
    for thr in thr_grid:
        pred = (prob >= thr).astype(int)
        f = f1_score(y_true, pred, average="macro")
        if f > best_f1:
            best_f1 = float(f)
            best_thr = float(thr)
            best_cm = confusion_matrix(y_true, pred).tolist()
    return best_thr, best_f1, best_cm

def get_git_hash():
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL).decode("utf-8").strip()
    except Exception:
        return None

# -----------------------------
# Load data
# -----------------------------
stays_train = pd.read_csv(DATA_DIR / "stays_train.csv")
stays_test  = pd.read_csv(DATA_DIR / "stays_test.csv")
patients    = pd.read_csv(DATA_DIR / "patients.csv")
vitals_feat = extract_vitals_features(DATA_DIR / "vitals_timeseries.json")

train = stays_train.merge(vitals_feat, on="stay_id", how="left").merge(patients, on="patient_id", how="left")
test  = stays_test.merge(vitals_feat, on="stay_id", how="left").merge(patients, on="patient_id", how="left")

# Hygiene + derived lengths
for df in (train, test):
    for c in ["note_all", "note_last3", "note_day10"]:
        df[c] = df[c].fillna("").astype(str)
    df["note_all_len"]   = df["note_all"].str.len()
    df["note_last3_len"] = df["note_last3"].str.len()
    df["note_day10_len"] = df["note_day10"].str.len()
    for c in ["unit_type","admission_reason","sex","insurance","zip3"]:
        df[c] = df[c].astype(str)

TARGET = "discharge_ready_day11"
y = train[TARGET].astype(int).copy()
X = train.drop(columns=[TARGET]).copy()

numeric_cols = [c for c in X.columns if any(c.startswith(v + "_") for v in VITAL_COLS)] + ["age", "note_all_len", "note_last3_len", "note_day10_len"]
cat_cols = ["unit_type", "admission_reason", "sex", "insurance", "zip3"]

# -----------------------------
# Pipeline: num + cat + multi-window TF-IDF -> LogisticRegression
# -----------------------------
preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), numeric_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
        ("txt_all",   TfidfVectorizer(**TFIDF_SPECS["txt_all"]),   "note_all"),
        ("txt_last3", TfidfVectorizer(**TFIDF_SPECS["txt_last3"]), "note_last3"),
        ("txt_day10", TfidfVectorizer(**TFIDF_SPECS["txt_day10"]), "note_day10"),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

clf = LogisticRegression(**MODEL_PARAMS)
pipe = Pipeline([("pre", preprocess), ("clf", clf)])

# -----------------------------
# Deterministic CV (OOF) + threshold tuning
# -----------------------------
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_prob = np.zeros(len(y), dtype=float)
fold_idx_list = []
fold_f1_at_05 = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), start=1):
    model = clone(pipe)
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    prob = model.predict_proba(X.iloc[va_idx])[:, 1]
    oof_prob[va_idx] = prob
    fold_idx_list.append(va_idx)

    pred_05 = (prob >= 0.5).astype(int)
    f1_05 = f1_score(y.iloc[va_idx], pred_05, average="macro")
    fold_f1_at_05.append(float(f1_05))
    print(f"Fold {fold}: macro-F1@0.5 = {f1_05:.4f}")

macro_f1_mean_at_05 = float(np.mean(fold_f1_at_05))

best_thr, macro_f1_oof_best, cm_best = threshold_sweep(y.values, oof_prob, THR_GRID)

fold_f1_at_best = []
for va_idx in fold_idx_list:
    pred_best = (oof_prob[va_idx] >= best_thr).astype(int)
    fold_f1_at_best.append(float(f1_score(y.iloc[va_idx], pred_best, average="macro")))

print("\n=== OOF Summary ===")
print(f"macro-F1@0.5 (mean over folds): {macro_f1_mean_at_05:.6f}")
print(f"Best threshold (grid): {best_thr:.2f}")
print(f"macro-F1 OOF best: {macro_f1_oof_best:.6f}")
print(f"macro-F1 per fold @best_thr: {[round(x,4) for x in fold_f1_at_best]}")
print(f"Confusion matrix @best_thr: {cm_best}")

# Save OOF artifacts (iteration-scoped)
oof_pred_path = ART_ROOT / f"iter{iteration_id:04d}_oof_predictions.csv"
oof_prob_path = ART_ROOT / f"iter{iteration_id:04d}_oof_prob.npy"
pd.DataFrame({"stay_id": train["stay_id"].values, "y_true": y.values, "oof_prob": oof_prob}).to_csv(oof_pred_path, index=False)
np.save(oof_prob_path, oof_prob)

# -----------------------------
# Fit final model on full train, predict test, write submission
# -----------------------------
final_model = clone(pipe)
final_model.fit(X, y)

test_prob = final_model.predict_proba(test)[:, 1]
test_pred = (test_prob >= best_thr).astype(int)

sub = pd.DataFrame({"stay_id": stays_test["stay_id"].values,
                    "discharge_ready_day11": test_pred.astype(int)})
sub.to_csv(SUBMISSION_PATH, index=False)

test_prob_path = ART_ROOT / f"iter{iteration_id:04d}_test_prob.npy"
np.save(test_prob_path, test_prob)

print(f"\nWrote submission: {SUBMISSION_PATH}")
print(f"Submission positive rate: {sub['discharge_ready_day11'].mean():.3f} | threshold={best_thr:.2f}")

# -----------------------------
# Checkpoint bundle + P* pointer
# -----------------------------
joblib.dump(final_model, ckpt_dir / "pipeline.joblib")

config = {
    "version": VERSION,
    "seed": SEED,
    "n_splits": N_SPLITS,
    "tfidf_specs": TFIDF_SPECS,
    "model_params": MODEL_PARAMS,
    "threshold_grid": THR_GRID.tolist(),
    "data_dir": str(DATA_DIR),
}

schema = {
    "numeric_cols": numeric_cols,
    "categorical_cols": cat_cols,
    "text_cols": ["note_all", "note_last3", "note_day10"],
}

metrics = {
    "macro_f1_mean_at_0.5": macro_f1_mean_at_05,
    "macro_f1_per_fold_at_0.5": fold_f1_at_05,
    "chosen_threshold": best_thr,
    "macro_f1_oof_best": macro_f1_oof_best,
    "macro_f1_per_fold_at_best_threshold": fold_f1_at_best,
    "confusion_matrix_oof_at_best_threshold": cm_best,
}

with open(ckpt_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
with open(ckpt_dir / "feature_schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=2, ensure_ascii=False)
with open(ckpt_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

# Copy submission into checkpoint for reproducibility
try:
    shutil.copy2(SUBMISSION_PATH, ckpt_dir / SUBMISSION_PATH.name)
except Exception:
    pass

# P* update logic
p_star_path = CKPT_ROOT / "p_star.json"
p_star = {"best_macro_f1_oof_best": None, "iteration_id": None, "checkpoint_dir": None}
if p_star_path.exists():
    try:
        p_star = json.load(open(p_star_path, "r", encoding="utf-8"))
    except Exception:
        pass

prev_best = p_star.get("best_macro_f1_oof_best")
is_new_pstar = (prev_best is None) or (macro_f1_oof_best > float(prev_best))

if is_new_pstar:
    p_star = {
        "best_macro_f1_oof_best": macro_f1_oof_best,
        "iteration_id": iteration_id,
        "checkpoint_dir": str(ckpt_dir),
        "updated_at": datetime.now(timezone.utc).isoformat(),
    }
    with open(p_star_path, "w", encoding="utf-8") as f:
        json.dump(p_star, f, indent=2, ensure_ascii=False)

# -----------------------------
# Iteration detail log (append-only)
# -----------------------------
hm_message = "HM reported leaderboard Macro-F1=0.7640 for latest submission; continue improving along this direction."
real_score = 0.7640

delta_note = "Initialized pipeline."
if prev_record is not None:
    delta_note = "Expanded vitals to (full10 + last3 + day10 lag) and used multi-window TF-IDF; threshold tuned on pooled OOF."

record = {
    "version": VERSION,
    "iteration_id": iteration_id,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "phase": "Modeling",
    "hm_message": hm_message,
    "real_score": real_score,
    "data_paths_used": {
        "stays_train": str(DATA_DIR / "stays_train.csv"),
        "stays_test": str(DATA_DIR / "stays_test.csv"),
        "patients": str(DATA_DIR / "patients.csv"),
        "vitals": str(DATA_DIR / "vitals_timeseries.json"),
    },
    "join_keys": {"patients": "patient_id", "vitals": "stay_id"},
    "join_notes": "Merged per-stay vitals features + per-patient demographics; confirmed no patient_id overlap between train and test.",
    "leakage_checks_performed": [
        "Features derived only from day1-10 vitals/notes (target is day11 readiness).",
        "No target-derived aggregates; per-stay feature extraction independent of labels.",
        "Train/test patient_id disjoint (prevents accidental memorization leakage).",
    ],
    "feature_summary": {
        "numeric_feature_count": int(len(numeric_cols)),
        "categorical_feature_count": int(len(cat_cols)),
        "text_feature_type": "TF-IDF (multi-window: all, last3, day10)",
        "tfidf_specs": TFIDF_SPECS,
        "vitals_aggregates": ["mean","std","min","max","first","last","delta","slope","missing"],
        "vitals_windows": ["full_10d", "last3d", "day10_lag"],
    },
    "models_attempted": [
        {
            "name": "LogisticRegression(saga) + ColumnTransformer(num+OHE+TFIDF_multiwindow)",
            "params": MODEL_PARAMS,
            "cv_macro_f1_mean_at_0.5": macro_f1_mean_at_05,
            "cv_macro_f1_per_fold_at_0.5": fold_f1_at_05,
            "cv_macro_f1_oof_at_best_threshold": macro_f1_oof_best,
            "cv_macro_f1_per_fold_at_best_threshold": fold_f1_at_best,
            "oof_best_threshold": best_thr,
            "notes": "Threshold tuned via grid search on pooled OOF probabilities (0.05..0.95 step 0.01).",
        }
    ],
    "selected_model": "LogisticRegression(saga) + num/cat + TF-IDF multi-window notes",
    "validation_protocol": {
        "splitter": "StratifiedKFold",
        "n_splits": N_SPLITS,
        "shuffle": True,
        "random_state": SEED,
        "threshold_selection": "Maximize macro-F1 on pooled out-of-fold probabilities.",
    },
    "metrics": metrics,
    "checkpoint": {
        "checkpoint_dir": str(ckpt_dir),
        "p_star_updated": bool(is_new_pstar),
        "p_star_path": str(p_star_path),
        "git_commit_hash": get_git_hash(),
        "submission_path": str(SUBMISSION_PATH),
        "oof_pred_path": str(oof_pred_path),
        "oof_prob_path": str(oof_prob_path),
        "test_prob_path": str(test_prob_path),
    },
    "deltas_vs_previous": delta_note,
    "known_defects_limitations": [
        "Threshold selected on pooled OOF may still be mildly optimistic; consider nested thresholding if needed.",
        "Linear model may miss non-linear interactions (but windows/lags partially mitigate).",
    ],
    "next_step_hypothesis": "Add day8/day9 vitals lags + mild elastic-net, and/or a small ensemble with a numeric-only tree model to push Macro-F1 toward ≥0.80.",
}

with open(ITER_LOG_PATH, "a", encoding="utf-8") as f:
    f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"\nCheckpoint saved to: {ckpt_dir}")
print(f"P* updated: {is_new_pstar} | best_macro_f1_oof_best={p_star.get('best_macro_f1_oof_best')}")
print(f"Iteration log appended: {ITER_LOG_PATH}")

Fold 1: macro-F1@0.5 = 0.7525
Fold 2: macro-F1@0.5 = 0.7673
Fold 3: macro-F1@0.5 = 0.7594
Fold 4: macro-F1@0.5 = 0.7333
Fold 5: macro-F1@0.5 = 0.7179

=== OOF Summary ===
macro-F1@0.5 (mean over folds): 0.746068
Best threshold (grid): 0.31
macro-F1 OOF best: 0.756702
macro-F1 per fold @best_thr: [0.8291, 0.7163, 0.7355, 0.7523, 0.7424]
Confusion matrix @best_thr: [[193, 151], [51, 605]]

Wrote submission: D:\AgentDs\agent_ds_healthcare\clai_ch3_v1_submission.csv
Submission positive rate: 0.770 | threshold=0.31

Checkpoint saved to: D:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v1\iter0004
P* updated: True | best_macro_f1_oof_best=0.7567015474745138
Iteration log appended: D:\AgentDs\agent_ds_healthcare\clai_ch3_v1_iteration_detail.jsonl


# Iteration 4
- 0.7773

In [24]:
# HOTFIX (does NOT increment iteration_id): make TF-IDF pipeline pickle-safe + add safe checkpoint dump guard
import os, re, glob, json, random, datetime
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone

import joblib

# -------------------------
# Config / deterministic
# -------------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

VERSION = "v1"
SUBMISSION_NAME = f"clai_ch3_{VERSION}_submission.csv"
LOG_NAME = f"clai_ch3_{VERSION}_iteration_detail.jsonl"

CKPT_ROOT = os.path.join("checkpoints", f"clai_ch3_{VERSION}")
ART_ROOT  = os.path.join("artifacts",  f"clai_ch3_{VERSION}")
os.makedirs(CKPT_ROOT, exist_ok=True)
os.makedirs(ART_ROOT,  exist_ok=True)

ERROR_SIGNATURE = "PicklingError: tfidf_pipe.<locals>.<lambda>"
FIX_SUMMARY = "Replace lambda-based text transformer with SimpleImputer+FunctionTransformer(np.ravel) in tfidf_pipe; add safe_dump fallback (joblib->cloudpickle) so checkpointing never crashes run."

# -------------------------
# Helpers
# -------------------------
def find_path(fname: str) -> str:
    candidates = [
        fname,
        os.path.join("data", fname),
        os.path.join("/mnt/data", fname),  # safe fallback for sandbox-style environments
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"Cannot find {fname}. Tried: {candidates}")

def detect_hotfix_iteration_id(log_path: str, ckpt_root: str, art_root: str) -> int:
    """
    Hotfix must NOT increment iteration_id.
    Best-effort detection:
      1) Prefer latest iterXXXX directory missing model_bundle.joblib (typical when joblib.dump crashed).
      2) Else fall back to max iteration_id found in JSONL.
      3) Else 0.
    """
    max_log = -1
    if os.path.exists(log_path):
        try:
            with open(log_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        obj = json.loads(line)
                        if isinstance(obj, dict) and "iteration_id" in obj:
                            max_log = max(max_log, int(obj["iteration_id"]))
                    except Exception:
                        # ignore malformed lines; keep robust
                        continue
        except Exception:
            pass

    # Look for iterXXXX dirs (ckpt/artifacts) and prefer one that looks "incomplete"
    iter_ids = set()
    for root in [ckpt_root, art_root]:
        if os.path.isdir(root):
            for d in glob.glob(os.path.join(root, "iter[0-9][0-9][0-9][0-9]")):
                if os.path.isdir(d):
                    m = re.search(r"iter(\d{4})$", d.replace("\\", "/"))
                    if m:
                        iter_ids.add(int(m.group(1)))

    for cid in sorted(iter_ids, reverse=True):
        ckpt_dir = os.path.join(ckpt_root, f"iter{cid:04d}")
        if os.path.isdir(ckpt_dir):
            # if model bundle is missing, that's likely the failed iteration we are hotfixing
            if not os.path.exists(os.path.join(ckpt_dir, "model_bundle.joblib")):
                return cid

    # Fallback: do not increment, use last logged id if any
    return max_log if max_log >= 0 else 0

def to_jsonable(x):
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, (np.ndarray,)):
        return x.tolist()
    if isinstance(x, (pd.Series,)):
        return x.tolist()
    if isinstance(x, (pd.DataFrame,)):
        return x.to_dict(orient="list")
    return x

def safe_dump(bundle: dict, joblib_path: str):
    """
    Try joblib.dump first; if it fails, fall back to cloudpickle.
    Always returns (backend, error_str_or_None, actual_path_written_or_None).
    """
    try:
        joblib.dump(bundle, joblib_path)
        return "joblib", None, joblib_path
    except Exception as e1:
        err1 = repr(e1)
        try:
            import cloudpickle
            cloud_path = joblib_path.replace(".joblib", ".cloudpickle")
            with open(cloud_path, "wb") as f:
                cloudpickle.dump(bundle, f)
            return "cloudpickle", err1, cloud_path
        except Exception as e2:
            return "failed", f"{err1} | cloudpickle_error={repr(e2)}", None

# -------------------------
# Load data
# -------------------------
stays_train_path = find_path("stays_train.csv")
stays_test_path  = find_path("stays_test.csv")
patients_path    = find_path("patients.csv")
vitals_path      = find_path("vitals_timeseries.json")

train = pd.read_csv(stays_train_path)
test  = pd.read_csv(stays_test_path)
patients = pd.read_csv(patients_path)

# Joins (no leakage: only demographics)
train = train.merge(patients, on="patient_id", how="left")
test  = test.merge(patients, on="patient_id", how="left")

# -------------------------
# Build vitals + note features
# -------------------------
with open(vitals_path, "r", encoding="utf-8") as f:
    vitals_list = json.load(f)

VITAL_COLS = ["hr", "sbp", "dbp", "temp_c", "rr"]
RECIPE = ["mean","std","min","max","last","missing","slope","last3_mean","last3_std","delta_last_first"]

rows = []
for rec in vitals_list:
    sid = rec.get("stay_id")
    days = rec.get("days", [])
    df = pd.DataFrame(days)
    feats = {"stay_id": sid}

    if df.empty:
        feats["note_all"] = ""
        for v in VITAL_COLS:
            for stat in RECIPE:
                feats[f"{v}_{stat}"] = np.nan
        rows.append(feats)
        continue

    df = df.sort_values("day")
    # text: use aggregated note across days (keep modeling direction: TF-IDF over notes)
    if "note" in df.columns:
        notes = df["note"].fillna("").astype(str).tolist()
        feats["note_all"] = " ".join([n.strip() for n in notes if isinstance(n, str) and n.strip()])
    else:
        feats["note_all"] = ""

    day_vals = pd.to_numeric(df["day"], errors="coerce").to_numpy(dtype=float) if "day" in df.columns else np.arange(1, len(df) + 1, dtype=float)

    for v in VITAL_COLS:
        arr = pd.to_numeric(df[v], errors="coerce").to_numpy(dtype=float) if v in df.columns else np.full(len(df), np.nan, dtype=float)

        feats[f"{v}_mean"] = float(np.nanmean(arr)) if np.isfinite(np.nanmean(arr)) else np.nan
        feats[f"{v}_std"]  = float(np.nanstd(arr))  if np.isfinite(np.nanstd(arr))  else np.nan
        feats[f"{v}_min"]  = float(np.nanmin(arr))  if np.isfinite(np.nanmin(arr))  else np.nan
        feats[f"{v}_max"]  = float(np.nanmax(arr))  if np.isfinite(np.nanmax(arr))  else np.nan

        last_val = arr[-1] if len(arr) else np.nan
        if not np.isfinite(last_val):
            valid = arr[np.isfinite(arr)]
            last_val = valid[-1] if len(valid) else np.nan
        feats[f"{v}_last"] = float(last_val) if np.isfinite(last_val) else np.nan

        feats[f"{v}_missing"] = int(np.sum(~np.isfinite(arr)))

        m = np.isfinite(arr) & np.isfinite(day_vals)
        feats[f"{v}_slope"] = float(np.polyfit(day_vals[m], arr[m], 1)[0]) if np.sum(m) >= 2 else np.nan

        last3 = arr[-3:] if len(arr) >= 3 else arr
        feats[f"{v}_last3_mean"] = float(np.nanmean(last3)) if np.isfinite(np.nanmean(last3)) else np.nan
        feats[f"{v}_last3_std"]  = float(np.nanstd(last3))  if np.isfinite(np.nanstd(last3))  else np.nan

        valid = arr[np.isfinite(arr)]
        feats[f"{v}_delta_last_first"] = float(valid[-1] - valid[0]) if len(valid) >= 2 else np.nan

    rows.append(feats)

vitals_feat = pd.DataFrame(rows)

train = train.merge(vitals_feat, on="stay_id", how="left")
test  = test.merge(vitals_feat, on="stay_id", how="left")

# ensure text column exists
if "note_all" not in train.columns:
    train["note_all"] = ""
    test["note_all"] = ""

# treat zip3 as categorical (string) if present
for df in (train, test):
    if "zip3" in df.columns:
        df["zip3"] = df["zip3"].astype("Int64").astype(str).replace("<NA>", "missing")

# -------------------------
# HOTFIX iteration id detection (DO NOT increment)
# -------------------------
hotfix_iter_id = detect_hotfix_iteration_id(LOG_NAME, CKPT_ROOT, ART_ROOT)
CKPT_DIR = os.path.join(CKPT_ROOT, f"iter{hotfix_iter_id:04d}")
ART_DIR  = os.path.join(ART_ROOT,  f"iter{hotfix_iter_id:04d}")
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(ART_DIR,  exist_ok=True)

# -------------------------
# Build model pipelines (NO lambdas in transformers)
# -------------------------
target = "discharge_ready_day11"
y = train[target].astype(int).values
X_train = train.drop(columns=[target]).copy()
X_test  = test.copy()

# define columns
id_cols = ["stay_id", "patient_id"]
text_cols = ["note_all"]
cat_cols = [c for c in ["unit_type","admission_reason","sex","insurance","zip3"] if c in X_train.columns]
num_cols = [c for c in X_train.columns if c not in (id_cols + text_cols + cat_cols)]

# coerce numeric
for c in num_cols:
    X_train[c] = pd.to_numeric(X_train[c], errors="coerce")
    X_test[c]  = pd.to_numeric(X_test[c],  errors="coerce")

def tfidf_pipe(max_features=6000, ngram_range=(1,2), min_df=2):
    """
    HOTFIX: pickle-safe text pipeline:
      - no lambda
      - uses built-in np.ravel
      - imputes missing to ""
    """
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="")),
        ("flatten", FunctionTransformer(np.ravel, validate=False)),
        ("tfidf", TfidfVectorizer(max_features=max_features, ngram_range=ngram_range, min_df=min_df)),
    ])

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler(with_mean=False)),
        ]), num_cols),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
        ("txt", tfidf_pipe(), text_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

pipe_logi = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", LogisticRegression(
        C=4.0,
        solver="liblinear",
        class_weight="balanced",
        max_iter=2000,
        random_state=SEED,
    ))
])

# Keep modeling direction: include a second "stronger" model if xgboost available; otherwise fast sparse-friendly fallback.
xgb_backend = "xgboost"
try:
    from xgboost import XGBClassifier
    xgb_clf = XGBClassifier(
        n_estimators=200,
        learning_rate=0.08,
        max_depth=3,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        min_child_weight=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=SEED,
        n_jobs=-1,
    )
except Exception as e:
    # fallback that still supports sparse matrices
    from sklearn.linear_model import SGDClassifier
    xgb_backend = f"SGDClassifier_fallback({repr(e)})"
    xgb_clf = SGDClassifier(
        loss="log_loss",
        alpha=1e-4,
        penalty="l2",
        max_iter=2000,
        random_state=SEED,
    )

pipe_xgb = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", xgb_clf)
])

# -------------------------
# Deterministic validation (OOF) + threshold
# -------------------------
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_logi = np.zeros(len(X_train), dtype=float)
oof_xgb  = np.zeros(len(X_train), dtype=float)
fold_f1s = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y), start=1):
    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    m1 = clone(pipe_logi)
    m2 = clone(pipe_xgb)

    m1.fit(X_tr, y_tr)
    m2.fit(X_tr, y_tr)

    p1 = m1.predict_proba(X_va)[:, 1]
    # SGDClassifier has predict_proba for log_loss; xgboost does as well
    p2 = m2.predict_proba(X_va)[:, 1]

    oof_logi[va_idx] = p1
    oof_xgb[va_idx]  = p2

    ens = (p1 + p2) / 2.0
    pred_05 = (ens >= 0.5).astype(int)
    fold_f1s.append(float(f1_score(y_va, pred_05, average="macro")))

oof_ens = (oof_logi + oof_xgb) / 2.0

# threshold search on OOF
ths = np.linspace(0.05, 0.95, 181)
best_th, best_f1 = 0.5, -1.0
for th in ths:
    pred = (oof_ens >= th).astype(int)
    f1 = float(f1_score(y, pred, average="macro"))
    if f1 > best_f1:
        best_f1, best_th = f1, float(th)

cm = confusion_matrix(y, (oof_ens >= best_th).astype(int)).tolist()
macro_f1_mean = float(np.mean(fold_f1s))

print("=== HOTFIX RUN (no new iteration id) ===")
print(f"hotfix_iter_id: {hotfix_iter_id:04d}")
print(f"OOF macro-F1 mean (threshold=0.5, per-fold): {macro_f1_mean:.4f} | folds={fold_f1s}")
print(f"Best OOF threshold: {best_th:.3f} | Best OOF macro-F1: {best_f1:.4f}")
print(f"Confusion matrix @ best_th: {cm}")
print(f"xgb_backend: {xgb_backend}")

# -------------------------
# Fit full + predict test + write submission
# -------------------------
pipe_logi.fit(X_train, y)
pipe_xgb.fit(X_train, y)

test_p1 = pipe_logi.predict_proba(X_test)[:, 1]
test_p2 = pipe_xgb.predict_proba(X_test)[:, 1]
test_prob = (test_p1 + test_p2) / 2.0
test_pred = (test_prob >= best_th).astype(int)

submission = pd.DataFrame({
    "stay_id": X_test["stay_id"].astype(int),
    "discharge_ready_day11": test_pred.astype(int),
})
submission.to_csv(SUBMISSION_NAME, index=False)
print(f"Wrote submission: {SUBMISSION_NAME} | shape={submission.shape}")

# -------------------------
# Save artifacts
# -------------------------
np.save(os.path.join(ART_DIR, f"iter{hotfix_iter_id:04d}_oof_prob.npy"), oof_ens)
pd.DataFrame({
    "stay_id": X_train["stay_id"].astype(int),
    "y_true": y.astype(int),
    "oof_prob": oof_ens.astype(float),
    "oof_pred": (oof_ens >= best_th).astype(int),
}).to_csv(os.path.join(ART_DIR, f"iter{hotfix_iter_id:04d}_oof_predictions.csv"), index=False)

# -------------------------
# Checkpoint bundle (safe dump)
# -------------------------
config = {
    "version": VERSION,
    "seed": SEED,
    "n_splits": N_SPLITS,
    "text_cols": text_cols,
    "cat_cols": cat_cols,
    "num_cols": num_cols,
    "tfidf": {"max_features": 6000, "ngram_range": [1,2], "min_df": 2},
    "logi": {"C": 4.0, "solver": "liblinear", "class_weight": "balanced", "max_iter": 2000},
    "xgb_backend": xgb_backend,
    "threshold_search": {"min": 0.05, "max": 0.95, "steps": 181},
}

metrics = {
    "macro_f1_mean": macro_f1_mean,
    "macro_f1_per_fold": fold_f1s,
    "best_oof_macro_f1": best_f1,
    "chosen_threshold": best_th,
    "confusion_matrix": cm,
}

schema = {
    "categorical_cols": cat_cols,
    "numeric_cols": num_cols,
    "text_cols": text_cols,
    "num_feature_recipe": RECIPE,
    "vital_signals": VITAL_COLS,
}

bundle = {
    "pipe_logi": pipe_logi,
    "pipe_xgb": pipe_xgb,
    "config": config,
    "metrics": metrics,
    "schema": schema,
}

model_bundle_path = os.path.join(CKPT_DIR, "model_bundle.joblib")
dump_backend, dump_err, written_path = safe_dump(bundle, model_bundle_path)

with open(os.path.join(CKPT_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
with open(os.path.join(CKPT_DIR, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
with open(os.path.join(CKPT_DIR, "schema.json"), "w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)

print(f"Checkpoint dump backend: {dump_backend} | path={written_path} | err={dump_err}")

# -------------------------
# Append HOTFIX log entry (must NOT increment iteration_id)
# -------------------------
timestamp = datetime.datetime.now(datetime.timezone.utc).isoformat()
log_entry = {
    "version": VERSION,
    "iteration_id": hotfix_iter_id,              # DO NOT increment
    "timestamp": timestamp,
    "phase": "Hotfix",
    "hotfix": True,
    "hotfix_for_iteration_id": hotfix_iter_id,
    "error_signature": ERROR_SIGNATURE,
    "fix_summary": FIX_SUMMARY,
    "data_paths_used": {
        "stays_train": stays_train_path,
        "stays_test": stays_test_path,
        "patients": patients_path,
        "vitals_timeseries": vitals_path,
    },
    "leakage_checks_performed": [
        "No joins to labels outside stays_train; only patient demographics + vitals/day notes by stay_id.",
        "Text pipeline uses only within-stay daily notes (days 1-10); target is day11 readiness.",
    ],
    "feature_summary": {
        "n_train": int(len(train)),
        "n_test": int(len(test)),
        "num_cols": int(len(num_cols)),
        "cat_cols": int(len(cat_cols)),
        "text_cols": int(len(text_cols)),
        "vital_recipe_len": int(len(RECIPE)),
    },
    "models_attempted": [
        {"name": "logistic_regression", "params": config["logi"]},
        {"name": "xgb_or_fallback", "params": {"backend": xgb_backend}},
    ],
    "selected_model": "avg_ensemble(pipe_logi, pipe_xgb)",
    "validation_protocol": {"type": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "metrics": metrics,
    "checkpoint": {
        "ckpt_dir": CKPT_DIR,
        "dump_backend": dump_backend,
        "dump_error": dump_err,
        "model_bundle_written": bool(written_path),
    },
}

with open(LOG_NAME, "a", encoding="utf-8") as f:
    f.write(json.dumps({k: to_jsonable(v) for k, v in log_entry.items()}, ensure_ascii=False) + "\n")

print(f"Appended HOTFIX entry to: {LOG_NAME} (iteration_id stays {hotfix_iter_id})")

=== HOTFIX RUN (no new iteration id) ===
hotfix_iter_id: 0004
OOF macro-F1 mean (threshold=0.5, per-fold): 0.7756 | folds=[0.8131868131868132, 0.7328800388538126, 0.7566909975669099, 0.7867803837953091, 0.7883234646299951]
Best OOF threshold: 0.470 | Best OOF macro-F1: 0.7841
Confusion matrix @ best_th: [[221, 123], [63, 593]]
xgb_backend: xgboost
Wrote submission: clai_ch3_v1_submission.csv | shape=(1000, 2)
Checkpoint dump backend: joblib | path=checkpoints\clai_ch3_v1\iter0004\model_bundle.joblib | err=None
Appended HOTFIX entry to: clai_ch3_v1_iteration_detail.jsonl (iteration_id stays 4)


# Iteration 5
- 0.7842

In [26]:
import os, json, random, joblib
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.base import clone
from xgboost import XGBClassifier

# ======================
# Config
# ======================
VX = "v1"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = Path(".").resolve()

def find_data_dir():
    candidates = [BASE_DIR, BASE_DIR / "data", Path("/mnt/data")]
    for c in candidates:
        if (c / "stays_train.csv").exists() and (c / "stays_test.csv").exists() and (c / "patients.csv").exists() and (c / "vitals_timeseries.json").exists():
            return c
    raise FileNotFoundError("Could not find required data files: stays_train.csv, stays_test.csv, patients.csv, vitals_timeseries.json")

DATA_DIR = find_data_dir()

submission_path = BASE_DIR / f"clai_ch3_{VX}_submission.csv"
log_path = BASE_DIR / f"clai_ch3_{VX}_iteration_detail.jsonl"
ckpt_root = BASE_DIR / "checkpoints" / f"clai_ch3_{VX}"
art_root = BASE_DIR / "artifacts" / f"clai_ch3_{VX}"
ckpt_root.mkdir(parents=True, exist_ok=True)
art_root.mkdir(parents=True, exist_ok=True)

def next_iteration_id(p: Path) -> int:
    if not p.exists():
        return 0
    mx = -1
    with p.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                mx = max(mx, int(obj.get("iteration_id", -1)))
            except Exception:
                continue
    return mx + 1

iteration_id = next_iteration_id(log_path)
timestamp = datetime.now().isoformat(timespec="seconds")

# ======================
# Helpers
# ======================
def safe_nan_stat(arr, fn, default=np.nan):
    arr = np.asarray(arr, dtype=float)
    if np.isfinite(arr).any():
        return float(fn(arr))
    return default

def safe_slope(day_idx, vals):
    x = np.asarray(day_idx, dtype=float)
    y = np.asarray(vals, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() >= 2:
        return float(np.polyfit(x[m], y[m], 1)[0])
    return np.nan

def best_threshold(y_true, prob):
    best_f1 = -1.0
    best_t = 0.5
    for t in np.linspace(0.05, 0.95, 181):
        f1 = f1_score(y_true, (prob >= t).astype(int), average="macro")
        if f1 > best_f1:
            best_f1 = float(f1)
            best_t = float(t)
    return best_f1, best_t

# ======================
# Load data
# ======================
st_train = pd.read_csv(DATA_DIR / "stays_train.csv")
st_test  = pd.read_csv(DATA_DIR / "stays_test.csv")
patients = pd.read_csv(DATA_DIR / "patients.csv")
with open(DATA_DIR / "vitals_timeseries.json", "r", encoding="utf-8") as f:
    vitals = json.load(f)

# ======================
# Build vitals + note features per stay_id
# ======================
vitals_cols = ["hr", "sbp", "dbp", "temp_c", "rr"]
num_recipe = ["mean","std","min","max","last","missing","slope","last3_mean","last3_std","delta_last_first"]

rows = []
day_vals = []
for rec in vitals:
    sid = rec.get("stay_id")
    days = sorted(rec.get("days", []), key=lambda d: d.get("day", 0))
    day_idx = [d.get("day", np.nan) for d in days]
    day_vals.extend([d.get("day", None) for d in days])

    feat = {"stay_id": sid}

    for col in vitals_cols:
        arr = np.array([np.nan if d.get(col) is None else float(d.get(col)) for d in days], dtype=float)
        feat[f"{col}_mean"] = safe_nan_stat(arr, np.nanmean)
        feat[f"{col}_std"]  = safe_nan_stat(arr, np.nanstd)
        feat[f"{col}_min"]  = safe_nan_stat(arr, np.nanmin)
        feat[f"{col}_max"]  = safe_nan_stat(arr, np.nanmax)
        feat[f"{col}_last"] = float(arr[-1]) if len(arr) else np.nan
        feat[f"{col}_missing"] = int(np.isnan(arr).sum())
        feat[f"{col}_slope"] = safe_slope(day_idx, arr)
        last3 = arr[-3:] if len(arr) >= 3 else arr
        feat[f"{col}_last3_mean"] = safe_nan_stat(last3, np.nanmean)
        feat[f"{col}_last3_std"]  = safe_nan_stat(last3, np.nanstd)
        if len(arr) and np.isfinite(arr[0]) and np.isfinite(arr[-1]):
            feat[f"{col}_delta_last_first"] = float(arr[-1] - arr[0])
        else:
            feat[f"{col}_delta_last_first"] = np.nan

    notes = [("" if d.get("note") is None else str(d.get("note"))) for d in days]
    feat["note_all"]   = " ".join(notes).strip()
    feat["note_last3"] = " ".join(notes[-3:]).strip()
    feat["note_day10"] = (notes[-1] if len(notes) else "").strip()
    rows.append(feat)

v_feat = pd.DataFrame(rows)

train_df = st_train.merge(patients, on="patient_id", how="left").merge(v_feat, on="stay_id", how="left")
test_df  = st_test.merge(patients, on="patient_id", how="left").merge(v_feat, on="stay_id", how="left")

# Normalize types
for c in ["unit_type", "admission_reason", "sex", "insurance", "zip3"]:
    train_df[c] = train_df[c].astype(str).fillna("UNK")
    test_df[c]  = test_df[c].astype(str).fillna("UNK")
for c in ["note_all", "note_last3", "note_day10"]:
    train_df[c] = train_df[c].fillna("").astype(str)
    test_df[c]  = test_df[c].fillna("").astype(str)

# Leakage-ish sanity (logged)
day_vals_clean = [d for d in day_vals if d is not None]
day_min = float(np.nanmin(day_vals_clean)) if len(day_vals_clean) else np.nan
day_max = float(np.nanmax(day_vals_clean)) if len(day_vals_clean) else np.nan

# ======================
# Feature columns
# ======================
target_col = "discharge_ready_day11"
cat_cols  = ["unit_type", "admission_reason", "sex", "insurance", "zip3"]
text_cols = ["note_all", "note_last3", "note_day10"]
num_cols  = ["age"] + [c for c in train_df.columns if any(c.startswith(v + "_") for v in vitals_cols)]
num_cols  = [c for c in num_cols if c not in text_cols + [target_col, "stay_id", "patient_id"]]

X = train_df.drop(columns=[target_col])
y = train_df[target_col].astype(int).to_numpy()
groups = train_df["patient_id"].to_numpy()
X_test = test_df.copy()

# ======================
# Preprocess (no lambdas / no closures persisted)
# ======================
tfidf_all   = dict(max_features=6000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
tfidf_last3 = dict(max_features=3000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
tfidf_day10 = dict(max_features=2000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler(with_mean=False))
        ]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("tfidf_all",   TfidfVectorizer(**tfidf_all),   "note_all"),
        ("tfidf_last3", TfidfVectorizer(**tfidf_last3), "note_last3"),
        ("tfidf_day10", TfidfVectorizer(**tfidf_day10), "note_day10"),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

# ======================
# Models
# ======================
logi_params = dict(
    solver="saga",
    max_iter=3000,
    class_weight="balanced",
    n_jobs=1,
    C=3.0,
    random_state=SEED,
)

xgb_params = dict(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.0,
    min_child_weight=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=SEED,
    n_jobs=1,
)

base_logi = Pipeline([("preprocess", preprocess), ("clf", LogisticRegression(**logi_params))])
base_xgb  = Pipeline([("preprocess", preprocess), ("clf", XGBClassifier(**xgb_params))])

# ======================
# CV OOF (deterministic)
# ======================
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_logi = np.zeros(len(train_df), dtype=float)
oof_xgb  = np.zeros(len(train_df), dtype=float)

for fold, (tr, va) in enumerate(cv.split(X, y, groups)):
    m1 = clone(base_logi)
    m2 = clone(base_xgb)
    m1.fit(X.iloc[tr], y[tr])
    m2.fit(X.iloc[tr], y[tr])
    oof_logi[va] = m1.predict_proba(X.iloc[va])[:, 1]
    oof_xgb[va]  = m2.predict_proba(X.iloc[va])[:, 1]
    print(f"fold {fold}: done")

oof_ens = (oof_logi + oof_xgb) / 2.0

f1_logi, t_logi = best_threshold(y, oof_logi)
f1_xgb,  t_xgb  = best_threshold(y, oof_xgb)
f1_ens,  t_ens  = best_threshold(y, oof_ens)

fold_f1s = []
for fold, (tr, va) in enumerate(cv.split(X, y, groups)):
    fold_f1s.append(float(f1_score(y[va], (oof_ens[va] >= t_ens).astype(int), average="macro")))
cm = confusion_matrix(y, (oof_ens >= t_ens).astype(int)).tolist()

print("\n=== CV Summary (OOF) ===")
print("Logi best Macro-F1:", f1_logi, "thr:", t_logi)
print("XGB  best Macro-F1:", f1_xgb,  "thr:", t_xgb)
print("ENS  best Macro-F1:", f1_ens,  "thr:", t_ens)
print("ENS per-fold:", fold_f1s)
print("ENS confusion_matrix:", cm)

# ======================
# Fit full + predict test
# ======================
final_logi = clone(base_logi).fit(X, y)
final_xgb  = clone(base_xgb).fit(X, y)

test_prob = (final_logi.predict_proba(X_test)[:, 1] + final_xgb.predict_proba(X_test)[:, 1]) / 2.0
test_pred = (test_prob >= t_ens).astype(int)

sub = pd.DataFrame({"stay_id": test_df["stay_id"].astype(int), "discharge_ready_day11": test_pred.astype(int)})
sub.to_csv(submission_path, index=False)
print("\nWrote submission:", submission_path)

# ======================
# Save artifacts + checkpoint
# ======================
iter_tag = f"iter{iteration_id:04d}"

# match prior naming style for easy comparison
np.save(art_root / f"{iter_tag}_oof_prob.npy", oof_ens)
pd.DataFrame({
    "stay_id": train_df["stay_id"].astype(int),
    "y_true": y.astype(int),
    "oof_prob_logi": oof_logi,
    "oof_prob_xgb": oof_xgb,
    "oof_prob": oof_ens,
    "oof_pred": (oof_ens >= t_ens).astype(int),
}).to_csv(art_root / f"{iter_tag}_oof_predictions.csv", index=False)

ckpt_dir = ckpt_root / iter_tag
ckpt_dir.mkdir(parents=True, exist_ok=True)

config = {
    "seed": SEED,
    "cat_cols": cat_cols,
    "num_cols": num_cols,
    "text_cols": text_cols,
    "tfidf_params": {"note_all": tfidf_all, "note_last3": tfidf_last3, "note_day10": tfidf_day10},
    "logi_params": logi_params,
    "xgb_params": xgb_params,
    "ensemble": {"type": "mean", "weights": {"logi": 0.5, "xgb": 0.5}},
}
metrics = {
    "macro_f1_mean": f1_ens,
    "macro_f1_per_fold": fold_f1s,
    "threshold": t_ens,
    "confusion_matrix": cm,
    "component_models": {
        "logi_best_macro_f1": f1_logi, "logi_thr": t_logi,
        "xgb_best_macro_f1": f1_xgb, "xgb_thr": t_xgb
    },
}

# P* update (prefer p_star.json; fallback to scanning log if missing)
p_star_path = ckpt_root / "p_star.json"
prev_best, prev_iter = -1.0, None
if p_star_path.exists():
    try:
        pstar = json.loads(p_star_path.read_text(encoding="utf-8"))
        prev_best = float(pstar.get("best_macro_f1", -1.0))
        prev_iter = pstar.get("iteration_id", None)
    except Exception:
        prev_best, prev_iter = -1.0, None
elif log_path.exists():
    try:
        bests = []
        with log_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                obj = json.loads(line)
                m = obj.get("metrics", {})
                if isinstance(m, dict) and m.get("macro_f1_mean") is not None:
                    bests.append(float(m["macro_f1_mean"]))
        if bests:
            prev_best = max(bests)
    except Exception:
        prev_best, prev_iter = -1.0, None

p_star_update = f1_ens > prev_best
if p_star_update:
    p_star_path.write_text(json.dumps({
        "best_macro_f1": f1_ens,
        "iteration_id": iteration_id,
        "checkpoint_dir": str(ckpt_dir)
    }, indent=2), encoding="utf-8")

bundle = {"pipe_logi": final_logi, "pipe_xgb": final_xgb, "config": config, "metrics": metrics}
joblib.dump(bundle, ckpt_dir / "model_bundle.joblib")
(ckpt_dir / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
(ckpt_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
schema = {"categorical_cols": cat_cols, "numeric_cols": num_cols, "text_cols": text_cols, "num_feature_recipe": num_recipe}
(ckpt_dir / "feature_schema.json").write_text(json.dumps(schema, indent=2), encoding="utf-8")

print("Saved checkpoint:", ckpt_dir, "| P* updated:", p_star_update, "| prev_best:", prev_best, "| prev_iter:", prev_iter)

# ======================
# Append iteration detail log (append-only)
# ======================
log_entry = {
    "version": VX,
    "iteration_id": iteration_id,
    "timestamp": timestamp,
    "phase": "Modeling",
    "data_paths_used": {
        "stays_train": str(DATA_DIR / "stays_train.csv"),
        "stays_test": str(DATA_DIR / "stays_test.csv"),
        "patients": str(DATA_DIR / "patients.csv"),
        "vitals_timeseries": str(DATA_DIR / "vitals_timeseries.json"),
    },
    "join_keys": {"patients": "patient_id", "vitals_timeseries": "stay_id"},
    "join_notes": "train/test stays joined to patients by patient_id; vitals aggregated to stay-level by stay_id; patient_id used ONLY for grouping in CV (not as feature).",
    "leakage_checks_performed": [
        f"vitals day range observed: min_day={day_min}, max_day={day_max} (expect 1..10)",
        "drop target column from X before modeling",
        "StratifiedGroupKFold with group=patient_id to prevent patient leakage across folds",
    ],
    "feature_summary": {
        "categorical_cols": cat_cols,
        "numeric_cols_count": len(num_cols),
        "text_cols": text_cols,
        "tfidf_max_features_total": int(tfidf_all["max_features"] + tfidf_last3["max_features"] + tfidf_day10["max_features"]),
        "num_feature_recipe": num_recipe,
        "n_train": int(train_df.shape[0]),
        "n_test": int(test_df.shape[0]),
        "pos_rate_train": float(np.mean(y)),
    },
    "models_attempted": [
        {"name": "LogisticRegression(saga) + TFIDF(all,last3,day10) + vitals trend + onehot cats",
         "params": logi_params,
         "cv_macro_f1_oof_best": f1_logi,
         "threshold_oof_best": t_logi,
         "failure_notes": None},
        {"name": "XGBClassifier(hist) + same preprocess (sparse)",
         "params": xgb_params,
         "cv_macro_f1_oof_best": f1_xgb,
         "threshold_oof_best": t_xgb,
         "failure_notes": None},
        {"name": "EnsembleMean(0.5*logi + 0.5*xgb)",
         "params": {"weights": {"logi": 0.5, "xgb": 0.5}},
         "cv_macro_f1_oof_best": f1_ens,
         "threshold_oof_best": t_ens,
         "failure_notes": None},
    ],
    "selected_model": {
        "name": "EnsembleMean(LogisticRegression + XGBClassifier)",
        "ensemble_rule": "mean",
        "weights": {"logi": 0.5, "xgb": 0.5},
        "threshold": t_ens,
    },
    "validation_protocol": {
        "cv": "StratifiedGroupKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": SEED,
        "group_col": "patient_id",
        "threshold_selection": "global_oof_max_macro_f1",
    },
    "metrics": metrics,
    "checkpoint": {
        "checkpoint_dir": str(ckpt_dir),
        "p_star_updated": bool(p_star_update),
        "p_star_prev_best": float(prev_best),
        "bundle_file": str(ckpt_dir / "model_bundle.joblib"),
        "submission_file": str(submission_path),
        "artifacts_oof_prob": str(art_root / f"{iter_tag}_oof_prob.npy"),
        "artifacts_oof_pred": str(art_root / f"{iter_tag}_oof_predictions.csv"),
    },
    "deltas_vs_previous": "Add XGBClassifier + mean-ensemble with LogisticRegression; add TFIDF on day10 note while keeping prior vitals trend + demographics + cat onehot setup; keep group-CV and global OOF threshold.",
    "known_defects_limitations": [
        "Threshold tuned on OOF may still be mildly optimistic; monitor real-score gap.",
        "XGB on sparse TFIDF can overfit; consider stronger regularization or early stopping if real score lags.",
    ],
    "next_step_hypothesis": "Tune ensemble weight and XGB regularization (max_depth/min_child_weight/reg_lambda) or introduce early stopping; consider median-of-fold thresholds to reduce optimism if needed.",
    "hm_message": "HM reported latest real Macro-F1=0.7773 (best so far). Continue improving from that base; aim to break 0.80 real Macro-F1.",
    "real_score": 0.7773,
}

with log_path.open("a", encoding="utf-8") as f:
    f.write(json.dumps(log_entry, ensure_ascii=False) + "\n")

print("Appended iteration log:", log_path)

fold 0: done
fold 1: done
fold 2: done
fold 3: done
fold 4: done

=== CV Summary (OOF) ===
Logi best Macro-F1: 0.7491524138784493 thr: 0.43999999999999995
XGB  best Macro-F1: 0.80649957735434 thr: 0.6699999999999999
ENS  best Macro-F1: 0.8069801369094378 thr: 0.5499999999999999
ENS per-fold: [0.7747747747747749, 0.8262078554049357, 0.7925537722458784, 0.8190863862505653, 0.8230923457954947]
ENS confusion_matrix: [[249, 95], [77, 579]]

Wrote submission: D:\AgentDs\agent_ds_healthcare\clai_ch3_v1_submission.csv
Saved checkpoint: D:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v1\iter0005 | P* updated: True | prev_best: -1.0 | prev_iter: 4
Appended iteration log: D:\AgentDs\agent_ds_healthcare\clai_ch3_v1_iteration_detail.jsonl


# Iteration 6
- 0.8244

In [28]:
# clai_ch3_v1 — Iteration (auto) — Weighted Ensemble + char-TFIDF + lexicon + reason_unit interaction
import os, json, re, math, random, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.base import clone
import joblib

# -----------------------
# Config / filenames
# -----------------------
VERSION = "v1"
TEAM = "clai"

SUBMISSION_PATH = Path(f"{TEAM}_ch3_{VERSION}_submission.csv")
ITER_LOG_PATH = Path(f"{TEAM}_ch3_{VERSION}_iteration_detail.jsonl")

CKPT_ROOT = Path("checkpoints") / f"{TEAM}_ch3_{VERSION}"
ART_ROOT = Path("artifacts") / f"{TEAM}_ch3_{VERSION}"
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
ART_ROOT.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

HM_MESSAGE = "HM: real F1=0.7842 (prev iter); request: keep direction, add bold improvements to break 0.80+."
REAL_SCORE = 0.7842

# -----------------------
# Utilities
# -----------------------
def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def to_jsonable(x):
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, dict):
        return {k: to_jsonable(v) for k, v in x.items()}
    if isinstance(x, list):
        return [to_jsonable(v) for v in x]
    return x

def append_jsonl(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(to_jsonable(obj), ensure_ascii=False) + "\n")

def get_next_iteration_id(log_path: Path, ckpt_root: Path, art_root: Path) -> int:
    max_id = -1
    if log_path.exists():
        with log_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    it = obj.get("iteration_id", None)
                    if isinstance(it, int):
                        max_id = max(max_id, it)
                except Exception:
                    continue

    # also scan existing checkpoint/artifacts to avoid collisions (e.g., if jsonl had duplicate ids)
    for p in ckpt_root.glob("iter[0-9][0-9][0-9][0-9]"):
        m = re.match(r"iter(\d{4})$", p.name)
        if m:
            max_id = max(max_id, int(m.group(1)))
    for p in art_root.glob("iter[0-9][0-9][0-9][0-9]_oof_prob*.npy"):
        m = re.match(r"iter(\d{4})_", p.name)
        if m:
            max_id = max(max_id, int(m.group(1)))
    for p in art_root.glob("iter[0-9][0-9][0-9][0-9]_oof_predictions*.csv"):
        m = re.match(r"iter(\d{4})_", p.name)
        if m:
            max_id = max(max_id, int(m.group(1)))
    return max_id + 1

def safe_dump(obj, path: Path) -> dict:
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        joblib.dump(obj, path)
        return {"dump_backend": "joblib", "dump_error": None, "model_bundle_written": True}
    except Exception as e1:
        # Fallback to cloudpickle if available; never crash the run due to checkpointing.
        try:
            import cloudpickle  # type: ignore
            with path.open("wb") as f:
                cloudpickle.dump(obj, f)
            return {"dump_backend": "cloudpickle", "dump_error": str(e1), "model_bundle_written": True}
        except Exception as e2:
            return {"dump_backend": None, "dump_error": f"{type(e1).__name__}: {e1} | fallback_failed: {type(e2).__name__}: {e2}", "model_bundle_written": False}

def to_1d(x):
    # picklable, no lambda
    return np.asarray(x).ravel()

# -----------------------
# Feature engineering from vitals_timeseries.json
# -----------------------
def safe_float(x):
    try:
        if x is None:
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def compute_slope(y):
    y = np.asarray(y, dtype=float)
    n = len(y)
    x = np.arange(n, dtype=float)
    mask = ~np.isnan(y)
    if mask.sum() < 2:
        return np.nan
    x = x[mask]
    y = y[mask]
    xm = x.mean()
    ym = y.mean()
    denom = ((x - xm) ** 2).sum()
    if denom == 0:
        return 0.0
    return float(((x - xm) * (y - ym)).sum() / denom)

def mean_abs_diff(y):
    y = np.asarray(y, dtype=float)
    y = y[~np.isnan(y)]
    if len(y) < 2:
        return np.nan
    d = np.abs(np.diff(y))
    return float(d.mean()) if len(d) else np.nan

def max_abs_diff(y):
    y = np.asarray(y, dtype=float)
    y = y[~np.isnan(y)]
    if len(y) < 2:
        return np.nan
    d = np.abs(np.diff(y))
    return float(d.max()) if len(d) else np.nan

def vitals_features(days_sorted, prefix="v_"):
    vitals_cols = ["hr", "sbp", "dbp", "temp_c", "rr"]
    arrs = {c: [safe_float(d.get(c)) for d in days_sorted] for c in vitals_cols}

    sbp = np.asarray(arrs["sbp"], dtype=float)
    dbp = np.asarray(arrs["dbp"], dtype=float)
    arrs["map"] = list((sbp + 2 * dbp) / 3.0)
    arrs["pp"] = list(sbp - dbp)

    feats = {}
    for c, v in arrs.items():
        vv = np.asarray(v, dtype=float)
        feats[f"{prefix}{c}_mean"] = float(np.nanmean(vv))
        feats[f"{prefix}{c}_std"] = float(np.nanstd(vv))
        feats[f"{prefix}{c}_min"] = float(np.nanmin(vv))
        feats[f"{prefix}{c}_max"] = float(np.nanmax(vv))
        feats[f"{prefix}{c}_first"] = float(vv[0]) if len(vv) and not np.isnan(vv[0]) else np.nan
        feats[f"{prefix}{c}_last"] = float(vv[-1]) if len(vv) and not np.isnan(vv[-1]) else np.nan
        feats[f"{prefix}{c}_delta"] = (feats[f"{prefix}{c}_last"] - feats[f"{prefix}{c}_first"]) if (not np.isnan(feats[f"{prefix}{c}_last"]) and not np.isnan(feats[f"{prefix}{c}_first"])) else np.nan
        feats[f"{prefix}{c}_slope"] = compute_slope(vv)
        feats[f"{prefix}{c}_missing"] = int(np.isnan(vv).sum())
        feats[f"{prefix}{c}_mean_abs_diff"] = mean_abs_diff(vv)
        feats[f"{prefix}{c}_max_abs_diff"] = max_abs_diff(vv)

        v3 = vv[-3:]
        feats[f"{prefix}{c}_last3_mean"] = float(np.nanmean(v3))
        feats[f"{prefix}{c}_last3_std"] = float(np.nanstd(v3))
        feats[f"{prefix}{c}_last3_min"] = float(np.nanmin(v3))
        feats[f"{prefix}{c}_last3_max"] = float(np.nanmax(v3))
        feats[f"{prefix}{c}_day10"] = float(vv[-1]) if len(vv) and not np.isnan(vv[-1]) else np.nan
    return feats

def text_windows(days_sorted):
    notes = [(d.get("day"), d.get("note") if d.get("note") is not None else "") for d in days_sorted]

    def join_days(valid_days):
        chunks = []
        for day, txt in notes:
            if day in valid_days and isinstance(txt, str) and txt.strip():
                chunks.append(txt.strip())
        return " ".join(chunks).strip()

    txt_all = join_days(set(range(1, 11)))
    txt_late3 = join_days({8, 9, 10})
    txt_day10 = join_days({10})

    return {
        "txt_all": txt_all,
        "txt_late3": txt_late3,
        "txt_day10": txt_day10,
        "note_len_all": int(len(txt_all)),
        "note_len_late": int(len(txt_late3)),
        "note_len_day10": int(len(txt_day10)),
        "note_nonempty_days": int(sum(1 for _, t in notes if isinstance(t, str) and t.strip())),
    }

# Lexicon features (small, dense)
POS_PATTERNS = [
    r"discharg", r"\bhome\b", r"\bready\b", r"\bstable\b", r"ambulat", r"\bpt\b", r"walk",
    r"afebrile", r"room air", r"\bra\b", r"wean", r"tolerat", r"\bpo\b", r"improv", r"pain"
]
NEG_PATTERNS = [
    r"\bicu\b", r"intubat", r"\bvent\b", r"pressor", r"unstable", r"worsen", r"fever", r"sepsis",
    r"tachy", r"hypoten", r"transfer", r"oxygen", r"\bo2\b", r"lactate", r"antibiotic", r"\bnpo\b"
]
POS_RE = [re.compile(p) for p in POS_PATTERNS]
NEG_RE = [re.compile(p) for p in NEG_PATTERNS]

def lexicon_counts(txt: str):
    t = (txt or "").lower()
    pos = sum(len(r.findall(t)) for r in POS_RE)
    neg = sum(len(r.findall(t)) for r in NEG_RE)
    return int(pos), int(neg), int(pos - neg)

def build_timeseries_features(stays_df: pd.DataFrame, vitals_by_stay: dict) -> pd.DataFrame:
    rows = []
    for sid in stays_df["stay_id"].astype(int).tolist():
        days = vitals_by_stay.get(int(sid), [])
        days_sorted = sorted(days, key=lambda d: d.get("day", 0))

        feats = {"stay_id": sid}
        feats.update(vitals_features(days_sorted, prefix="v_"))
        feats.update(text_windows(days_sorted))

        lp, ln, ld = lexicon_counts(feats["txt_late3"] + " " + feats["txt_day10"])
        feats["lex_pos"] = lp
        feats["lex_neg"] = ln
        feats["lex_diff"] = ld

        rows.append(feats)
    return pd.DataFrame(rows)

# -----------------------
# Load data
# -----------------------
stays_train = pd.read_csv("stays_train.csv")
stays_test = pd.read_csv("stays_test.csv")
patients = pd.read_csv("patients.csv")

with open("vitals_timeseries.json", "r", encoding="utf-8") as f:
    vitals_list = json.load(f)
vitals_by_stay = {int(item["stay_id"]): item["days"] for item in vitals_list}

# Feature build
ts_train = build_timeseries_features(stays_train, vitals_by_stay)
ts_test = build_timeseries_features(stays_test, vitals_by_stay)

train = stays_train.merge(patients, on="patient_id", how="left").merge(ts_train, on="stay_id", how="left")
test = stays_test.merge(patients, on="patient_id", how="left").merge(ts_test, on="stay_id", how="left")

# Basic joins / leakage guards
assert train.shape[0] == stays_train.shape[0], "Row mismatch after join (train)"
assert test.shape[0] == stays_test.shape[0], "Row mismatch after join (test)"
# patient_id disjoint check (as stated in prior logs)
train_pat = set(train["patient_id"].astype(int).tolist())
test_pat = set(test["patient_id"].astype(int).tolist())
patient_overlap = len(train_pat.intersection(test_pat))

# Derived categorical interaction
for df in (train, test):
    df["zip3"] = df["zip3"].astype(str)
    df["reason_unit"] = df["admission_reason"].astype(str) + "|" + df["unit_type"].astype(str)

# -----------------------
# Define feature columns
# -----------------------
text_cols = ["txt_all", "txt_late3", "txt_day10"]
cat_cols = ["unit_type", "admission_reason", "reason_unit", "sex", "insurance", "zip3"]

extra_num = ["age", "note_len_all", "note_len_late", "note_len_day10", "note_nonempty_days", "lex_pos", "lex_neg", "lex_diff"]
num_cols = [c for c in train.columns if c.startswith("v_")] + extra_num

# -----------------------
# Models
# -----------------------
# TF-IDF specs (keep moderate; tune later if needed)
tfidf_word_all = dict(max_features=2500, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
tfidf_word_late = dict(max_features=1500, ngram_range=(1, 2), min_df=1, sublinear_tf=True)
tfidf_word_day10 = dict(max_features=1200, ngram_range=(1, 2), min_df=1, sublinear_tf=True)
tfidf_char = dict(max_features=1500, analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)

# Logistic pipeline: num + OHE(cat) + word/char TF-IDF on multi-window notes
logi_pre = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler(with_mean=False))]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
        ("w_all", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="")),
                            ("to1d", FunctionTransformer(to_1d, validate=False)),
                            ("tfidf", TfidfVectorizer(**tfidf_word_all))]), ["txt_all"]),
        ("w_late3", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="")),
                              ("to1d", FunctionTransformer(to_1d, validate=False)),
                              ("tfidf", TfidfVectorizer(**tfidf_word_late))]), ["txt_late3"]),
        ("w_day10", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="")),
                              ("to1d", FunctionTransformer(to_1d, validate=False)),
                              ("tfidf", TfidfVectorizer(**tfidf_word_day10))]), ["txt_day10"]),
        ("c_late3", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="")),
                              ("to1d", FunctionTransformer(to_1d, validate=False)),
                              ("tfidf", TfidfVectorizer(**tfidf_char))]), ["txt_late3"]),
        ("c_day10", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="")),
                              ("to1d", FunctionTransformer(to_1d, validate=False)),
                              ("tfidf", TfidfVectorizer(**tfidf_char))]), ["txt_day10"]),
    ],
    sparse_threshold=0.3,
)

logi_clf = LogisticRegression(
    solver="saga",
    penalty="elasticnet",
    l1_ratio=0.05,
    C=0.6,
    class_weight="balanced",
    max_iter=20000,
    random_state=SEED,
    n_jobs=1,
)
pipe_logi = Pipeline([("preprocess", logi_pre), ("clf", logi_clf)])

# XGB (structured-only) pipeline: num + OHE(cat) (no TF-IDF)
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception as _e:
    HAS_XGB = False
    XGBClassifier = None

xgb_pre = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ],
    sparse_threshold=0.3,
)

if HAS_XGB:
    xgb_clf = XGBClassifier(
        n_estimators=450,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=2.0,
        reg_lambda=1.0,
        reg_alpha=0.0,
        gamma=0.0,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=SEED,
        n_jobs=1,           # determinism > speed
        tree_method="hist",
        verbosity=0,
    )
    pipe_xgb = Pipeline([("preprocess", xgb_pre), ("clf", xgb_clf)])
else:
    # fallback: a second logistic on structured only (keeps pipeline runnable)
    xgb_clf = LogisticRegression(
        solver="liblinear",
        penalty="l2",
        C=1.0,
        class_weight="balanced",
        max_iter=4000,
        random_state=SEED,
    )
    pipe_xgb = Pipeline([("preprocess", xgb_pre), ("clf", xgb_clf)])

# -----------------------
# CV OOF + ensemble weight/threshold tuning
# -----------------------
X = train.drop(columns=["discharge_ready_day11"])
y = train["discharge_ready_day11"].astype(int).values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof_prob_logi = np.zeros(len(y), dtype=float)
oof_prob_xgb = np.zeros(len(y), dtype=float)
fold_id = np.zeros(len(y), dtype=int)

fold_f1_logi_05 = []
fold_f1_xgb_05 = []

print(f"[Info] patient_id overlap train vs test: {patient_overlap} (expected 0)")
print("[CV] Running 5-fold StratifiedKFold...")

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    fold_id[va_idx] = fold

    m_logi = clone(pipe_logi)
    m_xgb = clone(pipe_xgb)

    m_logi.fit(X.iloc[tr_idx], y[tr_idx])
    m_xgb.fit(X.iloc[tr_idx], y[tr_idx])

    p_logi = m_logi.predict_proba(X.iloc[va_idx])[:, 1]
    p_xgb = m_xgb.predict_proba(X.iloc[va_idx])[:, 1]

    oof_prob_logi[va_idx] = p_logi
    oof_prob_xgb[va_idx] = p_xgb

    fold_f1_logi_05.append(f1_score(y[va_idx], (p_logi >= 0.5).astype(int), average="macro"))
    fold_f1_xgb_05.append(f1_score(y[va_idx], (p_xgb >= 0.5).astype(int), average="macro"))

def best_threshold(probs, y_true, grid):
    best_f = -1.0
    best_t = 0.5
    for t in grid:
        pred = (probs >= t).astype(int)
        f = f1_score(y_true, pred, average="macro")
        if f > best_f:
            best_f = f
            best_t = float(t)
    return float(best_f), float(best_t)

t_grid = np.linspace(0.05, 0.95, 91)

# Individual best thresholds (for reporting)
best_logi_f, best_logi_t = best_threshold(oof_prob_logi, y, t_grid)
best_xgb_f, best_xgb_t = best_threshold(oof_prob_xgb, y, t_grid)

# Ensemble weight + threshold search
w_grid = np.linspace(0.0, 1.0, 21)
best_ens_f = -1.0
best_w = 0.5
best_t = 0.5

for w in w_grid:
    comb = w * oof_prob_logi + (1.0 - w) * oof_prob_xgb
    f, t = best_threshold(comb, y, t_grid)
    if f > best_ens_f:
        best_ens_f = f
        best_w = float(w)
        best_t = float(t)

oof_prob_ens = best_w * oof_prob_logi + (1.0 - best_w) * oof_prob_xgb
oof_pred_ens = (oof_prob_ens >= best_t).astype(int)

fold_f1_ens = []
for fold in range(5):
    idx = (fold_id == fold)
    fold_f1_ens.append(f1_score(y[idx], (oof_prob_ens[idx] >= best_t).astype(int), average="macro"))

cm = confusion_matrix(y, oof_pred_ens)

print("\n[Results @0.5]")
print(f"  Logistic (0.5) fold macro-F1: {np.round(fold_f1_logi_05, 4).tolist()} | mean={np.mean(fold_f1_logi_05):.6f}")
print(f"  XGB/Structured (0.5) fold macro-F1: {np.round(fold_f1_xgb_05, 4).tolist()} | mean={np.mean(fold_f1_xgb_05):.6f}")

print("\n[Best threshold (OOF pooled)]")
print(f"  Logistic: best_f1={best_logi_f:.6f} @ t={best_logi_t:.3f}")
print(f"  XGB/Structured: best_f1={best_xgb_f:.6f} @ t={best_xgb_t:.3f}")

print("\n[Ensemble optimized on OOF pooled]")
print(f"  Best ensemble OOF macro-F1={best_ens_f:.6f} @ weight_logi={best_w:.2f}, threshold={best_t:.3f}")
print(f"  Ensemble per-fold macro-F1 @ best: {np.round(fold_f1_ens, 4).tolist()} | mean={np.mean(fold_f1_ens):.6f}")
print(f"  Confusion matrix (OOF @ best):\n{cm}")

# -----------------------
# Fit final models & predict test
# -----------------------
print("\n[Fit full] Training final models on full train...")
final_logi = clone(pipe_logi).fit(X, y)
final_xgb = clone(pipe_xgb).fit(X, y)

test_prob_logi = final_logi.predict_proba(test)[:, 1]
test_prob_xgb = final_xgb.predict_proba(test)[:, 1]
test_prob_ens = best_w * test_prob_logi + (1.0 - best_w) * test_prob_xgb

test_pred = (test_prob_ens >= best_t).astype(int)

submission = pd.DataFrame({
    "stay_id": test["stay_id"].astype(int).values,
    "discharge_ready_day11": test_pred.astype(int)
})
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"[Saved] submission -> {SUBMISSION_PATH.resolve()} (n={len(submission)})")

# -----------------------
# Save artifacts + checkpoint + P*
# -----------------------
iter_id = get_next_iteration_id(ITER_LOG_PATH, CKPT_ROOT, ART_ROOT)
ckpt_dir = CKPT_ROOT / f"iter{iter_id:04d}"
ckpt_dir.mkdir(parents=True, exist_ok=True)

oof_pred_path = ART_ROOT / f"iter{iter_id:04d}_oof_predictions.csv"
oof_prob_path = ART_ROOT / f"iter{iter_id:04d}_oof_prob.npy"
test_prob_path = ART_ROOT / f"iter{iter_id:04d}_test_prob.npy"
fold_id_path = ART_ROOT / f"iter{iter_id:04d}_fold_id.npy"

oof_df = pd.DataFrame({
    "stay_id": train["stay_id"].astype(int).values,
    "y_true": y.astype(int),
    "oof_prob_logi": oof_prob_logi,
    "oof_prob_xgb": oof_prob_xgb,
    "oof_prob": oof_prob_ens,
    "oof_pred": oof_pred_ens.astype(int),
})
oof_df.to_csv(oof_pred_path, index=False)
np.save(oof_prob_path, oof_prob_ens)
np.save(test_prob_path, test_prob_ens)
np.save(fold_id_path, fold_id)

config = {
    "seed": SEED,
    "cv": {"type": "StratifiedKFold", "n_splits": 5, "shuffle": True, "random_state": SEED},
    "features": {"num_cols": num_cols, "cat_cols": cat_cols, "text_cols": text_cols},
    "tfidf": {
        "word_all": tfidf_word_all,
        "word_late3": tfidf_word_late,
        "word_day10": tfidf_word_day10,
        "char_late3_day10": tfidf_char,
    },
    "logistic": {
        "solver": logi_clf.solver,
        "penalty": logi_clf.penalty,
        "l1_ratio": logi_clf.l1_ratio,
        "C": logi_clf.C,
        "class_weight": str(logi_clf.class_weight),
        "max_iter": logi_clf.max_iter,
        "random_state": logi_clf.random_state,
    },
    "xgb_or_fallback": {"has_xgb": HAS_XGB, "params": getattr(xgb_clf, "get_params", lambda: {})()},
    "ensemble_search": {
        "weight_grid": w_grid.tolist(),
        "threshold_grid": t_grid.tolist(),
        "best_weight_logi": best_w,
        "best_threshold": best_t,
    },
}

metrics = {
    "macro_f1_mean_at_0.5": {
        "logi": float(np.mean(fold_f1_logi_05)),
        "xgb": float(np.mean(fold_f1_xgb_05)),
    },
    "macro_f1_per_fold_at_0.5": {
        "logi": [float(x) for x in fold_f1_logi_05],
        "xgb": [float(x) for x in fold_f1_xgb_05],
    },
    "best_oof": {
        "logi_best_f1": best_logi_f,
        "logi_best_threshold": best_logi_t,
        "xgb_best_f1": best_xgb_f,
        "xgb_best_threshold": best_xgb_t,
        "ensemble_best_f1": best_ens_f,
        "ensemble_weight_logi": best_w,
        "ensemble_threshold": best_t,
    },
    "ensemble_per_fold_at_best": [float(x) for x in fold_f1_ens],
    "confusion_matrix_oof_at_best": cm.tolist(),
}

bundle_path = ckpt_dir / "model_bundle.joblib"
dump_info = safe_dump(
    {
        "pipe_logi": final_logi,
        "pipe_xgb": final_xgb,
        "ensemble": {"weight_logi": best_w, "threshold": best_t},
        "config": config,
        "metrics": metrics,
    },
    bundle_path,
)

with (ckpt_dir / "config.json").open("w", encoding="utf-8") as f:
    json.dump(to_jsonable(config), f, ensure_ascii=False, indent=2)
with (ckpt_dir / "metrics.json").open("w", encoding="utf-8") as f:
    json.dump(to_jsonable(metrics), f, ensure_ascii=False, indent=2)

# P* update
p_star_path = CKPT_ROOT / "p_star.json"
prev_best = -1.0
prev_iter = None
if p_star_path.exists():
    try:
        prev = json.loads(p_star_path.read_text(encoding="utf-8"))
        prev_best = float(prev.get("best_oof_macro_f1", -1.0))
        prev_iter = prev.get("iteration_id", None)
    except Exception:
        prev_best = -1.0

p_star_updated = bool(best_ens_f > prev_best + 1e-12)
if p_star_updated:
    p_star_obj = {
        "best_oof_macro_f1": float(best_ens_f),
        "iteration_id": int(iter_id),
        "checkpoint_dir": str(ckpt_dir),
        "timestamp": now_utc_iso(),
        "ensemble_weight_logi": float(best_w),
        "ensemble_threshold": float(best_t),
    }
    p_star_path.write_text(json.dumps(p_star_obj, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"\n[Checkpoint] {ckpt_dir.resolve()}")
print(f"[P*] previous best={prev_best:.6f} (iter={prev_iter}) | new best={best_ens_f:.6f} | updated={p_star_updated}")

# -----------------------
# Append iteration log
# -----------------------
log_entry = {
    "version": VERSION,
    "iteration_id": int(iter_id),
    "timestamp": now_utc_iso(),
    "phase": "Modeling",
    "hm_message": HM_MESSAGE,
    "real_score": REAL_SCORE,
    "data_paths_used": {
        "stays_train": "stays_train.csv",
        "stays_test": "stays_test.csv",
        "patients": "patients.csv",
        "vitals_timeseries": "vitals_timeseries.json",
    },
    "join_keys": {"patients": "patient_id", "vitals_timeseries": "stay_id"},
    "join_notes": "Merged per-patient demographics + per-stay vitals/day notes (days1-10).",
    "leakage_checks_performed": [
        "Only uses day1-10 vitals/notes; target is discharge readiness at day11.",
        "No joins to external label-bearing tables; no target-derived aggregates.",
        f"Train/test patient_id overlap = {patient_overlap} (expected 0).",
    ],
    "feature_summary": {
        "n_train": int(train.shape[0]),
        "n_test": int(test.shape[0]),
        "num_cols": int(len(num_cols)),
        "cat_cols": int(len(cat_cols)),
        "text_cols": text_cols,
        "added_features": ["reason_unit (interaction cat)", "lex_pos/lex_neg/lex_diff", "char TF-IDF on late/day10"],
    },
    "models_attempted": [
        {
            "name": "LogisticRegression(saga, elasticnet) + [num+OHE+TFIDF(word+char)]",
            "params": config["logistic"],
            "cv_macro_f1_mean_at_0.5": float(np.mean(fold_f1_logi_05)),
            "cv_macro_f1_per_fold_at_0.5": [float(x) for x in fold_f1_logi_05],
            "cv_macro_f1_oof_at_best_threshold": float(best_logi_f),
            "oof_best_threshold": float(best_logi_t),
        },
        {
            "name": "XGBClassifier(structured-only) + [num+OHE]",
            "params": {"has_xgb": HAS_XGB, "xgb_params_or_fallback": config["xgb_or_fallback"]},
            "cv_macro_f1_mean_at_0.5": float(np.mean(fold_f1_xgb_05)),
            "cv_macro_f1_per_fold_at_0.5": [float(x) for x in fold_f1_xgb_05],
            "cv_macro_f1_oof_at_best_threshold": float(best_xgb_f),
            "oof_best_threshold": float(best_xgb_t),
        },
        {
            "name": "WeightedEnsemble(proba) + global threshold",
            "params": {"weight_logi_grid": w_grid.tolist(), "threshold_grid": t_grid.tolist()},
            "best_oof_macro_f1": float(best_ens_f),
            "best_weight_logi": float(best_w),
            "best_threshold": float(best_t),
            "macro_f1_per_fold_at_best": [float(x) for x in fold_f1_ens],
        },
    ],
    "selected_model": f"weighted_ensemble(logi_text+struct, xgb_struct) @ weight_logi={best_w:.2f}, threshold={best_t:.3f}",
    "validation_protocol": {
        "cv": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": SEED,
        "threshold_selection": "Grid search on pooled OOF probs to maximize Macro-F1",
        "ensemble_weight_selection": "Grid search on pooled OOF probs to maximize Macro-F1",
    },
    "metrics": metrics,
    "checkpoint": {
        "checkpoint_dir": str(ckpt_dir),
        "p_star_updated": bool(p_star_updated),
        "p_star_path": str(p_star_path),
        "submission_path": str(SUBMISSION_PATH),
        "oof_pred_path": str(oof_pred_path),
        "oof_prob_path": str(oof_prob_path),
        "test_prob_path": str(test_prob_path),
        "fold_id_path": str(fold_id_path),
        "model_bundle_path": str(bundle_path),
        **dump_info,
    },
    "deltas_vs_previous": "Add char-TFIDF + lexicon counts + reason_unit interaction; optimize ensemble weight+threshold on pooled OOF.",
    "known_defects_limitations": [
        "Joint tuning of ensemble weight + threshold on pooled OOF can still be mildly optimistic; monitor real LB gap.",
        "Char n-grams increase dimensionality; keep regularization and max_features bounded if overfitting suspected.",
    ],
    "next_step_hypothesis": "If gains are small, add a calibrated LinearSVC text-only branch and stack/weight-tune against XGB to capture extra note signal.",
}
append_jsonl(ITER_LOG_PATH, log_entry)
print(f"[Logged] appended -> {ITER_LOG_PATH.resolve()}")

[Info] patient_id overlap train vs test: 0 (expected 0)
[CV] Running 5-fold StratifiedKFold...

[Results @0.5]
  Logistic (0.5) fold macro-F1: [0.7962, 0.7438, 0.785, 0.7394, 0.7802] | mean=0.768912
  XGB/Structured (0.5) fold macro-F1: [0.853, 0.7877, 0.7945, 0.8014, 0.8474] | mean=0.816813

[Best threshold (OOF pooled)]
  Logistic: best_f1=0.791959 @ t=0.330
  XGB/Structured: best_f1=0.820568 @ t=0.530

[Ensemble optimized on OOF pooled]
  Best ensemble OOF macro-F1=0.823975 @ weight_logi=0.35, threshold=0.600
  Ensemble per-fold macro-F1 @ best: [0.849, 0.8078, 0.8401, 0.8144, 0.8063] | mean=0.823511
  Confusion matrix (OOF @ best):
[[265  79]
 [ 80 576]]

[Fit full] Training final models on full train...
[Saved] submission -> D:\AgentDs\agent_ds_healthcare\clai_ch3_v1_submission.csv (n=1000)

[Checkpoint] D:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v1\iter0006
[P*] previous best=-1.000000 (iter=5) | new best=0.823975 | updated=True
[Logged] appended -> D:\AgentDs\agent_ds_h

# Iteration 7
- 0.7874

In [30]:
import os, json, time, re, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# =========================================================
# clai_ch3_v1 — Modeling iteration (post-EDA2)
# Base direction preserved: (TF-IDF Logistic) + (XGB structured) + OOF thresholding/ensemble
# Bold extension: patient-history features (CH1/CH2) + discharge_notes text + stacking option
# =========================================================

TEAM_TAG = "clai"
VERSION = "v1"
SEED = 42
N_SPLITS = 5  # deterministic CV (fixed seed)
TARGET_COL = "discharge_ready_day11"

TEXT_FULL_COL = "text_full"      # all 10-day notes + historical discharge notes
TEXT_RECENT_COL = "text_recent"  # last3-day notes + historical discharge notes
TEXT_SVD_COL = TEXT_FULL_COL     # SVD text for structured model

# -------------------------
# Utilities
# -------------------------
def now_ts():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def find_file(filename: str) -> Path:
    """Find a file by checking common locations, then falling back to a recursive search from CWD."""
    cand = Path(filename)
    if cand.exists():
        return cand

    for d in ["data", "ch3", "CH3", "datasets", "Dataset", "input"]:
        cand = Path(d) / filename
        if cand.exists():
            return cand

    cand = Path("/mnt/data") / filename  # harmless fallback if running in a sandbox
    if cand.exists():
        return cand

    for root, _, files in os.walk(Path.cwd()):
        if filename in files:
            return Path(root) / filename

    raise FileNotFoundError(f"Could not find {filename} from {Path.cwd()}")

def mode_str(s: pd.Series) -> str:
    m = s.dropna().astype(str).mode()
    return str(m.iloc[0]) if len(m) else "UNK"

def mode_num(s: pd.Series) -> float:
    m = s.dropna().mode()
    return float(m.iloc[0]) if len(m) else float("nan")

def lin_slope(x, y) -> float:
    if len(y) < 2:
        return 0.0
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x_mean = x.mean()
    y_mean = y.mean()
    denom = np.sum((x - x_mean) ** 2)
    if denom == 0:
        return 0.0
    return float(np.sum((x - x_mean) * (y - y_mean)) / denom)

def best_threshold_macro_f1(y_true, prob, thresholds=None):
    from sklearn.metrics import f1_score
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 91)
    best_t = 0.5
    best_f = -1.0
    for t in thresholds:
        pred = (prob >= t).astype(int)
        f = f1_score(y_true, pred, average="macro")
        if f > best_f:
            best_f = float(f)
            best_t = float(t)
    return best_f, best_t

# -------------------------
# Vitals feature extraction
# -------------------------
VITAL_VARS = ["hr", "sbp", "dbp", "temp_c", "rr"]

def extract_vitals(days_list):
    days_sorted = sorted(days_list, key=lambda d: d.get("day", 0))
    feats = {}

    for var in VITAL_VARS:
        xs, ys = [], []
        for d in days_sorted:
            val = d.get(var, None)
            if val is None:
                continue
            if isinstance(val, float) and np.isnan(val):
                continue
            xs.append(d.get("day", 0))
            ys.append(val)

        feats[f"{var}_count"] = int(len(ys))
        feats[f"{var}_missing"] = int(10 - len(ys))

        if len(ys) == 0:
            for stat in ["mean", "std", "min", "max", "first", "last", "range", "slope", "delta_last_first", "last3_mean", "last3_std"]:
                feats[f"{var}_{stat}"] = np.nan
        else:
            y_arr = np.asarray(ys, dtype=float)
            feats[f"{var}_mean"] = float(np.mean(y_arr))
            feats[f"{var}_std"] = float(np.std(y_arr)) if len(y_arr) > 1 else 0.0
            feats[f"{var}_min"] = float(np.min(y_arr))
            feats[f"{var}_max"] = float(np.max(y_arr))

            pairs = [(d.get("day", 0), d.get(var, None)) for d in days_sorted
                     if d.get(var, None) is not None and not (isinstance(d.get(var, None), float) and np.isnan(d.get(var, None)))]
            pairs_sorted = sorted(pairs, key=lambda t: t[0])

            feats[f"{var}_first"] = float(pairs_sorted[0][1])
            feats[f"{var}_last"] = float(pairs_sorted[-1][1])
            feats[f"{var}_range"] = feats[f"{var}_max"] - feats[f"{var}_min"]
            feats[f"{var}_slope"] = lin_slope(xs, ys)
            feats[f"{var}_delta_last_first"] = feats[f"{var}_last"] - feats[f"{var}_first"]

            last3 = [d.get(var, None) for d in days_sorted
                     if d.get("day", 0) >= 8 and d.get(var, None) is not None and not (isinstance(d.get(var, None), float) and np.isnan(d.get(var, None)))]
            if len(last3) == 0:
                feats[f"{var}_last3_mean"] = np.nan
                feats[f"{var}_last3_std"] = np.nan
            else:
                l3 = np.asarray(last3, dtype=float)
                feats[f"{var}_last3_mean"] = float(np.mean(l3))
                feats[f"{var}_last3_std"] = float(np.std(l3)) if len(l3) > 1 else 0.0

    # abnormal counts
    fever = tachy = hypo = tachyp = 0
    fever_l3 = tachy_l3 = hypo_l3 = tachyp_l3 = 0
    for d in days_sorted:
        day = d.get("day", 0)
        t = d.get("temp_c", None)
        hr = d.get("hr", None)
        sbp = d.get("sbp", None)
        rr = d.get("rr", None)

        if t is not None and not (isinstance(t, float) and np.isnan(t)) and t >= 38.0:
            fever += 1
            if day >= 8:
                fever_l3 += 1
        if hr is not None and not (isinstance(hr, float) and np.isnan(hr)) and hr >= 100:
            tachy += 1
            if day >= 8:
                tachy_l3 += 1
        if sbp is not None and not (isinstance(sbp, float) and np.isnan(sbp)) and sbp < 90:
            hypo += 1
            if day >= 8:
                hypo_l3 += 1
        if rr is not None and not (isinstance(rr, float) and np.isnan(rr)) and rr >= 20:
            tachyp += 1
            if day >= 8:
                tachyp_l3 += 1

    feats["abn_fever_cnt"] = fever
    feats["abn_tachy_cnt"] = tachy
    feats["abn_hypotension_cnt"] = hypo
    feats["abn_tachypnea_cnt"] = tachyp
    feats["abn_fever_last3"] = fever_l3
    feats["abn_tachy_last3"] = tachy_l3
    feats["abn_hypotension_last3"] = hypo_l3
    feats["abn_tachypnea_last3"] = tachyp_l3

    # text aggregation: all 10 days + last 3 days
    notes_all = []
    notes_last3 = []
    for d in days_sorted:
        note = d.get("note", "")
        if note is None:
            note = ""
        note = str(note)
        notes_all.append(note)
        if d.get("day", 0) >= 8:
            notes_last3.append(note)

    txt_all = " ".join(notes_all).strip()
    txt_last3 = " ".join(notes_last3).strip()

    feats["daily_note_text_all"] = txt_all
    feats["daily_note_text_last3"] = txt_last3
    feats["daily_note_len"] = int(len(txt_all))
    feats["daily_note_tok"] = int(len(re.findall(r"\w+", txt_all.lower())))
    return feats

# =========================
# Directories & iteration id
# =========================
log_path = Path(f"{TEAM_TAG}_ch3_{VERSION}_iteration_detail.jsonl")

prev_obj = None
max_iter_id = -1
if log_path.exists():
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception:
                continue
            prev_obj = obj
            if "iteration_id" in obj:
                try:
                    max_iter_id = max(max_iter_id, int(obj["iteration_id"]))
                except Exception:
                    pass
iter_id = max_iter_id + 1

ckpt_base = Path("checkpoints") / f"{TEAM_TAG}_ch3_{VERSION}"
art_base = Path("artifacts") / f"{TEAM_TAG}_ch3_{VERSION}"
ckpt_iter = ckpt_base / f"iter{iter_id:04d}"
ckpt_iter.mkdir(parents=True, exist_ok=True)
art_base.mkdir(parents=True, exist_ok=True)

# =========================
# Load data
# =========================
paths_used = {}
def _load_csv(name):
    p = find_file(name)
    paths_used[name] = str(p.resolve())
    return pd.read_csv(p)

stays_train = _load_csv("stays_train.csv")
stays_test = _load_csv("stays_test.csv")
patients = _load_csv("patients.csv")

# Extra sources (CH1/CH2) for safe patient-history joins
ad_train = _load_csv("admissions_train.csv")
ad_test = _load_csv("admissions_test.csv")
ed_train = _load_csv("ed_cost_train.csv")
ed_test = _load_csv("ed_cost_test.csv")

p_dn = find_file("discharge_notes.json")
paths_used["discharge_notes.json"] = str(p_dn.resolve())
discharge_notes = json.load(open(p_dn, "r", encoding="utf-8"))

p_vts = find_file("vitals_timeseries.json")
paths_used["vitals_timeseries.json"] = str(p_vts.resolve())
vitals_ts = json.load(open(p_vts, "r", encoding="utf-8"))

# Leakage check: patient overlap train vs test
patient_overlap = len(set(stays_train["patient_id"]).intersection(set(stays_test["patient_id"])))

# =========================
# Build vitals features (stay_id keyed)
# =========================
vital_rows = []
for item in vitals_ts:
    sid = int(item["stay_id"])
    feats = extract_vitals(item["days"])
    feats["stay_id"] = sid
    vital_rows.append(feats)
vitals_df = pd.DataFrame(vital_rows)

# =========================
# Build patient-history features (safe join; DROP CH1/CH2 targets)
# =========================
leakage_checks = []
ad_train_use = ad_train.copy()
if "readmit_30d" in ad_train_use.columns:
    ad_train_use = ad_train_use.drop(columns=["readmit_30d"])
    leakage_checks.append("Dropped admissions_train.readmit_30d")
else:
    leakage_checks.append("No admissions_train.readmit_30d column found (ok)")

ed_train_use = ed_train.copy()
if "ed_cost_next3y_usd" in ed_train_use.columns:
    ed_train_use = ed_train_use.drop(columns=["ed_cost_next3y_usd"])
    leakage_checks.append("Dropped ed_cost_train.ed_cost_next3y_usd")
else:
    leakage_checks.append("No ed_cost_train.ed_cost_next3y_usd column found (ok)")

ad_all = pd.concat([ad_train_use, ad_test], ignore_index=True)
ed_all = pd.concat([ed_train_use, ed_test], ignore_index=True)

ad_agg = ad_all.groupby("patient_id").agg(
    adm_count=("admission_id", "count"),
    los_mean=("los_days", "mean"),
    los_max=("los_days", "max"),
    los_min=("los_days", "min"),
    acuity_mean=("acuity_emergent", "mean"),
    charlson_mean=("charlson_band", "mean"),
    charlson_max=("charlson_band", "max"),
    ed_visits6m_mean=("ed_visits_6m", "mean"),
    ed_visits6m_max=("ed_visits_6m", "max"),
    discharge_wd_mode=("discharge_weekday", mode_num),
    primary_dx_mode=("primary_dx", mode_str),
).reset_index()

dn_df = pd.DataFrame(discharge_notes)
dn_join = dn_df.merge(ad_all[["admission_id", "patient_id"]], on="admission_id", how="left")

def join_notes(series: pd.Series) -> str:
    return " ".join([str(v) for v in series if v is not None])

dn_patient = dn_join.groupby("patient_id")["note"].apply(join_notes).reset_index().rename(columns={"note": "discharge_note_text"})

# =========================
# Merge into CH3 train/test
# =========================
JOIN_KEYS = {
    "stays_* + patients": "patient_id",
    "stays_* + vitals_ts": "stay_id",
    "stays_* + admissions_* agg": "patient_id",
    "stays_* + ed_cost_*": "patient_id",
    "stays_* + discharge_notes via admissions": "patient_id",
}

def assemble(base_df: pd.DataFrame) -> pd.DataFrame:
    df = (
        base_df.merge(patients, on="patient_id", how="left")
               .merge(vitals_df, on="stay_id", how="left")
               .merge(ad_agg, on="patient_id", how="left")
               .merge(ed_all, on="patient_id", how="left")
               .merge(dn_patient, on="patient_id", how="left")
    )

    # treat zip3 as categorical to allow non-linear interactions
    if "zip3" in df.columns:
        df["zip3"] = df["zip3"].astype("Int64").astype(str)

    # concatenate text sources (daily notes + historical discharge notes)
    df[TEXT_FULL_COL] = (df["daily_note_text_all"].fillna("") + " " + df["discharge_note_text"].fillna("")).str.strip()
    df[TEXT_RECENT_COL] = (df["daily_note_text_last3"].fillna("") + " " + df["discharge_note_text"].fillna("")).str.strip()
    return df

train_df = assemble(stays_train)
test_df = assemble(stays_test)

# =========================
# Feature columns
# =========================
exclude = {
    "stay_id", "patient_id", TARGET_COL,
    "daily_note_text_all", "daily_note_text_last3", "discharge_note_text",
    TEXT_FULL_COL, TEXT_RECENT_COL
}
num_cols = []
cat_cols = []
for c in train_df.columns:
    if c in exclude:
        continue
    if train_df[c].dtype == "object":
        cat_cols.append(c)
    else:
        num_cols.append(c)

# =========================
# Models
# =========================
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import TruncatedSVD
from sklearn.base import BaseEstimator, TransformerMixin
import joblib

try:
    from xgboost import XGBClassifier
except Exception as e:
    raise ImportError("xgboost is required for this pipeline. Please install xgboost.") from e

class TextColumnExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, col: str):
        self.col = col
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            s = X[self.col]
        else:
            s = pd.Series(X[:, 0])
        return s.fillna("").astype(str).values

# Logistic text models (two windows)
pipe_logi_full = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=40000, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=3000, C=4.0, solver="liblinear", class_weight="balanced", random_state=SEED)),
])
pipe_logi_recent = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=25000, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=3000, C=4.0, solver="liblinear", class_weight="balanced", random_state=SEED)),
])

# Structured model: num + cat + TF-IDF->SVD text
text_svd = Pipeline([
    ("sel", TextColumnExtractor(TEXT_SVD_COL)),
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=20000, sublinear_tf=True)),
    ("svd", TruncatedSVD(n_components=64, random_state=SEED)),
])

num_pipe = Pipeline([("imp", SimpleImputer(strategy="median"))])
cat_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                     ("ohe", OneHotEncoder(handle_unknown="ignore"))])

preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
    ("txt", text_svd, [TEXT_SVD_COL]),
], sparse_threshold=0.3)

pos = float(train_df[TARGET_COL].sum())
neg = float(len(train_df) - pos)
scale_pos_weight = (neg / pos) if pos > 0 else 1.0

pipe_xgb = Pipeline([
    ("preprocess", preprocess),
    ("clf", XGBClassifier(
        n_estimators=800,
        learning_rate=0.04,
        max_depth=4,
        min_child_weight=2,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.2,
        gamma=0.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=1,  # determinism
        random_state=SEED,
        scale_pos_weight=scale_pos_weight,
    ))
])

# =========================
# CV OOF
# =========================
y = train_df[TARGET_COL].astype(int).values
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_logi_full = np.zeros(len(train_df), dtype=float)
oof_logi_recent = np.zeros(len(train_df), dtype=float)
oof_xgb  = np.zeros(len(train_df), dtype=float)
fold_id  = np.full(len(train_df), -1, dtype=int)

fold_f1_logi_full_05 = []
fold_f1_logi_recent_05 = []
fold_f1_xgb_05 = []

for f, (tr, va) in enumerate(skf.split(train_df, y)):
    fold_id[va] = f
    Xtr, Xva = train_df.iloc[tr], train_df.iloc[va]
    ytr, yva = y[tr], y[va]

    pipe_logi_full.fit(Xtr[TEXT_FULL_COL].values, ytr)
    oof_logi_full[va] = pipe_logi_full.predict_proba(Xva[TEXT_FULL_COL].values)[:, 1]

    pipe_logi_recent.fit(Xtr[TEXT_RECENT_COL].values, ytr)
    oof_logi_recent[va] = pipe_logi_recent.predict_proba(Xva[TEXT_RECENT_COL].values)[:, 1]

    pipe_xgb.fit(Xtr, ytr)
    oof_xgb[va] = pipe_xgb.predict_proba(Xva)[:, 1]

    fold_f1_logi_full_05.append(float(f1_score(yva, (oof_logi_full[va] >= 0.5).astype(int), average="macro")))
    fold_f1_logi_recent_05.append(float(f1_score(yva, (oof_logi_recent[va] >= 0.5).astype(int), average="macro")))
    fold_f1_xgb_05.append(float(f1_score(yva, (oof_xgb[va] >= 0.5).astype(int), average="macro")))

best_f1_logi_full, best_t_logi_full = best_threshold_macro_f1(y, oof_logi_full)
best_f1_logi_recent, best_t_logi_recent = best_threshold_macro_f1(y, oof_logi_recent)
best_f1_xgb,  best_t_xgb  = best_threshold_macro_f1(y, oof_xgb)

# 2-model ensemble grid: (logi_full) + (xgb)
weights = np.linspace(0.0, 1.0, 21)
ths = np.linspace(0.05, 0.95, 91)
best_ens = {"f1": -1.0, "w_logi_full": 0.5, "t": 0.5}
for w in weights:
    p = w * oof_logi_full + (1.0 - w) * oof_xgb
    f1_w, t_w = best_threshold_macro_f1(y, p, thresholds=ths)
    if f1_w > best_ens["f1"]:
        best_ens = {"f1": float(f1_w), "w_logi_full": float(w), "t": float(t_w)}

# stacking meta (3-model): CV on OOF
meta_oof = np.zeros(len(train_df), dtype=float)
for f, (tr, va) in enumerate(skf.split(train_df, y)):
    Xtr_m = np.vstack([oof_logi_full[tr], oof_logi_recent[tr], oof_xgb[tr]]).T
    Xva_m = np.vstack([oof_logi_full[va], oof_logi_recent[va], oof_xgb[va]]).T
    meta = LogisticRegression(max_iter=3000, C=1.0, solver="lbfgs", class_weight="balanced", random_state=SEED)
    meta.fit(Xtr_m, y[tr])
    meta_oof[va] = meta.predict_proba(Xva_m)[:, 1]
best_f1_meta, best_t_meta = best_threshold_macro_f1(y, meta_oof)

cands = {
    "logi_full": (best_f1_logi_full, {"t": best_t_logi_full}),
    "logi_recent": (best_f1_logi_recent, {"t": best_t_logi_recent}),
    "xgb": (best_f1_xgb, {"t": best_t_xgb}),
    "ens_logi_full_xgb": (best_ens["f1"], {"w_logi_full": best_ens["w_logi_full"], "t": best_ens["t"]}),
    "stack_3": (best_f1_meta, {"t": best_t_meta}),
}
best_name = max(cands.keys(), key=lambda k: cands[k][0])
best_f1_pooled, best_params = cands[best_name]

if best_name == "logi_full":
    oof_final = oof_logi_full
    final_threshold = best_params["t"]
    blend_info = None
elif best_name == "logi_recent":
    oof_final = oof_logi_recent
    final_threshold = best_params["t"]
    blend_info = None
elif best_name == "xgb":
    oof_final = oof_xgb
    final_threshold = best_params["t"]
    blend_info = None
elif best_name == "ens_logi_full_xgb":
    w = best_params["w_logi_full"]
    oof_final = w * oof_logi_full + (1.0 - w) * oof_xgb
    final_threshold = best_params["t"]
    blend_info = {"w_logi_full": float(w)}
else:
    oof_final = meta_oof
    final_threshold = best_params["t"]
    blend_info = None

oof_pred = (oof_final >= final_threshold).astype(int)
cm = confusion_matrix(y, oof_pred)

per_fold_f1 = []
for f in range(N_SPLITS):
    idx = np.where(fold_id == f)[0]
    if len(idx) > 0:
        per_fold_f1.append(float(f1_score(y[idx], oof_pred[idx], average="macro")))
macro_f1_mean = float(np.mean(per_fold_f1)) if len(per_fold_f1) else float(f1_score(y, oof_pred, average="macro"))

# =========================
# Fit full & predict test
# =========================
pipe_logi_full.fit(train_df[TEXT_FULL_COL].values, y)
pipe_logi_recent.fit(train_df[TEXT_RECENT_COL].values, y)
pipe_xgb.fit(train_df, y)

p_logi_full_test = pipe_logi_full.predict_proba(test_df[TEXT_FULL_COL].values)[:, 1]
p_logi_recent_test = pipe_logi_recent.predict_proba(test_df[TEXT_RECENT_COL].values)[:, 1]
p_xgb_test  = pipe_xgb.predict_proba(test_df)[:, 1]

meta_full = None
if best_name == "logi_full":
    p_final_test = p_logi_full_test
elif best_name == "logi_recent":
    p_final_test = p_logi_recent_test
elif best_name == "xgb":
    p_final_test = p_xgb_test
elif best_name == "ens_logi_full_xgb":
    w = float(best_params["w_logi_full"])
    p_final_test = w * p_logi_full_test + (1.0 - w) * p_xgb_test
else:
    meta_full = LogisticRegression(max_iter=3000, C=1.0, solver="lbfgs", class_weight="balanced", random_state=SEED)
    meta_full.fit(np.vstack([oof_logi_full, oof_logi_recent, oof_xgb]).T, y)
    p_final_test = meta_full.predict_proba(np.vstack([p_logi_full_test, p_logi_recent_test, p_xgb_test]).T)[:, 1]

test_pred = (p_final_test >= final_threshold).astype(int)

sub_path = Path(f"{TEAM_TAG}_ch3_{VERSION}_submission.csv")
sub_df = pd.DataFrame({"stay_id": test_df["stay_id"].astype(int), TARGET_COL: test_pred.astype(int)})
sub_df.to_csv(sub_path, index=False)

# =========================
# Save artifacts
# =========================
np.save(art_base / f"iter{iter_id:04d}_fold_id.npy", fold_id)
np.save(art_base / f"iter{iter_id:04d}_oof_prob.npy", oof_final)
np.save(art_base / f"iter{iter_id:04d}_test_prob.npy", p_final_test)
oof_pred_path = art_base / f"iter{iter_id:04d}_oof_predictions.csv"
pd.DataFrame({
    "stay_id": train_df["stay_id"].astype(int),
    "y_true": y.astype(int),
    "fold_id": fold_id.astype(int),
    "oof_prob_final": oof_final,
    "oof_pred_final": oof_pred.astype(int),
    "oof_prob_logi_full": oof_logi_full,
    "oof_prob_logi_recent": oof_logi_recent,
    "oof_prob_xgb": oof_xgb,
}).to_csv(oof_pred_path, index=False)

# =========================
# Checkpoint bundle (v1 requirement)
# =========================
config = {
    "team_tag": TEAM_TAG,
    "version": VERSION,
    "seed": SEED,
    "n_splits": N_SPLITS,
    "text_cols": {"full": TEXT_FULL_COL, "recent": TEXT_RECENT_COL, "svd": TEXT_SVD_COL},
    "xgb_params": pipe_xgb.named_steps["clf"].get_params(),
    "logi_full_params": pipe_logi_full.named_steps["clf"].get_params(),
    "logi_recent_params": pipe_logi_recent.named_steps["clf"].get_params(),
}
metrics = {
    "macro_f1_mean": macro_f1_mean,
    "macro_f1_per_fold": per_fold_f1,
    "macro_f1_pooled_best": float(best_f1_pooled),
    "confusion_matrix": cm.tolist(),
    "selected_strategy": best_name,
    "selected_threshold": float(final_threshold),
    "blend_info": blend_info,
    "base_models_best_thresholds": {
        "logi_full": {"best_f1": float(best_f1_logi_full), "t": float(best_t_logi_full)},
        "logi_recent": {"best_f1": float(best_f1_logi_recent), "t": float(best_t_logi_recent)},
        "xgb": {"best_f1": float(best_f1_xgb), "t": float(best_t_xgb)},
        "ens_logi_full_xgb": best_ens,
        "stack_3": {"best_f1": float(best_f1_meta), "t": float(best_t_meta)},
    },
    "cv_at_0p5": {
        "logi_full_folds": fold_f1_logi_full_05,
        "logi_recent_folds": fold_f1_logi_recent_05,
        "xgb_folds": fold_f1_xgb_05,
        "logi_full_mean": float(np.mean(fold_f1_logi_full_05)),
        "logi_recent_mean": float(np.mean(fold_f1_logi_recent_05)),
        "xgb_mean": float(np.mean(fold_f1_xgb_05)),
    },
    "patient_id_overlap_train_test": int(patient_overlap),
}

schema = {
    "numeric_cols": num_cols,
    "categorical_cols": cat_cols,
    "text_cols": [TEXT_FULL_COL, TEXT_RECENT_COL],
    "vitals_recipe": ["mean","std","min","max","first","last","range","missing","slope","delta_last_first","last3_mean","last3_std","abnormal_counts"],
    "history_sources": ["admissions_* (drop readmit_30d)", "ed_cost_* (drop ed_cost_next3y_usd)", "discharge_notes via admission_id->patient_id"],
}

model_bundle_path = ckpt_iter / "model_bundle.joblib"
bundle = {
    "pipe_logi_full": pipe_logi_full,
    "pipe_logi_recent": pipe_logi_recent,
    "pipe_xgb": pipe_xgb,
    "meta_full": meta_full,  # may be None
    "config": config,
    "metrics": metrics,
    "schema": schema,
}
joblib_ok = True
try:
    joblib.dump(bundle, model_bundle_path)
except Exception as e:
    joblib_ok = False
    with open(ckpt_iter / "joblib_dump_error.txt", "w", encoding="utf-8") as f:
        f.write(repr(e))

with open(ckpt_iter / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
with open(ckpt_iter / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
with open(ckpt_iter / "schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)

# Update P* (best-known by deterministic validation macro_f1_mean)
pstar_path = ckpt_base / "pstar.json"
pstar = {"best_macro_f1_mean": -1.0, "iter_id": None}
if pstar_path.exists():
    try:
        pstar = json.load(open(pstar_path, "r", encoding="utf-8"))
    except Exception:
        pass

is_new_pstar = False
if macro_f1_mean > float(pstar.get("best_macro_f1_mean", -1.0)):
    pstar = {
        "best_macro_f1_mean": float(macro_f1_mean),
        "iter_id": int(iter_id),
        "timestamp": now_ts(),
        "selected_strategy": best_name,
        "selected_threshold": float(final_threshold),
        "blend_info": blend_info,
    }
    with open(pstar_path, "w", encoding="utf-8") as f:
        json.dump(pstar, f, ensure_ascii=False, indent=2)
    is_new_pstar = True

# =========================
# Append iteration detail log (append-only)
# =========================
hm_message = "HM: real F1=0.8244; Ensemble OOF macro-F1≈0.823975 with weight_logi=0.35, threshold=0.600 (prev run). Continue improving from this base."
real_score = 0.8244

prev_macro = None
if isinstance(prev_obj, dict):
    try:
        prev_macro = prev_obj.get("metrics", {}).get("macro_f1_mean", None)
    except Exception:
        prev_macro = None

deltas = {
    "added_patient_history_features": True,
    "added_discharge_notes_text": True,
    "added_dual_text_windows": True,
    "added_stacking_candidate": True,
    "prev_macro_f1_mean": prev_macro,
    "delta_macro_f1_mean": (float(macro_f1_mean) - float(prev_macro)) if prev_macro is not None else None,
}

iter_log = {
    "version": VERSION,
    "iteration_id": int(iter_id),
    "timestamp": now_ts(),
    "phase": "Modeling",
    "hm_message": hm_message,
    "real_score": real_score,

    "data_paths_used": paths_used,
    "join_keys": JOIN_KEYS,
    "leakage_checks_performed": [
        f"patient_id overlap train vs test = {patient_overlap} (expected 0)",
        "only days 1-10 vitals/notes used (vitals_timeseries.json has 10 days)",
        *leakage_checks,
    ],

    "feature_summary": {
        "n_train": int(len(train_df)),
        "n_test": int(len(test_df)),
        "n_num_cols": int(len(num_cols)),
        "n_cat_cols": int(len(cat_cols)),
        "text_cols": [TEXT_FULL_COL, TEXT_RECENT_COL],
        "tfidf": {
            "logi_full_max_features": 40000,
            "logi_recent_max_features": 25000,
            "svd_max_features": 20000,
            "svd_n_components": 64,
        },
    },

    "models_attempted": [
        {"name": "logistic_tfidf_full", "params": {"C": 4.0, "class_weight": "balanced", "ngram_range": [1,2]}},
        {"name": "logistic_tfidf_recent", "params": {"C": 4.0, "class_weight": "balanced", "ngram_range": [1,2]}},
        {"name": "xgb_structured_plus_svd_text", "params": config["xgb_params"]},
        {"name": "ensemble_weight_search", "params": {"grid_w": list(map(float, weights)), "grid_t": "0.05..0.95 step 0.01"}},
        {"name": "stacking_meta_logistic", "params": {"C": 1.0, "class_weight": "balanced"}},
    ],

    "selected_model": {
        "strategy": best_name,
        "threshold": float(final_threshold),
        "blend_info": blend_info,
    },

    "validation_protocol": {
        "cv": f"{N_SPLITS}-fold StratifiedKFold(shuffle=True, random_state={SEED})",
        "seed": SEED,
    },

    "metrics": {
        "macro_f1_mean": float(macro_f1_mean),
        "macro_f1_per_fold": per_fold_f1,
        "macro_f1_pooled_best": float(best_f1_pooled),
        "confusion_matrix": cm.tolist(),
        "pstar_updated": bool(is_new_pstar),
        "joblib_dump_ok": bool(joblib_ok),
        "submission_path": str(sub_path.resolve()),
    },

    "deltas_vs_prev": deltas,
    "known_defects_limitations": [
        "CV is still an estimate; monitor LB drift when adding cross-task history features.",
        "Discharge_notes text is from other admissions; could add noise if patient histories differ from CH3 stay context.",
        "Tree model determinism set via n_jobs=1; increase only if runtime is a blocker and determinism is not strict.",
    ],
    "next_step_hypothesis": "If this improves/stabilizes LB, next try: tune XGB depth/min_child_weight, and add explicit 'stability' features (e.g., last3 variance, trend sign bins).",
}

with open(log_path, "a", encoding="utf-8") as f:
    f.write(json.dumps(iter_log, ensure_ascii=False) + "\n")

# =========================
# Console summary
# =========================
print(f"[clai_ch3_{VERSION}] Iteration {iter_id} | Strategy={best_name}")
print(f"  OOF Macro-F1 (mean over folds @ best threshold) = {macro_f1_mean:.6f} | pooled-best={best_f1_pooled:.6f}")
print(f"  Selected threshold = {final_threshold:.3f} | blend_info={blend_info}")
print(f"  Confusion matrix (OOF) = {cm.tolist()}")
print(f"  patient_id overlap train vs test = {patient_overlap}")
print(f"  P* updated = {is_new_pstar} | P* file: {pstar_path.resolve()}")
print(f"  Submission written: {sub_path.resolve()}")
print(f"  Checkpoint dir: {ckpt_iter.resolve()}")
print(f"  Artifacts dir: {art_base.resolve()}")

[clai_ch3_v1] Iteration 7 | Strategy=stack_3
  OOF Macro-F1 (mean over folds @ best threshold) = 0.795994 | pooled-best=0.796426
  Selected threshold = 0.420 | blend_info=None
  Confusion matrix (OOF) = [[243, 101], [80, 576]]
  patient_id overlap train vs test = 0
  P* updated = True | P* file: D:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v1\pstar.json
  Submission written: D:\AgentDs\agent_ds_healthcare\clai_ch3_v1_submission.csv
  Checkpoint dir: D:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v1\iter0007
  Artifacts dir: D:\AgentDs\agent_ds_healthcare\artifacts\clai_ch3_v1


# Iteration 8
- 0.8244

In [32]:
import os, json, joblib, numpy as np, pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from sklearn.metrics import f1_score, confusion_matrix

# =====================
# Config (v1 rollback)
# =====================
VERSION = "v1"
TEAM_TAG = "clai"
TASK = "ch3"
TARGET = "discharge_ready_day11"
N_SPLITS = 5
SEED = 42

# HM instruction: rollback to last strong P* (iter0006 ensemble)
ROLLBACK_TO_ITER = 6
ROLLBACK_THRESHOLD = 0.600  # from iter0006 log
ROLLBACK_WEIGHT_LOGI = 0.35 # informational; iter0006 probs already ensembled

SUBMISSION_PATH = Path(f"{TEAM_TAG}_{TASK}_{VERSION}_submission.csv")
LOG_PATH = Path(f"{TEAM_TAG}_{TASK}_{VERSION}_iteration_detail.jsonl")

CKPT_ROOT = Path("checkpoints") / f"{TEAM_TAG}_{TASK}_{VERSION}"
ART_ROOT  = Path("artifacts")  / f"{TEAM_TAG}_{TASK}_{VERSION}"
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
ART_ROOT.mkdir(parents=True, exist_ok=True)

def _resolve_path(filename):
    """Try local project path first, then /mnt/data fallback (for notebook sandbox)."""
    cands = [Path(filename), Path("/mnt/data")/filename]
    for p in cands:
        if p.exists():
            return p
    raise FileNotFoundError(f"Cannot find required file: {filename} (tried {cands})")

def _find_artifact(fname):
    """Search common artifact locations for an existing file."""
    cands = [
        ART_ROOT / fname,
        Path(fname),
        Path("artifacts") / fname,                 # legacy
        Path("/mnt/data") / fname,                 # sandbox fallback
        Path("/mnt/data") / ("artifacts/" + fname) # another possible
    ]
    for p in cands:
        try:
            if p.exists():
                return p
        except Exception:
            pass
    return None

def _safe_read_last_iter_id(log_path, version):
    if not log_path.exists():
        return -1
    max_id = -1
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if obj.get("version") != version:
                continue
            iid = obj.get("iteration_id")
            if isinstance(iid, int) and iid > max_id:
                max_id = iid
    return max_id

def _safe_best_pstar(log_path, version):
    """Best known deterministic validation score from log, using metrics.macro_f1_pooled_best if present."""
    if not log_path.exists():
        return (-1.0, None)
    best_f1 = -1.0
    best_iter = None
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if obj.get("version") != version:
                continue
            met = obj.get("metrics", {})
            cand = None
            if isinstance(met, dict):
                cand = met.get("macro_f1_pooled_best", None)
                if cand is None:
                    cand = met.get("macro_f1_mean", None)
            if isinstance(cand, (int, float)) and float(cand) > best_f1:
                best_f1 = float(cand)
                best_iter = obj.get("iteration_id")
    return (best_f1, best_iter)

# =====================
# Load core data
# =====================
stays_train = pd.read_csv(_resolve_path("stays_train.csv"))
stays_test  = pd.read_csv(_resolve_path("stays_test.csv"))
patients    = pd.read_csv(_resolve_path("patients.csv"))

# leakage check: patient_id disjoint train vs test
overlap = set(stays_train["patient_id"]).intersection(set(stays_test["patient_id"]))
print(f"[Info] patient_id overlap train vs test: {len(overlap)} (expected 0)")
assert len(overlap) == 0, "Leakage risk: patient_id overlap train vs test should be 0."

# =====================
# Iteration bookkeeping
# =====================
prev_iter = _safe_read_last_iter_id(LOG_PATH, VERSION)
iteration_id = prev_iter + 1
best_known_f1, best_known_iter = _safe_best_pstar(LOG_PATH, VERSION)
print(f"[Iter] iteration_id={iteration_id} (prev={prev_iter}) | best_known_f1={best_known_f1:.6f} @ iter={best_known_iter}")

# =====================
# Rollback: load iter0006 probs if available (preferred)
# =====================
oof_prob = None
test_prob = None
fold_id = None
source_mode = "rollback_iter0006_artifacts"

oof_p  = _find_artifact(f"iter{ROLLBACK_TO_ITER:04d}_oof_prob.npy")
test_p = _find_artifact(f"iter{ROLLBACK_TO_ITER:04d}_test_prob.npy")
fold_p = _find_artifact(f"iter{ROLLBACK_TO_ITER:04d}_fold_id.npy")

if oof_p and test_p and fold_p:
    oof_prob = np.load(oof_p)
    test_prob = np.load(test_p)
    fold_id = np.load(fold_p)
    print(f"[Rollback] Loaded iter{ROLLBACK_TO_ITER:04d} artifacts:")
    print(f"  oof_prob:  {oof_p}")
    print(f"  test_prob: {test_p}")
    print(f"  fold_id:   {fold_p}")
else:
    # Fallback: train baseline (only if artifacts are missing)
    print("[Rollback] Missing one or more iter0006 artifacts. Falling back to retrain baseline CV ensemble (may differ slightly).")
    source_mode = "retrain_baseline_fallback"

    import random
    random.seed(SEED)
    np.random.seed(SEED)

    from sklearn.model_selection import StratifiedKFold
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    import xgboost as xgb

    with open(_resolve_path("vitals_timeseries.json"), "r", encoding="utf-8") as f:
        vitals = json.load(f)

    vital_names = ["hr","sbp","dbp","temp_c","rr"]

    # --- feature builder (no lambdas stored in transformers; pickle-safe patterns) ---
    def build_vitals_features(vitals_json):
        rows = []
        for rec in vitals_json:
            sid = rec["stay_id"]
            days = sorted(rec.get("days", []), key=lambda d: d.get("day", 0))  # ok: not serialized
            notes = [(d.get("note") or "") for d in days]
            note_all = " ".join([n for n in notes if n]).strip()
            note_last3 = " ".join([n for n in notes[-3:] if n]).strip()
            feat = {"stay_id": sid, "note_all": note_all, "note_last3": note_last3}

            # derived vitals
            def _to_arr(vals):
                return np.array([np.nan if v is None else float(v) for v in vals], dtype=float)

            sbp_arr = _to_arr([d.get("sbp") for d in days])
            dbp_arr = _to_arr([d.get("dbp") for d in days])
            pulse_pressure = sbp_arr - dbp_arr
            map_arr = dbp_arr + (pulse_pressure / 3.0)

            series_map = {"pulse_pressure": pulse_pressure, "map": map_arr}
            for vn in vital_names:
                series_map[vn] = _to_arr([d.get(vn) for d in days])

            idx = np.arange(1, 11, dtype=float)
            for name, arr in series_map.items():
                miss = np.isnan(arr)
                feat[f"{name}_missing"] = int(miss.sum())
                feat[f"{name}_missing_frac"] = float(miss.mean())
                if np.all(miss):
                    for stat in ["mean","std","min","max","first","last","delta_last_first","slope","last3_mean","last3_std","range"]:
                        feat[f"{name}_{stat}"] = np.nan
                    continue

                feat[f"{name}_mean"] = float(np.nanmean(arr))
                feat[f"{name}_std"]  = float(np.nanstd(arr))
                feat[f"{name}_min"]  = float(np.nanmin(arr))
                feat[f"{name}_max"]  = float(np.nanmax(arr))
                first = arr[~miss][0]
                last  = arr[~miss][-1]
                feat[f"{name}_first"] = float(first)
                feat[f"{name}_last"]  = float(last)
                feat[f"{name}_delta_last_first"] = float(last - first)
                feat[f"{name}_range"] = float(np.nanmax(arr) - np.nanmin(arr))
                if (~miss).sum() >= 2:
                    x = idx[~miss]; yv = arr[~miss]
                    x_mean = x.mean(); y_mean = yv.mean()
                    denom = float(((x - x_mean) ** 2).sum())
                    slope = 0.0 if denom == 0 else float(((x - x_mean) * (yv - y_mean)).sum() / denom)
                else:
                    slope = 0.0
                feat[f"{name}_slope"] = float(slope)
                last3 = arr[-3:]
                feat[f"{name}_last3_mean"] = float(np.nanmean(last3))
                feat[f"{name}_last3_std"]  = float(np.nanstd(last3))
            rows.append(feat)
        return pd.DataFrame(rows)

    vit_df = build_vitals_features(vitals)

    df_tr = stays_train.merge(patients, on="patient_id", how="left").merge(vit_df, on="stay_id", how="left")
    df_te = stays_test.merge(patients, on="patient_id", how="left").merge(vit_df, on="stay_id", how="left")
    y = df_tr[TARGET].astype(int).values

    # text
    df_tr["text_all"] = (
        df_tr["admission_reason"].fillna("").astype(str) + " " +
        df_tr["unit_type"].fillna("").astype(str) + " " +
        df_tr["note_all"].fillna("").astype(str)
    ).str.lower()
    df_te["text_all"] = (
        df_te["admission_reason"].fillna("").astype(str) + " " +
        df_te["unit_type"].fillna("").astype(str) + " " +
        df_te["note_all"].fillna("").astype(str)
    ).str.lower()

    # structured cols
    cat_cols = ["unit_type","admission_reason","sex","insurance","zip3"]
    num_cols = [c for c in df_tr.columns if any(c.startswith(v + "_") for v in (vital_names + ["pulse_pressure","map"]))] + ["age"]
    num_cols = [c for c in num_cols if c in df_tr.columns]
    num_cols = list(dict.fromkeys(num_cols))  # stable order

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_logi = np.zeros(len(df_tr), dtype=float)
    oof_xgb  = np.zeros(len(df_tr), dtype=float)
    test_logi = np.zeros(len(df_te), dtype=float)
    test_xgb  = np.zeros(len(df_te), dtype=float)
    fold_id = np.zeros(len(df_tr), dtype=int)

    pipe_logi = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=12000, ngram_range=(1,2), min_df=2)),
        ("clf", LogisticRegression(max_iter=500, C=4.0, class_weight="balanced", solver="liblinear"))
    ])

    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
    ], remainder="drop")

    for fold, (tr_idx, va_idx) in enumerate(skf.split(df_tr, y)):
        fold_id[va_idx] = fold
        X_tr = df_tr.iloc[tr_idx]
        X_va = df_tr.iloc[va_idx]
        y_tr = y[tr_idx]
        y_va = y[va_idx]

        # Logistic text
        pipe_logi.fit(X_tr["text_all"], y_tr)
        oof_logi[va_idx] = pipe_logi.predict_proba(X_va["text_all"])[:,1]
        test_logi += pipe_logi.predict_proba(df_te["text_all"])[:,1] / N_SPLITS

        # XGB structured (manual fit + early stopping)
        Xtr_mat = pre.fit_transform(X_tr[cat_cols + num_cols])
        Xva_mat = pre.transform(X_va[cat_cols + num_cols])
        Xte_mat = pre.transform(df_te[cat_cols + num_cols])

        xgb_clf = xgb.XGBClassifier(
            n_estimators=2500,
            learning_rate=0.03,
            max_depth=4,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=1.0,
            min_child_weight=1.0,
            gamma=0.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            n_jobs=-1,
            random_state=SEED + fold,
        )
        xgb_clf.fit(
            Xtr_mat, y_tr,
            eval_set=[(Xva_mat, y_va)],
            verbose=False,
            early_stopping_rounds=80
        )
        oof_xgb[va_idx] = xgb_clf.predict_proba(Xva_mat)[:,1]
        test_xgb += xgb_clf.predict_proba(Xte_mat)[:,1] / N_SPLITS

    # Ensemble weight + threshold search (deterministic)
    best = (-1.0, None, None)
    for w in np.linspace(0.0, 1.0, 21):  # 0.05 steps
        oof_ens = w * oof_logi + (1 - w) * oof_xgb
        for t in np.linspace(0.2, 0.8, 61):  # 0.01 steps
            f1 = f1_score(y, (oof_ens >= t).astype(int), average="macro")
            if f1 > best[0]:
                best = (float(f1), float(w), float(t))

    best_f1, best_w, best_t = best
    oof_prob = best_w * oof_logi + (1 - best_w) * oof_xgb
    test_prob = best_w * test_logi + (1 - best_w) * test_xgb
    ROLLBACK_WEIGHT_LOGI = best_w
    ROLLBACK_THRESHOLD = best_t

    print(f"[Fallback CV] Best ensemble pooled OOF macro-F1={best_f1:.6f} @ weight_logi={best_w:.3f}, threshold={best_t:.3f}")

# =====================
# Metrics on rollback model
# =====================
y_true = stays_train[TARGET].astype(int).values
assert oof_prob is not None and test_prob is not None, "Internal error: probabilities not produced."

threshold = float(ROLLBACK_THRESHOLD)
pred_oof = (oof_prob >= threshold).astype(int)
macro_f1_pooled = float(f1_score(y_true, pred_oof, average="macro"))
cm = confusion_matrix(y_true, pred_oof).tolist()

fold_f1s = []
macro_f1_mean = macro_f1_pooled
if fold_id is not None:
    fold_f1s = []
    for k in sorted(np.unique(fold_id).tolist()):
        idx = (fold_id == k)
        fold_f1s.append(float(f1_score(y_true[idx], pred_oof[idx], average="macro")))
    macro_f1_mean = float(np.mean(fold_f1s))

print(f"[Rollback Metrics] OOF Macro-F1 pooled={macro_f1_pooled:.6f} | mean_over_folds={macro_f1_mean:.6f} @ threshold={threshold:.3f}")
print(f"[Rollback Metrics] per-fold={fold_f1s}")
print(f"[Rollback Metrics] confusion_matrix={cm}")

is_new_pstar = (macro_f1_pooled > best_known_f1 + 1e-9)
print(f"[P*] is_new_pstar={is_new_pstar} (best_known_f1={best_known_f1:.6f})")

# =====================
# Write submission (rollback to P*)
# =====================
test_pred = (test_prob >= threshold).astype(int)
sub = pd.DataFrame({"stay_id": stays_test["stay_id"].astype(int), TARGET: test_pred.astype(int)})
sub.to_csv(SUBMISSION_PATH, index=False)
print(f"[Saved] submission -> {SUBMISSION_PATH.resolve()}")

# =====================
# Save artifacts + checkpoint
# =====================
np.save(ART_ROOT / f"iter{iteration_id:04d}_oof_prob.npy", oof_prob)
np.save(ART_ROOT / f"iter{iteration_id:04d}_test_prob.npy", test_prob)
if fold_id is not None:
    np.save(ART_ROOT / f"iter{iteration_id:04d}_fold_id.npy", fold_id)

oof_df = pd.DataFrame({
    "stay_id": stays_train["stay_id"].astype(int),
    "y_true": y_true.astype(int),
    "oof_prob": oof_prob.astype(float),
    "oof_pred": pred_oof.astype(int),
})
oof_df.to_csv(ART_ROOT / f"iter{iteration_id:04d}_oof_predictions.csv", index=False)

ckpt_dir = CKPT_ROOT / f"iter{iteration_id:04d}"
ckpt_dir.mkdir(parents=True, exist_ok=True)

config = {
    "team_tag": TEAM_TAG,
    "version": VERSION,
    "task": TASK,
    "seed": SEED,
    "n_splits": N_SPLITS,
    "rollback_to_iter": ROLLBACK_TO_ITER,
    "threshold": threshold,
    "weight_logi": float(ROLLBACK_WEIGHT_LOGI),
    "source_mode": source_mode,
    "note": "Rollback to last strong P* (iter0006). If artifacts missing, retrain baseline fallback.",
}
metrics = {
    "macro_f1_pooled_best": macro_f1_pooled,
    "macro_f1_mean": macro_f1_mean,
    "macro_f1_per_fold": fold_f1s,
    "confusion_matrix": cm,
    "threshold": threshold,
}
schema = {
    "categorical_cols": ["unit_type","admission_reason","sex","insurance","zip3"],
    "text_cols": ["daily notes (day1-10) via vitals_timeseries.json", "fallback only: admission_reason+unit_type+note_all -> text_all TF-IDF"],
    "numeric_feature_recipe": ["mean","std","min","max","first","last","missing","missing_frac","slope","last3_mean","last3_std","delta_last_first","range"],
    "notes": "Fallback retrain includes derived vitals pulse_pressure and MAP stats.",
}

joblib.dump(
    {"config": config, "metrics": metrics, "schema": schema, "oof_prob": oof_prob, "test_prob": test_prob, "fold_id": fold_id},
    ckpt_dir / "model_bundle.joblib"
)
with open(ckpt_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
with open(ckpt_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
with open(ckpt_dir / "schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)

print(f"[Saved] checkpoint -> {ckpt_dir.resolve()}")

# =====================
# Append iteration_detail.jsonl (HM score + rollback instruction)
# =====================
hm_message = (
    "HM: real F1=0.7874 for latest iteration (regression). Instruction: rollback to prior ~0.82x base (iter0006) "
    "and restart improvements only after local OOF sanity check."
)

log_entry = {
    "version": VERSION,
    "iteration_id": iteration_id,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "phase": "Modeling",
    "hm_message": hm_message,
    "real_score": 0.7874,
    "data_paths_used": {
        "stays_train": str(_resolve_path("stays_train.csv")),
        "stays_test": str(_resolve_path("stays_test.csv")),
        "patients": str(_resolve_path("patients.csv")),
        "vitals_timeseries": str(_resolve_path("vitals_timeseries.json")),
    },
    "join_keys": {"stays_patients": "patient_id", "stays_vitals": "stay_id"},
    "leakage_checks_performed": [
        "patient_id overlap train vs test == 0",
        "no target used in feature extraction (vitals/day1-10 only)",
    ],
    "feature_summary": {
        "core_stays_cols": ["stay_id","patient_id","unit_type","admission_reason"],
        "patient_cols": ["age","sex","insurance","zip3"],
        "vitals_numeric_recipe": schema["numeric_feature_recipe"],
        "notes_text": "daily notes (day1-10) concatenated; used in iter0006 artifacts; fallback retrain uses TF-IDF on text_all.",
    },
    "models_attempted": [
        {
            "name": "rollback_iter0006_ensemble",
            "source_mode": source_mode,
            "threshold": threshold,
            "weight_logi": float(ROLLBACK_WEIGHT_LOGI),
            "notes": "Loaded precomputed ensemble probs if available; otherwise retrained a baseline ensemble.",
        }
    ],
    "selected_model": "rollback_iter0006_ensemble",
    "validation_protocol": {
        "cv": f"{N_SPLITS}-fold StratifiedKFold",
        "seed": SEED,
        "oof": True,
        "threshold_selection": "fixed(iter0006) or fallback grid-search"
    },
    "metrics": metrics,
    "deltas_vs_previous_iteration": {
        "change": "Rollback to P* (iter0006). Removed failed changes from the regressed iteration.",
        "expected_effect": "Restore OOF≈0.824 and real-score stability.",
    },
    "p_star": {
        "best_known_before": {"macro_f1": best_known_f1, "iteration_id": best_known_iter},
        "current_macro_f1": macro_f1_pooled,
        "is_new_p_star": is_new_pstar,
        "rollback_to_iter": ROLLBACK_TO_ITER,
    },
    "known_defects_limitations": [
        "If using precomputed iter0006 probs, assumes probability array order matches stays_train/stays_test row order.",
        "Fallback retrain may not exactly match iter0006 due to model stochasticity / early stopping.",
    ],
    "next_step_hypothesis": "From this restored base, explore a third complementary text model (char n-grams) and/or CatBoost structured; only accept if pooled OOF macro-F1 improves over P* by >=0.003.",
}

with open(LOG_PATH, "a", encoding="utf-8") as f:
    f.write(json.dumps(log_entry, ensure_ascii=False) + "\n")

print(f"[Logged] appended -> {LOG_PATH.resolve()}")

[Info] patient_id overlap train vs test: 0 (expected 0)
[Iter] iteration_id=8 (prev=7) | best_known_f1=0.806980 @ iter=5
[Rollback] Loaded iter0006 artifacts:
  oof_prob:  artifacts\clai_ch3_v1\iter0006_oof_prob.npy
  test_prob: artifacts\clai_ch3_v1\iter0006_test_prob.npy
  fold_id:   artifacts\clai_ch3_v1\iter0006_fold_id.npy
[Rollback Metrics] OOF Macro-F1 pooled=0.823975 | mean_over_folds=0.823511 @ threshold=0.600
[Rollback Metrics] per-fold=[0.8489914788048754, 0.8077792853912257, 0.8401278976818545, 0.8143902172726281, 0.8062678062678064]
[Rollback Metrics] confusion_matrix=[[265, 79], [80, 576]]
[P*] is_new_pstar=True (best_known_f1=0.806980)
[Saved] submission -> D:\AgentDs\agent_ds_healthcare\clai_ch3_v1_submission.csv
[Saved] checkpoint -> D:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v1\iter0008
[Logged] appended -> D:\AgentDs\agent_ds_healthcare\clai_ch3_v1_iteration_detail.jsonl


# Iteration 9
- 0.7964

In [34]:
import os, json, time, random, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

import joblib

warnings.filterwarnings("ignore")

# =========================================================
# Config
# =========================================================
VX = "v1"
TEAM_TAG = "clai"
CHALLENGE = "ch3"
SEED = 42
N_SPLITS = 5

# Text vectorizer caps (balanced: fast-ish but strong)
TFIDF_MAX_ALL = 40000
TFIDF_MAX_LATE3 = 25000
TFIDF_MAX_D10 = 20000
CHAR_MAX_LATE3 = 25000
CHAR_MAX_D10 = 20000

# Ensemble search grids
WEIGHT_GRID = np.round(np.linspace(0.0, 1.0, 51), 4)   # step=0.02
THRESH_GRID = np.round(np.linspace(0.05, 0.95, 91), 4) # step=0.01

# XGB config (kept close to iter6, but slightly more expressive)
USE_XGBOOST = True
XGB_PARAMS = dict(
    objective="binary:logistic",
    n_estimators=650,
    learning_rate=0.05,
    max_depth=5,
    min_child_weight=1.0,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    reg_alpha=0.0,
    gamma=0.0,
    tree_method="hist",
    n_jobs=1,
    random_state=SEED,
    eval_metric="logloss",
    verbosity=0,
)

# Lexicon (lightweight heuristic features; complements TF-IDF)
LEX_POS = [
    "stable","improved","tolerating","ambulated","walked","mobilized","weaned","room air",
    "no new issues","afebrile","discharged","ready","normal limits","resting comfortably","slept better",
    "pain controlled","cleared","home","pt/ot","physical therapy","occupational therapy"
]
LEX_NEG = [
    "worsening","fever","hypotensive","hypoxia","respiratory","tachycard","tachycardic",
    "cultures pending","broad","antibiotic","elevated","drainage","purulence","delirium","confusion",
    "bandemia","infiltrate","bacteremia","sepsis","pain uncontrolled","shortness of breath","sob",
    "oxygen requirement","wheezing","desat","desaturation"
]

# =========================================================
# Paths & iteration id
# =========================================================
def choose_root() -> Path:
    for p in [Path("."), Path("/mnt/data")]:
        if (p / "stays_train.csv").exists() and (p / "vitals_timeseries.json").exists():
            return p.resolve()
    raise FileNotFoundError("Could not find stays_train.csv + vitals_timeseries.json in '.' or '/mnt/data'.")

ROOT = choose_root()

jsonl_path = ROOT / f"{TEAM_TAG}_{CHALLENGE}_{VX}_iteration_detail.jsonl"
ckpt_base = ROOT / "checkpoints" / f"{TEAM_TAG}_{CHALLENGE}_{VX}"
art_base = ROOT / "artifacts" / f"{TEAM_TAG}_{CHALLENGE}_{VX}"
ckpt_base.mkdir(parents=True, exist_ok=True)
art_base.mkdir(parents=True, exist_ok=True)

# Load past logs to infer next iteration_id and P*
past_recs = []
if jsonl_path.exists():
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                past_recs.append(json.loads(line))
            except Exception:
                # tolerate partially-written lines
                continue

last_iter = max([r.get("iteration_id", -1) for r in past_recs], default=-1)
iteration_id = last_iter + 1
iter_tag = f"iter{iteration_id:04d}"
timestamp = datetime.now(timezone.utc).isoformat()

def extract_best_f1(rec: dict) -> float:
    m = rec.get("metrics", {}) or {}
    if isinstance(m, dict):
        # Prefer pooled/ensemble best if present
        if "best_oof" in m and isinstance(m["best_oof"], dict) and "ensemble_best_f1" in m["best_oof"]:
            return float(m["best_oof"]["ensemble_best_f1"])
        if "macro_f1_pooled_best" in m:
            return float(m["macro_f1_pooled_best"])
        if "macro_f1_mean" in m:
            return float(m["macro_f1_mean"])
        if "macro_f1_mean_at_0.5" in m:
            return float(m["macro_f1_mean_at_0.5"])
    return float("-inf")

pstar_f1 = float("-inf")
pstar_iter = None
for r in past_recs:
    v = extract_best_f1(r)
    if v > pstar_f1:
        pstar_f1 = v
        pstar_iter = r.get("iteration_id", None)

# =========================================================
# Determinism
# =========================================================
random.seed(SEED)
np.random.seed(SEED)

# =========================================================
# Load data
# =========================================================
stays_train_path = ROOT / "stays_train.csv"
stays_test_path  = ROOT / "stays_test.csv"
patients_path    = ROOT / "patients.csv"
vitals_path      = ROOT / "vitals_timeseries.json"

stays_train = pd.read_csv(stays_train_path)
stays_test  = pd.read_csv(stays_test_path)
patients    = pd.read_csv(patients_path)
with open(vitals_path, "r", encoding="utf-8") as f:
    vitals_json = json.load(f)

# Leakage check: patient overlap train vs test (expected 0)
train_pat = set(stays_train["patient_id"].astype(int).tolist())
test_pat  = set(stays_test["patient_id"].astype(int).tolist())
overlap_pat = len(train_pat.intersection(test_pat))
print(f"[Info] patient_id overlap train vs test: {overlap_pat} (expected 0)")

# =========================================================
# Feature engineering from vitals_timeseries.json
# =========================================================
def _safe_float(x):
    try:
        if x is None:
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def _slope(day_idx, values):
    x = np.asarray(day_idx, dtype=float)
    y = np.asarray(values, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return np.nan
    x = x[mask]; y = y[mask]
    x = x - x.mean()
    denom = float((x**2).sum())
    if denom == 0.0:
        return np.nan
    return float((x * y).sum() / denom)

def _summarize(values, day_idx):
    arr = np.asarray(values, dtype=float)
    out = {}
    if np.isfinite(arr).any():
        out["mean"] = float(np.nanmean(arr))
        out["std"]  = float(np.nanstd(arr))
        out["min"]  = float(np.nanmin(arr))
        out["max"]  = float(np.nanmax(arr))
        out["median"] = float(np.nanmedian(arr))
        q25 = float(np.nanpercentile(arr, 25))
        q75 = float(np.nanpercentile(arr, 75))
        out["iqr"] = q75 - q25
        out["range"] = out["max"] - out["min"]
    else:
        out.update({k: np.nan for k in ["mean","std","min","max","median","iqr","range"]})

    out["last"] = float(arr[-1]) if len(arr) else np.nan
    out["first"] = float(arr[0]) if len(arr) else np.nan
    out["missing"] = float(np.isnan(arr).sum())
    out["slope"] = _slope(day_idx, arr)

    last3 = arr[-3:] if len(arr) >= 3 else arr
    if np.isfinite(last3).any():
        out["last3_mean"] = float(np.nanmean(last3))
        out["last3_std"]  = float(np.nanstd(last3))
        out["last3_min"]  = float(np.nanmin(last3))
        out["last3_max"]  = float(np.nanmax(last3))
        out["last3_range"] = out["last3_max"] - out["last3_min"]
    else:
        out.update({k: np.nan for k in ["last3_mean","last3_std","last3_min","last3_max","last3_range"]})

    if len(arr) > 1 and np.isfinite(arr[0]) and np.isfinite(arr[-1]):
        out["delta_last_first"] = float(arr[-1] - arr[0])
    else:
        out["delta_last_first"] = np.nan
    return out

def _count_kw(text, kws):
    t = (text or "").lower()
    return int(sum(t.count(k) for k in kws))

# Build a stay_id -> feature row dict
vital_rows = []
for entry in vitals_json:
    sid = int(entry["stay_id"])
    days = entry.get("days", []) or []
    by_day = {int(d.get("day")): d for d in days if d.get("day") is not None}

    # Collect series over day 1..10
    series = {k: [] for k in ["hr","sbp","dbp","temp_c","rr"]}
    notes = []
    for day in range(1, 11):
        d = by_day.get(day, {}) or {}
        notes.append((d.get("note") or "").strip())
        for k in series:
            series[k].append(_safe_float(d.get(k)))

    day_idx = list(range(1, 11))
    row = {"stay_id": sid}

    # Base vitals recipe
    for k, arr in series.items():
        summ = _summarize(arr, day_idx)
        for subk, v in summ.items():
            row[f"{k}_{subk}"] = v

    # Derived physiology interactions
    hr = np.asarray(series["hr"], dtype=float)
    sbp = np.asarray(series["sbp"], dtype=float)
    dbp = np.asarray(series["dbp"], dtype=float)
    rr  = np.asarray(series["rr"], dtype=float)
    temp = np.asarray(series["temp_c"], dtype=float)

    shock = hr / sbp
    mapv = (sbp + 2.0 * dbp) / 3.0
    pp = sbp - dbp

    for name, arr in [("shock", shock), ("map", mapv), ("pulse_pressure", pp)]:
        summ = _summarize(arr, day_idx)
        for subk, v in summ.items():
            row[f"{name}_{subk}"] = v

    # Abnormality counts (10d + last3)
    row["cnt_hr_gt100"] = float(np.nansum(hr > 100))
    row["cnt_sbp_lt90"] = float(np.nansum(sbp < 90))
    row["cnt_rr_gt20"] = float(np.nansum(rr > 20))
    row["cnt_temp_gt37_8"] = float(np.nansum(temp > 37.8))
    row["cnt_shock_gt0_9"] = float(np.nansum(shock > 0.9))

    row["cnt_hr_gt100_last3"] = float(np.nansum(hr[-3:] > 100))
    row["cnt_sbp_lt90_last3"] = float(np.nansum(sbp[-3:] < 90))
    row["cnt_rr_gt20_last3"] = float(np.nansum(rr[-3:] > 20))
    row["cnt_temp_gt37_8_last3"] = float(np.nansum(temp[-3:] > 37.8))
    row["cnt_shock_gt0_9_last3"] = float(np.nansum(shock[-3:] > 0.9))

    # Text fields
    txt_all = " ".join([n for n in notes if n])
    txt_late3 = " ".join([n for n in notes[-3:] if n])
    txt_day10 = notes[-1] if notes else ""
    row["txt_all"] = txt_all
    row["txt_late3"] = txt_late3
    row["txt_day10"] = txt_day10

    # Lexicon counts (keep iter6-compatible names + a few granular ones)
    row["lex_pos"] = float(_count_kw(txt_late3 + " " + txt_day10, LEX_POS))
    row["lex_neg"] = float(_count_kw(txt_late3 + " " + txt_day10, LEX_NEG))
    row["lex_diff"] = row["lex_pos"] - row["lex_neg"]
    row["lex_pos_late3"] = float(_count_kw(txt_late3, LEX_POS))
    row["lex_neg_late3"] = float(_count_kw(txt_late3, LEX_NEG))
    row["lex_pos_day10"] = float(_count_kw(txt_day10, LEX_POS))
    row["lex_neg_day10"] = float(_count_kw(txt_day10, LEX_NEG))

    # Note length stats
    row["note_len_all"] = float(len(txt_all))
    row["note_len_late3"] = float(len(txt_late3))
    row["note_len_day10"] = float(len(txt_day10))
    row["note_wordcount_all"] = float(len(txt_all.split())) if txt_all else 0.0
    row["note_wordcount_late3"] = float(len(txt_late3.split())) if txt_late3 else 0.0
    row["note_wordcount_day10"] = float(len(txt_day10.split())) if txt_day10 else 0.0

    vital_rows.append(row)

vitals_df = pd.DataFrame(vital_rows)

# =========================================================
# Merge
# =========================================================
def prep_base(df_stays: pd.DataFrame) -> pd.DataFrame:
    df = df_stays.merge(patients, on="patient_id", how="left")
    df = df.merge(vitals_df, on="stay_id", how="left")
    df["reason_unit"] = df["admission_reason"].astype(str) + "__" + df["unit_type"].astype(str)
    # Ensure text columns exist
    for c in ["txt_all","txt_late3","txt_day10"]:
        if c not in df.columns:
            df[c] = ""
        df[c] = df[c].fillna("").astype(str)
    return df

train_df = prep_base(stays_train)
test_df  = prep_base(stays_test)

y = train_df["discharge_ready_day11"].astype(int).values
X_train = train_df.drop(columns=["discharge_ready_day11"])
X_test  = test_df.copy()

# Define feature groups
text_cols = ["txt_all","txt_late3","txt_day10"]
cat_cols = ["unit_type","admission_reason","sex","insurance","zip3","reason_unit"]
id_cols = ["stay_id","patient_id"]
num_cols = [c for c in X_train.columns if c not in id_cols + text_cols + cat_cols]

# =========================================================
# Model definitions
# =========================================================
def make_ohe(sparse_out=True):
    # sklearn compatibility
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=sparse_out)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=sparse_out)

# Logistic (text-heavy)
pre_logi = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler(with_mean=False)),
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", make_ohe(True)),
        ]), cat_cols),
        ("w_all",  TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=TFIDF_MAX_ALL), "txt_all"),
        ("w_late", TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=TFIDF_MAX_LATE3), "txt_late3"),
        ("w_d10",  TfidfVectorizer(ngram_range=(1,2), min_df=1, max_features=TFIDF_MAX_D10), "txt_day10"),
        ("c_late", TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=2, max_features=CHAR_MAX_LATE3), "txt_late3"),
        ("c_d10",  TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=2, max_features=CHAR_MAX_D10), "txt_day10"),
    ],
    remainder="drop",
)

pipe_logi = Pipeline(steps=[
    ("pre", pre_logi),
    ("clf", LogisticRegression(
        solver="saga",
        penalty="elasticnet",
        l1_ratio=0.05,
        C=0.6,
        class_weight="balanced",
        max_iter=15000,
        random_state=SEED,
    )),
])

# XGB/Structured
if USE_XGBOOST:
    try:
        import xgboost as xgb
        xgb_clf = xgb.XGBClassifier(**XGB_PARAMS)
    except Exception:
        USE_XGBOOST = False

if not USE_XGBOOST:
    # deterministic fallback (we keep direction, but ensure code runs)
    from sklearn.ensemble import HistGradientBoostingClassifier
    xgb_clf = HistGradientBoostingClassifier(random_state=SEED)

pre_xgb = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", make_ohe(True)),
        ]), cat_cols),
    ],
    remainder="drop",
)

pipe_xgb = Pipeline(steps=[
    ("pre", pre_xgb),
    ("clf", xgb_clf),
])

# =========================================================
# CV + OOF
# =========================================================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

n = len(train_df)
oof_logi = np.zeros(n, dtype=float)
oof_xgb  = np.zeros(n, dtype=float)
fold_id  = np.zeros(n, dtype=int)

fold_f1_logi_05 = []
fold_f1_xgb_05  = []

print(f"[CV] Running {N_SPLITS}-fold StratifiedKFold...")

Xtr = X_train.drop(columns=id_cols).copy()
for fold, (tr_idx, va_idx) in enumerate(skf.split(Xtr, y)):
    fold_id[va_idx] = fold
    X_tr, y_tr = Xtr.iloc[tr_idx], y[tr_idx]
    X_va, y_va = Xtr.iloc[va_idx], y[va_idx]

    # Logistic
    pipe_logi.fit(X_tr, y_tr)
    p_va = pipe_logi.predict_proba(X_va)[:, 1]
    oof_logi[va_idx] = p_va
    fold_f1_logi_05.append(f1_score(y_va, (p_va >= 0.5).astype(int), average="macro"))

    # XGB
    pipe_xgb.fit(X_tr, y_tr)
    p_va = pipe_xgb.predict_proba(X_va)[:, 1]
    oof_xgb[va_idx] = p_va
    fold_f1_xgb_05.append(f1_score(y_va, (p_va >= 0.5).astype(int), average="macro"))

print("\n[Results @0.5]")
print(f"  Logistic (0.5) fold macro-F1: {[round(x,4) for x in fold_f1_logi_05]} | mean={np.mean(fold_f1_logi_05):.6f}")
print(f"  XGB/Structured (0.5) fold macro-F1: {[round(x,4) for x in fold_f1_xgb_05]} | mean={np.mean(fold_f1_xgb_05):.6f}")

def best_threshold(y_true, proba, grid):
    best_f1 = -1.0
    best_t = 0.5
    for t in grid:
        f = f1_score(y_true, (proba >= t).astype(int), average="macro")
        if f > best_f1:
            best_f1 = float(f)
            best_t = float(t)
    return best_f1, best_t

logi_best_f1, logi_best_t = best_threshold(y, oof_logi, THRESH_GRID)
xgb_best_f1,  xgb_best_t  = best_threshold(y, oof_xgb, THRESH_GRID)

print("\n[Best threshold (OOF pooled)]")
print(f"  Logistic: best_f1={logi_best_f1:.6f} @ t={logi_best_t:.3f}")
print(f"  XGB/Structured: best_f1={xgb_best_f1:.6f} @ t={xgb_best_t:.3f}")

# Ensemble weight + threshold search on pooled OOF
best_ens_f1 = -1.0
best_w = 0.5
best_t = 0.5
for w in WEIGHT_GRID:
    p = w * oof_logi + (1.0 - w) * oof_xgb
    f, t = best_threshold(y, p, THRESH_GRID)
    if f > best_ens_f1:
        best_ens_f1 = float(f)
        best_w = float(w)
        best_t = float(t)

p_ens = best_w * oof_logi + (1.0 - best_w) * oof_xgb
oof_pred_best = (p_ens >= best_t).astype(int)
cm = confusion_matrix(y, oof_pred_best)

# Per-fold macro-F1 using the globally-chosen (w, t)
ens_fold_f1 = []
for fold in range(N_SPLITS):
    idx = np.where(fold_id == fold)[0]
    ens_fold_f1.append(f1_score(y[idx], oof_pred_best[idx], average="macro"))

print("\n[Ensemble optimized on OOF pooled]")
print(f"  Best ensemble OOF macro-F1={best_ens_f1:.6f} @ weight_logi={best_w:.2f}, threshold={best_t:.3f}")
print(f"  Ensemble per-fold macro-F1 @ best: {[round(x,4) for x in ens_fold_f1]} | mean={np.mean(ens_fold_f1):.6f}")
print("  Confusion matrix (OOF @ best):")
print(cm)

# =========================================================
# Fit full models + predict test
# =========================================================
pipe_logi.fit(Xtr, y)
pipe_xgb.fit(Xtr, y)

Xte = X_test.drop(columns=id_cols).copy()
test_prob_logi = pipe_logi.predict_proba(Xte)[:, 1]
test_prob_xgb  = pipe_xgb.predict_proba(Xte)[:, 1]
test_prob_ens  = best_w * test_prob_logi + (1.0 - best_w) * test_prob_xgb
test_pred = (test_prob_ens >= best_t).astype(int)

submission = pd.DataFrame({
    "stay_id": X_test["stay_id"].astype(int),
    "discharge_ready_day11": test_pred.astype(int),
})
sub_path = ROOT / f"{TEAM_TAG}_{CHALLENGE}_{VX}_submission.csv"
submission.to_csv(sub_path, index=False)
print(f"\n[Saved] submission -> {sub_path}")

# Also save a tagged candidate submission
cand_path = art_base / f"{iter_tag}_candidate_submission.csv"
submission.to_csv(cand_path, index=False)

# =========================================================
# Save artifacts
# =========================================================
np.save(art_base / f"{iter_tag}_oof_prob_logi.npy", oof_logi)
np.save(art_base / f"{iter_tag}_oof_prob_xgb.npy",  oof_xgb)
np.save(art_base / f"{iter_tag}_oof_prob_ens.npy",  p_ens)
np.save(art_base / f"{iter_tag}_test_prob_ens.npy", test_prob_ens)
np.save(art_base / f"{iter_tag}_fold_id.npy", fold_id)

oof_pred_df = pd.DataFrame({
    "stay_id": X_train["stay_id"].astype(int),
    "y_true": y.astype(int),
    "oof_logi": oof_logi,
    "oof_xgb": oof_xgb,
    "oof_ens": p_ens,
    "pred_ens": oof_pred_best.astype(int),
    "fold": fold_id.astype(int),
})
oof_pred_df.to_csv(art_base / f"{iter_tag}_oof_predictions.csv", index=False)

# =========================================================
# Checkpoint saving (v1 mechanics)
# =========================================================
ckpt_dir = ckpt_base / iter_tag
ckpt_dir.mkdir(parents=True, exist_ok=True)

config = {
    "team_tag": TEAM_TAG,
    "challenge": CHALLENGE,
    "version": VX,
    "seed": SEED,
    "n_splits": N_SPLITS,
    "tfidf_caps": {
        "TFIDF_MAX_ALL": TFIDF_MAX_ALL,
        "TFIDF_MAX_LATE3": TFIDF_MAX_LATE3,
        "TFIDF_MAX_D10": TFIDF_MAX_D10,
        "CHAR_MAX_LATE3": CHAR_MAX_LATE3,
        "CHAR_MAX_D10": CHAR_MAX_D10,
    },
    "xgb_params": XGB_PARAMS if USE_XGBOOST else {"fallback": str(type(xgb_clf))},
    "ensemble_search": {
        "weight_grid": {"min": float(WEIGHT_GRID.min()), "max": float(WEIGHT_GRID.max()), "n": int(len(WEIGHT_GRID))},
        "thresh_grid": {"min": float(THRESH_GRID.min()), "max": float(THRESH_GRID.max()), "n": int(len(THRESH_GRID))},
    },
    "lexicon": {"pos_terms_n": len(LEX_POS), "neg_terms_n": len(LEX_NEG)},
}

schema = {
    "id_cols": id_cols,
    "categorical_cols": cat_cols,
    "numeric_cols": num_cols,
    "text_cols": text_cols,
}

metrics = {
    "macro_f1_mean_at_0.5": float(np.mean(ens_fold_f1)),  # ensemble @ best uses global (w,t) per-fold
    "macro_f1_per_fold_at_0.5": {
        "logi": [float(x) for x in fold_f1_logi_05],
        "xgb":  [float(x) for x in fold_f1_xgb_05],
    },
    "best_oof": {
        "logi_best_f1": float(logi_best_f1),
        "logi_best_threshold": float(logi_best_t),
        "xgb_best_f1": float(xgb_best_f1),
        "xgb_best_threshold": float(xgb_best_t),
        "ensemble_best_f1": float(best_ens_f1),
        "ensemble_weight_logi": float(best_w),
        "ensemble_threshold": float(best_t),
    },
    "ensemble_per_fold_at_best": [float(x) for x in ens_fold_f1],
    "confusion_matrix_oof_at_best": cm.tolist(),
}

with open(ckpt_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
with open(ckpt_dir / "schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)
with open(ckpt_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

bundle = {
    "pipe_logi": pipe_logi,
    "pipe_xgb": pipe_xgb,
    "ensemble": {"weight_logi": best_w, "threshold": best_t},
    "config": config,
    "metrics": metrics,
    "schema": schema,
}
joblib.dump(bundle, ckpt_dir / "model_bundle.joblib")

# Update P* if improved (deterministic validation definition)
pstar_updated = False
if best_ens_f1 > (pstar_f1 if np.isfinite(pstar_f1) else -1.0) + 1e-6:
    pstar_updated = True
    pstar_payload = {
        "iteration_id": iteration_id,
        "macro_f1_pooled_best": float(best_ens_f1),
        "timestamp_utc": timestamp,
        "checkpoint_dir": str(ckpt_dir),
        "model_bundle_path": str(ckpt_dir / "model_bundle.joblib"),
        "ensemble": {"weight_logi": best_w, "threshold": best_t},
    }
    with open(ckpt_base / "p_star.json", "w", encoding="utf-8") as f:
        json.dump(pstar_payload, f, ensure_ascii=False, indent=2)

# =========================================================
# Append iteration log (append-only)
# =========================================================
iter_log = {
    "version": VX,
    "iteration_id": iteration_id,
    "timestamp_utc": timestamp,
    "phase": "Modeling",
    "hm_message": "Rollback to iter0006 base (best known) and improve; avoid identical rerun; add new features + guardrails.",
    "real_score": None,

    "data_paths_used": {
        "stays_train": str(stays_train_path),
        "stays_test": str(stays_test_path),
        "patients": str(patients_path),
        "vitals_timeseries": str(vitals_path),
    },
    "join_keys": {
        "stays<->patients": "patient_id",
        "stays<->vitals_timeseries": "stay_id",
    },
    "leakage_checks": [
        f"patient_id overlap train vs test = {overlap_pat} (expected 0)",
        "vitals_timeseries used only day1..day10 vitals + notes; target is day11 readiness",
        "no use of discharge_ready_day11 in feature generation",
    ],
    "feature_summary": {
        "n_rows_train": int(train_df.shape[0]),
        "n_rows_test": int(test_df.shape[0]),
        "n_num_cols": int(len(num_cols)),
        "n_cat_cols": int(len(cat_cols)),
        "n_text_cols": int(len(text_cols)),
        "text_cols": text_cols,
        "cat_cols": cat_cols,
        "added_features": [
            "Derived physiology stats: shock=hr/sbp, MAP=(sbp+2*dbp)/3, pulse_pressure=sbp-dbp (full+last3)",
            "Abnormality counts: hr>100, sbp<90, rr>20, temp>37.8, shock>0.9 (full+last3)",
            "Expanded numeric summaries: median/IQR/range + last3 min/max/range",
            "Lexicon counts + note lengths (iter6-compatible names: lex_pos/lex_neg/lex_diff kept)",
        ],
    },
    "models_attempted": [
        {
            "name": "LogisticRegression(TFIDF word+char + structured)",
            "params": {"C": 0.6, "penalty": "elasticnet", "l1_ratio": 0.05, "class_weight": "balanced"},
            "cv_macro_f1_at_0.5_per_fold": [float(x) for x in fold_f1_logi_05],
            "cv_macro_f1_at_0.5_mean": float(np.mean(fold_f1_logi_05)),
            "oof_best_threshold": float(logi_best_t),
            "oof_best_macro_f1": float(logi_best_f1),
        },
        {
            "name": "XGBClassifier(structured + engineered vitals)",
            "params": XGB_PARAMS if USE_XGBOOST else {"fallback": str(type(xgb_clf))},
            "cv_macro_f1_at_0.5_per_fold": [float(x) for x in fold_f1_xgb_05],
            "cv_macro_f1_at_0.5_mean": float(np.mean(fold_f1_xgb_05)),
            "oof_best_threshold": float(xgb_best_t),
            "oof_best_macro_f1": float(xgb_best_f1),
        },
        {
            "name": "Ensemble(weighted prob avg; optimized on pooled OOF)",
            "params": {"weight_logi_grid_n": int(len(WEIGHT_GRID)), "threshold_grid_n": int(len(THRESH_GRID))},
            "oof_best_macro_f1": float(best_ens_f1),
            "oof_best_weight_logi": float(best_w),
            "oof_best_threshold": float(best_t),
            "cv_macro_f1_per_fold_at_best": [float(x) for x in ens_fold_f1],
            "confusion_matrix_oof_at_best": cm.tolist(),
        }
    ],
    "selected_model": {
        "name": "Ensemble(Logi + XGB)",
        "weight_logi": float(best_w),
        "threshold": float(best_t),
    },
    "validation_protocol": {
        "cv": "StratifiedKFold",
        "n_splits": N_SPLITS,
        "shuffle": True,
        "random_state": SEED,
        "oof": True,
        "threshold_optimized_on": "pooled OOF",
    },
    "metrics": metrics,

    "checkpoint": {
        "checkpoint_dir": str(ckpt_dir),
        "pstar_prev_iteration_id": pstar_iter,
        "pstar_prev_best_macro_f1": None if not np.isfinite(pstar_f1) else float(pstar_f1),
        "pstar_updated": bool(pstar_updated),
        "pstar_path": str(ckpt_base / "p_star.json"),
        "artifact_dir": str(art_base),
    },
    "deltas_vs_previous": [
        "Add physiology interaction features (shock/MAP/pulse pressure) + abnormality counts (10d + last3).",
        "Add richer robust summaries (median/IQR/range and last3 min/max/range).",
        "Slightly increase XGB capacity (depth=5, min_child_weight=1, more trees) while keeping hist + deterministic seed.",
        "Keep iter6 ensemble structure; refine (w, threshold) search resolution.",
    ],
    "known_defects_limitations": [
        "Lexicon is heuristic and may not generalize to unseen phrasing; TF-IDF still bag-of-words.",
        "Threshold/weight optimized on OOF can still overfit slightly; monitor real-vs-local drift.",
        "Feature explosion risk (many engineered stats) — may need pruning if instability appears.",
    ],
    "next_step_hypothesis": "If OOF improves and real holds, next explore adding low-rank text SVD features into XGB (as complementary signal) or a 3rd model for diversity.",
}

with open(jsonl_path, "a", encoding="utf-8") as f:
    f.write(json.dumps(iter_log, ensure_ascii=False) + "\n")

print(f"[Logged] appended iteration -> {jsonl_path}")
print(f"[Checkpoint] saved -> {ckpt_dir}")
print(f"[P*] previous_best={pstar_f1 if np.isfinite(pstar_f1) else None} | updated={pstar_updated} | current_iter={iteration_id}")

[Info] patient_id overlap train vs test: 0 (expected 0)
[CV] Running 5-fold StratifiedKFold...

[Results @0.5]
  Logistic (0.5) fold macro-F1: [0.8144, 0.7598, 0.789, 0.7748, 0.7707] | mean=0.781746
  XGB/Structured (0.5) fold macro-F1: [0.8519, 0.7877, 0.798, 0.8014, 0.8077] | mean=0.809318

[Best threshold (OOF pooled)]
  Logistic: best_f1=0.797119 @ t=0.340
  XGB/Structured: best_f1=0.818355 @ t=0.440

[Ensemble optimized on OOF pooled]
  Best ensemble OOF macro-F1=0.826790 @ weight_logi=0.48, threshold=0.400
  Ensemble per-fold macro-F1 @ best: [0.8736, 0.8007, 0.7914, 0.8429, 0.823] | mean=0.826337
  Confusion matrix (OOF @ best):
[[232 112]
 [ 35 621]]

[Saved] submission -> D:\AgentDs\agent_ds_healthcare\clai_ch3_v1_submission.csv
[Logged] appended iteration -> D:\AgentDs\agent_ds_healthcare\clai_ch3_v1_iteration_detail.jsonl
[Checkpoint] saved -> D:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v1\iter0009
[P*] previous_best=0.8239746523499384 | updated=True | current_iter=9

# Iteration 10
- 0.7782

In [38]:
# HOTFIX MODE (does NOT count as new iteration): fix PicklingError from lambda in tfidf_pipe + add robust safe_dump
import os, re, json, random, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MaxAbsScaler, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# -----------------------
# Config (do not change iteration_id in hotfix)
# -----------------------
VX = "v1"
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

DATA_STAYS_TRAIN = "stays_train.csv"
DATA_STAYS_TEST  = "stays_test.csv"
DATA_PATIENTS    = "patients.csv"
DATA_VITALS      = "vitals_timeseries.json"

SUBMISSION_PATH = f"clai_ch3_{VX}_submission.csv"
LOG_PATH = f"clai_ch3_{VX}_iteration_detail.jsonl"

CKPT_ROOT = Path("checkpoints") / f"clai_ch3_{VX}"
ART_ROOT  = Path("artifacts")   / f"clai_ch3_{VX}"
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
ART_ROOT.mkdir(parents=True, exist_ok=True)

def _read_jsonl(path: str):
    recs = []
    p = Path(path)
    if not p.exists():
        return recs
    with p.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                recs.append(json.loads(line))
            except Exception:
                # tolerate any corrupted line
                continue
    return recs

def _detect_hotfix_iteration_id(log_path: str, ckpt_root: Path) -> dict:
    """
    Hotfix should reuse the iteration id that already exists on disk/logs.
    If a run crashed during checkpoint dump, ckpt dir is often created but log line missing.
    We take max(max_logged, max_ckpt_dir) as the iteration to hotfix.
    """
    recs = _read_jsonl(log_path)
    max_logged = max([r.get("iteration_id", -1) for r in recs], default=-1)

    max_ckpt = -1
    if ckpt_root.exists():
        for p in ckpt_root.glob("iter*"):
            m = re.match(r"iter(\d{4})$", p.name)
            if m:
                max_ckpt = max(max_ckpt, int(m.group(1)))

    iter_to_fix = max(max_logged, max_ckpt)
    if iter_to_fix < 0:
        iter_to_fix = 0

    return {"iter_to_fix": int(iter_to_fix), "max_logged": int(max_logged), "max_ckpt": int(max_ckpt)}

detect_info = _detect_hotfix_iteration_id(LOG_PATH, CKPT_ROOT)
ITER_ID = detect_info["iter_to_fix"]
CKPT_DIR = CKPT_ROOT / f"iter{ITER_ID:04d}"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f"[Hotfix] Detected hotfix_for_iteration_id={ITER_ID} (max_logged={detect_info['max_logged']}, max_ckpt={detect_info['max_ckpt']})")
print(f"[Hotfix] Checkpoint dir: {CKPT_DIR}")

# -----------------------
# Feature extraction (matches the proven multi-window direction; key hotfix is: NO lambda in tfidf pipe)
# -----------------------
def build_vitals_features(vitals_list):
    vital_cols = ["hr", "sbp", "dbp", "temp_c", "rr"]

    # Simple clinical lexicon stems (kept lightweight and deterministic)
    pos_stems = ["stable", "improv", "tolerat", "ambulat", "walk", "discharg", "home", "afebrile", "ready", "pain controlled"]
    neg_stems = ["pain", "fever", "hypotens", "tachy", "infection", "sepsis", "bleed", "dyspnea", "sob", "oxygen", "confus", "deliri", "fall", "nausea", "vomit", "dizz", "weak"]

    def count_stems(text, stems):
        t = text.lower()
        return int(sum(t.count(s) for s in stems))

    rows = []
    for item in vitals_list:
        sid = item["stay_id"]
        days_sorted = sorted(item["days"], key=lambda d: d.get("day", 0))  # ok: lambda not stored/pickled
        day_map = {int(d.get("day")): d for d in days_sorted if d.get("day") is not None}

        row = {"stay_id": sid}
        arrs = {k: [] for k in vital_cols}
        notes_all, notes_late = [], []
        note_day10 = ""
        missing_total = 0

        for d in range(1, 11):
            rec = day_map.get(d, {})
            for k in vital_cols:
                v = rec.get(k)
                fv = np.nan if v is None else float(v)
                if v is None:
                    missing_total += 1
                row[f"{k}_d{d}"] = fv
                arrs[k].append(fv)

            note = rec.get("note")
            if note is not None:
                s = str(note)
                notes_all.append(s)
                if d >= 8:
                    notes_late.append(s)
                if d == 10:
                    note_day10 = s

        row["missing_total"] = int(missing_total)
        row["txt_all"] = " ".join(notes_all)
        row["txt_late3"] = " ".join(notes_late)
        row["txt_day10"] = note_day10

        # note lengths (numeric)
        for win in ["all", "late3", "day10"]:
            txt = row[f"txt_{win}"]
            row[f"len_chars_{win}"] = int(len(txt))
            row[f"len_words_{win}"] = int(len(txt.split())) if txt else 0

        # lex features (numeric) on late3 & day10
        for win in ["late3", "day10"]:
            txt = row[f"txt_{win}"]
            pos = count_stems(txt, pos_stems)
            neg = count_stems(txt, neg_stems)
            row[f"lex_pos_{win}"] = int(pos)
            row[f"lex_neg_{win}"] = int(neg)
            row[f"lex_diff_{win}"] = int(pos - neg)

        # event counts + shock index
        hr = np.array(arrs["hr"], dtype=float)
        sbp = np.array(arrs["sbp"], dtype=float)
        temp = np.array(arrs["temp_c"], dtype=float)
        rr = np.array(arrs["rr"], dtype=float)

        row["tachy_hr_cnt"] = int(np.nansum(hr > 100))
        row["hypotension_sbp_cnt"] = int(np.nansum(sbp < 90))
        row["fever_cnt"] = int(np.nansum(temp > 37.5))
        row["tachypnea_cnt"] = int(np.nansum(rr > 20))

        shock = hr / sbp
        row["shock_index_mean"] = float(np.nanmean(shock)) if np.isfinite(np.nanmean(shock)) else np.nan
        row["shock_index_last"] = float(shock[-1]) if not np.isnan(shock[-1]) else np.nan

        # summary stats recipe: mean,std,min,max,last(day10),missing,slope,last3_mean,last3_std,delta_last_first(day10-day1)
        for k in vital_cols:
            arr = np.array(arrs[k], dtype=float)
            non = arr[~np.isnan(arr)]
            if non.size == 0:
                stats = {
                    "mean": np.nan, "std": np.nan, "min": np.nan, "max": np.nan,
                    "last": np.nan, "missing": 10, "slope": 0.0,
                    "last3_mean": np.nan, "last3_std": np.nan,
                    "delta_last_first": np.nan
                }
            else:
                stats = {}
                stats["mean"] = float(np.nanmean(arr))
                stats["std"] = float(np.nanstd(arr))
                stats["min"] = float(np.nanmin(arr))
                stats["max"] = float(np.nanmax(arr))
                stats["last"] = float(arr[-1]) if not np.isnan(arr[-1]) else np.nan
                stats["missing"] = int(np.isnan(arr).sum())

                if non.size >= 2:
                    x_obs = np.array([d for d in range(1, 11) if not np.isnan(arr[d-1])], dtype=float)
                    y_obs = arr[~np.isnan(arr)]
                    x_mean = x_obs.mean()
                    denom = ((x_obs - x_mean) ** 2).sum()
                    slope = float(((x_obs - x_mean) * (y_obs - y_obs.mean())).sum() / denom) if denom != 0 else 0.0
                else:
                    slope = 0.0
                stats["slope"] = float(slope)

                last3 = arr[-3:]
                stats["last3_mean"] = float(np.nanmean(last3)) if np.isfinite(np.nanmean(last3)) else np.nan
                stats["last3_std"] = float(np.nanstd(last3)) if np.isfinite(np.nanstd(last3)) else np.nan

                if not np.isnan(arr[-1]) and not np.isnan(arr[0]):
                    stats["delta_last_first"] = float(arr[-1] - arr[0])
                else:
                    stats["delta_last_first"] = np.nan

            for sn, sv in stats.items():
                row[f"{k}_{sn}"] = sv

        rows.append(row)

    return pd.DataFrame(rows)

# -----------------------
# Load data
# -----------------------
train = pd.read_csv(DATA_STAYS_TRAIN)
test  = pd.read_csv(DATA_STAYS_TEST)
patients = pd.read_csv(DATA_PATIENTS)
with open(DATA_VITALS, "r", encoding="utf-8") as f:
    vitals = json.load(f)

vfeat = build_vitals_features(vitals)

train_df = train.merge(patients, on="patient_id", how="left").merge(vfeat, on="stay_id", how="left")
test_df  = test.merge(patients, on="patient_id", how="left").merge(vfeat, on="stay_id", how="left")

# expected: no overlap between train/test patient_id
overlap = set(train_df["patient_id"]).intersection(set(test_df["patient_id"]))
print(f"[Info] patient_id overlap train vs test: {len(overlap)} (expected 0)")

# derived categorical interaction
for df in [train_df, test_df]:
    df["zip3"] = df["zip3"].astype(str)
    df["reason_unit"] = (df["admission_reason"].astype(str) + "_" + df["unit_type"].astype(str))
    for col in ["txt_all", "txt_late3", "txt_day10"]:
        df[col] = df[col].fillna("").astype(str)

y = train_df["discharge_ready_day11"].astype(int).values
X = train_df.drop(columns=["discharge_ready_day11"])

cat_cols  = ["unit_type", "admission_reason", "sex", "insurance", "zip3", "reason_unit"]
text_cols = ["txt_all", "txt_late3", "txt_day10"]
exclude = ["stay_id", "patient_id"] + cat_cols + text_cols
num_cols = [c for c in X.columns if c not in exclude]

print(f"[Features] num_cols={len(num_cols)}, cat_cols={len(cat_cols)}, text_cols={text_cols}")

# -----------------------
# Pipelines (HOTFIX: NO lambda inside FunctionTransformer)
# -----------------------
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

def tfidf_word(max_features, ngram_range=(1, 2), min_df=1, sublinear_tf=True):
    # IMPORTANT: no lambda here; use np.ravel (pickle-safe)
    return Pipeline([
        ("to_1d", FunctionTransformer(np.ravel, validate=False)),
        ("tfidf", TfidfVectorizer(
            max_features=max_features,
            ngram_range=ngram_range,
            min_df=min_df,
            sublinear_tf=sublinear_tf
        ))
    ])

def tfidf_char(max_features, ngram_range=(3, 5), min_df=1, sublinear_tf=True):
    # IMPORTANT: no lambda here; use np.ravel (pickle-safe)
    return Pipeline([
        ("to_1d", FunctionTransformer(np.ravel, validate=False)),
        ("tfidf", TfidfVectorizer(
            max_features=max_features,
            analyzer="char",
            ngram_range=ngram_range,
            min_df=min_df,
            sublinear_tf=sublinear_tf
        ))
    ])

# Logistic (text+structured) — aligns with the “iter6 direction”
num_pipe_logi = Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", MaxAbsScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                     ("ohe", make_ohe())])

preprocess_logi = ColumnTransformer(
    transformers=[
        ("num", num_pipe_logi, num_cols),
        ("cat", cat_pipe, cat_cols),
        ("w_all",  tfidf_word(max_features=12000, ngram_range=(1, 2), min_df=2), "txt_all"),
        ("w_late", tfidf_word(max_features=8000,  ngram_range=(1, 2), min_df=1), "txt_late3"),
        ("w_d10",  tfidf_word(max_features=6000,  ngram_range=(1, 2), min_df=1), "txt_day10"),
        ("c_late", tfidf_char(max_features=6000,  ngram_range=(3, 5), min_df=1), "txt_late3"),
        ("c_d10",  tfidf_char(max_features=4000,  ngram_range=(3, 5), min_df=1), "txt_day10"),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

pipe_logi = Pipeline([
    ("preprocess", preprocess_logi),
    ("clf", LogisticRegression(
        solver="saga",
        penalty="elasticnet",
        l1_ratio=0.05,
        C=0.6,
        class_weight="balanced",
        max_iter=20000,
        random_state=SEED,
        n_jobs=1
    ))
])

# XGB (structured-only)
num_pipe_xgb = Pipeline([("imputer", SimpleImputer(strategy="median"))])
preprocess_xgb = ColumnTransformer(
    transformers=[
        ("num", num_pipe_xgb, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

has_xgb = True
xgb_params_or_fallback = {"has_xgb": None, "params": None, "fallback": None, "import_error": None}
try:
    import xgboost as xgb
    xgb_clf = xgb.XGBClassifier(
        n_estimators=450,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=2.0,
        reg_lambda=1.0,
        reg_alpha=0.0,
        gamma=0.0,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=SEED,
        n_jobs=1,
        tree_method="hist",
        verbosity=0
    )
    xgb_params_or_fallback = {"has_xgb": True, "params": xgb_clf.get_params(), "fallback": None, "import_error": None}
except Exception as e:
    has_xgb = False
    from sklearn.ensemble import HistGradientBoostingClassifier
    xgb_clf = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.06, max_iter=300, random_state=SEED)
    xgb_params_or_fallback = {"has_xgb": False, "params": None, "fallback": str(xgb_clf), "import_error": repr(e)}

pipe_xgb = Pipeline([("preprocess", preprocess_xgb), ("clf", xgb_clf)])

# -----------------------
# CV + OOF threshold/ensemble search
# -----------------------
print("[CV] Running 5-fold StratifiedKFold...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof_logi = np.zeros(len(train_df), dtype=float)
oof_xgb  = np.zeros(len(train_df), dtype=float)
fold_id  = np.full(len(train_df), -1, dtype=int)

fold_f1_logi_05, fold_f1_xgb_05 = [], []
for f, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    fold_id[va_idx] = f

    X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
    X_va, y_va = X.iloc[va_idx], y[va_idx]

    pipe_logi.fit(X_tr, y_tr)
    p1 = pipe_logi.predict_proba(X_va)[:, 1]
    oof_logi[va_idx] = p1
    fold_f1_logi_05.append(f1_score(y_va, (p1 >= 0.5).astype(int), average="macro"))

    pipe_xgb.fit(X_tr, y_tr)
    p2 = pipe_xgb.predict_proba(X_va)[:, 1]
    oof_xgb[va_idx] = p2
    fold_f1_xgb_05.append(f1_score(y_va, (p2 >= 0.5).astype(int), average="macro"))

print("\n[Results @0.5]")
print(f"  Logistic (0.5) fold macro-F1: {[round(x,4) for x in fold_f1_logi_05]} | mean={np.mean(fold_f1_logi_05):.6f}")
print(f"  XGB/Structured (0.5) fold macro-F1: {[round(x,4) for x in fold_f1_xgb_05]} | mean={np.mean(fold_f1_xgb_05):.6f}")

def best_threshold(y_true, prob, thresholds):
    best_f1, best_t = -1.0, 0.5
    for t in thresholds:
        pred = (prob >= t).astype(int)
        f1 = f1_score(y_true, pred, average="macro")
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_f1), float(best_t)

t_grid = np.round(np.arange(0.05, 0.951, 0.01), 2)

logi_best_f1, logi_best_t = best_threshold(y, oof_logi, t_grid)
xgb_best_f1,  xgb_best_t  = best_threshold(y, oof_xgb,  t_grid)

print("\n[Best threshold (OOF pooled)]")
print(f"  Logistic: best_f1={logi_best_f1:.6f} @ t={logi_best_t:.3f}")
print(f"  XGB/Structured: best_f1={xgb_best_f1:.6f} @ t={xgb_best_t:.3f}")

# Ensemble weight + threshold grid search on pooled OOF (same direction as iter6)
w_grid = np.round(np.arange(0.0, 1.001, 0.05), 2)
best_ens = {"f1": -1.0, "w_logi": 0.5, "t": 0.5}
for w in w_grid:
    prob = w * oof_logi + (1.0 - w) * oof_xgb
    f1, t = best_threshold(y, prob, t_grid)
    if f1 > best_ens["f1"]:
        best_ens = {"f1": float(f1), "w_logi": float(w), "t": float(t)}

ens_prob_oof = best_ens["w_logi"] * oof_logi + (1.0 - best_ens["w_logi"]) * oof_xgb
ens_pred_oof = (ens_prob_oof >= best_ens["t"]).astype(int)
cm = confusion_matrix(y, ens_pred_oof)

# per-fold f1 for ensemble @ best
ens_fold_f1 = []
for f in range(5):
    idx = np.where(fold_id == f)[0]
    ens_fold_f1.append(f1_score(y[idx], ens_pred_oof[idx], average="macro"))

print("\n[Ensemble optimized on OOF pooled]")
print(f"  Best ensemble OOF macro-F1={best_ens['f1']:.6f} @ weight_logi={best_ens['w_logi']:.2f}, threshold={best_ens['t']:.2f}")
print(f"  Ensemble per-fold macro-F1 @ best: {[round(x,4) for x in ens_fold_f1]} | mean={np.mean(ens_fold_f1):.6f}")
print(f"  Confusion matrix (OOF @ best):\n{cm}")

# -----------------------
# Fit full + predict test + write submission (always write submission even if checkpointing fails)
# -----------------------
pipe_logi.fit(X, y)
pipe_xgb.fit(X, y)

X_test = test_df.copy()
p_test_logi = pipe_logi.predict_proba(X_test)[:, 1]
p_test_xgb  = pipe_xgb.predict_proba(X_test)[:, 1]
p_test_ens  = best_ens["w_logi"] * p_test_logi + (1.0 - best_ens["w_logi"]) * p_test_xgb
pred_test   = (p_test_ens >= best_ens["t"]).astype(int)

sub = pd.DataFrame({"stay_id": test_df["stay_id"].astype(int), "discharge_ready_day11": pred_test.astype(int)})
sub.to_csv(SUBMISSION_PATH, index=False)
print(f"\n[IO] Wrote submission: {SUBMISSION_PATH} | shape={sub.shape}")

# -----------------------
# Checkpoint saving (HOTFIX: safe_dump + no lambda in pipeline)
# -----------------------
def safe_dump(obj, path_joblib: Path):
    """
    Guard: never crash the run on serialization.
    1) try joblib (fast/standard)
    2) fallback to cloudpickle for unpicklable objects (e.g., lambdas) and log the error
    """
    try:
        joblib.dump(obj, str(path_joblib))
        return {"dump_backend": "joblib", "dump_error": None, "model_bundle_path": str(path_joblib)}
    except Exception as e:
        dump_error = repr(e)
        alt_path = Path(str(path_joblib).replace(".joblib", ".cloudpickle"))
        try:
            from joblib.externals import cloudpickle
        except Exception:
            import cloudpickle  # type: ignore
        with open(alt_path, "wb") as f:
            cloudpickle.dump(obj, f)
        return {"dump_backend": "cloudpickle", "dump_error": dump_error, "model_bundle_path": str(alt_path)}

config = {
    "version": VX,
    "random_seed": SEED,
    "iteration_id": ITER_ID,
    "model_direction": "logi(text+structured) + xgb(structured), OOF-tuned weight+threshold",
    "best_ensemble": best_ens,
    "xgb_backend": xgb_params_or_fallback,
}

schema = {
    "categorical_cols": cat_cols,
    "numeric_cols": num_cols,
    "text_cols": text_cols,
    "text_vectorizers": {
        "word": {"analyzer": "word", "ngram_range": [1, 2]},
        "char": {"analyzer": "char", "ngram_range": [3, 5]},
    }
}

metrics = {
    "macro_f1_mean_at_0.5": {
        "logi": float(np.mean(fold_f1_logi_05)),
        "xgb": float(np.mean(fold_f1_xgb_05)),
    },
    "macro_f1_per_fold_at_0.5": {
        "logi": [float(x) for x in fold_f1_logi_05],
        "xgb": [float(x) for x in fold_f1_xgb_05],
    },
    "best_oof": {
        "logi_best_f1": logi_best_f1, "logi_best_threshold": logi_best_t,
        "xgb_best_f1": xgb_best_f1, "xgb_best_threshold": xgb_best_t,
        "ensemble_best_f1": best_ens["f1"], "ensemble_weight_logi": best_ens["w_logi"], "ensemble_threshold": best_ens["t"],
    },
    "ensemble_per_fold_at_best": [float(x) for x in ens_fold_f1],
    "confusion_matrix_oof_at_best": cm.tolist(),
}

# Write config + schema + metrics
with open(CKPT_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
with open(CKPT_DIR / "schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)
with open(CKPT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

# Save OOF probs for debugging
np.save(ART_ROOT / f"iter{ITER_ID:04d}_oof_prob_logi.npy", oof_logi)
np.save(ART_ROOT / f"iter{ITER_ID:04d}_oof_prob_xgb.npy",  oof_xgb)
np.save(ART_ROOT / f"iter{ITER_ID:04d}_oof_prob_ens.npy",  ens_prob_oof)
np.save(ART_ROOT / f"iter{ITER_ID:04d}_fold_id.npy", fold_id)
np.save(ART_ROOT / f"iter{ITER_ID:04d}_test_prob_ens.npy", p_test_ens)

bundle = {
    "pipe_logi": pipe_logi,
    "pipe_xgb": pipe_xgb,
    "config": config,
    "schema": schema,
    "metrics": metrics,
}
dump_info = safe_dump(bundle, CKPT_DIR / "model_bundle.joblib")
print(f"[CKPT] Saved model bundle via {dump_info['dump_backend']} to: {dump_info['model_bundle_path']}")
if dump_info["dump_error"] is not None:
    print(f"[CKPT] joblib failed, fallback used. Error: {dump_info['dump_error']}")

# -----------------------
# Append HOTFIX log entry (required fields)
# -----------------------
hotfix_entry = {
    "version": VX,
    "iteration_id": ITER_ID,  # do NOT increment; same as the iteration being hotfixed
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "phase": "Hotfix",
    "hotfix": True,
    "hotfix_for_iteration_id": ITER_ID,
    "error_signature": "PicklingError: tfidf_pipe.<locals>.<lambda>",
    "fix_summary": "Remove lambda from text pipeline (use FunctionTransformer(np.ravel) + pre-fill text as str); add safe_dump (joblib->cloudpickle fallback) so checkpointing never crashes.",
    "data_paths_used": {
        "stays_train": DATA_STAYS_TRAIN,
        "stays_test": DATA_STAYS_TEST,
        "patients": DATA_PATIENTS,
        "vitals_timeseries": DATA_VITALS,
    },
    "leakage_checks_performed": [
        "No label joins beyond stays_train target; features from patients demographics + vitals/day notes days1-10 only.",
        f"patient_id overlap train vs test = {len(overlap)} (expected 0).",
    ],
    "feature_summary": {
        "n_train": int(train_df.shape[0]),
        "n_test": int(test_df.shape[0]),
        "num_cols": int(len(num_cols)),
        "cat_cols": int(len(cat_cols)),
        "text_cols": text_cols,
    },
    "models_attempted": [
        {"name": "LogisticRegression(saga, elasticnet) + [num+OHE+TFIDF(word+char)]",
         "params": {"solver": "saga", "penalty": "elasticnet", "l1_ratio": 0.05, "C": 0.6, "class_weight": "balanced", "max_iter": 20000, "random_state": SEED}},
        {"name": "XGBClassifier(structured-only) + [num+OHE]",
         "params": xgb_params_or_fallback},
    ],
    "selected_model": f"weighted_ensemble(proba) @ w_logi={best_ens['w_logi']:.2f}, t={best_ens['t']:.2f}",
    "validation_protocol": {"type": "StratifiedKFold", "n_splits": 5, "shuffle": True, "random_state": SEED},
    "metrics": metrics,
    "checkpoint": {"ckpt_dir": str(CKPT_DIR), **dump_info},
}

with open(LOG_PATH, "a", encoding="utf-8") as f:
    f.write(json.dumps(hotfix_entry, ensure_ascii=False) + "\n")

print(f"[LOG] Appended hotfix entry to {LOG_PATH} (iteration_id={ITER_ID})")

[Hotfix] Detected hotfix_for_iteration_id=10 (max_logged=10, max_ckpt=10)
[Hotfix] Checkpoint dir: checkpoints\clai_ch3_v1\iter0010
[Info] patient_id overlap train vs test: 0 (expected 0)
[Features] num_cols=120, cat_cols=6, text_cols=['txt_all', 'txt_late3', 'txt_day10']
[CV] Running 5-fold StratifiedKFold...

[Results @0.5]
  Logistic (0.5) fold macro-F1: [0.7674, 0.7194, 0.7578, 0.7097, 0.7438] | mean=0.739636
  XGB/Structured (0.5) fold macro-F1: [0.8029, 0.7877, 0.7615, 0.7817, 0.7814] | mean=0.783040

[Best threshold (OOF pooled)]
  Logistic: best_f1=0.752782 @ t=0.440
  XGB/Structured: best_f1=0.788923 @ t=0.690

[Ensemble optimized on OOF pooled]
  Best ensemble OOF macro-F1=0.795684 @ weight_logi=0.25, threshold=0.64
  Ensemble per-fold macro-F1 @ best: [0.8118, 0.7998, 0.7724, 0.7946, 0.7981] | mean=0.795338
  Confusion matrix (OOF @ best):
[[247  97]
 [ 86 570]]

[IO] Wrote submission: clai_ch3_v1_submission.csv | shape=(1000, 2)
[CKPT] Saved model bundle via joblib to: chec

# Iteration 11
- 0.7757

In [40]:
import os, json, math, warnings, joblib, numpy as np, pandas as pd
from datetime import datetime, timezone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone

warnings.filterwarnings("ignore")

# =========================
# Config
# =========================
VERSION = "v1"
TEAM_TAG = "clai"
SEED = 42
N_SPLITS = 5

SUBMISSION_NAME = f"{TEAM_TAG}_ch3_{VERSION}_submission.csv"
ITER_LOG_NAME = f"{TEAM_TAG}_ch3_{VERSION}_iteration_detail.jsonl"
CKPT_ROOT = os.path.join("checkpoints", f"{TEAM_TAG}_ch3_{VERSION}")
ART_ROOT  = os.path.join("artifacts",  f"{TEAM_TAG}_ch3_{VERSION}")
os.makedirs(CKPT_ROOT, exist_ok=True)
os.makedirs(ART_ROOT, exist_ok=True)

def resolve_path(fname: str) -> str:
    """Prefer local project path; fallback to /mnt/data for sandbox."""
    if os.path.exists(fname):
        return fname
    alt = os.path.join("/mnt/data", fname)
    if os.path.exists(alt):
        return alt
    raise FileNotFoundError(f"Missing required file: {fname}")

# =========================
# Iteration ID (append-only; hotfix doesn't increment)
# =========================
log_path = ITER_LOG_NAME if os.path.exists(ITER_LOG_NAME) else (os.path.join("/mnt/data", ITER_LOG_NAME) if os.path.exists(os.path.join("/mnt/data", ITER_LOG_NAME)) else ITER_LOG_NAME)

prev_records = []
if os.path.exists(log_path):
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                prev_records.append(json.loads(line))
            except Exception:
                pass

prev_max_iter = max([r.get("iteration_id", -1) for r in prev_records], default=-1)
iteration_id = prev_max_iter + 1

# HM message / real score (from latest instruction)
hm_message = ("HM: latest submission regressed to real Macro-F1=0.7782 and requested rollback to Iter6 base (real 0.8244) "
              "then improve while avoiding known regression patterns; require local OOF sanity check before submitting.")
real_score = 0.7782

# =========================
# Load data
# =========================
stays_train = pd.read_csv(resolve_path("stays_train.csv"))
stays_test  = pd.read_csv(resolve_path("stays_test.csv"))
patients    = pd.read_csv(resolve_path("patients.csv"))
with open(resolve_path("vitals_timeseries.json"), "r", encoding="utf-8") as f:
    vitals_ts = json.load(f)

TARGET = "discharge_ready_day11"
IDCOL  = "stay_id"

# =========================
# Leakage sanity: patient overlap train vs test should be 0
# =========================
train_pat = set(stays_train["patient_id"].tolist())
test_pat  = set(stays_test["patient_id"].tolist())
overlap_pat = len(train_pat.intersection(test_pat))
print(f"[Info] patient_id overlap train vs test: {overlap_pat} (expected 0)")

# =========================
# Feature engineering (Iter6-style + robust additions)
# =========================
vital_keys = ["hr","sbp","dbp","temp_c","rr"]

POS_TERMS = [
    "stable","improving","improved","ready","discharge","home","ambulat","walk",
    "tolerat","po","diet","afebrile","no fever","no pain","controlled","resolved",
    "wean","off oxygen","room air","dc","d/c","follow up","outpatient"
]
NEG_TERMS = [
    "icu","intubat","vent","pressors","sepsis","shock","unstable","worsen","worsened",
    "fever","febrile","tachy","hypotens","hypoxia","respiratory failure","delir",
    "confus","bleed","infection","acute","cpap","bipap","high flow","transfer to icu"
]

def safe_float(x):
    try:
        if x is None:
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def linreg_slope(arr_1d):
    y = np.array(arr_1d, dtype=float)
    m = ~np.isnan(y)
    if m.sum() < 2:
        return np.nan
    x = np.arange(1, len(y)+1, dtype=float)[m]
    yy = y[m]
    xm, ym = x.mean(), yy.mean()
    denom = ((x - xm) ** 2).sum()
    if denom == 0:
        return 0.0
    return ((x - xm) * (yy - ym)).sum() / denom

def count_terms(text, terms):
    if not isinstance(text, str):
        return 0
    t = text.lower()
    c = 0
    for term in terms:
        if term in t:
            c += t.count(term)
    return int(c)

def build_vitals_features(vitals_list):
    rows = []
    for rec in vitals_list:
        stay_id = rec["stay_id"]
        days = sorted(rec["days"], key=lambda d: d["day"])
        per_key = {k: [safe_float(day.get(k)) for day in days] for k in vital_keys}
        notes = [str(day.get("note","")) for day in days]
        txt_all   = " ".join(notes)
        txt_late3 = " ".join(notes[-3:])
        txt_day10 = notes[-1] if notes else ""

        feats = {}
        # raw per-day + summary/trend
        for k, arr in per_key.items():
            arr = np.array(arr, dtype=float)
            for i, val in enumerate(arr, start=1):
                feats[f"{k}_d{i}"] = val
            feats[f"{k}_mean"] = float(np.nanmean(arr))
            feats[f"{k}_std"]  = float(np.nanstd(arr))
            feats[f"{k}_min"]  = float(np.nanmin(arr))
            feats[f"{k}_max"]  = float(np.nanmax(arr))
            feats[f"{k}_first"]= float(arr[0]) if len(arr) else np.nan
            feats[f"{k}_last"] = float(arr[-1]) if len(arr) else np.nan
            feats[f"{k}_missing"] = int(np.isnan(arr).sum())
            feats[f"{k}_slope"] = float(linreg_slope(arr))
            last3 = arr[-3:] if len(arr) >= 3 else arr
            feats[f"{k}_last3_mean"] = float(np.nanmean(last3))
            feats[f"{k}_last3_std"]  = float(np.nanstd(last3))
            if not np.isnan(feats[f"{k}_last"]) and not np.isnan(feats[f"{k}_first"]):
                feats[f"{k}_delta_last_first"] = float(feats[f"{k}_last"] - feats[f"{k}_first"])
            else:
                feats[f"{k}_delta_last_first"] = np.nan

        # abnormal day counts (simple clinical-ish thresholds)
        hr  = np.array(per_key["hr"], dtype=float)
        sbp = np.array(per_key["sbp"], dtype=float)
        dbp = np.array(per_key["dbp"], dtype=float)
        tmp = np.array(per_key["temp_c"], dtype=float)
        rr  = np.array(per_key["rr"], dtype=float)

        feats["abn_hr_gt100"]   = int(np.nansum(hr  > 100))
        feats["abn_sbp_lt90"]   = int(np.nansum(sbp <  90))
        feats["abn_temp_gt38"]  = int(np.nansum(tmp >  38))
        feats["abn_rr_gt20"]    = int(np.nansum(rr  >  20))
        if len(hr) >= 3:
            feats["abn_any_last3"] = int(((hr[-3:] > 100) | (sbp[-3:] < 90) | (tmp[-3:] > 38) | (rr[-3:] > 20)).any())
        else:
            feats["abn_any_last3"] = 0

        # pulse pressure
        pp = sbp - dbp
        for i, val in enumerate(pp, start=1):
            feats[f"pp_d{i}"] = val
        feats["pp_mean"]  = float(np.nanmean(pp))
        feats["pp_last"]  = float(pp[-1]) if len(pp) else np.nan
        feats["pp_slope"] = float(linreg_slope(pp))

        # text stats + lexicon counts (Iter6 signal)
        feats["txt_all_len"]   = int(len(txt_all))
        feats["txt_late3_len"] = int(len(txt_late3))
        feats["txt_day10_len"] = int(len(txt_day10))

        feats["lex_pos"]  = count_terms(txt_late3, POS_TERMS)
        feats["lex_neg"]  = count_terms(txt_late3, NEG_TERMS)
        feats["lex_diff"] = int(feats["lex_pos"] - feats["lex_neg"])

        feats["lex_pos_day10"]  = count_terms(txt_day10, POS_TERMS)
        feats["lex_neg_day10"]  = count_terms(txt_day10, NEG_TERMS)
        feats["lex_diff_day10"] = int(feats["lex_pos_day10"] - feats["lex_neg_day10"])

        row = {"stay_id": stay_id, "txt_all": txt_all, "txt_late3": txt_late3, "txt_day10": txt_day10}
        row.update(feats)
        rows.append(row)
    return pd.DataFrame(rows)

vitals_df = build_vitals_features(vitals_ts)

def make_dataset(stays_df):
    df = stays_df.merge(patients, on="patient_id", how="left")
    df = df.merge(vitals_df, on="stay_id", how="left")
    df["reason_unit"] = df["admission_reason"].astype(str) + "__" + df["unit_type"].astype(str)
    # duplicate fields for char TF-IDF (avoid custom lambdas for pickling)
    df["txt_day10_char"] = df["txt_day10"].fillna("")
    df["txt_late3_char"] = df["txt_late3"].fillna("")
    # SVD branch uses combined "late + day10"
    df["txt_svd"] = (df["txt_late3"].fillna("") + " " + df["txt_day10"].fillna("")).str.strip()
    return df

train_df = make_dataset(stays_train)
test_df  = make_dataset(stays_test)

# Columns
cat_cols  = ["unit_type","admission_reason","sex","insurance","zip3","reason_unit"]
text_cols = ["txt_all","txt_late3","txt_day10","txt_day10_char","txt_late3_char","txt_svd"]

exclude = set([IDCOL, "patient_id", TARGET] + text_cols)
num_cols = [c for c in train_df.columns if (c not in exclude and c not in cat_cols)]

print(f"[Info] n_train={train_df.shape[0]} n_test={test_df.shape[0]}")
print(f"[Info] n_num_cols={len(num_cols)} n_cat_cols={len(cat_cols)} text_cols(main)={['txt_all','txt_late3','txt_day10']}")

# =========================
# Models (Iter6 base + NEW XGB+SVD(text))
# =========================
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

# Logistic: num + OHE + word/char TF-IDF
num_pipe_logi = Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler(with_mean=False))])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                     ("ohe", make_ohe())])

tfidf_all = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=60000, sublinear_tf=True, dtype=np.float32)
tfidf_late= TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=40000, sublinear_tf=True, dtype=np.float32)
char_day10= TfidfVectorizer(analyzer="char", ngram_range=(3,5), min_df=2, max_features=50000, dtype=np.float32)
char_late = TfidfVectorizer(analyzer="char", ngram_range=(3,5), min_df=2, max_features=30000, dtype=np.float32)

pre_logi = ColumnTransformer(
    transformers=[
        ("num", num_pipe_logi, num_cols),
        ("cat", cat_pipe, cat_cols),
        ("tfidf_all", tfidf_all, "txt_all"),
        ("tfidf_late3", tfidf_late, "txt_late3"),
        ("char_day10", char_day10, "txt_day10_char"),
        ("char_late3", char_late, "txt_late3_char"),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

clf_logi = LogisticRegression(
    solver="saga",
    penalty="elasticnet",
    l1_ratio=0.05,
    C=0.6,
    class_weight="balanced",
    max_iter=20000,
    random_state=SEED,
    n_jobs=1
)

pipe_logi = Pipeline([("preprocess", pre_logi), ("clf", clf_logi)])

# XGB structured-only: num + OHE
try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False
    xgb = None

pre_xgb = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", make_ohe())]), cat_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

xgb_params = dict(
    n_estimators=450,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=SEED,
    n_jobs=1,
    verbosity=0
)

if HAS_XGB:
    pipe_xgb = Pipeline([("preprocess", pre_xgb), ("clf", xgb.XGBClassifier(**xgb_params))])
else:
    raise RuntimeError("xgboost not available; install xgboost to run this pipeline.")

# NEW: XGB + SVD text branch (structured + low-dim semantic text)
text_svd_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=30000, sublinear_tf=True, dtype=np.float32)),
    ("svd", TruncatedSVD(n_components=64, random_state=SEED)),
    ("scaler", StandardScaler(with_mean=True))
])

pre_xgb_mix = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", make_ohe())]), cat_cols),
        ("svd_text", text_svd_pipe, "txt_svd"),
    ],
    remainder="drop",
    sparse_threshold=1.0
)

# keep params close to earlier explored xgb+svd settings, but allow ensemble to ignore it if not helpful
neg = int((train_df[TARGET] == 0).sum())
pos = int((train_df[TARGET] == 1).sum())
scale_pos_weight = (neg / max(pos, 1))  # <1 here (pos majority) -> emphasize negative class

xgb_mix_params = dict(
    n_estimators=800,
    max_depth=4,
    learning_rate=0.04,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=2.0,
    reg_alpha=0.0,
    reg_lambda=1.2,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=SEED,
    n_jobs=1,
    verbosity=0,
    scale_pos_weight=scale_pos_weight
)

pipe_xgb_mix = Pipeline([("preprocess", pre_xgb_mix), ("clf", xgb.XGBClassifier(**xgb_mix_params))])

# =========================
# CV + OOF probs
# =========================
X = train_df.drop(columns=[TARGET])
y = train_df[TARGET].astype(int).values

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_logi = np.zeros(len(X), dtype=np.float32)
oof_xgb  = np.zeros(len(X), dtype=np.float32)
oof_mix  = np.zeros(len(X), dtype=np.float32)
fold_id  = np.zeros(len(X), dtype=int)

print(f"[CV] Running {N_SPLITS}-fold StratifiedKFold...")

for f, (tr, va) in enumerate(skf.split(X, y)):
    fold_id[va] = f
    X_tr, y_tr = X.iloc[tr], y[tr]
    X_va       = X.iloc[va]

    m_logi = clone(pipe_logi)
    m_xgb  = clone(pipe_xgb)
    m_mix  = clone(pipe_xgb_mix)

    m_logi.fit(X_tr, y_tr)
    m_xgb.fit(X_tr, y_tr)
    m_mix.fit(X_tr, y_tr)

    oof_logi[va] = m_logi.predict_proba(X_va)[:, 1].astype(np.float32)
    oof_xgb[va]  = m_xgb.predict_proba(X_va)[:, 1].astype(np.float32)
    oof_mix[va]  = m_mix.predict_proba(X_va)[:, 1].astype(np.float32)

# helper: evaluate per-fold macro-F1 at threshold t
def per_fold_f1(oof_p, thr):
    out = []
    for f in range(N_SPLITS):
        idx = np.where(fold_id == f)[0]
        pred = (oof_p[idx] >= thr).astype(int)
        out.append(float(f1_score(y[idx], pred, average="macro")))
    return out, float(np.mean(out))

# helper: best threshold on pooled OOF
def best_thr_on_oof(oof_p, thr_grid):
    best = (-1.0, 0.5)
    for t in thr_grid:
        pred = (oof_p >= t).astype(int)
        s = float(f1_score(y, pred, average="macro"))
        if s > best[0]:
            best = (s, float(t))
    return best

thr_grid = np.round(np.arange(0.05, 0.951, 0.01), 3)
# anchor old best threshold explicitly
thr_grid = np.unique(np.concatenate([thr_grid, np.array([0.60], dtype=float)]))

# Report @0.5
logi_pf_05, logi_m_05 = per_fold_f1(oof_logi, 0.5)
xgb_pf_05,  xgb_m_05  = per_fold_f1(oof_xgb,  0.5)
mix_pf_05,  mix_m_05  = per_fold_f1(oof_mix,  0.5)

print("\n[Results @0.5]")
print(f"  Logistic (0.5) fold macro-F1: {[round(x,4) for x in logi_pf_05]} | mean={logi_m_05:.6f}")
print(f"  XGB/Structured (0.5) fold macro-F1: {[round(x,4) for x in xgb_pf_05]} | mean={xgb_m_05:.6f}")
print(f"  XGB/Struct+SVDText (0.5) fold macro-F1: {[round(x,4) for x in mix_pf_05]} | mean={mix_m_05:.6f}")

# Best thresholds per model (pooled)
logi_best_f1, logi_best_t = best_thr_on_oof(oof_logi, thr_grid)
xgb_best_f1,  xgb_best_t  = best_thr_on_oof(oof_xgb,  thr_grid)
mix_best_f1,  mix_best_t  = best_thr_on_oof(oof_mix,  thr_grid)

print("\n[Best threshold (OOF pooled)]")
print(f"  Logistic: best_f1={logi_best_f1:.6f} @ t={logi_best_t:.3f}")
print(f"  XGB/Structured: best_f1={xgb_best_f1:.6f} @ t={xgb_best_t:.3f}")
print(f"  XGB/Struct+SVDText: best_f1={mix_best_f1:.6f} @ t={mix_best_t:.3f}")

# =========================
# Ensemble search (baseline 2-model + new 3-model) with regression guard
# =========================
w_grid = np.round(np.arange(0.0, 1.0001, 0.05), 2)
w_grid = np.unique(np.concatenate([w_grid, np.array([0.35], dtype=float)]))  # anchor Iter6 best weight

def search_2model(oof_a, oof_b, w_grid, thr_grid):
    best = (-1.0, None, None)
    for w in w_grid:
        p = w * oof_a + (1.0 - w) * oof_b
        for t in thr_grid:
            pred = (p >= t).astype(int)
            s = float(f1_score(y, pred, average="macro"))
            if s > best[0]:
                best = (s, float(w), float(t))
    return best

def search_3model(oof1, oof2, oof3, w_grid, thr_grid):
    # weights: w1 (logi), w3 (mix), w2 = 1 - w1 - w3 (struct)
    best = (-1.0, None, None, None)
    for w1 in w_grid:
        for w3 in w_grid:
            if w1 + w3 > 1.0:
                continue
            w2 = 1.0 - w1 - w3
            # include w3=0 path to preserve baseline
            p = w1 * oof1 + w2 * oof2 + w3 * oof3
            for t in thr_grid:
                pred = (p >= t).astype(int)
                s = float(f1_score(y, pred, average="macro"))
                if s > best[0]:
                    best = (s, float(w1), float(w2), float(w3), float(t))
    return best

base2_f1, base2_wlogi, base2_t = search_2model(oof_logi, oof_xgb, w_grid, thr_grid)
best3 = search_3model(oof_logi, oof_xgb, oof_mix, w_grid, thr_grid)
ens3_f1, w1, w2, w3, ens3_t = best3

# pick better ensemble (guard)
use_three = ens3_f1 >= base2_f1 - 1e-12
if use_three:
    ens_name = "Ensemble3(logi + xgb_struct + xgb_svdtext)"
    ens_best_f1 = ens3_f1
    ens_weights = {"w_logi": w1, "w_xgb_struct": w2, "w_xgb_svdtext": w3}
    ens_thr = ens3_t
    ens_oof = (w1 * oof_logi + w2 * oof_xgb + w3 * oof_mix).astype(np.float32)
else:
    ens_name = "Ensemble2(logi + xgb_struct)"
    ens_best_f1 = base2_f1
    ens_weights = {"w_logi": base2_wlogi, "w_xgb_struct": 1.0 - base2_wlogi, "w_xgb_svdtext": 0.0}
    ens_thr = base2_t
    ens_oof = (base2_wlogi * oof_logi + (1.0 - base2_wlogi) * oof_xgb).astype(np.float32)

ens_pf, ens_mean = per_fold_f1(ens_oof, ens_thr)
cm = confusion_matrix(y, (ens_oof >= ens_thr).astype(int)).tolist()

print("\n[Ensemble optimized on OOF pooled]")
if use_three:
    print(f"  Best 3-model OOF macro-F1={ens3_f1:.6f} @ w_logi={w1:.2f}, w_xgb_struct={w2:.2f}, w_xgb_svd={w3:.2f}, threshold={ens3_t:.3f}")
print(f"  Best baseline 2-model OOF macro-F1={base2_f1:.6f} @ w_logi={base2_wlogi:.2f}, threshold={base2_t:.3f}")
print(f"  SELECTED: {ens_name} | pooled-best={ens_best_f1:.6f} | threshold={ens_thr:.3f} | weights={ens_weights}")
print(f"  Ensemble per-fold macro-F1 @ best: {[round(x,4) for x in ens_pf]} | mean={ens_mean:.6f}")
print(f"  Confusion matrix (OOF @ best):\n{np.array(cm)}")

# =========================
# Fit final models on full train, predict test
# =========================
X_test = test_df.copy()

final_logi = clone(pipe_logi).fit(X, y)
final_xgb  = clone(pipe_xgb).fit(X, y)
final_mix  = clone(pipe_xgb_mix).fit(X, y)

test_logi = final_logi.predict_proba(X_test)[:, 1].astype(np.float32)
test_xgb  = final_xgb.predict_proba(X_test)[:, 1].astype(np.float32)
test_mix  = final_mix.predict_proba(X_test)[:, 1].astype(np.float32)

ens_test = (ens_weights["w_logi"] * test_logi +
            ens_weights["w_xgb_struct"] * test_xgb +
            ens_weights["w_xgb_svdtext"] * test_mix).astype(np.float32)

test_pred = (ens_test >= ens_thr).astype(int)

sub = pd.DataFrame({IDCOL: test_df[IDCOL].values, TARGET: test_pred.astype(int)})
sub.to_csv(SUBMISSION_NAME, index=False)
print(f"\n[Saved] submission -> {SUBMISSION_NAME} | positive_rate={test_pred.mean():.4f}")

# =========================
# Save artifacts + checkpoint
# =========================
ckpt_dir = os.path.join(CKPT_ROOT, f"iter{iteration_id:04d}")
os.makedirs(ckpt_dir, exist_ok=True)

np.save(os.path.join(ART_ROOT, f"iter{iteration_id:04d}_oof_prob_logi.npy"), oof_logi)
np.save(os.path.join(ART_ROOT, f"iter{iteration_id:04d}_oof_prob_xgb.npy"),  oof_xgb)
np.save(os.path.join(ART_ROOT, f"iter{iteration_id:04d}_oof_prob_mix.npy"),  oof_mix)
np.save(os.path.join(ART_ROOT, f"iter{iteration_id:04d}_oof_prob_ens.npy"),  ens_oof)
np.save(os.path.join(ART_ROOT, f"iter{iteration_id:04d}_test_prob_ens.npy"), ens_test)
np.save(os.path.join(ART_ROOT, f"iter{iteration_id:04d}_fold_id.npy"),       fold_id)

cand_path = os.path.join(ART_ROOT, f"iter{iteration_id:04d}_candidate_submission.csv")
sub.to_csv(cand_path, index=False)

config = {
    "version": VERSION,
    "iteration_id": iteration_id,
    "seed": SEED,
    "cv": {"n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "models": {
        "logistic": {"solver":"saga","penalty":"elasticnet","l1_ratio":0.05,"C":0.6,"class_weight":"balanced","max_iter":20000},
        "xgb_struct": xgb_params,
        "xgb_svdtext": xgb_mix_params,
    },
    "ensemble_selected": {"name": ens_name, "weights": ens_weights, "threshold": float(ens_thr)},
    "base_reference": {"intended_iter": 6, "known_iter6_anchor": {"w_logi":0.35,"threshold":0.60, "oof_pooled_best":0.8239746523499384}},
}

schema = {
    "categorical_cols": cat_cols,
    "numeric_cols": num_cols,
    "text_cols": ["txt_all","txt_late3","txt_day10"],
    "text_cols_aux": ["txt_day10_char","txt_late3_char","txt_svd"],
}

metrics = {
    "macro_f1_per_fold_at_0.5": {"logi": logi_pf_05, "xgb": xgb_pf_05, "mix": mix_pf_05},
    "macro_f1_mean_at_0.5": {"logi": logi_m_05, "xgb": xgb_m_05, "mix": mix_m_05},
    "best_threshold_oof_pooled": {
        "logi": {"best_f1": logi_best_f1, "best_t": logi_best_t},
        "xgb":  {"best_f1": xgb_best_f1,  "best_t": xgb_best_t},
        "mix":  {"best_f1": mix_best_f1,  "best_t": mix_best_t},
    },
    "ensemble": {
        "selected_name": ens_name,
        "pooled_best_macro_f1": float(ens_best_f1),
        "per_fold_macro_f1_at_best": ens_pf,
        "mean_macro_f1_at_best": float(ens_mean),
        "threshold": float(ens_thr),
        "weights": ens_weights,
        "confusion_matrix_oof_at_best": cm,
        "baseline2_pooled_best": float(base2_f1),
        "baseline2_best": {"w_logi": float(base2_wlogi), "threshold": float(base2_t)},
        "three_model_pooled_best": float(ens3_f1),
        "three_model_best": {"w_logi": float(w1), "w_xgb_struct": float(w2), "w_xgb_svdtext": float(w3), "threshold": float(ens3_t)},
    }
}

# P* update
pstar_path = os.path.join(CKPT_ROOT, "pstar.json")
pstar = {"iteration_id": None, "macro_f1_pooled_best": -1.0}
if os.path.exists(pstar_path):
    try:
        with open(pstar_path, "r", encoding="utf-8") as f:
            pstar = json.load(f)
    except Exception:
        pass

pstar_updated = False
if float(metrics["ensemble"]["pooled_best_macro_f1"]) > float(pstar.get("macro_f1_pooled_best", -1.0)) + 1e-12:
    pstar_updated = True
    pstar = {"iteration_id": iteration_id, "macro_f1_pooled_best": float(metrics["ensemble"]["pooled_best_macro_f1"])}
    with open(pstar_path, "w", encoding="utf-8") as f:
        json.dump(pstar, f, ensure_ascii=False, indent=2)

joblib_ok = True
try:
    joblib.dump(
        {
            "pipe_logi": final_logi,
            "pipe_xgb_struct": final_xgb,
            "pipe_xgb_svdtext": final_mix,
            "ensemble": {"name": ens_name, "weights": ens_weights, "threshold": float(ens_thr)},
            "config": config,
            "schema": schema,
            "metrics": metrics,
        },
        os.path.join(ckpt_dir, "model_bundle.joblib")
    )
except Exception as e:
    joblib_ok = False
    print(f"[WARN] joblib dump failed: {repr(e)}")

with open(os.path.join(ckpt_dir, "config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
with open(os.path.join(ckpt_dir, "schema.json"), "w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)
with open(os.path.join(ckpt_dir, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

# =========================
# Append iteration detail JSONL
# =========================
entry = {
    "version": VERSION,
    "iteration_id": iteration_id,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "phase": "Modeling",
    "hm_message": hm_message,
    "real_score": real_score,
    "data_paths_used": {
        "stays_train": "stays_train.csv",
        "stays_test": "stays_test.csv",
        "patients": "patients.csv",
        "vitals_timeseries": "vitals_timeseries.json",
    },
    "join_keys": {"patients": "patient_id", "vitals_timeseries": "stay_id"},
    "leakage_checks_performed": [
        "Only uses day1-10 vitals/notes; target is discharge readiness at day11.",
        "No joins to external label-bearing tables; no target-derived aggregates.",
        f"Train/test patient_id overlap = {overlap_pat} (expected 0).",
    ],
    "feature_summary": {
        "n_train": int(train_df.shape[0]),
        "n_test": int(test_df.shape[0]),
        "cat_cols": len(cat_cols),
        "num_cols": len(num_cols),
        "text_cols": ["txt_all","txt_late3","txt_day10"],
        "added_features": ["reason_unit (interaction cat)", "lex_pos/lex_neg/lex_diff (+day10 variants)", "char TF-IDF on late/day10", "NEW: SVD text for XGB mix"],
        "svd": {"n_components": 64, "max_features": 30000, "text_source": "txt_svd = late3 + day10"},
    },
    "models_attempted": [
        {
            "name": "LogisticRegression(saga, elasticnet) + [num+OHE+TFIDF(word+char)]",
            "params": {"C": 0.6, "l1_ratio": 0.05, "class_weight": "balanced", "max_iter": 20000, "solver": "saga", "random_state": SEED},
            "cv_macro_f1_mean_at_0.5": logi_m_05,
            "cv_macro_f1_per_fold_at_0.5": logi_pf_05,
            "cv_macro_f1_oof_at_best_threshold": logi_best_f1,
            "oof_best_threshold": logi_best_t,
        },
        {
            "name": "XGBClassifier(structured-only) + [num+OHE]",
            "params": xgb_params,
            "cv_macro_f1_mean_at_0.5": xgb_m_05,
            "cv_macro_f1_per_fold_at_0.5": xgb_pf_05,
            "cv_macro_f1_oof_at_best_threshold": xgb_best_f1,
            "oof_best_threshold": xgb_best_t,
        },
        {
            "name": "XGBClassifier(struct+SVDtext) + [num+OHE+SVD(TFIDF)]",
            "params": xgb_mix_params,
            "cv_macro_f1_mean_at_0.5": mix_m_05,
            "cv_macro_f1_per_fold_at_0.5": mix_pf_05,
            "cv_macro_f1_oof_at_best_threshold": mix_best_f1,
            "oof_best_threshold": mix_best_t,
        },
        {
            "name": "Ensemble baseline 2-model (logi+xgb_struct) weight+threshold search",
            "params": {"weight_logi_grid": w_grid.tolist(), "threshold_grid": thr_grid.tolist()},
            "best_oof_macro_f1": float(base2_f1),
            "best_weight_logi": float(base2_wlogi),
            "best_threshold": float(base2_t),
        },
        {
            "name": "Ensemble NEW 3-model (logi+xgb_struct+xgb_svdtext) weight+threshold search",
            "params": {"weight_grid": w_grid.tolist(), "threshold_grid": thr_grid.tolist(), "simplex_constraint": "w_logi+w_svd<=1"},
            "best_oof_macro_f1": float(ens3_f1),
            "best_weights": {"w_logi": float(w1), "w_xgb_struct": float(w2), "w_xgb_svdtext": float(w3)},
            "best_threshold": float(ens3_t),
        },
    ],
    "selected_model": {"name": ens_name, "weights": ens_weights, "threshold": float(ens_thr)},
    "validation_protocol": {"splitter": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "metrics": {
        "macro_f1_mean": float(ens_mean),
        "macro_f1_per_fold": ens_pf,
        "macro_f1_pooled_best": float(ens_best_f1),
        "confusion_matrix": cm,
        "threshold": float(ens_thr),
        "pstar_updated": bool(pstar_updated),
        "joblib_dump_ok": bool(joblib_ok),
        "submission_path": SUBMISSION_NAME,
        "candidate_submission_path": cand_path,
        "checkpoint_dir": ckpt_dir,
    },
    "deltas_vs_prev": {
        "intent": "Rollback to Iter6 base direction; add only a complementary third branch to help (or be ignored) by ensemble search.",
        "main_change": "Add XGB(struct+SVD(text)) branch and 3-way ensemble optimization; keep Iter6 baseline as explicit comparator/fallback.",
        "guard": "Select max(2-model, 3-model) by pooled OOF macro-F1; include Iter6 anchor (w=0.35,t=0.60) in grids."
    },
    "known_defects_limitations": [
        "Pooled OOF threshold/weight search can mildly overfit; keep grids moderate and compare to baseline 2-model.",
        "No group split needed since train/test patient_id overlap is 0; still, distribution shift risk remains.",
    ],
    "next_step_hypothesis": "If 3-way ensemble yields consistent lift, try small hyperparameter sweep for xgb_svdtext (depth/eta/reg) or add calibrated linear SVM text branch."
}

# Ensure append-only
os.makedirs(os.path.dirname(log_path) or ".", exist_ok=True)
with open(log_path, "a", encoding="utf-8") as f:
    f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"[Log] appended -> {log_path}")
print(f"[P*] {pstar} (updated={pstar_updated})")

[Info] patient_id overlap train vs test: 0 (expected 0)
[Info] n_train=1000 n_test=1000
[Info] n_num_cols=133 n_cat_cols=6 text_cols(main)=['txt_all', 'txt_late3', 'txt_day10']
[CV] Running 5-fold StratifiedKFold...


# Iteration 12
- 0.7986

In [42]:
# clai_ch3_v1 | Modeling iteration: rollback-to-iter6 base + small safe tweak w/ guard (NO silent regression)

import os, json, time, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

import joblib
import xgboost as xgb

warnings.filterwarnings("ignore")

# -----------------------
# Config
# -----------------------
VERSION = "v1"
EXP_NAME = f"clai_ch3_{VERSION}"
SEED = 42
N_SPLITS = 5

# HM message (recorded in this iteration log entry)
HM_REAL_F1_LAST = 0.7757
HM_MESSAGE = "Rollback to iter6(real=0.8244) base; do ONLY small iteration; add self-eval guard to avoid negative-improve regressions."

SUBMISSION_PATH = Path(f"{EXP_NAME}_submission.csv")
ITER_LOG_PATH = Path(f"{EXP_NAME}_iteration_detail.jsonl")

CKPT_ROOT = Path("checkpoints") / EXP_NAME
ART_ROOT = Path("artifacts") / EXP_NAME
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
ART_ROOT.mkdir(parents=True, exist_ok=True)

# Determinism (best effort)
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

def utc_now_iso():
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())

def find_file(fname: str) -> Path:
    # Robust locator: project root / data / /mnt/data (sandbox)
    candidates = [
        Path(fname),
        Path("data") / fname,
        Path("/mnt/data") / fname
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Cannot find required file: {fname} (searched: {[str(c) for c in candidates]})")

def next_iteration_id(log_path: Path) -> int:
    if not log_path.exists():
        return 0
    max_id = -1
    with log_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict) and "iteration_id" in obj:
                    max_id = max(max_id, int(obj["iteration_id"]))
            except Exception:
                continue
    return max_id + 1

ITER_ID = next_iteration_id(ITER_LOG_PATH)
ITER_TAG = f"iter{ITER_ID:04d}"
CKPT_DIR = CKPT_ROOT / ITER_TAG
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f"[Run] {EXP_NAME} | iteration_id={ITER_ID} | tag={ITER_TAG}")
print(f"[Paths] submission -> {SUBMISSION_PATH}")
print(f"[Paths] checkpoint  -> {CKPT_DIR}")
print(f"[Seed] {SEED} | CV={N_SPLITS}-fold StratifiedKFold")

# -----------------------
# Load data
# -----------------------
stays_train_path = find_file("stays_train.csv")
stays_test_path  = find_file("stays_test.csv")
patients_path    = find_file("patients.csv")
vitals_path      = find_file("vitals_timeseries.json")

st_train = pd.read_csv(stays_train_path)
st_test  = pd.read_csv(stays_test_path)
patients = pd.read_csv(patients_path)

with open(vitals_path, "r", encoding="utf-8") as f:
    vitals_raw = json.load(f)

assert "discharge_ready_day11" in st_train.columns
assert set(["stay_id","patient_id","unit_type","admission_reason"]).issubset(st_train.columns)
assert set(["stay_id","patient_id","unit_type","admission_reason"]).issubset(st_test.columns)

# -----------------------
# Feature engineering from vitals_timeseries.json
# -----------------------
VITALS = ["hr","sbp","dbp","temp_c","rr"]
BASE_RECIPE = ["mean","std","min","max","last","missing","slope","last3_mean","last3_std","delta_last_first"]  # iter6-style core
# Candidate small add-ons (kept limited)
CAND_EXTRA_RECIPE = ["first","missing_frac","range","delta_last_prev","slope_last3","max_last3","min_last3"]  # small safe extensions

def _safe_float(x):
    try:
        if x is None:
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def _nanstats(arr: np.ndarray):
    n = len(arr)
    mask = ~np.isnan(arr)
    miss = int(n - mask.sum())
    if mask.sum() == 0:
        return dict(
            mean=np.nan, std=np.nan, min=np.nan, max=np.nan,
            first=np.nan, last=np.nan, missing=miss, missing_frac=miss / n,
            slope=np.nan, last3_mean=np.nan, last3_std=np.nan,
            delta_last_first=np.nan, range=np.nan
        )
    mean = float(np.nanmean(arr))
    std  = float(np.nanstd(arr))
    mn   = float(np.nanmin(arr))
    mx   = float(np.nanmax(arr))
    first = float(arr[0]) if not np.isnan(arr[0]) else float(arr[mask][0])
    last  = float(arr[-1]) if not np.isnan(arr[-1]) else float(arr[mask][-1])

    # slope on available points vs day index 1..n
    if mask.sum() >= 2:
        x = np.arange(1, n+1)[mask]
        y = arr[mask]
        slope = float(np.cov(x, y, bias=True)[0,1] / np.var(x))
    else:
        slope = np.nan

    last3 = arr[-3:]
    if np.sum(~np.isnan(last3)) > 0:
        last3_mean = float(np.nanmean(last3))
        last3_std  = float(np.nanstd(last3))
    else:
        last3_mean = np.nan
        last3_std  = np.nan

    delta_last_first = float(last - first) if (not np.isnan(last) and not np.isnan(first)) else np.nan
    rng = float(mx - mn) if (not np.isnan(mx) and not np.isnan(mn)) else np.nan

    return dict(
        mean=mean, std=std, min=mn, max=mx,
        first=first, last=last, missing=miss, missing_frac=miss / n,
        slope=slope, last3_mean=last3_mean, last3_std=last3_std,
        delta_last_first=delta_last_first, range=rng
    )

def _extras(arr: np.ndarray):
    n = len(arr)
    # day10-day9
    delta_last_prev = np.nan
    if n >= 2 and not np.isnan(arr[-1]) and not np.isnan(arr[-2]):
        delta_last_prev = float(arr[-1] - arr[-2])

    # slope on last3 using index (n-2..n)
    last3 = arr[-3:]
    mask = ~np.isnan(last3)
    if mask.sum() >= 2:
        x = np.arange(n-2, n+1)[mask]
        y = last3[mask]
        slope_last3 = float(np.cov(x, y, bias=True)[0,1] / np.var(x))
    else:
        slope_last3 = np.nan

    if mask.sum() > 0:
        max_last3 = float(np.nanmax(last3))
        min_last3 = float(np.nanmin(last3))
    else:
        max_last3 = np.nan
        min_last3 = np.nan

    return dict(delta_last_prev=delta_last_prev, slope_last3=slope_last3, max_last3=max_last3, min_last3=min_last3)

rows = []
for obj in vitals_raw:
    stay_id = obj["stay_id"]
    days = obj.get("days", [])
    # map day -> record
    day_map = {int(d.get("day")): d for d in days if d.get("day") is not None}
    # enforce 10-day window (1..10)
    notes = []
    feats = {"stay_id": stay_id}
    for v in VITALS:
        arr = []
        for day in range(1, 11):
            rec = day_map.get(day, {})
            arr.append(_safe_float(rec.get(v)))
        arr = np.array(arr, dtype=float)
        s = _nanstats(arr)
        # base stats
        for k in BASE_RECIPE:
            feats[f"{v}_{k}"] = s.get(k, np.nan)
        # candidate extras (we compute anyway; selection controls usage)
        ex = _extras(arr)
        feats[f"{v}_first"] = s.get("first", np.nan)
        feats[f"{v}_missing_frac"] = s.get("missing_frac", np.nan)
        feats[f"{v}_range"] = s.get("range", np.nan)
        feats[f"{v}_delta_last_prev"] = ex.get("delta_last_prev", np.nan)
        feats[f"{v}_slope_last3"] = ex.get("slope_last3", np.nan)
        feats[f"{v}_max_last3"] = ex.get("max_last3", np.nan)
        feats[f"{v}_min_last3"] = ex.get("min_last3", np.nan)

    # notes text windows
    for day in range(1, 11):
        rec = day_map.get(day, {})
        note = rec.get("note")
        if note is None:
            note = ""
        notes.append(str(note))
    text_all = " ".join([t for t in notes if t.strip() != ""])
    text_last3 = " ".join([t for t in notes[-3:] if t.strip() != ""])
    text_day10 = notes[-1] if len(notes) else ""

    feats["note_all"] = text_all
    feats["note_last3"] = text_last3
    feats["note_day10"] = text_day10

    # lightweight note length stats (candidate structured only)
    lens = [len(t) for t in notes]
    feats["note_len_mean"] = float(np.mean(lens))
    feats["note_len_std"]  = float(np.std(lens))
    feats["note_len_last"] = float(lens[-1]) if lens else 0.0
    feats["note_days_nonempty"] = int(sum(1 for t in notes if t.strip() != ""))

    rows.append(feats)

vitals_df = pd.DataFrame(rows)

# Merge
train_df = st_train.merge(vitals_df, on="stay_id", how="left").merge(patients, on="patient_id", how="left")
test_df  = st_test.merge(vitals_df, on="stay_id", how="left").merge(patients, on="patient_id", how="left")

y = train_df["discharge_ready_day11"].astype(int).values

# Leakage / sanity checks
train_pat = set(train_df["patient_id"].unique().tolist())
test_pat  = set(test_df["patient_id"].unique().tolist())
overlap_pat = len(train_pat.intersection(test_pat))
print(f"[Info] patient_id overlap train vs test: {overlap_pat} (expected 0)")

# -----------------------
# Column sets
# -----------------------
cat_cols = ["unit_type","admission_reason","sex","insurance","zip3"]

# Base numeric cols (iter6-style): 10 recipe * 5 vitals + age
num_cols_base = []
for v in VITALS:
    for k in BASE_RECIPE:
        num_cols_base.append(f"{v}_{k}")
num_cols_base.append("age")

# Candidate numeric cols (small add-ons): base + extra vitals + note length stats
num_cols_cand = list(num_cols_base)
for v in VITALS:
    for k in CAND_EXTRA_RECIPE:
        num_cols_cand.append(f"{v}_{k}")
num_cols_cand += ["note_len_mean","note_len_std","note_len_last","note_days_nonempty"]

text_cols = ["note_all","note_last3","note_day10"]

# -----------------------
# Helpers: picklable text selector (avoid lambda to prevent joblib PicklingError)
# -----------------------
def select_text_1d(X):
    # X: pd.DataFrame with single column or array shape (n,1)
    if isinstance(X, pd.DataFrame):
        s = X.iloc[:, 0]
        return s.fillna("").astype(str).values
    arr = np.asarray(X)
    if arr.ndim == 2 and arr.shape[1] == 1:
        arr = arr[:, 0]
    return pd.Series(arr).fillna("").astype(str).values

# -----------------------
# Build Logistic (structured + text TF-IDF) - baseline component (keep stable)
# -----------------------
logi_pre = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", SimpleImputer(strategy="median"), num_cols_base),
        ("txt_all",
         Pipeline([
             ("sel", FunctionTransformer(select_text_1d, validate=False)),
             ("tfidf", TfidfVectorizer(max_features=8000, ngram_range=(1,2), min_df=2, sublinear_tf=True))
         ]),
         ["note_all"]),
        ("txt_last3",
         Pipeline([
             ("sel", FunctionTransformer(select_text_1d, validate=False)),
             ("tfidf", TfidfVectorizer(max_features=2000, ngram_range=(1,2), min_df=2, sublinear_tf=True))
         ]),
         ["note_last3"]),
        ("txt_day10",
         Pipeline([
             ("sel", FunctionTransformer(select_text_1d, validate=False)),
             ("tfidf", TfidfVectorizer(max_features=1000, ngram_range=(1,2), min_df=1, sublinear_tf=True))
         ]),
         ["note_day10"]),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

logi_clf = LogisticRegression(
    solver="saga",
    penalty="l2",
    C=2.0,
    max_iter=3000,
    tol=1e-3,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=1
)

logi_pipe = Pipeline([("pre", logi_pre), ("clf", logi_clf)])

# -----------------------
# Build XGB structured - baseline + candidate (small tweak)
# -----------------------
def make_xgb_params(seed: int):
    # keep stable; small bagging uses different seeds
    return dict(
        n_estimators=900,
        learning_rate=0.03,
        max_depth=4,
        min_child_weight=1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        reg_alpha=0.0,
        gamma=0.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=1,
        random_state=int(seed)
    )

def make_xgb_pre(num_cols):
    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
            ("num", SimpleImputer(strategy="median"), num_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3
    )

# -----------------------
# CV + OOF
# -----------------------
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

n = len(train_df)
oof_logi = np.zeros(n, dtype=float)
oof_xgb_base = np.zeros(n, dtype=float)
oof_xgb_cand = np.zeros(n, dtype=float)
fold_id = np.zeros(n, dtype=int)

print("[CV] Running 5-fold StratifiedKFold...")

for f, (tr_idx, va_idx) in enumerate(skf.split(train_df, y)):
    fold_id[va_idx] = f
    Xtr = train_df.iloc[tr_idx].copy()
    Xva = train_df.iloc[va_idx].copy()
    ytr = y[tr_idx]

    # Logistic (stable baseline component)
    logi_pipe.fit(Xtr, ytr)
    oof_logi[va_idx] = logi_pipe.predict_proba(Xva)[:, 1]

    # XGB baseline (single seed, base numeric)
    xgb_pre_base = make_xgb_pre(num_cols_base)
    Xtr_base = xgb_pre_base.fit_transform(Xtr)
    Xva_base = xgb_pre_base.transform(Xva)

    xgb_base = xgb.XGBClassifier(**make_xgb_params(SEED))
    xgb_base.fit(Xtr_base, ytr)
    oof_xgb_base[va_idx] = xgb_base.predict_proba(Xva_base)[:, 1]

    # XGB candidate (2-seed bagging, extended numeric)
    xgb_pre_cand = make_xgb_pre(num_cols_cand)
    Xtr_cand = xgb_pre_cand.fit_transform(Xtr)
    Xva_cand = xgb_pre_cand.transform(Xva)

    preds = []
    for s in [SEED, SEED + 13]:
        mdl = xgb.XGBClassifier(**make_xgb_params(s))
        mdl.fit(Xtr_cand, ytr)
        preds.append(mdl.predict_proba(Xva_cand)[:, 1])
    oof_xgb_cand[va_idx] = np.mean(preds, axis=0)

# -----------------------
# Metrics utilities
# -----------------------
def macro_f1_foldwise(y_true, y_pred, fold_id, n_splits):
    scores = []
    for f in range(n_splits):
        idx = (fold_id == f)
        scores.append(float(f1_score(y_true[idx], y_pred[idx], average="macro")))
    return scores, float(np.mean(scores))

def best_threshold_on_grid(y_true, prob, fold_id, n_splits, thresholds):
    best = None
    for t in thresholds:
        pred = (prob >= t).astype(int)
        pooled = float(f1_score(y_true, pred, average="macro"))
        fold_scores, mean_score = macro_f1_foldwise(y_true, pred, fold_id, n_splits)
        if best is None or pooled > best["pooled"]:
            best = dict(threshold=float(t), pooled=pooled, mean=mean_score, per_fold=fold_scores)
    return best

def search_ensemble(y_true, p_logi, p_xgb, fold_id, n_splits, weight_grid, thr_grid, objective="mean"):
    best = None
    for w in weight_grid:
        p = w * p_logi + (1.0 - w) * p_xgb
        for t in thr_grid:
            pred = (p >= t).astype(int)
            pooled = float(f1_score(y_true, pred, average="macro"))
            fold_scores, mean_score = macro_f1_foldwise(y_true, pred, fold_id, n_splits)
            score = mean_score if objective == "mean" else pooled
            if best is None or score > best["score"] + 1e-12:
                best = dict(score=float(score), pooled=pooled, mean=mean_score,
                            weight_logi=float(w), threshold=float(t),
                            per_fold=fold_scores,
                            confusion=confusion_matrix(y_true, pred).tolist())
    return best

# -----------------------
# Evaluate components + baseline vs candidate ensembles
# -----------------------
thr_grid_single = np.linspace(0.20, 0.80, 121)
logi_best = best_threshold_on_grid(y, oof_logi, fold_id, N_SPLITS, thr_grid_single)
xgb_base_best = best_threshold_on_grid(y, oof_xgb_base, fold_id, N_SPLITS, thr_grid_single)
xgb_cand_best = best_threshold_on_grid(y, oof_xgb_cand, fold_id, N_SPLITS, thr_grid_single)

# Iter6 known-good defaults (used as baseline reference point)
ITER6_W_DEFAULT = 0.35
ITER6_T_DEFAULT = 0.60
p_base_default = ITER6_W_DEFAULT * oof_logi + (1.0 - ITER6_W_DEFAULT) * oof_xgb_base
pred_base_default = (p_base_default >= ITER6_T_DEFAULT).astype(int)
base_default_pooled = float(f1_score(y, pred_base_default, average="macro"))
base_default_per_fold, base_default_mean = macro_f1_foldwise(y, pred_base_default, fold_id, N_SPLITS)
base_default_cm = confusion_matrix(y, pred_base_default).tolist()

# Candidate ensemble search around iter6 neighborhood (small safe)
weight_grid = np.round(np.linspace(0.10, 0.55, 10), 3)   # includes ~0.35
thr_grid = np.round(np.linspace(0.45, 0.75, 61), 3)      # includes ~0.60
ens_cand_best = search_ensemble(
    y_true=y, p_logi=oof_logi, p_xgb=oof_xgb_cand,
    fold_id=fold_id, n_splits=N_SPLITS,
    weight_grid=weight_grid, thr_grid=thr_grid,
    objective="mean"  # reduce optimistic pooled-overfit
)

print("\n[Results @0.5]")
# component @0.5
for name, prob in [("Logistic", oof_logi), ("XGB/Structured(base)", oof_xgb_base), ("XGB/Structured(cand)", oof_xgb_cand)]:
    pred = (prob >= 0.5).astype(int)
    f1s, mean_f1 = macro_f1_foldwise(y, pred, fold_id, N_SPLITS)
    print(f"  {name} (0.5) fold macro-F1: {[round(x,4) for x in f1s]} | mean={mean_f1:.6f}")

print("\n[Best threshold (OOF pooled)]")
print(f"  Logistic: best_f1={logi_best['pooled']:.6f} @ t={logi_best['threshold']:.3f}")
print(f"  XGB base: best_f1={xgb_base_best['pooled']:.6f} @ t={xgb_base_best['threshold']:.3f}")
print(f"  XGB cand: best_f1={xgb_cand_best['pooled']:.6f} @ t={xgb_cand_best['threshold']:.3f}")

print("\n[Baseline iter6-default ensemble (fixed w=0.35, t=0.60)]")
print(f"  OOF pooled macro-F1={base_default_pooled:.6f} | mean over folds={base_default_mean:.6f}")
print(f"  per-fold={ [round(x,4) for x in base_default_per_fold] }")
print(f"  CM={base_default_cm}")

print("\n[Candidate ensemble optimized on OOF (objective=fold-mean)]")
print(f"  Best cand mean macro-F1={ens_cand_best['mean']:.6f} | pooled={ens_cand_best['pooled']:.6f} @ weight_logi={ens_cand_best['weight_logi']:.3f}, threshold={ens_cand_best['threshold']:.3f}")
print(f"  per-fold={ [round(x,4) for x in ens_cand_best['per_fold']] }")
print(f"  CM={ens_cand_best['confusion']}")

# -----------------------
# Guard: select best (no regression)
# -----------------------
# If candidate does not strictly beat baseline-by-mean, we keep baseline default.
EPS = 0.0005
use_candidate = (ens_cand_best["mean"] > base_default_mean + EPS)

selected = {
    "name": "candidate_small_xgb_bagging" if use_candidate else "baseline_iter6_default",
    "ensemble_weight_logi": float(ens_cand_best["weight_logi"] if use_candidate else ITER6_W_DEFAULT),
    "threshold": float(ens_cand_best["threshold"] if use_candidate else ITER6_T_DEFAULT),
    "selection_rule": f"cand_mean > base_mean + {EPS}" if use_candidate else "fallback_to_baseline_default",
}

print("\n[Selection]")
print(selected)

# -----------------------
# Fit final models on full train and predict test
# -----------------------
# Logistic full fit
logi_pipe.fit(train_df, y)
test_logi_prob = logi_pipe.predict_proba(test_df)[:, 1]

# XGB full fit (base or candidate)
if use_candidate:
    xgb_pre_final = make_xgb_pre(num_cols_cand)
    Xtr_final = xgb_pre_final.fit_transform(train_df)
    Xte_final = xgb_pre_final.transform(test_df)
    xgb_models = []
    te_preds = []
    for s in [SEED, SEED + 13]:
        mdl = xgb.XGBClassifier(**make_xgb_params(s))
        mdl.fit(Xtr_final, y)
        xgb_models.append(mdl)
        te_preds.append(mdl.predict_proba(Xte_final)[:, 1])
    test_xgb_prob = np.mean(te_preds, axis=0)
    final_num_cols = num_cols_cand
else:
    xgb_pre_final = make_xgb_pre(num_cols_base)
    Xtr_final = xgb_pre_final.fit_transform(train_df)
    Xte_final = xgb_pre_final.transform(test_df)
    mdl = xgb.XGBClassifier(**make_xgb_params(SEED))
    mdl.fit(Xtr_final, y)
    xgb_models = [mdl]
    test_xgb_prob = mdl.predict_proba(Xte_final)[:, 1]
    final_num_cols = num_cols_base

w = selected["ensemble_weight_logi"]
t = selected["threshold"]
test_ens_prob = w * test_logi_prob + (1.0 - w) * test_xgb_prob
test_pred = (test_ens_prob >= t).astype(int)

sub = pd.DataFrame({"stay_id": test_df["stay_id"].astype(int).values,
                    "discharge_ready_day11": test_pred.astype(int)})
sub.to_csv(SUBMISSION_PATH, index=False)
print(f"\n[Saved] submission -> {SUBMISSION_PATH} | shape={sub.shape}")

# -----------------------
# Save artifacts
# -----------------------
np.save(ART_ROOT / f"{ITER_TAG}_fold_id.npy", fold_id)
np.save(ART_ROOT / f"{ITER_TAG}_oof_prob_logi.npy", oof_logi)
np.save(ART_ROOT / f"{ITER_TAG}_oof_prob_xgb_base.npy", oof_xgb_base)
np.save(ART_ROOT / f"{ITER_TAG}_oof_prob_xgb_cand.npy", oof_xgb_cand)
np.save(ART_ROOT / f"{ITER_TAG}_test_prob_logi.npy", test_logi_prob)
np.save(ART_ROOT / f"{ITER_TAG}_test_prob_xgb.npy", test_xgb_prob)
np.save(ART_ROOT / f"{ITER_TAG}_test_prob_ens.npy", test_ens_prob)

# -----------------------
# Checkpoint bundle (P* update logic from historical log)
# -----------------------
def load_prev_best_macro_f1(log_path: Path):
    if not log_path.exists():
        return None
    best = None
    best_iter = None
    with log_path.open("r", encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if not line:
                continue
            try:
                obj=json.loads(line)
            except Exception:
                continue
            if not isinstance(obj, dict):
                continue
            met=obj.get("metrics", {})
            if not isinstance(met, dict):
                continue
            cand = None
            # prefer pooled-best if present
            if "macro_f1_pooled_best" in met and met["macro_f1_pooled_best"] is not None:
                cand=float(met["macro_f1_pooled_best"])
            elif "macro_f1_mean" in met and met["macro_f1_mean"] is not None:
                cand=float(met["macro_f1_mean"])
            if cand is not None:
                if best is None or cand > best:
                    best = cand
                    best_iter = obj.get("iteration_id")
    return {"best_macro_f1": best, "iteration_id": best_iter}

prev_best = load_prev_best_macro_f1(ITER_LOG_PATH)
# define current pooled-best metric for P* comparison
current_pooled_best = float(ens_cand_best["pooled"] if use_candidate else base_default_pooled)
pstar_updated = False
if prev_best is None or prev_best["best_macro_f1"] is None:
    pstar_updated = True
elif current_pooled_best > float(prev_best["best_macro_f1"]) + 1e-12:
    pstar_updated = True

# Metrics object for logging & checkpoint
metrics = {
    "macro_f1_mean": float(ens_cand_best["mean"] if use_candidate else base_default_mean),
    "macro_f1_per_fold": (ens_cand_best["per_fold"] if use_candidate else base_default_per_fold),
    "macro_f1_pooled_best": current_pooled_best,
    "confusion_matrix": (ens_cand_best["confusion"] if use_candidate else base_default_cm),
    "threshold": float(t),
    "ensemble_weight_logi": float(w),
    "component_best_thresholds": {
        "logistic_best_pooled": logi_best,
        "xgb_base_best_pooled": xgb_base_best,
        "xgb_cand_best_pooled": xgb_cand_best
    },
    "baseline_default": {
        "weight_logi": float(ITER6_W_DEFAULT),
        "threshold": float(ITER6_T_DEFAULT),
        "pooled": float(base_default_pooled),
        "mean": float(base_default_mean),
        "per_fold": base_default_per_fold
    },
    "candidate_best_mean_objective": ens_cand_best,
    "pstar_updated": bool(pstar_updated),
    "prev_best": prev_best
}

schema = {
    "categorical_cols": cat_cols,
    "numeric_cols_base": num_cols_base,
    "numeric_cols_candidate": num_cols_cand,
    "final_numeric_cols_used": final_num_cols,
    "text_cols": text_cols,
    "tfidf_max_features_total": 8000 + 2000 + 1000,
    "base_vitals_recipe": BASE_RECIPE,
    "candidate_extra_recipe": CAND_EXTRA_RECIPE
}

config = {
    "exp_name": EXP_NAME,
    "version": VERSION,
    "seed": SEED,
    "cv": {"n_splits": N_SPLITS, "type": "StratifiedKFold", "shuffle": True, "random_state": SEED},
    "iter6_defaults": {"weight_logi": ITER6_W_DEFAULT, "threshold": ITER6_T_DEFAULT},
    "selection": selected,
    "xgb_params_base": make_xgb_params(SEED),
    "xgb_params_candidate_seeds": [make_xgb_params(SEED), make_xgb_params(SEED+13)],
}

# Save checkpoint files
with open(CKPT_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
with open(CKPT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
with open(CKPT_DIR / "schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)

bundle = {
    "logi_pipe": logi_pipe,
    "xgb_pre": xgb_pre_final,
    "xgb_models": xgb_models,
    "ensemble_weight_logi": float(w),
    "threshold": float(t),
    "config": config,
    "metrics": metrics,
    "schema": schema,
}
joblib_ok = True
try:
    joblib.dump(bundle, CKPT_DIR / "model_bundle.joblib")
except Exception as e:
    joblib_ok = False
    with open(CKPT_DIR / "joblib_dump_error.txt", "w", encoding="utf-8") as f:
        f.write(str(e))

metrics["joblib_dump_ok"] = joblib_ok

print(f"[Checkpoint] model_bundle.joblib saved: {joblib_ok}")

# -----------------------
# Append iteration detail JSONL (append-only)
# -----------------------
log_entry = {
    "version": VERSION,
    "iteration_id": int(ITER_ID),
    "timestamp": utc_now_iso(),
    "phase": "Modeling",
    "hm_message": HM_MESSAGE,
    "real_score": HM_REAL_F1_LAST,
    "data_paths_used": {
        "stays_train": str(stays_train_path),
        "stays_test": str(stays_test_path),
        "patients": str(patients_path),
        "vitals_timeseries": str(vitals_path),
    },
    "join_keys": {
        "stays<->vitals": "stay_id",
        "stays<->patients": "patient_id"
    },
    "leakage_checks_performed": [
        f"patient_id overlap train vs test = {overlap_pat} (expected 0)",
        "No use of discharge_notes.json / any post-discharge artifacts",
        "Only day1-10 vitals + day1-10 daily notes used (target is day11 readiness)"
    ],
    "feature_summary": {
        "categorical_cols": cat_cols,
        "text_cols": text_cols,
        "num_cols_base_count": len(num_cols_base),
        "num_cols_candidate_count": len(num_cols_cand),
        "n_train": int(train_df.shape[0]),
        "n_test": int(test_df.shape[0]),
        "pos_rate_train": float(np.mean(y))
    },
    "models_attempted": [
        {
            "name": "baseline_iter6_default",
            "details": {
                "logistic": {"tfidf_total_max_features": 11000, "C": 2.0, "solver": "saga", "class_weight": "balanced"},
                "xgb": {"numeric_cols": "base", "seeds": [SEED]},
                "ensemble": {"weight_logi": ITER6_W_DEFAULT, "threshold": ITER6_T_DEFAULT},
            },
            "cv_macro_f1": {
                "pooled": float(base_default_pooled),
                "mean": float(base_default_mean),
                "per_fold": base_default_per_fold
            }
        },
        {
            "name": "candidate_small_xgb_bagging",
            "details": {
                "logistic": "same_as_baseline",
                "xgb": {"numeric_cols": "candidate_extended", "seeds": [SEED, SEED+13]},
                "ensemble_search": {"objective": "fold-mean", "weight_grid": weight_grid.tolist(), "thr_grid": thr_grid.tolist()}
            },
            "cv_macro_f1": {
                "pooled_best": float(ens_cand_best["pooled"]),
                "mean_best": float(ens_cand_best["mean"]),
                "best_params": {"weight_logi": float(ens_cand_best["weight_logi"]), "threshold": float(ens_cand_best["threshold"])}
            }
        }
    ],
    "selected_model": selected,
    "validation_protocol": {
        "cv": "5-fold StratifiedKFold",
        "seed": SEED,
        "selection_guard": f"candidate must beat baseline mean by > {EPS}; else fallback baseline default"
    },
    "metrics": metrics,
    "deltas_vs_previous_iteration": {
        "rollback_to_iter6_base": True,
        "small_tweak_only": "XGB gets small extra vitals delta/slope/range + note length + 2-seed bagging; Logistic unchanged",
        "added_guard": "baseline vs candidate OOF comparison; auto-fallback to baseline if no improvement"
    },
    "known_defects_limitations": [
        "Compute cost: 5-fold CV trains Logistic + XGB(base) + XGB(cand bagging) per fold.",
        "Leaderboard correlation can still drift; guard reduces but cannot eliminate."
    ],
    "next_step_hypothesis": "If candidate wins OOF reliably, next small step: add 1 more XGB seed (3-seed bagging) or add 1 shallow XGB variant and re-optimize weights with same guard."
}

with ITER_LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(json.dumps(log_entry, ensure_ascii=False) + "\n")

print(f"[Log] appended -> {ITER_LOG_PATH}")

[Run] clai_ch3_v1 | iteration_id=12 | tag=iter0012
[Paths] submission -> clai_ch3_v1_submission.csv
[Paths] checkpoint  -> checkpoints\clai_ch3_v1\iter0012
[Seed] 42 | CV=5-fold StratifiedKFold
[Info] patient_id overlap train vs test: 0 (expected 0)
[CV] Running 5-fold StratifiedKFold...

[Results @0.5]
  Logistic (0.5) fold macro-F1: [0.7089, 0.6746, 0.7042, 0.6742, 0.6604] | mean=0.684478
  XGB/Structured(base) (0.5) fold macro-F1: [0.7379, 0.7143, 0.7289, 0.7654, 0.7207] | mean=0.733447
  XGB/Structured(cand) (0.5) fold macro-F1: [0.8191, 0.7808, 0.7847, 0.818, 0.7792] | mean=0.796362

[Best threshold (OOF pooled)]
  Logistic: best_f1=0.706450 @ t=0.425
  XGB base: best_f1=0.745394 @ t=0.585
  XGB cand: best_f1=0.820413 @ t=0.755

[Baseline iter6-default ensemble (fixed w=0.35, t=0.60)]
  OOF pooled macro-F1=0.747512 | mean over folds=0.746978
  per-fold=[0.7473, 0.7197, 0.738, 0.7627, 0.7673]
  CM=[[233, 111], [118, 538]]

[Candidate ensemble optimized on OOF (objective=fold-mean)]

# Iteration 13
- 0.7813

In [44]:
import os, json, math, random
from pathlib import Path

import numpy as np
import pandas as pd

# ============================================================
# clai_ch3_v1 — Modeling iteration (rollback to iter0006 base)
# One-feature change: add char-level TF-IDF on txt_all
# ============================================================
VERSION = "v1"
SEED = 42
N_SPLITS = 5
np.random.seed(SEED)
random.seed(SEED)

# ONE feature change vs iter0006 (keep everything else same)
ADD_CHAR_TFIDF_ALL = True

# ---------- Path helpers ----------
def resolve_path(filename: str) -> Path:
    """Prefer local working dir; fallback to /mnt/data (for sandbox)."""
    p = Path(filename)
    if p.exists():
        return p
    alt = Path("/mnt/data") / filename
    if alt.exists():
        return alt
    alt2 = Path("..") / filename
    if alt2.exists():
        return alt2
    return p  # will fail later with clear error

stays_train_path = resolve_path("stays_train.csv")
stays_test_path  = resolve_path("stays_test.csv")
patients_path    = resolve_path("patients.csv")
vitals_path      = resolve_path("vitals_timeseries.json")

submission_path = Path(f"clai_ch3_{VERSION}_submission.csv")
log_path = Path(f"clai_ch3_{VERSION}_iteration_detail.jsonl")

ckpt_root = Path("checkpoints") / f"clai_ch3_{VERSION}"
art_root  = Path("artifacts") / f"clai_ch3_{VERSION}"
ckpt_root.mkdir(parents=True, exist_ok=True)
art_root.mkdir(parents=True, exist_ok=True)

def next_iteration_id(log_file: Path) -> int:
    """Hotfix-safe: hotfix entries do not increment iteration_id."""
    if not log_file.exists():
        return 0
    max_id = -1
    with log_file.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception:
                continue
            if obj.get("hotfix") is True:
                continue
            it = obj.get("iteration_id")
            if isinstance(it, int):
                max_id = max(max_id, it)
    return max_id + 1

ITER_ID = next_iteration_id(log_path)
ITER_TAG = f"iter{ITER_ID:04d}"
ckpt_dir = ckpt_root / ITER_TAG
ckpt_dir.mkdir(parents=True, exist_ok=True)

# ---------- Load data ----------
st_train = pd.read_csv(stays_train_path)
st_test  = pd.read_csv(stays_test_path)
patients = pd.read_csv(patients_path)

with open(vitals_path, "r", encoding="utf-8") as f:
    vitals_list = json.load(f)
vitals_map = {int(obj["stay_id"]): obj["days"] for obj in vitals_list}

# Leakage guard: patient_id overlap train vs test expected 0
train_pids = set(st_train["patient_id"].unique().tolist())
test_pids  = set(st_test["patient_id"].unique().tolist())
overlap = len(train_pids & test_pids)
print(f"[Info] patient_id overlap train vs test: {overlap} (expected 0)")

# ---------- Optional baseline reference (iter0006) ----------
def find_baseline_file(rel_name: str):
    candidates = [
        art_root / rel_name,
        Path(rel_name),
        Path("/mnt/data") / rel_name,
        Path("..") / rel_name,
    ]
    for c in candidates:
        if c.exists():
            return c
    return None

baseline_oof_path = find_baseline_file("iter0006_oof_predictions.csv")
baseline_ref = None
if baseline_oof_path is not None:
    try:
        bdf = pd.read_csv(baseline_oof_path)
        yb = bdf["y_true"].values.astype(int)
        pl = bdf["oof_prob_logi"].values.astype(float)
        px = bdf["oof_prob_xgb"].values.astype(float)
        thr_grid_b = np.round(np.arange(0.05, 0.951, 0.01), 2)
        w_grid_b = np.round(np.arange(0.0, 1.001, 0.05), 2)

        from sklearn.metrics import f1_score
        def best_threshold(y_true, prob, grid):
            best_f, best_t = -1.0, 0.5
            for t in grid:
                f = f1_score(y_true, (prob >= t).astype(int), average="macro")
                if f > best_f:
                    best_f, best_t = f, t
            return float(best_f), float(best_t)

        best = (-1.0, 0.5, 0.5)
        for w in w_grid_b:
            p = w * pl + (1.0 - w) * px
            f, t = best_threshold(yb, p, thr_grid_b)
            if f > best[0]:
                best = (f, float(w), float(t))
        baseline_ref = {
            "path": str(baseline_oof_path),
            "pooled_best_macro_f1": float(best[0]),
            "weight_logi": float(best[1]),
            "threshold": float(best[2]),
        }
        print(f"[Baseline iter0006] pooled_best={baseline_ref['pooled_best_macro_f1']:.6f} @ w={baseline_ref['weight_logi']:.2f}, t={baseline_ref['threshold']:.3f}")
    except Exception as e:
        print(f"[WARN] Failed to read baseline iter0006 OOF file: {e}")

# ---------- Feature engineering ----------
VITALS = ["hr", "sbp", "dbp", "temp_c", "rr"]
DAY_IDX = np.arange(1, 11)

POS_TERMS = [
    "stable", "improving", "improved", "weaned", "room air", "tolerating",
    "ambulated", "walked", "out of bed", "up in chair", "pain controlled",
    "no new issues", "ready for discharge", "discharged"
]
NEG_TERMS = [
    "worsening", "febrile", "fever", "spiked", "hypotension", "tachycard",
    "tachypnea", "oxygen requirement", "increased oxygen", "sirs", "sepsis",
    "icu", "awaiting placement", "rehab", "snf", "delirium", "confusion"
]

def _safe_nan_stat(arr: np.ndarray, fn, default=np.nan):
    arr = np.asarray(arr, dtype=float)
    if np.all(np.isnan(arr)):
        return default
    try:
        return float(fn(arr))
    except Exception:
        return default

def _linreg_slope(x: np.ndarray, y: np.ndarray) -> float:
    if len(x) < 2:
        return 0.0
    x = x.astype(float)
    y = y.astype(float)
    x0 = x - x.mean()
    y0 = y - y.mean()
    denom = float(np.sum(x0 ** 2))
    if denom == 0.0:
        return 0.0
    return float(np.sum(x0 * y0) / denom)

def _count_terms(text: str, terms) -> int:
    t = (text or "").lower()
    c = 0
    for term in terms:
        if term in t:
            c += 1
    return int(c)

def extract_one_stay(stay_id: int) -> dict:
    days = vitals_map.get(int(stay_id), [])
    rec_by_day = {int(d.get("day")): d for d in days if d.get("day") is not None}

    feats = {}

    # vitals numeric: summary recipe + raw per-day values (iter0006 family)
    for v in VITALS:
        arr = np.array([rec_by_day.get(int(d), {}).get(v, np.nan) for d in DAY_IDX], dtype=float)

        feats[f"{v}_mean"] = _safe_nan_stat(arr, np.nanmean)
        feats[f"{v}_std"]  = _safe_nan_stat(arr, np.nanstd)
        feats[f"{v}_min"]  = _safe_nan_stat(arr, np.nanmin)
        feats[f"{v}_max"]  = _safe_nan_stat(arr, np.nanmax)
        feats[f"{v}_last"] = float(arr[-1]) if not np.isnan(arr[-1]) else np.nan
        feats[f"{v}_missing"] = int(np.isnan(arr).sum())

        mask = ~np.isnan(arr)
        feats[f"{v}_slope"] = _linreg_slope(DAY_IDX[mask], arr[mask]) if mask.sum() >= 2 else 0.0

        last3 = arr[-3:]
        feats[f"{v}_last3_mean"] = _safe_nan_stat(last3, np.nanmean)
        feats[f"{v}_last3_std"]  = _safe_nan_stat(last3, np.nanstd)

        first = arr[0]
        if np.isnan(first) and mask.any():
            first = arr[mask][0]
        feats[f"{v}_delta_last_first"] = float(arr[-1] - first) if (not np.isnan(first) and not np.isnan(arr[-1])) else 0.0

        for i in range(1, 11):
            val = arr[i - 1]
            feats[f"{v}_d{i}"] = float(val) if not np.isnan(val) else np.nan

    # notes -> 3 text windows
    notes = []
    notes_late3 = []
    note_day10 = ""
    for d in DAY_IDX:
        n = rec_by_day.get(int(d), {}).get("note", "")
        if n is None:
            n = ""
        n = str(n)
        if n:
            notes.append(n)
        if d >= 8 and n:
            notes_late3.append(n)
        if d == 10:
            note_day10 = n

    txt_all = " ".join(notes)
    txt_late3 = " ".join(notes_late3)
    txt_day10 = note_day10

    feats["txt_all"] = txt_all
    feats["txt_late3"] = txt_late3
    feats["txt_day10"] = txt_day10

    # small numeric helpers (iter0006 family)
    feats["len_all"] = int(len(txt_all))
    feats["len_late3"] = int(len(txt_late3))
    feats["len_day10"] = int(len(txt_day10))

    lp = _count_terms(txt_day10, POS_TERMS)
    ln = _count_terms(txt_day10, NEG_TERMS)
    feats["lex_pos"] = lp
    feats["lex_neg"] = ln
    feats["lex_diff"] = lp - ln

    return feats

def build_feature_frame(stays_df: pd.DataFrame) -> pd.DataFrame:
    feat_rows = [extract_one_stay(int(sid)) for sid in stays_df["stay_id"].values]
    feats_df = pd.DataFrame(feat_rows)

    base = stays_df.merge(patients, on="patient_id", how="left")
    out = pd.concat([base.reset_index(drop=True), feats_df.reset_index(drop=True)], axis=1)
    out["reason_unit"] = out["admission_reason"].astype(str) + "__" + out["unit_type"].astype(str)

    for c in ["txt_all", "txt_late3", "txt_day10"]:
        out[c] = out[c].fillna("")
    return out

X_train = build_feature_frame(st_train.drop(columns=["discharge_ready_day11"]))
X_test  = build_feature_frame(st_test.copy())
y = st_train["discharge_ready_day11"].astype(int).values

CAT_COLS = ["unit_type", "admission_reason", "sex", "insurance", "zip3", "reason_unit"]
TEXT_COLS = ["txt_all", "txt_late3", "txt_day10"]
ID_COLS = ["stay_id", "patient_id"]
NUM_COLS = [c for c in X_train.columns if c not in CAT_COLS + TEXT_COLS + ID_COLS]

# ---------- Modeling ----------
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
import joblib

import xgboost as xgb

# Logistic pipeline (iter0006 base + ONE CHANGE: char TF-IDF on txt_all)
text_transformers = [
    ("txt_all_w",  TfidfVectorizer(max_features=6000, ngram_range=(1, 2), min_df=2, sublinear_tf=True), "txt_all"),
    ("txt_late_w", TfidfVectorizer(max_features=2500, ngram_range=(1, 2), min_df=2, sublinear_tf=True), "txt_late3"),
    ("txt_d10_w",  TfidfVectorizer(max_features=2500, ngram_range=(1, 2), min_df=2, sublinear_tf=True), "txt_day10"),
    ("txt_late_c", TfidfVectorizer(max_features=2000, analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True), "txt_late3"),
    ("txt_d10_c",  TfidfVectorizer(max_features=2000, analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True), "txt_day10"),
]
if ADD_CHAR_TFIDF_ALL:
    text_transformers.append(
        ("txt_all_c", TfidfVectorizer(max_features=2000, analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True), "txt_all")
    )

logi_pre = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler(with_mean=False))]), NUM_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
        *text_transformers,
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

pipe_logi = Pipeline([
    ("pre", logi_pre),
    ("clf", LogisticRegression(
        solver="saga",
        penalty="elasticnet",
        l1_ratio=0.05,
        C=0.6,
        class_weight="balanced",
        max_iter=20000,
        n_jobs=1,
        random_state=SEED,
    ))
])

# XGB structured pipeline (kept fixed)
xgb_pre = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), NUM_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
    ],
    remainder="drop",
    sparse_threshold=0.0,
)

xgb_params = dict(
    objective="binary:logistic",
    learning_rate=0.05,
    max_depth=4,
    n_estimators=450,
    min_child_weight=2.0,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="logloss",
    random_state=SEED,
    n_jobs=1,
    reg_lambda=1.0,
    reg_alpha=0.0,
    gamma=0.0,
    verbosity=0,
)

pipe_xgb = Pipeline([("pre", xgb_pre), ("clf", xgb.XGBClassifier(**xgb_params))])

# CV
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_prob_logi = np.zeros(len(X_train), dtype=float)
oof_prob_xgb  = np.zeros(len(X_train), dtype=float)
fold_id = np.zeros(len(X_train), dtype=int)

fold_f1_logi = []
fold_f1_xgb  = []

print(f"\n[CV] Running {N_SPLITS}-fold StratifiedKFold...")
for f, (tr_idx, va_idx) in enumerate(skf.split(X_train, y)):
    fold_id[va_idx] = f
    X_tr, y_tr = X_train.iloc[tr_idx], y[tr_idx]
    X_va, y_va = X_train.iloc[va_idx], y[va_idx]

    pl = clone(pipe_logi)
    px = clone(pipe_xgb)

    pl.fit(X_tr, y_tr)
    px.fit(X_tr, y_tr)

    oof_prob_logi[va_idx] = pl.predict_proba(X_va)[:, 1]
    oof_prob_xgb[va_idx]  = px.predict_proba(X_va)[:, 1]

    fold_f1_logi.append(f1_score(y_va, (oof_prob_logi[va_idx] >= 0.5).astype(int), average="macro"))
    fold_f1_xgb.append( f1_score(y_va, (oof_prob_xgb[va_idx]  >= 0.5).astype(int), average="macro"))

print("\n[Results @0.5]")
print(f"  Logistic (0.5) fold macro-F1: {[round(x,4) for x in fold_f1_logi]} | mean={np.mean(fold_f1_logi):.6f}")
print(f"  XGB/Structured (0.5) fold macro-F1: {[round(x,4) for x in fold_f1_xgb]} | mean={np.mean(fold_f1_xgb):.6f}")

def best_threshold(y_true, prob, grid):
    best_f, best_t = -1.0, 0.5
    for t in grid:
        f = f1_score(y_true, (prob >= t).astype(int), average="macro")
        if f > best_f:
            best_f, best_t = f, t
    return float(best_f), float(best_t)

thr_grid = np.round(np.arange(0.05, 0.951, 0.01), 2)
w_grid = np.round(np.arange(0.0, 1.001, 0.05), 2)

logi_best_f1, logi_best_t = best_threshold(y, oof_prob_logi, thr_grid)
xgb_best_f1,  xgb_best_t  = best_threshold(y, oof_prob_xgb,  thr_grid)

print("\n[Best threshold (OOF pooled)]")
print(f"  Logistic: best_f1={logi_best_f1:.6f} @ t={logi_best_t:.3f}")
print(f"  XGB/Structured: best_f1={xgb_best_f1:.6f} @ t={xgb_best_t:.3f}")

# Ensemble optimize (w, t)
best = (-1.0, 0.5, 0.5)  # (f1, w, t)
for w in w_grid:
    p = w * oof_prob_logi + (1.0 - w) * oof_prob_xgb
    f, t = best_threshold(y, p, thr_grid)
    if f > best[0]:
        best = (f, float(w), float(t))

best_f1_ens, best_w_logi, best_t_ens = best
oof_prob_ens = best_w_logi * oof_prob_logi + (1.0 - best_w_logi) * oof_prob_xgb
oof_pred_ens = (oof_prob_ens >= best_t_ens).astype(int)

ens_fold_f1 = []
for f in range(N_SPLITS):
    idx = (fold_id == f)
    ens_fold_f1.append(f1_score(y[idx], oof_pred_ens[idx], average="macro"))

cm = confusion_matrix(y, oof_pred_ens)

print("\n[Ensemble optimized on OOF pooled]")
print(f"  Best ensemble OOF macro-F1={best_f1_ens:.6f} @ weight_logi={best_w_logi:.2f}, threshold={best_t_ens:.3f}")
print(f"  Ensemble per-fold macro-F1 @ best: {[round(x,4) for x in ens_fold_f1]} | mean={np.mean(ens_fold_f1):.6f}")
print(f"  Confusion matrix (OOF @ best):\n{cm}")

if baseline_ref is not None:
    delta = best_f1_ens - baseline_ref["pooled_best_macro_f1"]
    print(f"[Compare vs iter0006] delta_pooled_best={delta:+.6f}")

# Fit full models
pipe_logi.fit(X_train, y)
pipe_xgb.fit(X_train, y)

test_prob_logi = pipe_logi.predict_proba(X_test)[:, 1]
test_prob_xgb  = pipe_xgb.predict_proba(X_test)[:, 1]
test_prob_ens  = best_w_logi * test_prob_logi + (1.0 - best_w_logi) * test_prob_xgb
test_pred = (test_prob_ens >= best_t_ens).astype(int)

# Submission
sub = pd.DataFrame({
    "stay_id": st_test["stay_id"].astype(int).values,
    "discharge_ready_day11": test_pred.astype(int),
})
sub.to_csv(submission_path, index=False)
print(f"\n[Saved] submission -> {submission_path.resolve()}")

# Artifacts
oof_df = pd.DataFrame({
    "stay_id": st_train["stay_id"].astype(int).values,
    "y_true": y,
    "oof_prob_logi": oof_prob_logi,
    "oof_prob_xgb": oof_prob_xgb,
    "oof_prob": oof_prob_ens,
    "oof_pred": oof_pred_ens.astype(int),
})
oof_pred_path = art_root / f"{ITER_TAG}_oof_predictions.csv"
oof_prob_path = art_root / f"{ITER_TAG}_oof_prob.npy"
test_prob_path = art_root / f"{ITER_TAG}_test_prob.npy"
fold_id_path = art_root / f"{ITER_TAG}_fold_id.npy"

oof_df.to_csv(oof_pred_path, index=False)
np.save(oof_prob_path, oof_prob_ens)
np.save(test_prob_path, test_prob_ens)
np.save(fold_id_path, fold_id)

# Checkpoint bundle
config = {
    "version": VERSION,
    "iteration_id": ITER_ID,
    "seed": SEED,
    "n_splits": N_SPLITS,
    "rollback_base": "iter0006",
    "feature_change": "ADD_CHAR_TFIDF_ALL=True (char TF-IDF on txt_all)",
    "logi_params": {
        "solver": "saga",
        "penalty": "elasticnet",
        "l1_ratio": 0.05,
        "C": 0.6,
        "class_weight": "balanced",
        "max_iter": 20000,
        "n_jobs": 1,
        "random_state": SEED,
    },
    "xgb_params": xgb_params,
    "tfidf_params": {
        "txt_all_w": {"max_features": 6000, "ngram_range": [1,2], "min_df": 2, "sublinear_tf": True},
        "txt_late_w": {"max_features": 2500, "ngram_range": [1,2], "min_df": 2, "sublinear_tf": True},
        "txt_d10_w": {"max_features": 2500, "ngram_range": [1,2], "min_df": 2, "sublinear_tf": True},
        "txt_late_c": {"max_features": 2000, "analyzer": "char_wb", "ngram_range": [3,5], "min_df": 2, "sublinear_tf": True},
        "txt_d10_c": {"max_features": 2000, "analyzer": "char_wb", "ngram_range": [3,5], "min_df": 2, "sublinear_tf": True},
        "txt_all_c": {"enabled": bool(ADD_CHAR_TFIDF_ALL), "max_features": 2000, "analyzer": "char_wb", "ngram_range": [3,5], "min_df": 2, "sublinear_tf": True},
    },
    "ensemble_search": {"weight_grid": w_grid.tolist(), "threshold_grid": thr_grid.tolist()},
}

schema = {
    "categorical_cols": CAT_COLS,
    "numeric_cols": NUM_COLS,
    "text_cols": TEXT_COLS,
    "num_feature_recipe": ["mean","std","min","max","last","missing","slope","last3_mean","last3_std","delta_last_first","raw_day_values(1..10)","len_*","lex_*"],
}

metrics = {
    "macro_f1_pooled_best": float(best_f1_ens),
    "macro_f1_mean": float(np.mean(ens_fold_f1)),
    "macro_f1_per_fold": [float(x) for x in ens_fold_f1],
    "confusion_matrix": cm.tolist(),
    "threshold": float(best_t_ens),
    "weight_logi": float(best_w_logi),
    "baseline_ref": baseline_ref,
}

p_star_path = ckpt_root / "p_star.json"
p_star = None
if p_star_path.exists():
    try:
        p_star = json.loads(p_star_path.read_text(encoding="utf-8"))
    except Exception:
        p_star = None
prev_best = p_star.get("best_macro_f1_pooled_best") if isinstance(p_star, dict) else None

p_star_updated = False
if (prev_best is None) or (float(best_f1_ens) > float(prev_best) + 1e-9):
    p_star_updated = True
    p_star = {
        "best_iteration_id": ITER_ID,
        "best_iter_tag": ITER_TAG,
        "best_macro_f1_pooled_best": float(best_f1_ens),
        "best_macro_f1_mean": float(np.mean(ens_fold_f1)),
        "timestamp": pd.Timestamp.utcnow().isoformat(),
        "checkpoint_dir": str(ckpt_dir),
    }
    p_star_path.write_text(json.dumps(p_star, ensure_ascii=False, indent=2), encoding="utf-8")

dump_error = None
model_bundle_path = ckpt_dir / "model_bundle.joblib"
try:
    joblib.dump(
        {"pipe_logi": pipe_logi, "pipe_xgb": pipe_xgb, "config": config, "metrics": metrics, "schema": schema},
        model_bundle_path
    )
except Exception as e:
    dump_error = repr(e)
    print(f"[WARN] joblib.dump failed: {dump_error}")

(ckpt_dir / "config.json").write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
(ckpt_dir / "schema.json").write_text(json.dumps(schema, ensure_ascii=False, indent=2), encoding="utf-8")
(ckpt_dir / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

# Append iteration log
hm_message = (
    "HM: realF1=0.7986 (regression). Instruction: rollback to iter0006 (real~0.8244) and now do ONLY "
    "one feature change per iteration after local OOF sanity-check."
)

log_entry = {
    "version": VERSION,
    "iteration_id": ITER_ID,
    "timestamp": pd.Timestamp.utcnow().isoformat(),
    "phase": "Modeling",
    "hm_message": hm_message,
    "real_score": 0.7986,
    "data_paths_used": {
        "stays_train": str(stays_train_path),
        "stays_test": str(stays_test_path),
        "patients": str(patients_path),
        "vitals_timeseries": str(vitals_path),
    },
    "join_keys": {
        "stays_to_patients": "patient_id",
        "stays_to_vitals_timeseries": "stay_id",
    },
    "join_notes": "Left-join patients on patient_id; vitals_timeseries features mapped by stay_id; text from day notes only.",
    "leakage_checks_performed": [
        f"patient_id overlap train vs test = {overlap} (expected 0)",
        "No use of discharge_notes.json for CH3 modeling.",
        "All features use only days 1-10 vitals + daily notes; target is day11 readiness.",
    ],
    "feature_summary": {
        "n_train": int(len(st_train)),
        "n_test": int(len(st_test)),
        "pos_rate_train": float(np.mean(y)),
        "num_cols": int(len(NUM_COLS)),
        "cat_cols": CAT_COLS,
        "text_cols": TEXT_COLS,
        "added_features_this_iter": ["char_tfidf(txt_all)"] if ADD_CHAR_TFIDF_ALL else [],
    },
    "models_attempted": [
        {
            "name": "LogisticRegression(saga, elasticnet) + [num+OHE+TFIDF(word+char)]",
            "params": config["logi_params"],
            "cv_macro_f1_mean_at_0.5": float(np.mean(fold_f1_logi)),
            "cv_macro_f1_per_fold_at_0.5": [float(x) for x in fold_f1_logi],
            "cv_macro_f1_oof_at_best_threshold": float(logi_best_f1),
            "oof_best_threshold": float(logi_best_t),
        },
        {
            "name": "XGBClassifier(structured-only) + [num+OHE]",
            "params": {"xgb_params": xgb_params},
            "cv_macro_f1_mean_at_0.5": float(np.mean(fold_f1_xgb)),
            "cv_macro_f1_per_fold_at_0.5": [float(x) for x in fold_f1_xgb],
            "cv_macro_f1_oof_at_best_threshold": float(xgb_best_f1),
            "oof_best_threshold": float(xgb_best_t),
        },
        {
            "name": "WeightedEnsemble(proba) + global threshold",
            "params": {"weight_logi_grid": w_grid.tolist(), "threshold_grid": thr_grid.tolist()},
            "best_oof_macro_f1": float(best_f1_ens),
            "best_weight_logi": float(best_w_logi),
            "best_threshold": float(best_t_ens),
            "macro_f1_per_fold_at_best": [float(x) for x in ens_fold_f1],
        },
    ],
    "selected_model": f"weighted_ensemble(logi_text+struct, xgb_struct) @ weight_logi={best_w_logi:.2f}, threshold={best_t_ens:.3f}",
    "validation_protocol": {
        "cv": "StratifiedKFold",
        "n_splits": int(N_SPLITS),
        "shuffle": True,
        "random_state": SEED,
        "threshold_selection": "Grid search on pooled OOF probs to maximize Macro-F1",
        "ensemble_weight_selection": "Grid search on pooled OOF probs to maximize Macro-F1",
    },
    "metrics": {
        "macro_f1_pooled_best": float(best_f1_ens),
        "macro_f1_mean": float(np.mean(ens_fold_f1)),
        "macro_f1_per_fold": [float(x) for x in ens_fold_f1],
        "confusion_matrix": cm.tolist(),
        "threshold": float(best_t_ens),
    },
    "checkpoint": {
        "checkpoint_dir": str(ckpt_dir),
        "p_star_updated": bool(p_star_updated),
        "p_star_path": str(p_star_path),
        "submission_path": str(submission_path),
        "oof_pred_path": str(oof_pred_path),
        "oof_prob_path": str(oof_prob_path),
        "test_prob_path": str(test_prob_path),
        "fold_id_path": str(fold_id_path),
        "model_bundle_path": str(model_bundle_path),
        "dump_backend": "joblib",
        "dump_error": dump_error,
        "model_bundle_written": dump_error is None,
    },
    "deltas_vs_previous": [
        "Rollback to iter0006 direction.",
        "ONE feature change: add char-level TF-IDF on txt_all (logistic branch only).",
        "No new structured interaction/abnormality/robust-summary features (previous regressions).",
    ],
    "known_defects_limitations": [
        "High-dimensional TF-IDF may overfit; rely on pooled OOF Macro-F1 and keep single-change iterations.",
        "Leaderboard Macro-F1 can differ; keep validation deterministic, consider future calibration if drift persists.",
        "XGBoost determinism can be sensitive to threading; we force n_jobs=1.",
    ],
    "next_step_hypothesis": "If this improves vs iter0006, next safe step is ONE additional text window (e.g., days 4-7). If it regresses, revert this and try a different single text feature.",
}

with log_path.open("a", encoding="utf-8") as f:
    f.write(json.dumps(log_entry, ensure_ascii=False) + "\n")

print(f"[Log] appended -> {log_path.resolve()}")
print(f"[Artifacts] {oof_pred_path.name}, {oof_prob_path.name}, {test_prob_path.name}, {fold_id_path.name}")
print(f"[Checkpoint] {ckpt_dir}")
print(f"[P*] updated={p_star_updated} | best_macro_f1_pooled_best={p_star.get('best_macro_f1_pooled_best') if isinstance(p_star, dict) else None}")

[Info] patient_id overlap train vs test: 0 (expected 0)
[Baseline iter0006] pooled_best=0.823975 @ w=0.35, t=0.600

[CV] Running 5-fold StratifiedKFold...

[Results @0.5]
  Logistic (0.5) fold macro-F1: [0.7559, 0.7419, 0.7551, 0.7426, 0.7721] | mean=0.753536
  XGB/Structured (0.5) fold macro-F1: [0.8078, 0.7828, 0.7981, 0.7852, 0.752] | mean=0.785167

[Best threshold (OOF pooled)]
  Logistic: best_f1=0.759681 @ t=0.380
  XGB/Structured: best_f1=0.790896 @ t=0.750

[Ensemble optimized on OOF pooled]
  Best ensemble OOF macro-F1=0.798423 @ weight_logi=0.35, threshold=0.550
  Ensemble per-fold macro-F1 @ best: [0.823, 0.803, 0.7916, 0.7883, 0.7847] | mean=0.798138
  Confusion matrix (OOF @ best):
[[240 104]
 [ 74 582]]
[Compare vs iter0006] delta_pooled_best=-0.025552

[Saved] submission -> D:\AgentDs\agent_ds_healthcare\clai_ch3_v1_submission.csv
[Log] appended -> D:\AgentDs\agent_ds_healthcare\clai_ch3_v1_iteration_detail.jsonl
[Artifacts] iter0013_oof_predictions.csv, iter0013_oof_pro

# Iteration 14
- 0.8244

In [46]:
import os, json, time, shutil, random
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.base import clone

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

import joblib

# =========================
# Config (v1)
# =========================
VX = "v1"
TEAM_TAG = "clai"
CH = "ch3"
SEED = 42
N_SPLITS = 5

# HM instruction: rollback to iter0006 baseline and do only 1 feature change
BASELINE_ITER = 6  # iter0006 is the 0.8244-real / 0.823975-OOF base in prior logs

# The ONE allowed change this iteration:
ADD_ONE_FEATURE = True
ONE_FEATURE_NAME = "barrier_day10"  # derived from day10 note (single binary feature)

# Ensemble neighborhood guard (small changes only)
BASE_W = 0.35
BASE_T = 0.60
WEIGHT_GRID = np.round(np.arange(max(0.0, BASE_W - 0.20), min(1.0, BASE_W + 0.20) + 1e-9, 0.05), 2)
THR_GRID = np.round(np.arange(max(0.05, BASE_T - 0.15), min(0.95, BASE_T + 0.15) + 1e-9, 0.01), 2)

# I/O
log_path = f"{TEAM_TAG}_{CH}_{VX}_iteration_detail.jsonl"
submission_path = f"{TEAM_TAG}_{CH}_{VX}_submission.csv"
ckpt_root = os.path.join("checkpoints", f"{TEAM_TAG}_{CH}_{VX}")
art_root = os.path.join("artifacts", f"{TEAM_TAG}_{CH}_{VX}")
os.makedirs(ckpt_root, exist_ok=True)
os.makedirs(art_root, exist_ok=True)

# Determinism
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# =========================
# Helpers
# =========================
def now_iso():
    return datetime.now(timezone.utc).isoformat()

def read_jsonl_max_iter(path):
    if not os.path.exists(path):
        return -1
    mx = -1
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                mx = max(mx, int(obj.get("iteration_id", -1)))
            except Exception:
                continue
    return mx

def append_jsonl(path, obj):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def macro_f1_at_threshold(y_true, proba, thr):
    y_pred = (proba >= thr).astype(int)
    return f1_score(y_true, y_pred, average="macro")

def find_best_threshold(y_true, proba, thresholds=np.round(np.arange(0.05, 0.951, 0.01), 2)):
    best_thr = 0.5
    best_f1 = -1.0
    for t in thresholds:
        f1v = macro_f1_at_threshold(y_true, proba, float(t))
        if f1v > best_f1:
            best_f1 = float(f1v)
            best_thr = float(t)
    return best_f1, best_thr

def optimize_ensemble(y_true, proba_logi, proba_xgb, weight_grid, thr_grid):
    best = {"f1": -1.0, "w": float(BASE_W), "t": float(BASE_T)}
    for w in weight_grid:
        p = w * proba_logi + (1.0 - w) * proba_xgb
        for t in thr_grid:
            f1v = macro_f1_at_threshold(y_true, p, float(t))
            if f1v > best["f1"]:
                best = {"f1": float(f1v), "w": float(w), "t": float(t)}
    return best

def flatten_text(x):
    # NOTE: no lambda -> pickle-safe
    if isinstance(x, pd.DataFrame):
        x = x.iloc[:, 0]
    if isinstance(x, pd.Series):
        return x.astype(str).fillna("").values
    arr = np.asarray(x)
    if arr.ndim == 2 and arr.shape[1] == 1:
        arr = arr[:, 0]
    return pd.Series(arr).astype(str).fillna("").values

def safe_stats(arr):
    arr = np.asarray(arr, dtype=float)
    valid = arr[~np.isnan(arr)]
    if valid.size == 0:
        return dict(mean=np.nan, std=np.nan, min=np.nan, max=np.nan, first=np.nan, last=np.nan, missing=int(np.isnan(arr).sum()), range=np.nan)
    return dict(
        mean=float(np.nanmean(arr)),
        std=float(np.nanstd(arr)),
        min=float(np.nanmin(arr)),
        max=float(np.nanmax(arr)),
        first=float(arr[0]) if not np.isnan(arr[0]) else np.nan,
        last=float(arr[-1]) if not np.isnan(arr[-1]) else np.nan,
        missing=int(np.isnan(arr).sum()),
        range=float(np.nanmax(arr) - np.nanmin(arr)),
    )

def slope_from_days(days, values):
    xs, ys = [], []
    for d, v in zip(days, values):
        if v is None:
            continue
        try:
            fv = float(v)
        except Exception:
            continue
        if np.isnan(fv):
            continue
        xs.append(float(d))
        ys.append(fv)
    if len(xs) < 2:
        return np.nan
    x = np.asarray(xs, dtype=float)
    y = np.asarray(ys, dtype=float)
    xm = x.mean()
    denom = ((x - xm) ** 2).sum()
    if denom == 0:
        return 0.0
    return float(((x - xm) * (y - y.mean())).sum() / denom)

def build_vitals_features(stays_df, vitals_map, include_barrier_day10=False):
    vital_cols = ["hr", "sbp", "dbp", "temp_c", "rr"]
    barrier_terms = [
        "awaiting placement", "awaiting", "placement", "rehab", "snf",
        "iv antibiotics", "oxygen requirement", "hypotensive", "febrile",
        "delirium", "fall risk"
    ]

    rows = []
    for _, r in stays_df.iterrows():
        sid = int(r["stay_id"])
        days = vitals_map.get(sid, [])
        day_dict = {int(d.get("day")): d for d in days if d.get("day") is not None}

        feats = {"stay_id": sid}

        # raw day vitals (day1..day10)
        for day in range(1, 11):
            dd = day_dict.get(day, {})
            for vc in vital_cols:
                v = dd.get(vc, np.nan)
                feats[f"{vc}_d{day}"] = (np.nan if v is None else v)

        # summary stats per vital + last3 stats + slope + delta
        for vc in vital_cols:
            arr = np.array([np.nan if day_dict.get(day, {}).get(vc, None) is None else float(day_dict.get(day, {}).get(vc)) for day in range(1, 11)], dtype=float)
            st = safe_stats(arr)
            feats[f"{vc}_mean"] = st["mean"]
            feats[f"{vc}_std"] = st["std"]
            feats[f"{vc}_min"] = st["min"]
            feats[f"{vc}_max"] = st["max"]
            feats[f"{vc}_first"] = st["first"]
            feats[f"{vc}_last"] = st["last"]
            feats[f"{vc}_missing"] = st["missing"]
            feats[f"{vc}_range"] = st["range"]
            feats[f"{vc}_slope"] = slope_from_days(list(range(1, 11)), arr.tolist())
            last3 = arr[-3:]
            st3 = safe_stats(last3)
            feats[f"{vc}_last3_mean"] = st3["mean"]
            feats[f"{vc}_last3_std"] = st3["std"]
            feats[f"{vc}_delta_last_first"] = (arr[-1] - arr[0]) if (not np.isnan(arr[-1]) and not np.isnan(arr[0])) else np.nan

        # text windows
        notes = []
        for day in range(1, 11):
            note = day_dict.get(day, {}).get("note", "")
            note = "" if note is None else str(note)
            notes.append(note)
        txt_all = " ".join([n for n in notes if n and n.lower() != "nan"])
        txt_late3 = " ".join([str(day_dict.get(day, {}).get("note", "") or "") for day in range(8, 11)])
        txt_day10 = str(day_dict.get(10, {}).get("note", "") or "")

        feats["txt_all"] = txt_all
        feats["txt_late3"] = txt_late3
        feats["txt_day10"] = txt_day10
        feats["len_all"] = len(txt_all)
        feats["len_late3"] = len(txt_late3)
        feats["len_day10"] = len(txt_day10)

        # iter0006-compatible lex counts (kept unchanged)
        pos_words = ["discharged", "home", "stable", "improved", "independent", "ready", "cleared", "pain controlled", "vitals stable", "pt cleared", "ot cleared"]
        neg_words = ["awaiting placement", "rehab", "snf", "needs assistance", "unstable", "hypotensive", "febrile", "infection", "cultures", "oxygen requirement", "delirium", "fall risk", "iv antibiotics"]
        low_txt = txt_all.lower()
        pos = sum(low_txt.count(w) for w in pos_words)
        neg = sum(low_txt.count(w) for w in neg_words)
        feats["lex_pos"] = pos
        feats["lex_neg"] = neg
        feats["lex_diff"] = pos - neg

        # THE ONE NEW FEATURE (binary)
        if include_barrier_day10:
            low10 = txt_day10.lower()
            feats[ONE_FEATURE_NAME] = int(any(t in low10 for t in barrier_terms))

        rows.append(feats)

    return pd.DataFrame(rows)

def locate_file(candidates):
    for p in candidates:
        if p and os.path.exists(p):
            return p
    return None

# =========================
# Iteration bookkeeping
# =========================
prev_max_iter = read_jsonl_max_iter(log_path)
iteration_id = prev_max_iter + 1
iter_tag = f"iter{iteration_id:04d}"
ckpt_dir = os.path.join(ckpt_root, iter_tag)
os.makedirs(ckpt_dir, exist_ok=True)

hm_message = (
    "HM: rollback to iter0006 (real≈0.8244, OOF pooled-best≈0.823975). "
    "From this base, only ONE feature change per iteration; avoid risky expansions."
)

# =========================
# Load data
# =========================
stays_train = pd.read_csv("stays_train.csv")
stays_test = pd.read_csv("stays_test.csv")
patients = pd.read_csv("patients.csv")
with open("vitals_timeseries.json", "r", encoding="utf-8") as f:
    vitals_list = json.load(f)
vitals_map = {int(x["stay_id"]): x["days"] for x in vitals_list}

# Leakage checks
train_pat = set(stays_train["patient_id"].tolist())
test_pat = set(stays_test["patient_id"].tolist())
pat_overlap = len(train_pat.intersection(test_pat))
stay_overlap = len(set(stays_train["stay_id"]).intersection(set(stays_test["stay_id"])))

print(f"[Info] patient_id overlap train vs test: {pat_overlap} (expected 0)")
print(f"[Info] stay_id overlap train vs test: {stay_overlap} (expected 0)")
print(f"[Data] stays_train: {stays_train.shape} | stays_test: {stays_test.shape} | patients: {patients.shape}")

# =========================
# Load baseline (iter0006) safety net
# =========================
baseline_oof_path = locate_file([
    os.path.join(art_root, f"iter{BASELINE_ITER:04d}_oof_prob.npy"),
    f"iter{BASELINE_ITER:04d}_oof_prob.npy",
])
baseline_test_path = locate_file([
    os.path.join(art_root, f"iter{BASELINE_ITER:04d}_test_prob.npy"),
    f"iter{BASELINE_ITER:04d}_test_prob.npy",
])
baseline_fold_path = locate_file([
    os.path.join(art_root, f"iter{BASELINE_ITER:04d}_fold_id.npy"),
    f"iter{BASELINE_ITER:04d}_fold_id.npy",
])

y = stays_train["discharge_ready_day11"].values.astype(int)

baseline_available = baseline_oof_path is not None and baseline_test_path is not None
baseline_best_f1 = None
baseline_best_t = BASE_T
if baseline_available:
    base_oof = np.load(baseline_oof_path)
    baseline_best_f1, baseline_best_t = find_best_threshold(y, base_oof)
    print(f"[Baseline iter{BASELINE_ITER:04d}] pooled_best={baseline_best_f1:.6f} @ t={baseline_best_t:.3f} | oof_path={baseline_oof_path}")
else:
    print(f"[Baseline iter{BASELINE_ITER:04d}] artifacts not found -> will compute baseline via CV as fallback (slower).")

# =========================
# Build train/test features (candidate only; one feature change)
# =========================
# Base joins
train_base = stays_train.merge(patients, on="patient_id", how="left")
test_base = stays_test.merge(patients, on="patient_id", how="left")

# Add reason_unit and zip3 as string for OHE
for df in (train_base, test_base):
    df["zip3"] = df["zip3"].astype(str)
    df["reason_unit"] = df["admission_reason"].astype(str) + "__" + df["unit_type"].astype(str)

# Candidate vitals/text features
train_vfeat = build_vitals_features(train_base[["stay_id"]], vitals_map, include_barrier_day10=ADD_ONE_FEATURE)
test_vfeat  = build_vitals_features(test_base[["stay_id"]],  vitals_map, include_barrier_day10=ADD_ONE_FEATURE)

train_df = train_base.merge(train_vfeat, on="stay_id", how="left")
test_df  = test_base.merge(test_vfeat,  on="stay_id", how="left")

text_cols = ["txt_all", "txt_late3", "txt_day10"]
cat_cols = ["unit_type", "admission_reason", "sex", "insurance", "zip3", "reason_unit"]
exclude = set(["stay_id", "patient_id", "discharge_ready_day11"] + text_cols + cat_cols)
num_cols = [c for c in train_df.columns if c not in exclude and pd.api.types.is_numeric_dtype(train_df[c])]

# Feature summary sanity
print(f"[Features] num_cols={len(num_cols)} | cat_cols={len(cat_cols)} | text_cols={len(text_cols)} | one_feature_added={ADD_ONE_FEATURE}")

# =========================
# Build models (same direction as iter0006)
# =========================
# TF-IDF configs (kept moderate; iter0006-style word+char)
tfidf_word_all = TfidfVectorizer(max_features=6000, ngram_range=(1,2), min_df=2, sublinear_tf=True, strip_accents="unicode")
tfidf_word_small = TfidfVectorizer(max_features=3000, ngram_range=(1,2), min_df=2, sublinear_tf=True, strip_accents="unicode")
tfidf_char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=2, sublinear_tf=True, max_features=8000)

txt_all_pipe = Pipeline([
    ("flatten", FunctionTransformer(flatten_text, validate=False)),
    ("tfidf", tfidf_word_all),
])
txt_late_pipe = FeatureUnion([
    ("w", Pipeline([("flatten", FunctionTransformer(flatten_text, validate=False)), ("tfidf", tfidf_word_small)])),
    ("c", Pipeline([("flatten", FunctionTransformer(flatten_text, validate=False)), ("tfidf", tfidf_char)])),
])
txt_day10_pipe = FeatureUnion([
    ("w", Pipeline([("flatten", FunctionTransformer(flatten_text, validate=False)), ("tfidf", tfidf_word_small)])),
    ("c", Pipeline([("flatten", FunctionTransformer(flatten_text, validate=False)), ("tfidf", tfidf_char)])),
])

preprocess_logi = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler(with_mean=False))]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("txt_all", txt_all_pipe, "txt_all"),
        ("txt_late3", txt_late_pipe, "txt_late3"),
        ("txt_day10", txt_day10_pipe, "txt_day10"),
    ],
    sparse_threshold=0.3,
)

logi = LogisticRegression(
    solver="saga",
    penalty="elasticnet",
    l1_ratio=0.05,
    C=0.6,
    class_weight="balanced",
    max_iter=20000,
    random_state=SEED,
    n_jobs=1,
)
pipe_logi = Pipeline([("pre", preprocess_logi), ("clf", logi)])

# XGB structured-only (fallback to sklearn if xgboost missing)
has_xgb = True
try:
    import xgboost as xgb
except Exception as e:
    has_xgb = False
    from sklearn.ensemble import HistGradientBoostingClassifier

preprocess_xgb = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    sparse_threshold=0.3,
)

if has_xgb:
    xgb_clf = xgb.XGBClassifier(
        n_estimators=450,
        learning_rate=0.05,
        max_depth=4,
        min_child_weight=2.0,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=SEED,
        n_jobs=1,
        reg_alpha=0.0,
        reg_lambda=1.0,
        gamma=0.0,
    )
else:
    xgb_clf = HistGradientBoostingClassifier(random_state=SEED)

pipe_xgb = Pipeline([("pre", preprocess_xgb), ("clf", xgb_clf)])

# =========================
# CV + OOF evaluation (candidate)
# =========================
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_logi = np.zeros(len(train_df), dtype=float)
oof_xgb  = np.zeros(len(train_df), dtype=float)
fold_id  = np.zeros(len(train_df), dtype=int)

fold_f1_logi = []
fold_f1_xgb  = []

print(f"[CV] Running {N_SPLITS}-fold StratifiedKFold...")

for f, (tr_idx, va_idx) in enumerate(cv.split(train_df, y)):
    fold_id[va_idx] = f
    X_tr = train_df.iloc[tr_idx]
    y_tr = y[tr_idx]
    X_va = train_df.iloc[va_idx]
    y_va = y[va_idx]

    m_logi = clone(pipe_logi)
    m_xgb  = clone(pipe_xgb)

    m_logi.fit(X_tr, y_tr)
    m_xgb.fit(X_tr, y_tr)

    p1 = m_logi.predict_proba(X_va)[:, 1]
    p2 = m_xgb.predict_proba(X_va)[:, 1]

    oof_logi[va_idx] = p1
    oof_xgb[va_idx]  = p2

    fold_f1_logi.append(macro_f1_at_threshold(y_va, p1, 0.5))
    fold_f1_xgb.append(macro_f1_at_threshold(y_va, p2, 0.5))

print("\n[Results @0.5]")
print(f"  Logistic (0.5) fold macro-F1: {[round(x,4) for x in fold_f1_logi]} | mean={np.mean(fold_f1_logi):.6f}")
print(f"  XGB/Structured (0.5) fold macro-F1: {[round(x,4) for x in fold_f1_xgb]} | mean={np.mean(fold_f1_xgb):.6f}")

# Best thresholds (pooled OOF)
logi_best_f1, logi_best_t = find_best_threshold(y, oof_logi)
xgb_best_f1,  xgb_best_t  = find_best_threshold(y, oof_xgb)

print("\n[Best threshold (OOF pooled)]")
print(f"  Logistic: best_f1={logi_best_f1:.6f} @ t={logi_best_t:.3f}")
print(f"  XGB/Structured: best_f1={xgb_best_f1:.6f} @ t={xgb_best_t:.3f}")

# Ensemble (restricted neighborhood grid; small change guard)
ens_best = optimize_ensemble(y, oof_logi, oof_xgb, WEIGHT_GRID, THR_GRID)
oof_ens = ens_best["w"] * oof_logi + (1.0 - ens_best["w"]) * oof_xgb
cm = confusion_matrix(y, (oof_ens >= ens_best["t"]).astype(int))

# Per-fold ensemble F1 at best
ens_fold_f1 = []
for f in range(N_SPLITS):
    idx = np.where(fold_id == f)[0]
    ens_fold_f1.append(macro_f1_at_threshold(y[idx], oof_ens[idx], ens_best["t"]))

print("\n[Ensemble optimized on OOF pooled (guarded neighborhood)]")
print(f"  Best ensemble OOF macro-F1={ens_best['f1']:.6f} @ weight_logi={ens_best['w']:.2f}, threshold={ens_best['t']:.3f}")
print(f"  Ensemble per-fold macro-F1 @ best: {[round(x,4) for x in ens_fold_f1]} | mean={np.mean(ens_fold_f1):.6f}")
print(f"  Confusion matrix (OOF @ best):\n{cm}")

candidate_best_f1 = ens_best["f1"]

# =========================
# Baseline fallback compute (only if artifacts missing)
# =========================
baseline_computed = False
if not baseline_available:
    # Build baseline feature tables by recomputing without the one feature
    train_vfeat_base = build_vitals_features(train_base[["stay_id"]], vitals_map, include_barrier_day10=False)
    test_vfeat_base  = build_vitals_features(test_base[["stay_id"]],  vitals_map, include_barrier_day10=False)
    train_df_base = train_base.merge(train_vfeat_base, on="stay_id", how="left")
    test_df_base  = test_base.merge(test_vfeat_base,  on="stay_id", how="left")

    exclude_b = set(["stay_id", "patient_id", "discharge_ready_day11"] + text_cols + cat_cols)
    num_cols_b = [c for c in train_df_base.columns if c not in exclude_b and pd.api.types.is_numeric_dtype(train_df_base[c])]

    preprocess_logi_b = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler(with_mean=False))]), num_cols_b),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
            ("txt_all", txt_all_pipe, "txt_all"),
            ("txt_late3", txt_late_pipe, "txt_late3"),
            ("txt_day10", txt_day10_pipe, "txt_day10"),
        ],
        sparse_threshold=0.3,
    )
    pipe_logi_b = Pipeline([("pre", preprocess_logi_b), ("clf", clone(logi))])

    preprocess_xgb_b = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), num_cols_b),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ],
        sparse_threshold=0.3,
    )
    if has_xgb:
        pipe_xgb_b = Pipeline([("pre", preprocess_xgb_b), ("clf", clone(xgb_clf))])
    else:
        pipe_xgb_b = Pipeline([("pre", preprocess_xgb_b), ("clf", clone(xgb_clf))])

    oof_logi_b = np.zeros(len(train_df_base), dtype=float)
    oof_xgb_b  = np.zeros(len(train_df_base), dtype=float)

    for f, (tr_idx, va_idx) in enumerate(cv.split(train_df_base, y)):
        X_tr = train_df_base.iloc[tr_idx]
        y_tr = y[tr_idx]
        X_va = train_df_base.iloc[va_idx]

        m1 = clone(pipe_logi_b).fit(X_tr, y_tr)
        m2 = clone(pipe_xgb_b).fit(X_tr, y_tr)

        oof_logi_b[va_idx] = m1.predict_proba(X_va)[:, 1]
        oof_xgb_b[va_idx]  = m2.predict_proba(X_va)[:, 1]

    ens_best_b = optimize_ensemble(y, oof_logi_b, oof_xgb_b, WEIGHT_GRID, THR_GRID)
    baseline_best_f1 = ens_best_b["f1"]
    baseline_best_t  = ens_best_b["t"]
    baseline_computed = True
    print(f"\n[Baseline recomputed] pooled_best={baseline_best_f1:.6f} @ w={ens_best_b['w']:.2f}, t={baseline_best_t:.3f}")

# =========================
# Select model for submission (never regress vs baseline)
# =========================
EPS = 1e-4
use_candidate = False
if baseline_best_f1 is None:
    # Shouldn't happen, but be safe
    use_candidate = True
else:
    use_candidate = (candidate_best_f1 > baseline_best_f1 + EPS)

print("\n[Selection]")
print(f"  baseline_best_f1={baseline_best_f1:.6f} | candidate_best_f1={candidate_best_f1:.6f} | use_candidate={use_candidate}")

# =========================
# Fit final model(s) and predict test
# =========================
# Candidate final predictions
pipe_logi_full = clone(pipe_logi).fit(train_df, y)
pipe_xgb_full  = clone(pipe_xgb).fit(train_df, y)
test_proba_logi = pipe_logi_full.predict_proba(test_df)[:, 1]
test_proba_xgb  = pipe_xgb_full.predict_proba(test_df)[:, 1]
test_proba_ens  = ens_best["w"] * test_proba_logi + (1.0 - ens_best["w"]) * test_proba_xgb
test_pred_cand  = (test_proba_ens >= ens_best["t"]).astype(int)

# Baseline fallback predictions (from artifacts if available)
baseline_pred = None
baseline_test_prob = None
if baseline_available:
    baseline_test_prob = np.load(baseline_test_path)
    baseline_pred = (baseline_test_prob >= baseline_best_t).astype(int)

# Choose final
if use_candidate:
    final_pred = test_pred_cand
    final_meta = {"name": "candidate_iter"+iter_tag, "one_feature_added": ONE_FEATURE_NAME, "ensemble_w": ens_best["w"], "threshold": ens_best["t"], "oof_best_f1": candidate_best_f1}
else:
    if baseline_pred is not None:
        final_pred = baseline_pred
        final_meta = {"name": f"baseline_iter{BASELINE_ITER:04d}_fallback", "threshold": baseline_best_t, "oof_best_f1": baseline_best_f1}
    else:
        # If baseline artifacts missing, fallback to candidate (but this should be rare)
        final_pred = test_pred_cand
        final_meta = {"name": "candidate_fallback_no_baseline", "one_feature_added": ONE_FEATURE_NAME, "ensemble_w": ens_best["w"], "threshold": ens_best["t"], "oof_best_f1": candidate_best_f1}

# Write submission
sub = pd.DataFrame({"stay_id": test_df["stay_id"].astype(int), "discharge_ready_day11": final_pred.astype(int)})
sub.to_csv(submission_path, index=False)
print(f"\n[Saved] submission -> {os.path.abspath(submission_path)}")

# =========================
# Save artifacts for this iteration
# =========================
oof_ens_path = os.path.join(art_root, f"{iter_tag}_oof_prob.npy")
test_ens_path = os.path.join(art_root, f"{iter_tag}_test_prob.npy")
fold_path = os.path.join(art_root, f"{iter_tag}_fold_id.npy")
oof_pred_csv = os.path.join(art_root, f"{iter_tag}_oof_predictions.csv")

np.save(oof_ens_path, oof_ens)
np.save(test_ens_path, test_proba_ens)
np.save(fold_path, fold_id)

oof_df = pd.DataFrame({
    "stay_id": train_df["stay_id"].astype(int),
    "y_true": y.astype(int),
    "proba_logi": oof_logi,
    "proba_xgb": oof_xgb,
    "proba_ens": oof_ens,
    "pred_ens": (oof_ens >= ens_best["t"]).astype(int),
    "fold": fold_id.astype(int),
})
oof_df.to_csv(oof_pred_csv, index=False)

print(f"[Artifacts] {oof_pred_csv}, {oof_ens_path}, {test_ens_path}, {fold_path}")

# =========================
# Checkpoint: save config, schema, metrics, model bundle
# =========================
config = {
    "version": VX,
    "seed": SEED,
    "n_splits": N_SPLITS,
    "baseline_iter": BASELINE_ITER,
    "one_feature_added": ADD_ONE_FEATURE,
    "one_feature_name": ONE_FEATURE_NAME if ADD_ONE_FEATURE else None,
    "ensemble_grid_guard": {"base_w": BASE_W, "base_t": BASE_T, "weight_grid": WEIGHT_GRID.tolist(), "thr_grid": THR_GRID.tolist()},
    "tfidf": {
        "word_all_max_features": 6000,
        "word_small_max_features": 3000,
        "char_max_features": 8000,
        "char_ngram_range": [3,5],
        "word_ngram_range": [1,2],
        "min_df": 2,
        "sublinear_tf": True,
    },
    "logi_params": {
        "solver": "saga", "penalty": "elasticnet", "l1_ratio": 0.05, "C": 0.6, "class_weight": "balanced",
        "max_iter": 20000, "random_state": SEED, "n_jobs": 1,
    },
    "xgb_params": (xgb_clf.get_params() if has_xgb else {"fallback": "HistGradientBoostingClassifier"}),
}

schema = {
    "cat_cols": cat_cols,
    "text_cols": text_cols,
    "num_cols_count": len(num_cols),
    "one_feature_name": ONE_FEATURE_NAME if ADD_ONE_FEATURE else None,
}

metrics = {
    "baseline": {"available": baseline_available, "pooled_best_f1": baseline_best_f1, "best_threshold": baseline_best_t},
    "candidate": {
        "macro_f1_mean_at_0.5": {"logi": float(np.mean(fold_f1_logi)), "xgb": float(np.mean(fold_f1_xgb))},
        "macro_f1_per_fold_at_0.5": {"logi": [float(x) for x in fold_f1_logi], "xgb": [float(x) for x in fold_f1_xgb]},
        "best_oof": {"logi_best_f1": logi_best_f1, "logi_best_threshold": logi_best_t, "xgb_best_f1": xgb_best_f1, "xgb_best_threshold": xgb_best_t},
        "ensemble_best": {"ensemble_best_f1": candidate_best_f1, "ensemble_weight_logi": ens_best["w"], "ensemble_threshold": ens_best["t"]},
        "ensemble_per_fold_at_best": [float(x) for x in ens_fold_f1],
        "confusion_matrix_oof_at_best": cm.tolist(),
    },
    "selected": final_meta,
}

# P* guard file (never regress)
pstar_path = os.path.join(ckpt_root, "pstar.json")
pstar = {"best_macro_f1_pooled_best": None, "best_iteration_id": None}
if os.path.exists(pstar_path):
    try:
        with open(pstar_path, "r", encoding="utf-8") as f:
            pstar = json.load(f)
    except Exception:
        pass

# enforce baseline as minimum P* per HM rollback instruction
best_so_far = pstar.get("best_macro_f1_pooled_best")
if best_so_far is None or (baseline_best_f1 is not None and float(best_so_far) < float(baseline_best_f1)):
    pstar["best_macro_f1_pooled_best"] = float(baseline_best_f1) if baseline_best_f1 is not None else best_so_far
    pstar["best_iteration_id"] = f"iter{BASELINE_ITER:04d}"

pstar_updated = False
if use_candidate and candidate_best_f1 is not None:
    if pstar.get("best_macro_f1_pooled_best") is None or float(candidate_best_f1) > float(pstar["best_macro_f1_pooled_best"]) + EPS:
        pstar["best_macro_f1_pooled_best"] = float(candidate_best_f1)
        pstar["best_iteration_id"] = iter_tag
        pstar_updated = True

with open(pstar_path, "w", encoding="utf-8") as f:
    json.dump(pstar, f, ensure_ascii=False, indent=2)

# Save checkpoint files
with open(os.path.join(ckpt_dir, "config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
with open(os.path.join(ckpt_dir, "schema.json"), "w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)
with open(os.path.join(ckpt_dir, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

joblib_dump_ok = True
joblib_error = None

try:
    bundle_path = os.path.join(ckpt_dir, "model_bundle.joblib")
    if use_candidate:
        joblib.dump({"pipe_logi": pipe_logi_full, "pipe_xgb": pipe_xgb_full, "ensemble": ens_best, "config": config, "metrics": metrics}, bundle_path)
    else:
        # If baseline is selected, try to copy baseline checkpoint bundle if it exists; else still save candidate bundle (as audit) but mark selection as baseline.
        baseline_bundle = locate_file([
            os.path.join(ckpt_root, f"iter{BASELINE_ITER:04d}", "model_bundle.joblib"),
            os.path.join("checkpoints", f"{TEAM_TAG}_{CH}_{VX}", f"iter{BASELINE_ITER:04d}", "model_bundle.joblib"),
        ])
        if baseline_bundle is not None:
            shutil.copy2(baseline_bundle, bundle_path)
        else:
            # fallback: store candidate bundle but selection metadata indicates baseline fallback
            joblib.dump({"pipe_logi": pipe_logi_full, "pipe_xgb": pipe_xgb_full, "ensemble": ens_best, "config": config, "metrics": metrics}, bundle_path)
except Exception as e:
    joblib_dump_ok = False
    joblib_error = str(e)

print(f"[Checkpoint] {ckpt_dir}")
print(f"[P*] updated={pstar_updated} | best_macro_f1_pooled_best={pstar.get('best_macro_f1_pooled_best')} | best_iter={pstar.get('best_iteration_id')}")
if not joblib_dump_ok:
    print(f"[Warn] joblib dump failed: {joblib_error}")

# =========================
# Append iteration log JSONL
# =========================
log_entry = {
    "version": VX,
    "iteration_id": iteration_id,
    "timestamp": now_iso(),
    "phase": "Modeling",
    "hm_message": hm_message,
    "real_score": None,

    "data_paths_used": {
        "stays_train": "stays_train.csv",
        "stays_test": "stays_test.csv",
        "patients": "patients.csv",
        "vitals_timeseries": "vitals_timeseries.json",
    },
    "join_keys": {"stays_to_patients": "patient_id", "stays_to_vitals": "stay_id"},
    "leakage_checks_performed": {
        "patient_id_overlap_train_test": pat_overlap,
        "stay_id_overlap_train_test": stay_overlap,
        "notes_used_days": "day1-10 only (no day11+ text present)",
    },

    "feature_summary": {
        "n_train": int(stays_train.shape[0]),
        "n_test": int(stays_test.shape[0]),
        "num_cols": int(len(num_cols)),
        "cat_cols": int(len(cat_cols)),
        "text_cols": text_cols,
        "added_features": [ONE_FEATURE_NAME] if ADD_ONE_FEATURE else [],
        "rollback_base": f"iter{BASELINE_ITER:04d}",
    },

    "models_attempted": [
        {"name": f"Baseline iter{BASELINE_ITER:04d} (loaded if available)", "oof_pooled_best_f1": baseline_best_f1, "best_threshold": baseline_best_t, "available": baseline_available},
        {"name": "LogisticRegression(saga, elasticnet) + [num+OHE+TFIDF(word+char)]", "params": config["logi_params"], "cv_macro_f1_mean_at_0.5": float(np.mean(fold_f1_logi)), "cv_macro_f1_per_fold_at_0.5": [float(x) for x in fold_f1_logi], "oof_best_threshold": logi_best_t, "cv_macro_f1_oof_at_best_threshold": logi_best_f1},
        {"name": "XGBClassifier(structured-only) + [num+OHE]", "params": {"has_xgb": has_xgb, "params": (xgb_clf.get_params() if has_xgb else {})}, "cv_macro_f1_mean_at_0.5": float(np.mean(fold_f1_xgb)), "cv_macro_f1_per_fold_at_0.5": [float(x) for x in fold_f1_xgb], "oof_best_threshold": xgb_best_t, "cv_macro_f1_oof_at_best_threshold": xgb_best_f1},
        {"name": "Guarded weighted ensemble (neighborhood search)", "ensemble_best_f1": candidate_best_f1, "ensemble_weight_logi": ens_best["w"], "ensemble_threshold": ens_best["t"], "ensemble_per_fold_at_best": [float(x) for x in ens_fold_f1]},
    ],

    "selected_model": final_meta,

    "validation_protocol": {
        "cv": f"StratifiedKFold(n_splits={N_SPLITS}, shuffle=True, random_state={SEED})",
        "threshold_tuning": "OOF pooled search (guarded neighborhood)",
        "seeds": {"numpy": SEED, "random": SEED},
    },

    "metrics": {
        "baseline_pooled_best": baseline_best_f1,
        "candidate_pooled_best": candidate_best_f1,
        "selected": final_meta,
        "confusion_matrix_oof_at_candidate_best": cm.tolist(),
        "joblib_dump_ok": joblib_dump_ok,
        "joblib_error": joblib_error,
        "submission_path": os.path.abspath(submission_path),
        "pstar_updated": pstar_updated,
        "pstar": pstar,
    },

    "deltas_vs_previous_iteration": {
        "delta_candidate_vs_baseline": (float(candidate_best_f1) - float(baseline_best_f1)) if (candidate_best_f1 is not None and baseline_best_f1 is not None) else None,
        "guardrail": "never ship worse-than-baseline; fallback to iter0006 probs if candidate not better",
    },

    "checkpoint": {"dir": os.path.abspath(ckpt_dir), "pstar_path": os.path.abspath(pstar_path)},
    "known_defects_limitations": [
        "Single new keyword-based feature may be redundant with TF-IDF; might not improve.",
        "If baseline artifacts are missing, fallback baseline recomputation is slower.",
    ],
    "next_step_hypothesis": "If this single-feature change doesn't beat baseline, try exactly one alternative (e.g., shock_index_day10) while keeping models/threshold grid identical."
}

append_jsonl(log_path, log_entry)
print(f"[Log] appended -> {os.path.abspath(log_path)}")

[Info] patient_id overlap train vs test: 0 (expected 0)
[Info] stay_id overlap train vs test: 0 (expected 0)
[Data] stays_train: (1000, 5) | stays_test: (1000, 4) | patients: (4000, 5)
[Baseline iter0006] pooled_best=0.823975 @ t=0.600 | oof_path=artifacts\clai_ch3_v1\iter0006_oof_prob.npy
[Features] num_cols=118 | cat_cols=6 | text_cols=3 | one_feature_added=True
[CV] Running 5-fold StratifiedKFold...

[Results @0.5]
  Logistic (0.5) fold macro-F1: [0.75, 0.7307, 0.7316, 0.7287, 0.7566] | mean=0.739524
  XGB/Structured (0.5) fold macro-F1: [0.8078, 0.769, 0.7914, 0.7932, 0.7669] | mean=0.785656

[Best threshold (OOF pooled)]
  Logistic: best_f1=0.762280 @ t=0.370
  XGB/Structured: best_f1=0.797624 @ t=0.680

[Ensemble optimized on OOF pooled (guarded neighborhood)]
  Best ensemble OOF macro-F1=0.799178 @ weight_logi=0.25, threshold=0.630
  Ensemble per-fold macro-F1 @ best: [0.8264, 0.7949, 0.7932, 0.7898, 0.79] | mean=0.798843
  Confusion matrix (OOF @ best):
[[249  95]
 [ 85 571]]



# Iteration 15
- 0.8244

In [48]:
import os, json, time, datetime, random, shutil
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# -----------------------------
# Config
# -----------------------------
VERSION = "v1"
TEAM_TAG = "clai"
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

SUBMISSION_PATH = f"{TEAM_TAG}_ch3_{VERSION}_submission.csv"
LOG_PATH = f"{TEAM_TAG}_ch3_{VERSION}_iteration_detail.jsonl"

ART_DIR = os.path.join("artifacts", f"{TEAM_TAG}_ch3_{VERSION}")
CKPT_ROOT = os.path.join("checkpoints", f"{TEAM_TAG}_ch3_{VERSION}")
os.makedirs(ART_DIR, exist_ok=True)
os.makedirs(CKPT_ROOT, exist_ok=True)

# Baseline (iter0006) anchor (from prior best-known run)
BASELINE_ITER_DIRNAME = "iter0006"
BASELINE_W = 0.35
BASELINE_T = 0.60

# HM message (from user)
HM_MESSAGE = "HM: real F1=0.8244 (iter0006 base). Instruction: rollback to iter0006 and do only small, single-change iteration with strict guard to avoid regressions."
REAL_SCORE = 0.8244

# -----------------------------
# Helpers
# -----------------------------
def utc_now_iso():
    return datetime.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"

def find_file(filename, roots):
    # direct
    for r in roots:
        p = os.path.join(r, filename)
        if os.path.exists(p):
            return p
    # walk
    for r in roots:
        if r and os.path.isdir(r):
            for dirpath, _, filenames in os.walk(r):
                if filename in filenames:
                    return os.path.join(dirpath, filename)
    return None

def safe_jsonable(x):
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.ndarray,)):
        return x.tolist()
    return x

def best_threshold_macro_f1(y_true, proba, t_grid):
    best_f1, best_t = -1.0, 0.5
    for t in t_grid:
        pred = (proba >= t).astype(int)
        f1 = f1_score(y_true, pred, average="macro")
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_f1), float(best_t)

def ensemble_search(y_true, p_logi, p_xgb, w_grid, t_grid):
    best = {"f1": -1.0, "w": None, "t": None}
    for w in w_grid:
        p = w * p_logi + (1.0 - w) * p_xgb
        f1, t = best_threshold_macro_f1(y_true, p, t_grid)
        if f1 > best["f1"]:
            best = {"f1": float(f1), "w": float(w), "t": float(t)}
    return best

def build_vitals_features(vitals_json_path):
    import re
    with open(vitals_json_path, "r", encoding="utf-8") as f:
        vitals = json.load(f)

    pos_words = set(["ambulated","home","tolerated","afebrile","good","well","stable","improving","improved","resolved","discharge","ready","walking","chair"])
    neg_words = set(["worsening","pain","hypotensive","tachycardic","sepsis","fever","icu","intubated","dyspnea","sob","unstable","critical"])

    def ts_stats(arr):
        arr = np.array(arr, dtype=float)
        arr = np.where(np.isfinite(arr), arr, np.nan)
        out = {}
        if np.all(np.isnan(arr)):
            out.update({
                "mean": np.nan, "std": np.nan, "min": np.nan, "max": np.nan,
                "last": np.nan, "missing": 1.0, "slope": np.nan,
                "last3_mean": np.nan, "last3_std": np.nan, "delta_last_first": np.nan
            })
            return out

        out["mean"] = float(np.nanmean(arr))
        out["std"]  = float(np.nanstd(arr))
        out["min"]  = float(np.nanmin(arr))
        out["max"]  = float(np.nanmax(arr))
        out["missing"] = float(np.mean(np.isnan(arr)))

        # last non-missing
        last_val = np.nan
        for v in arr[::-1]:
            if not np.isnan(v):
                last_val = float(v); break
        out["last"] = last_val

        # slope
        idx = np.arange(len(arr))
        msk = ~np.isnan(arr)
        if msk.sum() >= 2:
            out["slope"] = float(np.polyfit(idx[msk], arr[msk], 1)[0])
        else:
            out["slope"] = 0.0

        last3 = arr[-3:]
        out["last3_mean"] = float(np.nanmean(last3))
        out["last3_std"]  = float(np.nanstd(last3))

        first_val = np.nan
        for v in arr:
            if not np.isnan(v):
                first_val = float(v); break
        out["delta_last_first"] = float(last_val - first_val) if (not np.isnan(last_val) and not np.isnan(first_val)) else np.nan
        return out

    rows = []
    for rec in vitals:
        sid = rec["stay_id"]
        days = sorted(rec["days"], key=lambda x: x.get("day", 0))
        notes = [(d.get("note") or "") for d in days]
        txt_all = " ".join(notes).strip()
        txt_late3 = " ".join(notes[-3:]).strip()
        txt_day10 = (notes[-1].strip() if len(notes) else "")

        # NEW (single-change) feature group: note length stats
        len_all = len(txt_all)
        len_late3 = len(txt_late3)
        len_day10 = len(txt_day10)

        words = re.findall(r"[a-zA-Z/]+", txt_all.lower())
        lex_pos = sum(w in pos_words for w in words)
        lex_neg = sum(w in neg_words for w in words)
        lex_diff = lex_pos - lex_neg

        feats = {
            "stay_id": sid,
            "txt_all": txt_all,
            "txt_late3": txt_late3,
            "txt_day10": txt_day10,
            "lex_pos": lex_pos,
            "lex_neg": lex_neg,
            "lex_diff": lex_diff,
            "note_len_all": len_all,
            "note_len_late3": len_late3,
            "note_len_day10": len_day10,
        }

        for vital in ["hr","sbp","dbp","temp_c","rr"]:
            arr = []
            for d in days:
                v = d.get(vital, None)
                arr.append(np.nan if v is None else float(v))
            stats = ts_stats(arr)
            for k, v in stats.items():
                feats[f"{vital}_{k}"] = v

        rows.append(feats)

    return pd.DataFrame(rows)

# -----------------------------
# Iteration id
# -----------------------------
iteration_id = 0
if os.path.exists(LOG_PATH):
    try:
        with open(LOG_PATH, "r", encoding="utf-8") as f:
            ids = []
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    if "iteration_id" in obj:
                        ids.append(int(obj["iteration_id"]))
                except Exception:
                    pass
            if ids:
                iteration_id = max(ids) + 1
    except Exception:
        iteration_id = 0

iter_name = f"iter{iteration_id:04d}"
print(f"[Iter] {iter_name}")

# dirs for this iter
iter_art_dir = ART_DIR
iter_ckpt_dir = os.path.join(CKPT_ROOT, iter_name)
os.makedirs(iter_ckpt_dir, exist_ok=True)

# -----------------------------
# Load data
# -----------------------------
root = os.getcwd()
search_roots = [
    root,
    os.path.join(root, ART_DIR),
    os.path.join(root, "artifacts"),
    os.path.join(root, CKPT_ROOT),
    os.path.join(root, "checkpoints"),
    "/mnt/data",
]

stays_train_path = find_file("stays_train.csv", search_roots)
stays_test_path  = find_file("stays_test.csv", search_roots)
patients_path    = find_file("patients.csv", search_roots)
vitals_path      = find_file("vitals_timeseries.json", search_roots)

if not all([stays_train_path, stays_test_path, patients_path, vitals_path]):
    raise FileNotFoundError(f"Missing required files. Found: stays_train={stays_train_path}, stays_test={stays_test_path}, patients={patients_path}, vitals={vitals_path}")

stays_train = pd.read_csv(stays_train_path)
stays_test  = pd.read_csv(stays_test_path)
patients    = pd.read_csv(patients_path)

print(f"[Data] stays_train: {stays_train.shape} | stays_test: {stays_test.shape} | patients: {patients.shape}")

# Leakage checks
patient_overlap = len(set(stays_train["patient_id"]).intersection(set(stays_test["patient_id"])))
stay_overlap = len(set(stays_train["stay_id"]).intersection(set(stays_test["stay_id"])))
print(f"[Info] patient_id overlap train vs test: {patient_overlap} (expected 0)")
print(f"[Info] stay_id overlap train vs test: {stay_overlap} (expected 0)")

# Vitals features
vitals_df = build_vitals_features(vitals_path)

# Merge
train_df = stays_train.merge(patients, on="patient_id", how="left").merge(vitals_df, on="stay_id", how="left")
test_df  = stays_test.merge(patients, on="patient_id", how="left").merge(vitals_df, on="stay_id", how="left")

# Categorical interaction (baseline-compatible)
train_df["reason_unit"] = train_df["admission_reason"].astype(str) + "__" + train_df["unit_type"].astype(str)
test_df["reason_unit"]  = test_df["admission_reason"].astype(str) + "__" + test_df["unit_type"].astype(str)

# Ensure categorical dtypes are stable
for c in ["unit_type","admission_reason","sex","insurance","zip3","reason_unit"]:
    train_df[c] = train_df[c].astype(str)
    test_df[c]  = test_df[c].astype(str)

y = train_df["discharge_ready_day11"].astype(int).values

text_cols = ["txt_all", "txt_late3", "txt_day10"]
cat_cols  = ["unit_type","admission_reason","sex","insurance","zip3","reason_unit"]
exclude = set(["stay_id","patient_id","discharge_ready_day11"] + text_cols + cat_cols)
num_cols = [c for c in train_df.columns if c not in exclude]

print(f"[Features] num_cols={len(num_cols)} | cat_cols={len(cat_cols)} | text_cols={len(text_cols)} | one_feature_group_added=note_len_*")

# -----------------------------
# Load baseline artifacts (iter0006)
# -----------------------------
baseline_oof_pred_path = find_file("iter0006_oof_predictions.csv", search_roots)
baseline_oof_prob_path = find_file("iter0006_oof_prob.npy", search_roots)
baseline_test_prob_path = find_file("iter0006_test_prob.npy", search_roots)
baseline_fold_id_path = find_file("iter0006_fold_id.npy", search_roots)

baseline_best_f1 = 0.8239746523499384  # fallback
baseline_cm = None

if baseline_oof_pred_path and os.path.exists(baseline_oof_pred_path):
    base_oof = pd.read_csv(baseline_oof_pred_path)
    if {"y_true","oof_pred"}.issubset(base_oof.columns):
        baseline_best_f1 = float(f1_score(base_oof["y_true"].astype(int).values, base_oof["oof_pred"].astype(int).values, average="macro"))
        baseline_cm = confusion_matrix(base_oof["y_true"].astype(int).values, base_oof["oof_pred"].astype(int).values).tolist()

print(f"[Baseline {BASELINE_ITER_DIRNAME}] pooled_best={baseline_best_f1:.6f} @ w={BASELINE_W:.2f}, t={BASELINE_T:.2f} | oof_pred_path={baseline_oof_pred_path}")

# fold_id alignment
fold_id = None
fold_id_source = None
if baseline_fold_id_path and os.path.exists(baseline_fold_id_path) and baseline_oof_pred_path and os.path.exists(baseline_oof_pred_path):
    fid = np.load(baseline_fold_id_path)
    base_oof = pd.read_csv(baseline_oof_pred_path)[["stay_id"]]
    if len(fid) == len(base_oof) == len(train_df):
        fold_map = dict(zip(base_oof["stay_id"].astype(int).tolist(), fid.astype(int).tolist()))
        aligned = train_df["stay_id"].astype(int).map(fold_map)
        if aligned.isna().sum() == 0:
            fold_id = aligned.astype(int).values
            fold_id_source = "baseline_iter0006_fold_id_aligned_by_stay_id"
        else:
            fold_id = None

if fold_id is None:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_id = np.zeros(len(train_df), dtype=int)
    for f, (_, va_idx) in enumerate(skf.split(train_df, y)):
        fold_id[va_idx] = f
    fold_id_source = "generated_stratifiedkfold_seed42"

print(f"[CV] Running 5-fold StratifiedKFold... | fold_id_source={fold_id_source}")

# -----------------------------
# Build models (same direction as iter0006)
# -----------------------------
num_pipe_logi = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore")),
])

# TF-IDF (kept stable; no lambdas; picklable)
tfidf_all = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.95, max_features=40000, dtype=np.float32)
tfidf_char_late = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=2, max_features=20000, dtype=np.float32)
tfidf_char_day10 = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=2, max_features=15000, dtype=np.float32)

preprocess_logi = ColumnTransformer(
    transformers=[
        ("num", num_pipe_logi, num_cols),
        ("cat", cat_pipe, cat_cols),
        ("txt_all", tfidf_all, "txt_all"),
        ("txt_late3", tfidf_char_late, "txt_late3"),
        ("txt_day10", tfidf_char_day10, "txt_day10"),
    ],
    sparse_threshold=0.3
)

logi = LogisticRegression(
    solver="saga",
    penalty="elasticnet",
    l1_ratio=0.05,
    C=0.6,
    class_weight="balanced",
    max_iter=20000,
    random_state=SEED,
    n_jobs=1
)
pipe_logi = Pipeline([("prep", preprocess_logi), ("clf", logi)])

# XGB (structured only) — keep params aligned to iter0006
HAS_XGB = True
try:
    from xgboost import XGBClassifier
except Exception:
    HAS_XGB = False

preprocess_xgb = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    sparse_threshold=0.3
)

if not HAS_XGB:
    raise ImportError("xgboost is required for this pipeline but is not available in the environment.")

xgb = XGBClassifier(
    objective="binary:logistic",
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=2.0,
    subsample=0.8,
    colsample_bytree=0.8,
    n_estimators=450,
    tree_method="hist",
    n_jobs=1,
    random_state=SEED,
    eval_metric="logloss",
    verbosity=0,
    reg_alpha=0.0,
    reg_lambda=1.0,
)
pipe_xgb = Pipeline([("prep", preprocess_xgb), ("clf", xgb)])

# -----------------------------
# CV training
# -----------------------------
n_splits = 5
oof_logi = np.zeros(len(train_df), dtype=float)
oof_xgb  = np.zeros(len(train_df), dtype=float)
fold_f1_logi = []
fold_f1_xgb  = []

for f in range(n_splits):
    tr_idx = np.where(fold_id != f)[0]
    va_idx = np.where(fold_id == f)[0]
    X_tr = train_df.iloc[tr_idx]
    y_tr = y[tr_idx]
    X_va = train_df.iloc[va_idx]
    y_va = y[va_idx]

    pipe_logi.fit(X_tr, y_tr)
    p_logi = pipe_logi.predict_proba(X_va)[:, 1]
    oof_logi[va_idx] = p_logi
    fold_f1_logi.append(float(f1_score(y_va, (p_logi >= 0.5).astype(int), average="macro")))

    pipe_xgb.fit(X_tr, y_tr)
    p_xgb = pipe_xgb.predict_proba(X_va)[:, 1]
    oof_xgb[va_idx] = p_xgb
    fold_f1_xgb.append(float(f1_score(y_va, (p_xgb >= 0.5).astype(int), average="macro")))

print("\n[Results @0.5]")
print(f"  Logistic (0.5) fold macro-F1: {[round(x,4) for x in fold_f1_logi]} | mean={np.mean(fold_f1_logi):.6f}")
print(f"  XGB/Structured (0.5) fold macro-F1: {[round(x,4) for x in fold_f1_xgb]} | mean={np.mean(fold_f1_xgb):.6f}")

# Best thresholds (pooled)
t_grid = np.round(np.arange(0.05, 0.951, 0.01), 2)
logi_best_f1, logi_best_t = best_threshold_macro_f1(y, oof_logi, t_grid)
xgb_best_f1, xgb_best_t   = best_threshold_macro_f1(y, oof_xgb, t_grid)
print("\n[Best threshold (OOF pooled)]")
print(f"  Logistic: best_f1={logi_best_f1:.6f} @ t={logi_best_t:.3f}")
print(f"  XGB/Structured: best_f1={xgb_best_f1:.6f} @ t={xgb_best_t:.3f}")

# Ensemble anchored + neighborhood (guard)
p_ens_anchor = BASELINE_W * oof_logi + (1.0 - BASELINE_W) * oof_xgb
f1_anchor = float(f1_score(y, (p_ens_anchor >= BASELINE_T).astype(int), average="macro"))

w_grid = np.round(np.arange(0.25, 0.451, 0.05), 2)   # guarded neighborhood around 0.35
t_grid_guard = np.round(np.arange(0.55, 0.651, 0.01), 2)  # guarded neighborhood around 0.60
best_ens = ensemble_search(y, oof_logi, oof_xgb, w_grid, t_grid_guard)

# Per-fold at best ensemble
p_ens_best = best_ens["w"] * oof_logi + (1.0 - best_ens["w"]) * oof_xgb
fold_f1_ens_best = []
for f in range(n_splits):
    va_idx = np.where(fold_id == f)[0]
    fold_f1_ens_best.append(float(f1_score(y[va_idx], (p_ens_best[va_idx] >= best_ens["t"]).astype(int), average="macro")))
cm_best = confusion_matrix(y, (p_ens_best >= best_ens["t"]).astype(int)).tolist()

print("\n[Ensemble optimized on OOF pooled (guarded neighborhood)]")
print(f"  Anchor (w={BASELINE_W:.2f}, t={BASELINE_T:.2f}) OOF macro-F1={f1_anchor:.6f}")
print(f"  Best guarded OOF macro-F1={best_ens['f1']:.6f} @ weight_logi={best_ens['w']:.2f}, threshold={best_ens['t']:.2f}")
print(f"  Ensemble per-fold macro-F1 @ best: {[round(x,4) for x in fold_f1_ens_best]} | mean={np.mean(fold_f1_ens_best):.6f}")
print(f"  Confusion matrix (OOF @ best):\n{np.array(cm_best)}")

# Selection rule: only accept if clear improvement over baseline best
EPS_IMPROVE = 0.0010
use_candidate = (best_ens["f1"] >= baseline_best_f1 + EPS_IMPROVE) and (f1_anchor >= baseline_best_f1 - 0.003)

print("\n[Selection]")
print(f"  baseline_best_f1={baseline_best_f1:.6f} | candidate_best_f1={best_ens['f1']:.6f} | anchor_f1={f1_anchor:.6f} | use_candidate={use_candidate}")

# -----------------------------
# Fit full models + predict test (or fallback to baseline)
# -----------------------------
if use_candidate:
    pipe_logi.fit(train_df, y)
    pipe_xgb.fit(train_df, y)
    p_test_logi = pipe_logi.predict_proba(test_df)[:, 1]
    p_test_xgb  = pipe_xgb.predict_proba(test_df)[:, 1]
    w_final = best_ens["w"]
    t_final = best_ens["t"]
    p_test_ens = w_final * p_test_logi + (1.0 - w_final) * p_test_xgb
    selection_name = "candidate_note_length_features_guarded"
else:
    # prefer baseline test probs to guarantee non-regression
    if baseline_test_prob_path and os.path.exists(baseline_test_prob_path):
        p_test_ens = np.load(baseline_test_prob_path).astype(float)
        w_final = BASELINE_W
        t_final = BASELINE_T
        selection_name = "baseline_iter0006_test_prob_fallback"
    else:
        # last-resort fallback: use candidate model but baseline parameters
        pipe_logi.fit(train_df, y)
        pipe_xgb.fit(train_df, y)
        p_test_logi = pipe_logi.predict_proba(test_df)[:, 1]
        p_test_xgb  = pipe_xgb.predict_proba(test_df)[:, 1]
        w_final = BASELINE_W
        t_final = BASELINE_T
        p_test_ens = w_final * p_test_logi + (1.0 - w_final) * p_test_xgb
        selection_name = "fallback_candidate_model_baseline_params (baseline_test_prob_missing)"

y_test_pred = (p_test_ens >= t_final).astype(int)

sub = pd.DataFrame({
    "stay_id": stays_test["stay_id"].astype(int).values,
    "discharge_ready_day11": y_test_pred.astype(int)
})
sub.to_csv(SUBMISSION_PATH, index=False)
print(f"\n[Saved] submission -> {os.path.abspath(SUBMISSION_PATH)}")

# -----------------------------
# Save artifacts
# -----------------------------
oof_pred_ens = (p_ens_best >= best_ens["t"]).astype(int)
oof_pred_df = pd.DataFrame({
    "stay_id": train_df["stay_id"].astype(int).values,
    "y_true": y.astype(int),
    "oof_prob_logi": oof_logi.astype(float),
    "oof_prob_xgb": oof_xgb.astype(float),
    "oof_prob_ens_best": p_ens_best.astype(float),
    "oof_pred_ens_best": oof_pred_ens.astype(int),
})
oof_csv_path = os.path.join(iter_art_dir, f"{iter_name}_oof_predictions.csv")
oof_pred_df.to_csv(oof_csv_path, index=False)

np.save(os.path.join(iter_art_dir, f"{iter_name}_oof_prob_logi.npy"), oof_logi)
np.save(os.path.join(iter_art_dir, f"{iter_name}_oof_prob_xgb.npy"), oof_xgb)
np.save(os.path.join(iter_art_dir, f"{iter_name}_oof_prob_ens.npy"), p_ens_best)
np.save(os.path.join(iter_art_dir, f"{iter_name}_test_prob_ens.npy"), p_test_ens)
np.save(os.path.join(iter_art_dir, f"{iter_name}_fold_id.npy"), fold_id.astype(int))

print(f"[Artifacts] {oof_csv_path} + npy saved under {iter_art_dir}")

# -----------------------------
# Checkpoint (avoid lambda pickling; store what was used)
# -----------------------------
# If we used baseline fallback, try to copy baseline checkpoint bundle for reproducibility.
joblib_dump_ok = False
ckpt_note = None

try:
    import joblib
    if use_candidate:
        bundle = {
            "pipe_logi": pipe_logi,
            "pipe_xgb": pipe_xgb,
            "weight_logi": w_final,
            "threshold": t_final,
            "selection_name": selection_name,
        }
        joblib.dump(bundle, os.path.join(iter_ckpt_dir, "model_bundle.joblib"), compress=3)
        joblib_dump_ok = True
        ckpt_note = "Saved candidate pipelines + ensemble params."
    else:
        # copy baseline model bundle if exists
        baseline_model_bundle = find_file(os.path.join("iter0006", "model_bundle.joblib"), [CKPT_ROOT, os.path.join(root, CKPT_ROOT), os.path.join(root, "checkpoints"), "/mnt/data"])
        if baseline_model_bundle and os.path.exists(baseline_model_bundle):
            shutil.copy2(baseline_model_bundle, os.path.join(iter_ckpt_dir, "model_bundle.joblib"))
            joblib_dump_ok = True
            ckpt_note = f"Copied baseline model_bundle.joblib from {baseline_model_bundle}"
        else:
            # fallback: still save candidate pipelines (even if baseline prob used)
            bundle = {
                "pipe_logi_candidate": pipe_logi,
                "pipe_xgb_candidate": pipe_xgb,
                "weight_logi_used": w_final,
                "threshold_used": t_final,
                "selection_name": selection_name,
                "warning": "baseline model bundle not found; saved candidate models; submission may have used baseline test prob if available."
            }
            joblib.dump(bundle, os.path.join(iter_ckpt_dir, "model_bundle.joblib"), compress=3)
            joblib_dump_ok = True
            ckpt_note = "Baseline model bundle not found; saved candidate pipelines as fallback."
except Exception as e:
    joblib_dump_ok = False
    ckpt_note = f"joblib dump failed: {repr(e)}"

# Save config/metrics/schema
config = {
    "version": VERSION,
    "seed": SEED,
    "baseline_anchor": {"iter": BASELINE_ITER_DIRNAME, "weight_logi": BASELINE_W, "threshold": BASELINE_T, "baseline_best_f1": baseline_best_f1},
    "feature_change": "ADD note_len_* numeric features (all/late3/day10) to help structured model; keep rest unchanged",
    "guard": {
        "ensemble_weight_grid": w_grid.tolist(),
        "ensemble_threshold_grid": t_grid_guard.tolist(),
        "eps_improve": EPS_IMPROVE,
        "accept_requires": "candidate_best_f1 >= baseline_best_f1 + eps AND anchor_f1 >= baseline_best_f1 - 0.003"
    }
}

metrics = {
    "macro_f1_mean_at_0.5": float((np.mean(fold_f1_logi) + np.mean(fold_f1_xgb)) / 2.0),
    "macro_f1_per_fold_at_0.5": {"logi": fold_f1_logi, "xgb": fold_f1_xgb},
    "best_oof": {
        "logi_best_f1": logi_best_f1, "logi_best_threshold": logi_best_t,
        "xgb_best_f1": xgb_best_f1, "xgb_best_threshold": xgb_best_t,
        "ensemble_anchor_f1": f1_anchor, "ensemble_anchor_weight": BASELINE_W, "ensemble_anchor_threshold": BASELINE_T,
        "ensemble_best_f1_guarded": best_ens["f1"], "ensemble_weight_logi": best_ens["w"], "ensemble_threshold": best_ens["t"],
    },
    "ensemble_per_fold_at_best": fold_f1_ens_best,
    "confusion_matrix_oof_at_best": cm_best,
    "selection": {"use_candidate": bool(use_candidate), "selection_name": selection_name, "weight_final": float(w_final), "threshold_final": float(t_final)},
    "checkpoint": {"joblib_dump_ok": bool(joblib_dump_ok), "note": ckpt_note},
    "paths": {"submission": os.path.abspath(SUBMISSION_PATH), "oof_csv": os.path.abspath(oof_csv_path), "ckpt_dir": os.path.abspath(iter_ckpt_dir)}
}

schema = {"numeric_cols": num_cols, "categorical_cols": cat_cols, "text_cols": text_cols}

with open(os.path.join(iter_ckpt_dir, "config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
with open(os.path.join(iter_ckpt_dir, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
with open(os.path.join(iter_ckpt_dir, "schema.json"), "w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)

print(f"[Checkpoint] {os.path.abspath(iter_ckpt_dir)}")
print(f"[Checkpoint] joblib_dump_ok={joblib_dump_ok} | note={ckpt_note}")

# -----------------------------
# P* tracking (simple from logs)
# -----------------------------
best_before = {"macro_f1": None, "iteration_id": None}
best_macro = -1.0
best_iter = None

if os.path.exists(LOG_PATH):
    with open(LOG_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                it = obj.get("iteration_id", None)
                m = obj.get("metrics", {})
                # try common fields
                cand = None
                if isinstance(m, dict):
                    if "macro_f1_pooled_best" in m:
                        cand = m["macro_f1_pooled_best"]
                    elif "best_oof" in m and isinstance(m["best_oof"], dict):
                        cand = m["best_oof"].get("ensemble_best_f1", None) or m["best_oof"].get("ensemble_best_f1_guarded", None)
                if cand is not None:
                    cand = float(cand)
                    if cand > best_macro:
                        best_macro = cand
                        best_iter = it
            except Exception:
                pass

if best_macro < 0:
    best_macro = baseline_best_f1
    best_iter = 6

best_before = {"macro_f1": float(best_macro), "iteration_id": int(best_iter) if best_iter is not None else None}

# candidate current score to compare for P*
current_score = best_ens["f1"] if use_candidate else baseline_best_f1
is_new_pstar = current_score > best_before["macro_f1"] + 1e-12

pstar_obj = {
    "best_known_before": best_before,
    "current_macro_f1": float(current_score),
    "is_new_p_star": bool(is_new_pstar),
    "best_iter_after": int(iteration_id) if is_new_pstar else int(best_before["iteration_id"]) if best_before["iteration_id"] is not None else None
}

# -----------------------------
# Append iteration log
# -----------------------------
log_entry = {
    "version": VERSION,
    "iteration_id": int(iteration_id),
    "timestamp_utc": utc_now_iso(),
    "phase": "Modeling",
    "hm_message": HM_MESSAGE,
    "real_score": REAL_SCORE,
    "data_paths_used": {
        "stays_train": stays_train_path,
        "stays_test": stays_test_path,
        "patients": patients_path,
        "vitals_timeseries": vitals_path,
        "baseline_oof_predictions": baseline_oof_pred_path,
        "baseline_test_prob": baseline_test_prob_path,
        "baseline_fold_id": baseline_fold_id_path,
    },
    "join_keys": {"stays_patients": "patient_id", "stays_vitals": "stay_id"},
    "leakage_checks_performed": {
        "patient_id_overlap_train_test": int(patient_overlap),
        "stay_id_overlap_train_test": int(stay_overlap),
        "note": "Expected both overlaps to be 0."
    },
    "feature_summary": {
        "n_train": int(len(train_df)),
        "n_test": int(len(test_df)),
        "num_cols": int(len(num_cols)),
        "cat_cols": int(len(cat_cols)),
        "text_cols": text_cols,
        "added_features": ["note_len_all", "note_len_late3", "note_len_day10"],
        "cat_encoding": "OneHotEncoder(handle_unknown='ignore')",
        "text_vectorization": {
            "txt_all": {"type": "word_tfidf", "ngram_range": [1,2], "max_features": 40000},
            "txt_late3": {"type": "char_wb_tfidf", "ngram_range": [3,5], "max_features": 20000},
            "txt_day10": {"type": "char_wb_tfidf", "ngram_range": [3,5], "max_features": 15000},
        }
    },
    "models_attempted": [
        {
            "name": "LogisticRegression(saga, elasticnet) + [num+OHE+TFIDF(word+char)]",
            "params": {"solver":"saga","penalty":"elasticnet","l1_ratio":0.05,"C":0.6,"class_weight":"balanced","max_iter":20000,"random_state":SEED},
            "cv_macro_f1_mean_at_0.5": float(np.mean(fold_f1_logi)),
            "cv_macro_f1_per_fold_at_0.5": fold_f1_logi,
            "cv_macro_f1_oof_at_best_threshold": logi_best_f1,
            "oof_best_threshold": logi_best_t,
        },
        {
            "name": "XGBClassifier(structured-only) + [num+OHE]",
            "params": {"has_xgb": True, "xgb_params": {k: safe_jsonable(v) for k,v in xgb.get_params().items()}},
            "cv_macro_f1_mean_at_0.5": float(np.mean(fold_f1_xgb)),
            "cv_macro_f1_per_fold_at_0.5": fold_f1_xgb,
            "cv_macro_f1_oof_at_best_threshold": xgb_best_f1,
            "oof_best_threshold": xgb_best_t,
        },
        {
            "name": "WeightedEnsemble(proba) + guarded neighborhood threshold search",
            "params": {"weight_grid": w_grid.tolist(), "threshold_grid": t_grid_guard.tolist(), "anchor": {"w": BASELINE_W, "t": BASELINE_T}},
            "baseline_best_f1": baseline_best_f1,
            "anchor_f1": f1_anchor,
            "candidate_best_f1_guarded": best_ens["f1"],
            "best_weight_logi": best_ens["w"],
            "best_threshold": best_ens["t"],
        }
    ],
    "selected_model": {
        "name": "Ensemble(Logi + XGB)",
        "selection_name": selection_name,
        "use_candidate": bool(use_candidate),
        "weight_logi": float(w_final),
        "threshold": float(t_final),
    },
    "validation_protocol": {
        "cv": "5-fold StratifiedKFold",
        "seed": SEED,
        "fold_id_source": fold_id_source,
        "thresholding": "global threshold optimized on pooled OOF; guarded neighborhood around iter0006 anchor",
    },
    "metrics": metrics,
    "deltas_vs_previous_iteration": {
        "change": "One small change: add note_len_* numeric features (all/late3/day10) to provide non-TFIDF length signal; keep model family the same; add strict guarded selection vs iter0006 baseline.",
        "expected_effect": "Improve structured model + ensemble slightly without shifting threshold/weight drastically; prevent real-score regression by fallback to iter0006 test_prob.",
    },
    "p_star": pstar_obj,
    "known_defects_limitations": [
        "OOF-optimized thresholds can still overfit; mitigated by guarded neighborhood + baseline fallback.",
        "If baseline checkpoint/test_prob files are missing, fallback may not perfectly reproduce iter0006 submission.",
        "TF-IDF + XGB CV can be compute-heavy; keep an eye on runtime."
    ],
    "next_step_hypothesis": "If note_len_* improves OOF and holds on real-score, next try a similarly-stable micro-feature: late3_len - early3_len (trend) or per-day length slope, still under same guarded selection."
}

with open(LOG_PATH, "a", encoding="utf-8") as f:
    f.write(json.dumps(log_entry, ensure_ascii=False) + "\n")

print(f"[Log] appended -> {os.path.abspath(LOG_PATH)}")

[Iter] iter0016
[Data] stays_train: (1000, 5) | stays_test: (1000, 4) | patients: (4000, 5)
[Info] patient_id overlap train vs test: 0 (expected 0)
[Info] stay_id overlap train vs test: 0 (expected 0)
[Features] num_cols=57 | cat_cols=6 | text_cols=3 | one_feature_group_added=note_len_*
[Baseline iter0006] pooled_best=0.823975 @ w=0.35, t=0.60 | oof_pred_path=d:\AgentDs\agent_ds_healthcare\artifacts\clai_ch3_v1\iter0006_oof_predictions.csv
[CV] Running 5-fold StratifiedKFold... | fold_id_source=baseline_iter0006_fold_id_aligned_by_stay_id

[Results @0.5]
  Logistic (0.5) fold macro-F1: [0.736, 0.7426, 0.727, 0.7347, 0.7533] | mean=0.738745
  XGB/Structured (0.5) fold macro-F1: [0.8092, 0.7644, 0.7817, 0.7932, 0.7546] | mean=0.780629

[Best threshold (OOF pooled)]
  Logistic: best_f1=0.758279 @ t=0.330
  XGB/Structured: best_f1=0.795334 @ t=0.690

[Ensemble optimized on OOF pooled (guarded neighborhood)]
  Anchor (w=0.35, t=0.60) OOF macro-F1=0.789800
  Best guarded OOF macro-F1=0.79823

# New Iter

Iterations: 0.7070, 0.7640, 0.7417, 0.7773, 0.7842, 0.8244, 0.7874, 0.8244, 0.7964, 0.7782, 0.7757, 0.7986, 0.7813, 0.8244, 0.8244

# Submission

In [49]:
# 3. Submit Predictions

# Submit predictions to the competition
print("🚀 Submitting predictions...")

try:
    # result = client.submit_prediction("Healthcare", 3, "artifacts/clai_ch3_v1/clai_ch3_v1_submission_alt_thr0.55.csv")
    result = client.submit_prediction("Healthcare", 3, "clai_ch3_v1_submission.csv")
    
    if result['success']:
        print("✅ Submission successful!")
        print(f"   📊 Score: {result['score']:.4f}")
        print(f"   📏 Metric: {result['metric_name']}")
        print(f"   ✔️  Validation: {'Passed' if result['validation_passed'] else 'Failed'}")
    else:
        print("❌ Submission failed!")
        print(f"   Error details: {result.get('details', {}).get('validation_errors', 'Unknown error')}")
        
except Exception as e:
    print(f"💥 Submission error: {e}")
    print("🔧 Check your API key and team name are correct!")

print("\n🎯 Next steps:")
print("   1. Try incorporating relevant information outside this table!")
print("   2. You've completed all Healthcare challenges!")


🚀 Submitting predictions...
✅ Prediction submitted successfully!
📊 Score: 0.8244 (Macro-F1)
✅ Validation passed
✅ Submission successful!
   📊 Score: 0.8244
   📏 Metric: Macro-F1
   ✔️  Validation: Passed

🎯 Next steps:
   1. Try incorporating relevant information outside this table!
   2. You've completed all Healthcare challenges!
